# Cleaned XAI Coupled NMF Master Runbook

This notebook was cleaned from the uploaded original Colab notebook:

`4Copy_of_xai_coupled_nmf_master_runbook.ipynb`

Cleaning actions applied:

- removed cells with traceback/error outputs;
- removed empty cells;
- removed exact duplicate code cells;
- cleared all execution outputs and execution counters;
- inserted section headings to make the workflow easier to navigate.

Important note: the original notebook was developed interactively in Colab and may have been executed non-linearly. This cleaned version is therefore best treated as an archival runbook. For public reproducibility, use the repository scripts, configuration files, final CSV outputs, tables, and figures.

## 1. Project setup and Google Drive mounting

Original cell index starts around `0`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
if os.path.exists(PROJECT_ROOT):
    print(f"Project directory found: {PROJECT_ROOT}")
else:
    print(f"Project directory NOT found: {PROJECT_ROOT}. Please ensure the path is correct.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
PROJECT_ROOT

In [ ]:
import os

folders = [
    f"{PROJECT_ROOT}",
    f"{PROJECT_ROOT}/data",
    f"{PROJECT_ROOT}/data/raw",
    f"{PROJECT_ROOT}/data/processed",
    f"{PROJECT_ROOT}/results",
    f"{PROJECT_ROOT}/results/csv",
    f"{PROJECT_ROOT}/results/figures",
    f"{PROJECT_ROOT}/results/logs",
    f"{PROJECT_ROOT}/results/tables",
    f"{PROJECT_ROOT}/paper",
    f"{PROJECT_ROOT}/paper/figures",
    f"{PROJECT_ROOT}/paper/tables",
    f"{PROJECT_ROOT}/paper/sections",
    f"{PROJECT_ROOT}/src",
    f"{PROJECT_ROOT}/notebooks",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders ready.")

In [ ]:
%cd /content/drive/MyDrive/xai_coupled_nmf_project

In [ ]:
!find /content/drive/MyDrive/xai_coupled_nmf_project -maxdepth 2 -type d | sort

In [ ]:
import zipfile

zip_path = f"{PROJECT_ROOT}/coupled_nmf_script_skeleton_pack.zip"
extract_to = PROJECT_ROOT

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

    print("Skeleton pack extracted.")

In [ ]:
!find /content/drive/MyDrive/xai_coupled_nmf_project/src -maxdepth 1 -type f | sort

In [ ]:
!python src/build_dataset_summary.py \
  --dataset ml100k \
    --input_ratings data/processed/ml100k_ratings.csv \
      --input_descriptors data/processed/ml100k_item_descriptors.csv \
        --output_csv results/csv/ml100k_dataset_summary.csv \
          --log_file results/logs/ml100k_dataset_summary.log \
            --overwrite true

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
%cd /content/drive/MyDrive/xai_coupled_nmf_project

In [ ]:
!find /content/drive/MyDrive/xai_coupled_nmf_project -name "build_dataset_summary.py"

In [ ]:
import os
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target_src = PROJECT_ROOT / "src"
target_src.mkdir(parents=True, exist_ok=True)

matches = list(PROJECT_ROOT.rglob("build_dataset_summary.py"))

if not matches:
    print("build_dataset_summary.py was not found. The ZIP may not have been uploaded or extracted correctly.")
else:
    script_path = matches[0]
    source_src = script_path.parent

    print("Found script at:", script_path)
    print("Source src folder:", source_src)
    print("Target src folder:", target_src)

    if source_src.resolve() != target_src.resolve():
        for item in source_src.iterdir():
            dest = target_src / item.name
            if item.is_file():
                shutil.copy2(item, dest)
            elif item.is_dir():
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(item, dest)
        print("Copied skeleton scripts into the correct src folder.")
    else:
        print("Scripts are already in the correct src folder.")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"

DATASETS = {
    "main": {
        "dataset_id": "ml_latest_small",
        "raw_folder": "ml-latest-small",
        "descriptor_source": "tags_genres",
        "role": "main"
    },
    "secondary": {
        "dataset_id": "ml1m",
        "raw_folder": "ml-1m",
        "descriptor_source": "genres",
        "role": "secondary"
    },
    "optional_extended": {
        "dataset_id": "ml20m",
        "raw_folder": "ml-20m",
        "descriptor_source": "tag_genome",
        "role": "optional_extended"
    }
}

DATASETS

## 2. MovieLens latest-small preprocessing and split generation

Original cell index starts around `15`.

In [ ]:
# Create:
# data/processed/ml_latest_small_ratings.csv
# data/processed/ml_latest_small_item_descriptors.csv

In [ ]:
!python src/build_dataset_summary.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --output_csv results/csv/ml_latest_small_dataset_summary.csv \
          --log_file results/logs/ml_latest_small_dataset_summary.log \
            --overwrite true

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
%cd /content/drive/MyDrive/xai_coupled_nmf_project

!mkdir -p data/raw

# Main dataset: MovieLens latest-small
!wget -O data/raw/ml-latest-small.zip https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o data/raw/ml-latest-small.zip -d data/raw/

# Secondary dataset: MovieLens 1M
!wget -O data/raw/ml-1m.zip https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -o data/raw/ml-1m.zip -d data/raw/

# Check folders
!find data/raw -maxdepth 2 -type f | sort

In [ ]:
# Optional extended dataset: MovieLens 20M
# Run this only after ml_latest_small and ml1m are fully working.

!wget -O data/raw/ml-20m.zip https://files.grouplens.org/datasets/movielens/ml-20m.zip
!unzip -o data/raw/ml-20m.zip -d data/raw/

!find data/raw/ml-20m -maxdepth 1 -type f | sort

In [ ]:
!ls -lh data/raw/ml-latest-small

In [ ]:
!python src/preprocess_ratings.py \
  --dataset ml_latest_small \
    --input_ratings data/raw/ml-latest-small/ratings.csv \
      --output_csv data/processed/ml_latest_small_ratings.csv \
        --min_user_ratings 20 \
          --min_item_ratings 20 \
            --log_file results/logs/ml_latest_small_preprocess_ratings.log

In [ ]:
!python src/build_item_descriptors.py \
  --dataset ml_latest_small \
    --input_items data/raw/ml-latest-small/movies.csv \
      --input_tags data/raw/ml-latest-small/tags.csv \
        --descriptor_source tags_genres \
          --output_csv data/processed/ml_latest_small_item_descriptors.csv \
            --top_n_tags 200 \
              --log_file results/logs/ml_latest_small_build_item_descriptors.log

In [ ]:
!ls -lh data/processed/

In [ ]:
import pandas as pd

ratings = pd.read_csv("data/processed/ml_latest_small_ratings.csv")
desc = pd.read_csv("data/processed/ml_latest_small_item_descriptors.csv")

print("Ratings shape:", ratings.shape)
print("Descriptors shape:", desc.shape)

display(ratings.head())
display(desc.head())

In [ ]:
pd.read_csv("results/csv/ml_latest_small_dataset_summary.csv")

In [ ]:
!python src/generate_splits.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --split_type random_per_user \
        --seed_list 42,123,2026,7,99 \
          --output_csv results/csv/ml_latest_small_split_registry.csv \
            --log_file results/logs/ml_latest_small_generate_splits.log

In [ ]:
!python src/check_evaluation_pipeline.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --split_registry results/csv/ml_latest_small_split_registry.csv \
        --top_k 5,10 \
          --relevance_threshold 4.0 \
            --log_file results/logs/ml_latest_small_check_evaluation_pipeline.log

In [ ]:
!ls -lh results/csv/

In [ ]:
import pandas as pd

ratings = pd.read_csv("data/processed/ml_latest_small_ratings.csv")
descriptors = pd.read_csv("data/processed/ml_latest_small_item_descriptors.csv")
splits = pd.read_csv("results/csv/ml_latest_small_split_registry.csv")
summary = pd.read_csv("results/csv/ml_latest_small_dataset_summary.csv")

print("Ratings columns:")
print(ratings.columns.tolist())
print("\nRatings shape:", ratings.shape)
display(ratings.head())

print("\nDescriptor columns:")
print(descriptors.columns[:20].tolist())
print("\nDescriptors shape:", descriptors.shape)
display(descriptors.head())

print("\nSplit registry columns:")
print(splits.columns.tolist())
print("\nSplit registry shape:", splits.shape)
display(splits.head())

print("\nDataset summary:")
display(summary)

## 3. Baseline benchmark scripts and initial benchmark execution

Original cell index starts around `31`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

ratings = pd.read_csv("data/processed/ml_latest_small_ratings.csv")
splits = pd.read_csv("results/csv/ml_latest_small_split_registry.csv")

seed = 42
split_seed = splits[splits["seed"] == seed].copy()

train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

train = ratings.merge(train_keys, on=["user_id", "item_id"], how="inner")
test = ratings.merge(test_keys, on=["user_id", "item_id"], how="inner")

global_mean = train["rating"].mean()
item_means = train.groupby("item_id")["rating"].mean()

test["pred_global_mean"] = global_mean
test["pred_item_mean"] = test["item_id"].map(item_means).fillna(global_mean)

rows = []

for model_name, pred_col in [
    ("global_mean", "pred_global_mean"),
    ("item_mean", "pred_item_mean"),
]:
    rmse = np.sqrt(mean_squared_error(test["rating"], test[pred_col]))
    mae = mean_absolute_error(test["rating"], test[pred_col])
    rows.append({
        "dataset_name": "ml_latest_small",
        "model_name": model_name,
        "seed": seed,
        "rmse": rmse,
        "mae": mae,
        "num_train": len(train),
        "num_test": len(test),
    })

smoke = pd.DataFrame(rows)
display(smoke)

Path("results/csv").mkdir(parents=True, exist_ok=True)
smoke.to_csv("results/csv/ml_latest_small_baseline_smoke_test.csv", index=False)
print("Saved: results/csv/ml_latest_small_baseline_smoke_test.csv")

In [ ]:
%%writefile src/run_main_benchmarks.py
import argparse
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

def parse_seed_list(seed_list: str) -> List[int]:
    return [int(x.strip()) for x in seed_list.split(",") if x.strip()]

def parse_top_k(top_k: str) -> List[int]:
    return [int(x.strip()) for x in top_k.split(",") if x.strip()]

def ensure_parent(path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

def write_log(log_file: str | None, message: str) -> None:
    if log_file:
        ensure_parent(log_file)
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(message + "\n")
    print(message)

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def load_split_data(
    ratings: pd.DataFrame,
    split_registry: pd.DataFrame,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_seed = split_registry[split_registry["seed"] == seed].copy()

    train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
    val_keys = split_seed[split_seed["split"] == "val"][["user_id", "item_id"]]
    test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

    train = ratings.merge(train_keys, on=["user_id", "item_id"], how="inner")
    val = ratings.merge(val_keys, on=["user_id", "item_id"], how="inner")
    test = ratings.merge(test_keys, on=["user_id", "item_id"], how="inner")

    return train, val, test

def build_id_maps(ratings: pd.DataFrame) -> Tuple[Dict, Dict, Dict, Dict]:
    users = sorted(ratings["user_id"].unique())
    items = sorted(ratings["item_id"].unique())

    user_to_idx = {u: idx for idx, u in enumerate(users)}
    item_to_idx = {i: idx for idx, i in enumerate(items)}

    idx_to_user = {idx: u for u, idx in user_to_idx.items()}
    idx_to_item = {idx: i for i, idx in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item

def make_train_matrix(
    train: pd.DataFrame,
    user_to_idx: Dict,
    item_to_idx: Dict,
) -> np.ndarray:
    mat = np.zeros((len(user_to_idx), len(item_to_idx)), dtype=np.float32)

    for row in train.itertuples(index=False):
        u = user_to_idx[row.user_id]
        i = item_to_idx[row.item_id]
        mat[u, i] = float(row.rating)

    return mat

def prediction_metrics(test: pd.DataFrame, pred: np.ndarray) -> Tuple[float, float]:
    y_true = test["rating"].to_numpy(dtype=float)
    y_pred = np.asarray(pred, dtype=float)
    y_pred = np.clip(y_pred, 0.5, 5.0)
    return rmse(y_true, y_pred), mae(y_true, y_pred)

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def ranking_metrics_for_scores(
    scores_by_user: Dict,
    train: pd.DataFrame,
    test: pd.DataFrame,
    all_items: List,
    top_k_values: List[int],
    relevance_threshold: float,
) -> Dict[str, float]:
    train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()

    relevant_test = test[test["rating"] >= relevance_threshold]
    relevant_by_user = relevant_test.groupby("user_id")["item_id"].apply(set).to_dict()

    metric_store = {}
    for k in top_k_values:
        metric_store[f"precision_at_{k}"] = []
        metric_store[f"recall_at_{k}"] = []
        metric_store[f"ndcg_at_{k}"] = []

    for user_id, relevant_items in relevant_by_user.items():
        if len(relevant_items) == 0:
            continue

        train_seen = train_items_by_user.get(user_id, set())
        candidate_items = [i for i in all_items if i not in train_seen]

        if user_id not in scores_by_user:
            continue

        user_scores = scores_by_user[user_id]
        candidate_scores = [(i, user_scores.get(i, -np.inf)) for i in candidate_items]
        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        for k in top_k_values:
            top_items = [i for i, _ in candidate_scores[:k]]
            hits = [1 if i in relevant_items else 0 for i in top_items]

            precision = sum(hits) / k
            recall = sum(hits) / len(relevant_items)

            dcg = 0.0
            for rank, hit in enumerate(hits, start=1):
                if hit:
                    dcg += 1.0 / np.log2(rank + 1)

            ideal_count = min(len(relevant_items), k)
            idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_count + 1))
            ndcg = dcg / idcg if idcg > 0 else 0.0

            metric_store[f"precision_at_{k}"].append(precision)
            metric_store[f"recall_at_{k}"].append(recall)
            metric_store[f"ndcg_at_{k}"].append(ndcg)

    return {
        key: float(np.mean(values)) if values else 0.0
        for key, values in metric_store.items()
    }

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def run_item_cf(
    train: pd.DataFrame,
    test: pd.DataFrame,
    all_items: List,
    top_k_values: List[int],
    relevance_threshold: float,
) -> Dict[str, float]:
    global_mean = train["rating"].mean()
    item_mean = train.groupby("item_id")["rating"].mean().to_dict()

    all_data = pd.concat(
        [
            train[["user_id", "item_id", "rating"]],
            test[["user_id", "item_id", "rating"]],
        ]
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    train_mat = make_train_matrix(train, user_to_idx, item_to_idx)

    item_norms = np.linalg.norm(train_mat, axis=0) + 1e-12
    sim = (train_mat.T @ train_mat) / np.outer(item_norms, item_norms)
    np.fill_diagonal(sim, 0.0)

    preds = []
    scores_by_user = {}

    for user_id, group in train.groupby("user_id"):
        rated_items = group["item_id"].tolist()
        rated_ratings = group["rating"].to_numpy(dtype=float)

        rated_indices = [item_to_idx[i] for i in rated_items if i in item_to_idx]

        user_scores = {}

        for item_id in all_items:
            if item_id not in item_to_idx:
                user_scores[item_id] = item_mean.get(item_id, global_mean)
                continue

            j = item_to_idx[item_id]
            sims = sim[j, rated_indices]

            denom = np.sum(np.abs(sims))

            if denom > 1e-12:
                score = float(np.dot(sims, rated_ratings[:len(sims)]) / denom)
            else:
                score = item_mean.get(item_id, global_mean)

            user_scores[item_id] = score

        scores_by_user[user_id] = user_scores

    for row in test.itertuples(index=False):
        if row.user_id in scores_by_user:
            pred = scores_by_user[row.user_id].get(
                row.item_id,
                item_mean.get(row.item_id, global_mean),
            )
        else:
            pred = item_mean.get(row.item_id, global_mean)

        preds.append(pred)

    out = {}
    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

    out.update(
        ranking_metrics_for_scores(
            scores_by_user=scores_by_user,
            train=train,
            test=test,
            all_items=all_items,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
        )
    )

    return out

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def train_biased_mf(
    train: pd.DataFrame,
    user_to_idx: Dict,
    item_to_idx: Dict,
    k: int = 30,
    epochs: int = 25,
    lr: float = 0.01,
    reg: float = 0.05,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    mu = float(train["rating"].mean())
    bu = np.zeros(n_users, dtype=np.float32)
    bi = np.zeros(n_items, dtype=np.float32)

    P = 0.05 * rng.standard_normal((n_users, k)).astype(np.float32)
    Q = 0.05 * rng.standard_normal((n_items, k)).astype(np.float32)

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for _ in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = mu + bu[u] + bi[i] + float(P[u] @ Q[i])
            err = rating - pred

            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])

            old_p = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * old_p - reg * Q[i])

    return mu, bu, bi, P, Q

def run_biased_mf(
    train: pd.DataFrame,
    test: pd.DataFrame,
    all_items: List,
    top_k_values: List[int],
    relevance_threshold: float,
    seed: int,
) -> Dict[str, float]:
    all_data = pd.concat(
        [
            train[["user_id", "item_id", "rating"]],
            test[["user_id", "item_id", "rating"]],
        ]
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    mu, bu, bi, P, Q = train_biased_mf(
        train=train,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        seed=seed,
    )

    preds = []

    for row in test.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(mu)
        else:
            preds.append(mu + bu[u] + bi[i] + float(P[u] @ Q[i]))

    scores_by_user = {}

    for user_id, u in user_to_idx.items():
        scores = mu + bu[u] + bi + (Q @ P[u])
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    out = {}
    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

    out.update(
        ranking_metrics_for_scores(
            scores_by_user=scores_by_user,
            train=train,
            test=test,
            all_items=all_items,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
        )
    )

    return out

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def train_plain_nmf(
    train: pd.DataFrame,
    user_to_idx: Dict,
    item_to_idx: Dict,
    k: int = 30,
    epochs: int = 35,
    lr: float = 0.005,
    reg: float = 0.03,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    U = rng.random((n_users, k), dtype=np.float32) * 0.1
    V = rng.random((n_items, k), dtype=np.float32) * 0.1

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for _ in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = float(U[u] @ V[i])
            err = rating - pred

            old_u = U[u].copy()

            U[u] += lr * (err * V[i] - reg * U[u])
            V[i] += lr * (err * old_u - reg * V[i])

            U[u] = np.maximum(U[u], 1e-8)
            V[i] = np.maximum(V[i], 1e-8)

    return U, V

def run_plain_nmf(
    train: pd.DataFrame,
    test: pd.DataFrame,
    all_items: List,
    top_k_values: List[int],
    relevance_threshold: float,
    seed: int,
) -> Dict[str, float]:
    all_data = pd.concat(
        [
            train[["user_id", "item_id", "rating"]],
            test[["user_id", "item_id", "rating"]],
        ]
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V = train_plain_nmf(
        train=train,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        seed=seed,
    )

    global_mean = train["rating"].mean()

    preds = []

    for row in test.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    scores_by_user = {}

    for user_id, u in user_to_idx.items():
        scores = V @ U[u]
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    out = {}
    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

    out.update(
        ranking_metrics_for_scores(
            scores_by_user=scores_by_user,
            train=train,
            test=test,
            all_items=all_items,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
        )
    )

    return out

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def aggregate_results(run_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    metric_cols = [
        "rmse",
        "mae",
        "precision_at_5",
        "precision_at_10",
        "recall_at_5",
        "recall_at_10",
        "ndcg_at_5",
        "ndcg_at_10",
    ]

    rows = []

    for model_name, group in run_df.groupby("model_name"):
        row = {
            "dataset_name": dataset_name,
            "model_name": model_name,
        }

        for col in metric_cols:
            row[col] = group[col].mean()
            row[f"{col}_std"] = group[col].std(ddof=0)

        rows.append(row)

    return pd.DataFrame(rows)

def export_main_comparison(
    dataset: str,
    input_ratings: str,
    input_descriptors: str,
    seed_list: str,
    descriptor_source: str,
    top_k: str,
    relevance_threshold: float,
    output_csv: str,
    log_file: str | None = None,
    overwrite: str = "false",
):
    del input_descriptors
    del descriptor_source
    del overwrite

    ensure_parent(output_csv)

    ratings = pd.read_csv(input_ratings)
    split_path = f"results/csv/{dataset}_split_registry.csv"

    if not os.path.exists(split_path):
        raise FileNotFoundError(f"Split registry not found: {split_path}")

    split_registry = pd.read_csv(split_path)

    seeds = parse_seed_list(seed_list)
    top_k_values = parse_top_k(top_k)

    required_topk = {5, 10}
    if not required_topk.issubset(set(top_k_values)):
        raise ValueError("This workflow expects --top_k to include 5 and 10.")

    all_items = sorted(ratings["item_id"].unique())

    run_rows = []

    for seed in seeds:
        write_log(log_file, f"Running seed={seed}")
        train, val, test = load_split_data(ratings, split_registry, seed)

        models = {
            "item_cf": lambda: run_item_cf(
                train=train,
                test=test,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
            ),
            "biased_mf": lambda: run_biased_mf(
                train=train,
                test=test,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=seed,
            ),
            "plain_nmf": lambda: run_plain_nmf(
                train=train,
                test=test,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=seed,
            ),
        }

        for model_name, runner in models.items():
            write_log(log_file, f"  Model: {model_name}")
            metrics = runner()

            row = {
                "dataset_name": dataset,
                "model_name": model_name,
                "seed": seed,
            }

            row.update(metrics)
            run_rows.append(row)

    run_df = pd.DataFrame(run_rows)

    run_level_path = f"results/csv/{dataset}_run_level_metrics.csv"
    ensure_parent(run_level_path)
    run_df.to_csv(run_level_path, index=False)

    summary_df = aggregate_results(run_df, dataset)
    summary_df.to_csv(output_csv, index=False)

    write_log(log_file, f"Saved run-level metrics: {run_level_path}")
    write_log(log_file, f"Saved main comparison: {output_csv}")

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--dataset", required=True)
    parser.add_argument("--input_ratings", required=True)
    parser.add_argument("--input_descriptors", required=True)
    parser.add_argument("--seed_list", required=True)
    parser.add_argument("--descriptor_source", default="tags_genres")
    parser.add_argument("--top_k", default="5,10")
    parser.add_argument("--relevance_threshold", type=float, default=4.0)
    parser.add_argument("--output_csv", required=True)
    parser.add_argument("--log_file", default=None)
    parser.add_argument("--overwrite", default="false")

    args = parser.parse_args()
    export_main_comparison(**vars(args))


if __name__ == "__main__":
    main()

In [ ]:
!python -m py_compile src/run_main_benchmarks.py
print("Syntax check passed.")

In [ ]:
!python src/run_main_benchmarks.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --seed_list 42,123,2026,7,99 \
          --descriptor_source tags_genres \
            --top_k 5,10 \
              --relevance_threshold 4.0 \
                --output_csv results/csv/ml_latest_small_main_comparison.csv \
                  --log_file results/logs/ml_latest_small_main_comparison.log \
                    --overwrite true

In [ ]:
from pathlib import Path

bad_file = Path("src/run_main_benchmarks.py")
if bad_file.exists():
    bad_file.unlink()
    print("Deleted corrupted src/run_main_benchmarks.py")
else:
    print("No existing run_main_benchmarks.py found.")

In [ ]:
from pathlib import Path
import textwrap

Path("src").mkdir(parents=True, exist_ok=True)

def write_chunk(path, content, mode="a"):
    text = textwrap.dedent(content).strip() + "\n\n"
    if mode == "w":
        Path(path).write_text(text, encoding="utf-8")
    else:
        with open(path, "a", encoding="utf-8") as f:
            f.write(text)

In [ ]:
write_chunk(
    "src/benchmark_utils.py",
    """
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

def parse_seed_list(seed_list: str) -> List[int]:
    return [int(x.strip()) for x in seed_list.split(",") if x.strip()]

def parse_top_k(top_k: str) -> List[int]:
    return [int(x.strip()) for x in top_k.split(",") if x.strip()]

def ensure_parent(path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

def write_log(log_file: str | None, message: str) -> None:
    if log_file:
        ensure_parent(log_file)
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(message + "\n")
    print(message)

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))
""",
    mode="w",
)

In [ ]:
write_chunk(
      "src/benchmark_utils.py",
          """
              def load_split_data(
                      ratings: pd.DataFrame,
                              split_registry: pd.DataFrame,
                                      seed: int,
                                          ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

                                                  split_seed = split_registry[split_registry["seed"] == seed].copy()

                                                          train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
                                                                  val_keys = split_seed[split_seed["split"] == "val"][["user_id", "item_id"]]
                                                                          test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

                                                                                  train = ratings.merge(train_keys, on=["user_id", "item_id"], how="inner")
                                                                                          val = ratings.merge(val_keys, on=["user_id", "item_id"], how="inner")
                                                                                                  test = ratings.merge(test_keys, on=["user_id", "item_id"], how="inner")

                                                                                                          return train, val, test
                                                                                                              """,
                                                                                                              )

In [ ]:
write_chunk(
      "src/benchmark_utils.py",
          """
              def build_id_maps(ratings: pd.DataFrame):
                      users = sorted(ratings["user_id"].unique())
                              items = sorted(ratings["item_id"].unique())

                                      user_to_idx = {u: idx for idx, u in enumerate(users)}
                                              item_to_idx = {i: idx for idx, i in enumerate(items)}

                                                      idx_to_user = {idx: u for u, idx in user_to_idx.items()}
                                                              idx_to_item = {idx: i for i, idx in item_to_idx.items()}

                                                                      return user_to_idx, item_to_idx, idx_to_user, idx_to_item


                                                                          def prediction_metrics(test: pd.DataFrame, pred: np.ndarray):
                                                                                  y_true = test["rating"].to_numpy(dtype=float)
                                                                                          y_pred = np.asarray(pred, dtype=float)
                                                                                                  y_pred = np.clip(y_pred, 0.5, 5.0)

                                                                                                          return rmse(y_true, y_pred), mae(y_true, y_pred)
                                                                                                              """
                                                                                                              )

In [ ]:
write_chunk(
      "src/benchmark_utils.py",
          """
              def ranking_metrics_for_scores(
                      scores_by_user: Dict,
                              train: pd.DataFrame,
                                      test: pd.DataFrame,
                                              all_items: List,
                                                      top_k_values: List[int],
                                                              relevance_threshold: float,
                                                                  ) -> Dict[str, float]:

                                                                          train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()

                                                                                  relevant_test = test[test["rating"] >= relevance_threshold]
                                                                                          relevant_by_user = relevant_test.groupby("user_id")["item_id"].apply(set).to_dict()

                                                                                                  metric_store = {}
                                                                                                          for k in top_k_values:
                                                                                                                      metric_store[f"precision_at_{k}"] = []
                                                                                                                                  metric_store[f"recall_at_{k}"] = []
                                                                                                                                              metric_store[f"ndcg_at_{k}"] = []
                                                                                                                                                  """,
                                                                                                                                                  )

In [ ]:
write_chunk(
      "src/benchmark_utils.py",
          """
                  for user_id, relevant_items in relevant_by_user.items():
                              train_seen = train_items_by_user.get(user_id, set())
                                          candidate_items = [i for i in all_items if i not in train_seen]

                                                      if user_id not in scores_by_user:
                                                                      continue

                                                                                  user_scores = scores_by_user[user_id]
                                                                                              candidate_scores = [(i, user_scores.get(i, -np.inf)) for i in candidate_items]
                                                                                                          candidate_scores.sort(key=lambda x: x[1], reverse=True)

                                                                                                                      for k in top_k_values:
                                                                                                                                      top_items = [i for i, _ in candidate_scores[:k]]
                                                                                                                                                      hits = [1 if i in relevant_items else 0 for i in top_items]

                                                                                                                                                                      precision = sum(hits) / k
                                                                                                                                                                                      recall = sum(hits) / len(relevant_items)

                                                                                                                                                                                                      dcg = 0.0
                                                                                                                                                                                                                      for rank, hit in enumerate(hits, start=1):
                                                                                                                                                                                                                                          if hit:
                                                                                                                                                                                                                                                                  dcg += 1.0 / np.log2(rank + 1)

                                                                                                                                                                                                                                                                                  ideal_count = min(len(relevant_items), k)
                                                                                                                                                                                                                                                                                                  idcg = sum(
                                                                                                                                                                                                                                                                                                                      1.0 / np.log2(rank + 1)
                                                                                                                                                                                                                                                                                                                                          for rank in range(1, ideal_count + 1)
                                                                                                                                                                                                                                                                                                                                                          )

                                                                                                                                                                                                                                                                                                                                                                          ndcg = dcg / idcg if idcg > 0 else 0.0

                                                                                                                                                                                                                                                                                                                                                                                          metric_store[f"precision_at_{k}"].append(precision)
                                                                                                                                                                                                                                                                                                                                                                                                          metric_store[f"recall_at_{k}"].append(recall)
                                                                                                                                                                                                                                                                                                                                                                                                                          metric_store[f"ndcg_at_{k}"].append(ndcg)

                                                                                                                                                                                                                                                                                                                                                                                                                                  return {
                                                                                                                                                                                                                                                                                                                                                                                                                                              key: float(np.mean(values)) if values else 0.0
                                                                                                                                                                                                                                                                                                                                                                                                                                                          for key, values in metric_store.items()
                                                                                                                                                                                                                                                                                                                                                                                                                                                                  }
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      """
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
from typing import Dict, List

import numpy as np
import pandas as pd

from benchmark_utils import (
        build_id_maps,
                prediction_metrics,
                        ranking_metrics_for_scores,
                            )
""",
                                                              mode="w"
                                                              )

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
def run_item_mean(
        train: pd.DataFrame,
                test: pd.DataFrame,
                        all_items: List,
                                top_k_values: List[int],
                                        relevance_threshold: float,
                                            ) -> Dict[str, float]:

    global_mean = train["rating"].mean()
            item_mean = train.groupby("item_id")["rating"].mean().to_dict()

            preds = [
                        item_mean.get(row.item_id, global_mean)
                                    for row in test.itertuples(index=False)
                                            ]

                                                    scores_by_user = {}
                                                            users = sorted(train["user_id"].unique())

                                                                    for user_id in users:
                                                                                scores_by_user[user_id] = {
                                                                                                item_id: item_mean.get(item_id, global_mean)
                                                                                                                for item_id in all_items
                                                                                                                            }

                                                                                                                                    out = {}
                                                                                                                                            out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

                                                                                                                                                    out.update(
                                                                                                                                                                ranking_metrics_for_scores(
                                                                                                                                                                                scores_by_user=scores_by_user,
                                                                                                                                                                                                train=train,
                                                                                                                                                                                                                test=test,
                                                                                                                                                                                                                                all_items=all_items,
                                                                                                                                                                                                                                                top_k_values=top_k_values,
                                                                                                                                                                                                                                                                relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                                            )
                                                                                                                                                                                                                                                                                    )

                                                                                                                                                                                                                                                                                            return out
"""
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
def train_biased_mf(
        train: pd.DataFrame,
                user_to_idx: Dict,
                        item_to_idx: Dict,
                                k: int = 30,
                                        epochs: int = 20,
                                                lr: float = 0.01,
                                                        reg: float = 0.05,
                                                                seed: int = 42,
                                                                    ):
    rng = np.random.default_rng(seed)

            n_users = len(user_to_idx)
                    n_items = len(item_to_idx)

                            mu = float(train["rating"].mean())
                                    bu = np.zeros(n_users, dtype=np.float32)
                                            bi = np.zeros(n_items, dtype=np.float32)

                                                    P = 0.05 * rng.standard_normal((n_users, k)).astype(np.float32)
                                                            Q = 0.05 * rng.standard_normal((n_items, k)).astype(np.float32)

                                                                    triples = [
                                                                                (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
                                                                                            for r in train.itertuples(index=False)
                                                                                                        if r.user_id in user_to_idx and r.item_id in item_to_idx
                                                                                                                ]
"""
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
    for _ in range(epochs):
                rng.shuffle(triples)

                            for u, i, rating in triples:
                                            pred = mu + bu[u] + bi[i] + float(P[u] @ Q[i])
                                                            err = rating - pred

                                                                            bu[u] += lr * (err - reg * bu[u])
                                                                                            bi[i] += lr * (err - reg * bi[i])

                                                                                                            old_p = P[u].copy()
                                                                                                                            P[u] += lr * (err * Q[i] - reg * P[u])
                                                                                                                                            Q[i] += lr * (err * old_p - reg * Q[i])

                                                                                                                                                    return mu, bu, bi, P, Q
"""
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
def run_biased_mf(
        train: pd.DataFrame,
                test: pd.DataFrame,
                        all_items: List,
                                top_k_values: List[int],
                                        relevance_threshold: float,
                                                seed: int,
                                                    ) -> Dict[str, float]:

    all_data = pd.concat(
                [
                                train[["user_id", "item_id", "rating"]],
                                                test[["user_id", "item_id", "rating"]],
                                                            ],
                                                                        ignore_index=True,
                                                                                )

                                                                                        user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

                                                                                                mu, bu, bi, P, Q = train_biased_mf(
                                                                                                            train=train,
                                                                                                                        user_to_idx=user_to_idx,
                                                                                                                                    item_to_idx=item_to_idx,
                                                                                                                                                seed=seed,
                                                                                                                                                        )

                                                                                                                                                                preds = []
                                                                                                                                                                        for row in test.itertuples(index=False):
                                                                                                                                                                                    u = user_to_idx.get(row.user_id)
                                                                                                                                                                                                i = item_to_idx.get(row.item_id)

                                                                                                                                                                                                            if u is None or i is None:
                                                                                                                                                                                                                            preds.append(mu)
                                                                                                                                                                                                                                        else:
                                                                                                                                                                                                                                                        preds.append(mu + bu[u] + bi[i] + float(P[u] @ Q[i]))

                                                                                                                                                                                                                                                scores_by_user = {}
                                                                                                                                                                                                                                                        for user_id, u in user_to_idx.items():
                                                                                                                                                                                                                                                                    scores = mu + bu[u] + bi + (Q @ P[u])
                                                                                                                                                                                                                                                                                scores_by_user[user_id] = {
                                                                                                                                                                                                                                                                                                idx_to_item[i]: float(scores[i])
                                                                                                                                                                                                                                                                                                                for i in range(len(idx_to_item))
                                                                                                                                                                                                                                                                                                                            }

                                                                                                                                                                                                                                                                                                                                    out = {}
                                                                                                                                                                                                                                                                                                                                            out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

                                                                                                                                                                                                                                                                                                                                                    out.update(
                                                                                                                                                                                                                                                                                                                                                                ranking_metrics_for_scores(
                                                                                                                                                                                                                                                                                                                                                                                scores_by_user=scores_by_user,
                                                                                                                                                                                                                                                                                                                                                                                                train=train,
                                                                                                                                                                                                                                                                                                                                                                                                                test=test,
                                                                                                                                                                                                                                                                                                                                                                                                                                all_items=all_items,
                                                                                                                                                                                                                                                                                                                                                                                                                                                top_k_values=top_k_values,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                            )
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    )

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            return out
"""
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
def train_plain_nmf(
        train: pd.DataFrame,
                user_to_idx: Dict,
                        item_to_idx: Dict,
                                k: int = 30,
                                        epochs: int = 30,
                                                lr: float = 0.005,
                                                        reg: float = 0.03,
                                                                seed: int = 42,
                                                                    ):
    rng = np.random.default_rng(seed)

            n_users = len(user_to_idx)
                    n_items = len(item_to_idx)

                            U = rng.random((n_users, k), dtype=np.float32) * 0.1
                                    V = rng.random((n_items, k), dtype=np.float32) * 0.1

                                            triples = [
                                                        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
                                                                    for r in train.itertuples(index=False)
                                                                                if r.user_id in user_to_idx and r.item_id in item_to_idx
                                                                                        ]

                                                                                                for _ in range(epochs):
                                                                                                            rng.shuffle(triples)

                                                                                                                        for u, i, rating in triples:
                                                                                                                                        pred = float(U[u] @ V[i])
                                                                                                                                                        err = rating - pred

                                                                                                                                                                        old_u = U[u].copy()

                                                                                                                                                                                        U[u] += lr * (err * V[i] - reg * U[u])
                                                                                                                                                                                                        V[i] += lr * (err * old_u - reg * V[i])

                                                                                                                                                                                                                        U[u] = np.maximum(U[u], 1e-8)
                                                                                                                                                                                                                                        V[i] = np.maximum(V[i], 1e-8)

                                                                                                                                                                                                                                                return U, V
"""
)

In [ ]:
write_chunk(
      "src/benchmark_models.py",
          """
def run_plain_nmf(
        train: pd.DataFrame,
                test: pd.DataFrame,
                        all_items: List,
                                top_k_values: List[int],
                                        relevance_threshold: float,
                                                seed: int,
                                                    ) -> Dict[str, float]:

    all_data = pd.concat(
                [
                                train[["user_id", "item_id", "rating"]],
                                                test[["user_id", "item_id", "rating"]],
                                                            ],
                                                                        ignore_index=True,
                                                                                )

                                                                                        user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

                                                                                                U, V = train_plain_nmf(
                                                                                                            train=train,
                                                                                                                        user_to_idx=user_to_idx,
                                                                                                                                    item_to_idx=item_to_idx,
                                                                                                                                                seed=seed,
                                                                                                                                                        )

                                                                                                                                                                global_mean = train["rating"].mean()

                                                                                                                                                                        preds = []
                                                                                                                                                                                for row in test.itertuples(index=False):
                                                                                                                                                                                            u = user_to_idx.get(row.user_id)
                                                                                                                                                                                                        i = item_to_idx.get(row.item_id)

                                                                                                                                                                                                                    if u is None or i is None:
                                                                                                                                                                                                                                    preds.append(global_mean)
                                                                                                                                                                                                                                else:
                                                                                                                                                                                                                                                    preds.append(float(U[u] @ V[i]))

                                                                                                                                                                                                                                            scores_by_user = {}
                                                                                                                                                                                                                                                    for user_id, u in user_to_idx.items():
                                                                                                                                                                                                                                                                scores = V @ U[u]
                                                                                                                                                                                                                                                                            scores_by_user[user_id] = {
                                                                                                                                                                                                                                                                                            idx_to_item[i]: float(scores[i])
                                                                                                                                                                                                                                                                                                            for i in range(len(idx_to_item))
                                                                                                                                                                                                                                                                                                                        }

                                                                                                                                                                                                                                                                                                                                out = {}
                                                                                                                                                                                                                                                                                                                                        out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))

                                                                                                                                                                                                                                                                                                                                                out.update(
                                                                                                                                                                                                                                                                                                                                                            ranking_metrics_for_scores(
                                                                                                                                                                                                                                                                                                                                                                            scores_by_user=scores_by_user,
                                                                                                                                                                                                                                                                                                                                                                                            train=train,
                                                                                                                                                                                                                                                                                                                                                                                                            test=test,
                                                                                                                                                                                                                                                                                                                                                                                                                            all_items=all_items,
                                                                                                                                                                                                                                                                                                                                                                                                                                            top_k_values=top_k_values,
                                                                                                                                                                                                                                                                                                                                                                                                                                                            relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        )
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    )

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            return out
"""
)

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
              import argparse
                  import os

                      import pandas as pd

                          from benchmark_models import (
                                  run_biased_mf,
                                          run_item_mean,
                                                  run_plain_nmf,
                                                      )

                                                          from benchmark_utils import (
                                                                  ensure_parent,
                                                                          load_split_data,
                                                                                  parse_seed_list,
                                                                                          parse_top_k,
                                                                                                  write_log,
                                                                                                      )
                                                                                                          """,
                                                                                                              mode="w",
                                                                                                              )

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
def aggregate_results(run_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    metric_cols = [
        "rmse",
        "mae",
        "precision_at_5",
        "precision_at_10",
        "recall_at_5",
        "recall_at_10",
        "ndcg_at_5",
        "ndcg_at_10",
    ]

    rows = []

    for model_name, group in run_df.groupby("model_name"):
        row = {
            "dataset_name": dataset_name,
            "model_name": model_name,
        }

        for col in metric_cols:
            row[col] = group[col].mean()
            row[f"{col}_std"] = group[col].std(ddof=0)

        rows.append(row)

    return pd.DataFrame(rows)
""",
)

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
              def export_main_comparison(
                      dataset: str,
                              input_ratings: str,
                                      input_descriptors: str,
                                              seed_list: str,
                                                      descriptor_source: str,
                                                              top_k: str,
                                                                      relevance_threshold: float,
                                                                              output_csv: str,
                                                                                      log_file: str | None = None,
                                                                                              overwrite: str = "false",
                                                                                                  ):
                                                                                                          del input_descriptors
                                                                                                                  del descriptor_source
                                                                                                                          del overwrite

                                                                                                                                  ensure_parent(output_csv)

                                                                                                                                          ratings = pd.read_csv(input_ratings)
                                                                                                                                                  split_path = f"results/csv/{dataset}_split_registry.csv"

                                                                                                                                                          if not os.path.exists(split_path):
                                                                                                                                                                      raise FileNotFoundError(f"Split registry not found: {split_path}")

                                                                                                                                                                              split_registry = pd.read_csv(split_path)

                                                                                                                                                                                      seeds = parse_seed_list(seed_list)
                                                                                                                                                                                              top_k_values = parse_top_k(top_k)

                                                                                                                                                                                                      required_topk = {5, 10}
                                                                                                                                                                                                              if not required_topk.issubset(set(top_k_values)):
                                                                                                                                                                                                                          raise ValueError("This workflow expects --top_k to include 5 and 10.")

                                                                                                                                                                                                                                  all_items = sorted(ratings["item_id"].unique())
                                                                                                                                                                                                                                          run_rows = []
                                                                                                                                                                                                                                              """,
                                                                                                                                                                                                                                              )

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
                  for seed in seeds:
                              write_log(log_file, f"Running seed={seed}")

                                          train, val, test = load_split_data(
                                                          ratings=ratings,
                                                                          split_registry=split_registry,
                                                                                          seed=seed,
                                                                                                      )

                                                                                                                  models = {
                                                                                                                                  "item_mean": lambda: run_item_mean(
                                                                                                                                                      train=train,
                                                                                                                                                                          test=test,
                                                                                                                                                                                              all_items=all_items,
                                                                                                                                                                                                                  top_k_values=top_k_values,
                                                                                                                                                                                                                                      relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                      ),
                                                                                                                                                                                                                                                                      "biased_mf": lambda: run_biased_mf(
                                                                                                                                                                                                                                                                                          train=train,
                                                                                                                                                                                                                                                                                                              test=test,
                                                                                                                                                                                                                                                                                                                                  all_items=all_items,
                                                                                                                                                                                                                                                                                                                                                      top_k_values=top_k_values,
                                                                                                                                                                                                                                                                                                                                                                          relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                                                                                                                                                              seed=seed,
                                                                                                                                                                                                                                                                                                                                                                                                              ),
                                                                                                                                                                                                                                                                                                                                                                                                                              "plain_nmf": lambda: run_plain_nmf(
                                                                                                                                                                                                                                                                                                                                                                                                                                                  train=train,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      test=test,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          all_items=all_items,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              top_k_values=top_k_values,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  relevance_threshold=relevance_threshold,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      seed=seed,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      ),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  }
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      """
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      )

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
                      for model_name, runner in models.items():
                                      write_log(log_file, f"  Model: {model_name}")
                                                      metrics = runner()

                                                                      row = {
                                                                                          "dataset_name": dataset,
                                                                                                              "model_name": model_name,
                                                                                                                                  "seed": seed,
                                                                                                                                                  }

                                                                                                                                                                  row.update(metrics)
                                                                                                                                                                                  run_rows.append(row)

                                                                                                                                                                                          run_df = pd.DataFrame(run_rows)

                                                                                                                                                                                                  run_level_path = f"results/csv/{dataset}_run_level_metrics.csv"
                                                                                                                                                                                                          ensure_parent(run_level_path)
                                                                                                                                                                                                                  run_df.to_csv(run_level_path, index=False)

                                                                                                                                                                                                                          summary_df = aggregate_results(run_df, dataset)
                                                                                                                                                                                                                                  summary_df.to_csv(output_csv, index=False)

                                                                                                                                                                                                                                          write_log(log_file, f"Saved run-level metrics: {run_level_path}")
                                                                                                                                                                                                                                                  write_log(log_file, f"Saved main comparison: {output_csv}")
                                                                                                                                                                                                                                                      """
                                                                                                                                                                                                                                                      )

In [ ]:
write_chunk(
      "src/run_main_benchmarks.py",
          """
              def main():
                      parser = argparse.ArgumentParser()

                              parser.add_argument("--dataset", required=True)
                                      parser.add_argument("--input_ratings", required=True)
                                              parser.add_argument("--input_descriptors", required=True)
                                                      parser.add_argument("--seed_list", required=True)
                                                              parser.add_argument("--descriptor_source", default="tags_genres")
                                                                      parser.add_argument("--top_k", default="5,10")
                                                                              parser.add_argument("--relevance_threshold", type=float, default=4.0)
                                                                                      parser.add_argument("--output_csv", required=True)
                                                                                              parser.add_argument("--log_file", default=None)
                                                                                                      parser.add_argument("--overwrite", default="false")

                                                                                                              args = parser.parse_args()
                                                                                                                      export_main_comparison(**vars(args))


                                                                                                                          if __name__ == "__main__":
                                                                                                                                  main()
                                                                                                                                      """,
                                                                                                                                      )

In [ ]:
!python -m py_compile src/benchmark_utils.py
!python -m py_compile src/benchmark_models.py
!python -m py_compile src/run_main_benchmarks.py
print("Syntax check passed for all benchmark scripts.")

In [ ]:
import pandas as pd

main_comparison_df = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")
display(main_comparison_df)

In [ ]:
from pathlib import Path

benchmark_utils_file = Path("src/benchmark_utils.py")
if benchmark_utils_file.exists():
    benchmark_utils_file.unlink()
    print("Deleted redundant src/benchmark_utils.py")
else:
    print("No redundant src/benchmark_utils.py found to delete.")

In [ ]:
%%writefile -a src/run_main_benchmarks.py

def load_split_data(
    ratings: pd.DataFrame,
    split_registry: pd.DataFrame,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_seed = split_registry[split_registry["seed"] == seed].copy()

    train_keys = split_seed[split_seed["split"] == "train"][[("user_id"), "item_id"]]
    val_keys = split_seed[split_seed["split"] == "val"][[("user_id"), "item_id"]]
    test_keys = split_seed[split_seed["split"] == "test"][[("user_id"), "item_id"]]

    train = ratings.merge(train_keys, on=[("user_id"), "item_id"], how="inner")
    val = ratings.merge(val_keys, on=[("user_id"), "item_id"], how="inner")
    test = ratings.merge(test_keys, on=[("user_id"), "item_id"], how="inner")

    return train, val, test

def build_id_maps(ratings: pd.DataFrame) -> Tuple[Dict, Dict, Dict, Dict]:
    users = sorted(ratings["user_id"].unique())
    items = sorted(ratings["item_id"].unique())

    user_to_idx = {u: idx for idx, u in enumerate(users)}
    item_to_idx = {i: idx for idx, i in enumerate(items)}

    idx_to_user = {idx: u for u, idx in user_to_idx.items()}
    idx_to_item = {idx: i for i, idx in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item

def make_train_matrix(
    train: pd.DataFrame,
    user_to_idx: Dict,
    item_to_idx: Dict,
) -> np.ndarray:
    mat = np.zeros((len(user_to_idx), len(item_to_idx)), dtype=np.float32)

    for row in train.itertuples(index=False):
        u = user_to_idx[row.user_id]
        i = item_to_idx[row.item_id]
        mat[u, i] = float(row.rating)

    return mat

def prediction_metrics(test: pd.DataFrame, pred: np.ndarray) -> Tuple[float, float]:
    y_true = test["rating"].to_numpy(dtype=float)
    y_pred = np.asarray(pred, dtype=float)
    y_pred = np.clip(y_pred, 0.5, 5.0)
    return rmse(y_true, y_pred), mae(y_true, y_pred)

In [ ]:
!rm -f src/benchmark_utils.py src/benchmark_models.py src/run_main_benchmarks.py
!ls -lh src | grep benchmark || true

In [ ]:
%%bash
cat > src/benchmark_utils.py <<'PY'
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


def parse_seed_list(seed_list: str) -> List[int]:
    return [int(x.strip()) for x in seed_list.split(",") if x.strip()]


def parse_top_k(top_k: str) -> List[int]:
    return [int(x.strip()) for x in top_k.split(",") if x.strip()]


def ensure_parent(path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def write_log(log_file, message: str) -> None:
    if log_file:
        ensure_parent(log_file)
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(message + "\n")
    print(message)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def load_split_data(
    ratings: pd.DataFrame,
    split_registry: pd.DataFrame,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split_seed = split_registry[split_registry["seed"] == seed].copy()

    train_keys = split_seed[split_seed["split"] == "train"][[("user_id"), "item_id"]]
    val_keys = split_seed[split_seed["split"] == "val"][[("user_id"), "item_id"]]
    test_keys = split_seed[split_seed["split"] == "test"][[("user_id"), "item_id"]]

    train = ratings.merge(train_keys, on=[("user_id"), "item_id"], how="inner")
    val = ratings.merge(val_keys, on=[("user_id"), "item_id"], how="inner")
    test = ratings.merge(test_keys, on=[("user_id"), "item_id"], how="inner")

    return train, val, test


def build_id_maps(ratings: pd.DataFrame):
    users = sorted(ratings["user_id"].unique())
    items = sorted(ratings["item_id"].unique())

    user_to_idx = {u: idx for idx, u in enumerate(users)}
    item_to_idx = {i: idx for idx, i in enumerate(items)}

    idx_to_user = {idx: u for u, idx in user_to_idx.items()}
    idx_to_item = {idx: i for i, idx in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


def prediction_metrics(test: pd.DataFrame, pred: np.ndarray):
    y_true = test["rating"].to_numpy(dtype=float)
    y_pred = np.asarray(pred, dtype=float)
    y_pred = np.clip(y_pred, 0.5, 5.0)

    return rmse(y_true, y_pred), mae(y_true, y_pred)


def ranking_metrics_for_scores(
    scores_by_user: Dict,
    train: pd.DataFrame,
    test: pd.DataFrame,
    all_items: List,
    top_k_values: List[int],
    relevance_threshold: float,
) -> Dict[str, float]:
    train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()

    relevant_test = test[test["rating"] >= relevance_threshold]
    relevant_by_user = relevant_test.groupby("user_id")["item_id"].apply(set).to_dict()

    metric_store = {}
    for k in top_k_values:
        metric_store[f"precision_at_{k}"] = []
        metric_store[f"recall_at_{k}"] = []
        metric_store[f"ndcg_at_{k}"] = []

    for user_id, relevant_items in relevant_by_user.items():
        train_seen = train_items_by_user.get(user_id, set())
        candidate_items = [i for i in all_items if i not in train_seen]

        if user_id not in scores_by_user:
            continue

        user_scores = scores_by_user[user_id]
        candidate_scores = [(i, user_scores.get(i, -np.inf)) for i in candidate_items]
        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        for k in top_k_values:
            top_items = [i for i, _ in candidate_scores[:k]]
            hits = [1 if i in relevant_items else 0 for i in top_items]

            precision = sum(hits) / k
            recall = sum(hits) / len(relevant_items)

            dcg = 0.0
            for rank, hit in enumerate(hits, start=1):
                if hit:
                    dcg += 1.0 / np.log2(rank + 1)

            ideal_count = min(len(relevant_items), k)
            idcg = sum(
                1.0 / np.log2(rank + 1)
                for rank in range(1, ideal_count + 1)
            )

            ndcg = dcg / idcg if idcg > 0 else 0.0

            metric_store[f"precision_at_{k}"].append(precision)
            metric_store[f"recall_at_{k}"].append(recall)
            metric_store[f"ndcg_at_{k}"].append(ndcg)

    return {
        key: float(np.mean(values)) if values else 0.0
        for key, values in metric_store.items()
    }
PY

In [ ]:
from pathlib import Path

model_file = Path("src/benchmark_models.py")
model_file.parent.mkdir(parents=True, exist_ok=True)

if model_file.exists():
    model_file.unlink()
    print("Deleted existing src/benchmark_models.py")

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines('src/benchmark_models.py', [
    'from typing import Dict, List',
    '',
    'import numpy as np',
    'import pandas as pd',
    '',
    'from benchmark_utils import (',
    '    build_id_maps,',
    '    prediction_metrics,',
    '    ranking_metrics_for_scores,',
    ')',
])

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines("src/benchmark_models.py", [
    "def run_item_mean(",
    "    train: pd.DataFrame,",
    "    test: pd.DataFrame,",
    "    all_items: List,",
    "    top_k_values: List[int],",
    "    relevance_threshold: float,",
    ") -> Dict[str, float]:",
    "    global_mean = train[\"rating\"].mean()",
    "    item_mean = train.groupby(\"{item_id}\")[\"rating\"].mean().to_dict()".format(item_id='item_id'),
    "",
    "    preds = [",
    "        item_mean.get(row.item_id, global_mean)",
    "        for row in test.itertuples(index=False)",
    "    ]",
    "",
    "    scores_by_user = {}",
    "    users = sorted(train[\"user_id\"].unique())",
    "",
    "    for user_id in users:",
    "        scores_by_user[user_id] = {",
    "            item_id: item_mean.get(item_id, global_mean)",
    "            for item_id in all_items",
    "        }",
    "",
    "    out = {}",
    "    out[\"rmse\"], out[\"mae\"] = prediction_metrics(test, np.array(preds))",
    "",
    "    out.update(",
    "        ranking_metrics_for_scores(",
    "            scores_by_user=scores_by_user,",
    "            train=train,",
    "            test=test,",
    "            all_items=all_items,",
    "            top_k_values=top_k_values,",
    "            relevance_threshold=relevance_threshold,",
    "        )",
    "    )",
    "",
    "    return out",
])

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines('src/benchmark_models.py', [
    'def train_biased_mf(',
    '    train: pd.DataFrame,',
    '    user_to_idx: Dict,',
    '    item_to_idx: Dict,',
    '    k: int = 30,',
    '    epochs: int = 20,',
    '    lr: float = 0.01,',
    '    reg: float = 0.05,',
    '    seed: int = 42,',
    '):',
    '    rng = np.random.default_rng(seed)',
    '',
    '    n_users = len(user_to_idx)',
    '    n_items = len(item_to_idx)',
    '',
    '    mu = float(train["rating"].mean())',
    '    bu = np.zeros(n_users, dtype=np.float32)',
    '    bi = np.zeros(n_items, dtype=np.float32)',
    '',
    '    P = 0.05 * rng.standard_normal((n_users, k)).astype(np.float32)',
    '    Q = 0.05 * rng.standard_normal((n_items, k)).astype(np.float32)',
    '',
    '    triples = [',
    '        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))',
    '        for r in train.itertuples(index=False)',
    '        if r.user_id in user_to_idx and r.item_id in item_to_idx',
    '    ]',
    '',
    '    for _ in range(epochs):',
    '        rng.shuffle(triples)',
    '',
    '        for u, i, rating in triples:',
    '            pred = mu + bu[u] + bi[i] + float(P[u] @ Q[i])',
    '            err = rating - pred',
    '',
    '            bu[u] += lr * (err - reg * bu[u])',
    '            bi[i] += lr * (err - reg * bi[i])',
    '',
    '            old_p = P[u].copy()',
    '            P[u] += lr * (err * Q[i] - reg * P[u])',
    '            Q[i] += lr * (err * old_p - reg * Q[i])',
    '',
    '    return mu, bu, bi, P, Q',
])

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines('src/benchmark_models.py', [
    'def run_biased_mf(',
    '    train: pd.DataFrame,',
    '    test: pd.DataFrame,',
    '    all_items: List,',
    '    top_k_values: List[int],',
    '    relevance_threshold: float,',
    '    seed: int,',
    ') -> Dict[str, float]:',
    '    all_data = pd.concat([',
    '        train[["user_id", "item_id", "rating"]],',
    '        test[["user_id", "item_id", "rating"]],',
    '    ], ignore_index=True)',
    '    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)',
    '    mu, bu, bi, P, Q = train_biased_mf(train, user_to_idx, item_to_idx, seed=seed)',
    '    preds = []',
    '    for row in test.itertuples(index=False):',
    '        u = user_to_idx.get(row.user_id)',
    '        i = item_to_idx.get(row.item_id)',
    '        if u is None or i is None:',
    '            preds.append(mu)',
    '        else:',
    '            preds.append(mu + bu[u] + bi[i] + float(P[u] @ Q[i]))',
    '    scores_by_user = {}',
    '    for user_id, u in user_to_idx.items():',
    '        scores = mu + bu[u] + bi + (Q @ P[u])',
    '        scores_by_user[user_id] = {idx_to_item[i]: float(scores[i]) for i in range(len(idx_to_item))}',
    '    out = {}',
    '    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))',
    '    out.update(ranking_metrics_for_scores(',
    '        scores_by_user=scores_by_user, train=train, test=test,',
    '        all_items=all_items, top_k_values=top_k_values, relevance_threshold=relevance_threshold))',
    '    return out',
])

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines("src/benchmark_models.py", [
    "def train_plain_nmf(",
    "    train: pd.DataFrame,",
    "    user_to_idx: Dict,",
    "    item_to_idx: Dict,",
    "    k: int = 30,",
    "    epochs: int = 30,",
    "    lr: float = 0.005,",
    "    reg: float = 0.03,",
    "    seed: int = 42,",
    "):",
    "    rng = np.random.default_rng(seed)",
    "",
    "    n_users = len(user_to_idx)",
    "    n_items = len(item_to_idx)",
    "",
    "    U = rng.random((n_users, k), dtype=np.float32) * 0.1",
    "    V = rng.random((n_items, k), dtype=np.float32) * 0.1",
    "",
    "    triples = [",
    "        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))",
    "        for r in train.itertuples(index=False)",
    "        if r.user_id in user_to_idx and r.item_id in item_to_idx",
    "    ]",
    "",
    "    for _ in range(epochs):",
    "        rng.shuffle(triples)",
    "",
    "        for u, i, rating in triples:",
    "            pred = float(U[u] @ V[i])",
    "            err = rating - pred",
    "",
    "            old_u = U[u].copy()",
    "",
    "            U[u] += lr * (err * V[i] - reg * U[u])",
    "            V[i] += lr * (err * old_u - reg * V[i])",
    "",
    "            U[u] = np.maximum(U[u], 1e-8)",
    "            V[i] = np.maximum(V[i], 1e-8)",
    "",
    "    return U, V",
])

In [ ]:
def append_lines(path, lines):
    with open(path, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n\n')

append_lines('src/benchmark_models.py', [
    'def run_plain_nmf(',
    '    train: pd.DataFrame,',
    '    test: pd.DataFrame,',
    '    all_items: List,',
    '    top_k_values: List[int],',
    '    relevance_threshold: float,',
    '    seed: int,',
    ') -> Dict[str, float]:',
    '    all_data = pd.concat([',
    '        train[["user_id", "item_id", "rating"]],',
    '        test[["user_id", "item_id", "rating"]],',
    '    ], ignore_index=True)',
    '    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)',
    '    U, V = train_plain_nmf(train, user_to_idx, item_to_idx, seed=seed)',
    '    global_mean = train["rating"].mean()',
    '    preds = []',
    '    for row in test.itertuples(index=False):',
    '        u = user_to_idx.get(row.user_id)',
    '        i = item_to_idx.get(row.item_id)',
    '        if u is None or i is None:',
    '            preds.append(global_mean)',
    '        else:',
    '            preds.append(float(U[u] @ V[i]))',
    '    scores_by_user = {}',
    '    for user_id, u in user_to_idx.items():',
    '        scores = V @ U[u]',
    '        scores_by_user[user_id] = {idx_to_item[i]: float(scores[i]) for i in range(len(idx_to_item))}',
    '    out = {}',
    '    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))',
    '    out.update(ranking_metrics_for_scores(',
    '        scores_by_user=scores_by_user, train=train, test=test,',
    '        all_items=all_items, top_k_values=top_k_values, relevance_threshold=relevance_threshold))',
    '    return out',
])

In [ ]:
!python -m py_compile src/benchmark_models.py

In [ ]:
!python -m py_compile src/benchmark_utils.py
!python -m py_compile src/benchmark_models.py
!python -m py_compile src/run_main_benchmarks.py

In [ ]:
!python src/run_coupled_nmf.py \
  --dataset ml_latest_small \
  --input_ratings data/processed/ml_latest_small_ratings.csv \
  --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
  --seed_list 42,123,2026,7,99 \
  --output_csv results/csv/ml_latest_small_main_comparison.csv \
  --log_file results/logs/ml_latest_small_run_coupled_nmf.log

In [ ]:
import pandas as pd

main_results = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")
run_level = pd.read_csv("results/csv/ml_latest_small_run_level_metrics.csv")

display(main_results)
display(run_level.head())

print("Run-level shape:", run_level.shape)

In [ ]:
import pandas as pd

main_results = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")
run_level = pd.read_csv("results/csv/ml_latest_small_run_level_metrics.csv")

display(main_results)
print("Run-level shape:", run_level.shape)
display(run_level.head(10))

In [ ]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("data/processed/ml_latest_small_ratings.csv")
desc = pd.read_csv("data/processed/ml_latest_small_item_descriptors.csv")

print("Ratings shape:", ratings.shape)
print("Descriptor shape:", desc.shape)

print("\nRatings columns:")
print(ratings.columns.tolist())

print("\nFirst 20 descriptor columns:")
print(desc.columns[:20].tolist())

numeric_cols = [c for c in desc.columns if c != "item_id" and pd.api.types.is_numeric_dtype(desc[c])]
non_numeric_cols = [c for c in desc.columns if c != "item_id" and not pd.api.types.is_numeric_dtype(desc[c])]

print("\nNumber of numeric descriptor columns:", len(numeric_cols))
print("Number of non-numeric descriptor columns:", len(non_numeric_cols))

if non_numeric_cols:
    print("Non-numeric descriptor columns:")
    print(non_numeric_cols[:20])

    print("\nAny missing values in descriptor matrix?")
    print(desc[numeric_cols].isna().sum().sum())

    display(desc.head())

## 4. Coupled NMF script construction, execution, and correction

Original cell index starts around `107`.

In [ ]:
from pathlib import Path

coupled_file = Path("src/run_coupled_nmf.py")
coupled_file.parent.mkdir(parents=True, exist_ok=True)

# Initialize a fresh file
coupled_file.write_text("", encoding="utf-8")
print("Reset and initialized:", coupled_file)

def append_lines(path, lines):
    with open(path, "a", encoding="utf-8") as f:
        f.write("\n".join(lines))
        f.write("\n\n")

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "import argparse",
    "import os",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "from benchmark_utils import (",
    "    build_id_maps,",
    "    ensure_parent,",
    "    load_split_data,",
    "    parse_seed_list,",
    "    parse_top_k,",
    "    prediction_metrics,",
    "    ranking_metrics_for_scores,",
    "    write_log,",
    "    )"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def build_descriptor_matrix(descriptors, item_to_idx):",
    "    feature_cols = [",
    "        c for c in descriptors.columns",
    "        if c != \"item_id\" and pd.api.types.is_numeric_dtype(descriptors[c])",
    "    ]",
    "",
    "    X = np.zeros((len(item_to_idx), len(feature_cols)), dtype=np.float32)",
    "    desc_indexed = descriptors.set_index(\"item_id\")",
    "",
    "    for item_id, idx in item_to_idx.items():",
    "        if item_id in desc_indexed.index:",
    "            values = desc_indexed.loc[item_id, feature_cols].to_numpy(dtype=np.float32)",
    "            X[idx, :] = values",
    "",
    "    col_max = np.maximum(X.max(axis=0), 1e-8)",
    "    X = X / col_max",
    "",
    "    return X, feature_cols"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def train_coupled_nmf(",
    "    train,",
    "    descriptors,",
    "    user_to_idx,",
    "    item_to_idx,",
    "    k=30,",
    "    epochs=30,",
    "    lr_rating=0.005,",
    "    lr_semantic=0.01,",
    "    alpha=0.1,",
    "    lambda_reg=0.03,",
    "    beta=0.001,",
    "    seed=42,",
    "): ",
    "    rng = np.random.default_rng(seed)",
    "",
    "    n_users = len(user_to_idx)",
    "    n_items = len(item_to_idx)",
    "",
    "    X, feature_cols = build_descriptor_matrix(descriptors, item_to_idx)",
    "    n_features = X.shape[1]",
    "",
    "    U = rng.random((n_users, k), dtype=np.float32) * 0.1",
    "    V = rng.random((n_items, k), dtype=np.float32) * 0.1",
    "    B = rng.random((n_features, k), dtype=np.float32) * 0.1",
    "",
    "    triples = [",
    "        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))",
    "        for r in train.itertuples(index=False)",
    "        if r.user_id in user_to_idx and r.item_id in item_to_idx",
    "    ]"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "    for _ in range(epochs):",
    "        rng.shuffle(triples)",
    "",
    "        for u, i, rating in triples:",
    "            pred = float(U[u] @ V[i])",
    "            err = rating - pred",
    "",
    "            old_u = U[u].copy()",
    "",
    "            U[u] += lr_rating * (err * V[i] - lambda_reg * U[u])",
    "            V[i] += lr_rating * (err * old_u - lambda_reg * V[i])",
    "",
    "            U[u] = np.maximum(U[u], 1e-8)",
    "            V[i] = np.maximum(V[i], 1e-8)",
    "",
    "        E = X - (V @ B.T)",
    "",
    "        grad_V_sem = (-2.0 * alpha / max(n_features, 1)) * (E @ B)",
    "        grad_B_sem = (-2.0 * alpha / max(n_items, 1)) * (E.T @ V)",
    "",
    "        grad_V = grad_V_sem + 2.0 * lambda_reg * V",
    "        grad_B = grad_B_sem + 2.0 * lambda_reg * B + beta",
    "",
    "        V -= lr_semantic * grad_V",
    "        B -= lr_semantic * grad_B",
    "",
    "        V = np.maximum(V, 1e-8)",
    "        B = np.maximum(B, 1e-8)",
    "",
    "    return U, V, B, feature_cols"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def run_coupled_nmf_one_seed(",
    "    train,",
    "    test,",
    "    descriptors,",
    "    all_items,",
    "    top_k_values,",
    "    relevance_threshold,",
    "    seed,",
    "): ",
    "    all_data = pd.concat(",
    "        [",
    "            train[['user_id', 'item_id', 'rating']],",
    "            test[['user_id', 'item_id', 'rating']],",
    "        ],",
    "        ignore_index=True,",
    "    )",
    "",
    "    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)",
    "",
    "    U, V, B, feature_cols = train_coupled_nmf(",
    "        train=train,",
    "        descriptors=descriptors,",
    "        user_to_idx=user_to_idx,",
    "        item_to_idx=item_to_idx,",
    "        seed=seed,",
    "    )",
    "",
    "    global_mean = train['rating'].mean()",
    "    preds = []",
    "",
    "    for row in test.itertuples(index=False):",
    "        u = user_to_idx.get(row.user_id)",
    "        i = item_to_idx.get(row.item_id)",
    "",
    "        if u is None or i is None:",
    "            preds.append(global_mean)",
    "        else:",
    "            preds.append(float(U[u] @ V[i]))",
    "",
    "    scores_by_user = {}",
    "    for user_id, u_idx in user_to_idx.items():",
    "        scores = V @ U[u_idx]",
    "        scores_by_user[user_id] = {idx_to_item[i]: float(scores[i]) for i in range(len(idx_to_item))}",
    "",
    "    out = {}",
    "    out['rmse'], out['mae'] = prediction_metrics(test, np.array(preds))",
    "    out.update(ranking_metrics_for_scores(",
    "        scores_by_user=scores_by_user, train=train, test=test,",
    "        all_items=all_items, top_k_values=top_k_values, relevance_threshold=relevance_threshold))",
    "",
    "    return out"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "    scores_by_user = {}",
    "",
    "    for user_id, u in user_to_idx.items():",
    "        scores = V @ U[u]",
    "        scores_by_user[user_id] = {",
    "            idx_to_item[i]: float(scores[i])",
    "            for i in range(len(idx_to_item))",
    "        }",
    "",
    "    out = {}",
    "    out[\"rmse\"], out[\"mae\"] = prediction_metrics(test, np.array(preds))",
    "",
    "    out.update(",
    "        ranking_metrics_for_scores(",
    "            scores_by_user=scores_by_user,",
    "            train=train,",
    "            test=test,",
    "            all_items=all_items,",
    "            top_k_values=top_k_values,",
    "            relevance_threshold=relevance_threshold,",
    "        )",
    "    )",
    "",
    "    model_objects = {",
    "        \"U\": U,",
    "        \"V\": V,",
    "        \"B\": B,",
    "        \"feature_cols\": feature_cols,",
    "        \"user_to_idx\": user_to_idx,",
    "        \"item_to_idx\": item_to_idx,",
    "        \"idx_to_user\": idx_to_user,",
    "        \"idx_to_item\": idx_to_item,",
    "        \"scores_by_user\": scores_by_user,",
    "    }",
    "",
    "    return out, model_objects",
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def aggregate_results(run_df, dataset_name):",
    "    metric_cols = [",
    "        \"rmse\",",
    "        \"mae\",",
    "        \"precision_at_5\",",
    "        \"precision_at_10\",",
    "        \"recall_at_5\",",
    "        \"recall_at_10\",",
    "        \"ndcg_at_5\",",
    "        \"ndcg_at_10\",",
    "    ]",
    "",
    "    rows = []",
    "",
    "    for model_name, group in run_df.groupby(\"model_name\"):",
    "        row = {",
    "            \"dataset_name\": dataset_name,",
    "            \"model_name\": model_name,",
    "        }",
    "",
    "        for col in metric_cols:",
    "            row[col] = group[col].mean()",
    "            row[f\"{col}_std\"] = group[col].std(ddof=0)",
    "",
    "        rows.append(row)",
    "",
    "    return pd.DataFrame(rows)",
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def export_factor_descriptor_weights(dataset, model_objects):",
    "    B = model_objects['B']",
    "    feature_cols = model_objects['feature_cols']",
    "",
    "    rows = []",
    "    for factor_idx in range(B.shape[1]):",
    "        for desc_idx, desc_name in enumerate(feature_cols):",
    "            rows.append({",
    "                'dataset_name': dataset,",
    "                'factor_id': factor_idx + 1,",
    "                'descriptor_name': desc_name,",
    "                'weight': float(B[desc_idx, factor_idx]),",
    "            })",
    "",
    "    out_path = f'results/csv/{dataset}_factor_descriptor_weights.csv'",
    "    ensure_parent(out_path)",
    "    pd.DataFrame(rows).to_csv(out_path, index=False)",
    "    return out_path"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def export_factor_keywords(dataset, model_objects, top_m=5):",
          "    B = model_objects[\"B\"]",
              "    feature_cols = model_objects[\"feature_cols\"]",
                  "",
                      "    rows = []",
                          "    for factor_idx in range(B.shape[1]):",
                              "        weights = B[:, factor_idx]",
                                  "        top_indices = np.argsort(weights)[::-1][:top_m]",
                                      "        descriptors = [feature_cols[i] for i in top_indices]",
                                          "",
                                              "        row = {",
                                                  "            \"dataset_name\": dataset,",
                                                      "            \"factor_id\": factor_idx + 1,",
                                                          "        }",
                                                              "",
                                                                  "        for j in range(top_m):",
                                                                      "            row[f\"descriptor_{j + 1}\"] = descriptors[j] if j < len(descriptors) else \"\"",
                                                                          "",
                                                                              "        row[\"interpretation\"] = \"\"",
                                                                                  "        rows.append(row)",
                                                                                      "",
                                                                                          "    out_path = f\"results/csv/{dataset}_factor_keywords.csv\"",
                                                                                              "    ensure_parent(out_path)",
                                                                                                  "    pd.DataFrame(rows).to_csv(out_path, index=False)",
                                                                                                      "    return out_path",
                                                                                                      ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def export_recommendation_traces(dataset, train, test, model_objects, top_k=10):",
          "    U = model_objects[\"U\"]",
              "    V = model_objects[\"V\"]",
                  "    idx_to_item = model_objects[\"idx_to_item\"]",
                      "    user_to_idx = model_objects[\"user_to_idx\"]",
                          "",
                              "    train_items_by_user = train.groupby(\"user_id\")[\"item_id\"].apply(set).to_dict()",
                                  "    all_items = list(idx_to_item.values())",
                                      "",
                                          "    rows = []",
                                              "    for user_id, u in list(user_to_idx.items())[:50]:",
                                                  "        seen = train_items_by_user.get(user_id, set())",
                                                      "        scores = V @ U[u]",
                                                          "",
                                                              "        candidate_pairs = []",
                                                                  "        for i_idx, item_id in idx_to_item.items():",
                                                                      "            if item_id not in seen:",
                                                                          "                candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))",
                                                                              "",
                                                                                  "        candidate_pairs.sort(key=lambda x: x[1], reverse=True)",
                                                                                      "",
                                                                                          "        for item_id, score, i_idx in candidate_pairs[:top_k]:",
                                                                                              "            contrib = U[u] * V[i_idx]",
                                                                                                  "            top_factors = np.argsort(contrib)[::-1][:3] + 1",
                                                                                                      "            rows.append({",
                                                                                                          "                \"dataset_name\": dataset,",
                                                                                                              "                \"user_id\": user_id,",
                                                                                                                  "                \"item_id\": item_id,",
                                                                                                                      "                \"score\": score,",
                                                                                                                          "                \"dominant_factors\": \",\".join([f\"f{x}\" for x in top_factors]),",
                                                                                                                              "            })",
                                                                                                                                  "",
                                                                                                                                      "    out_path = f\"results/csv/{dataset}_recommendation_traces.csv\"",
                                                                                                                                          "    ensure_parent(out_path)",
                                                                                                                                              "    pd.DataFrame(rows).to_csv(out_path, index=False)",
                                                                                                                                                  "    return out_path",
                                                                                                                                                  ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def export_coupled_nmf(",
    "    dataset,",
    "    input_ratings,",
    "    input_descriptors,",
    "    seed_list,",
    "    descriptor_source,",
    "    top_k,",
    "    relevance_threshold,",
    "    output_csv,",
    "    log_file=None,",
    "    overwrite=\"false\",",
    "):",
    "    del descriptor_source",
    "    del overwrite",
    "",
    "    ratings = pd.read_csv(input_ratings)",
    "    descriptors = pd.read_csv(input_descriptors)",
    "",
    "    split_path = f\"results/csv/{dataset}_split_registry.csv\"",
    "    if not os.path.exists(split_path):",
    "        raise FileNotFoundError(f\"Split registry not found: {split_path}\")",
    "",
    "    split_registry = pd.read_csv(split_path)",
    "    seeds = parse_seed_list(seed_list)",
    "    top_k_values = parse_top_k(top_k)",
    "    all_items = sorted(ratings[\"item_id\"].unique())",
    "",
    "    run_rows = []",
    "    first_model_objects = None",
    "    first_train = None",
    "    first_test = None",
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "    for seed in seeds:",
          "        write_log(log_file, f\"Running coupled_nmf seed={seed}\")",
              "",
                  "        train, val, test = load_split_data(",
                      "            ratings=ratings,",
                          "            split_registry=split_registry,",
                              "            seed=seed,",
                                  "        )",
                                      "",
                                          "        metrics, model_objects = run_coupled_nmf_one_seed(",
                                              "            train=train,",
                                                  "            test=test,",
                                                      "            descriptors=descriptors,",
                                                          "            all_items=all_items,",
                                                              "            top_k_values=top_k_values,",
                                                                  "            relevance_threshold=relevance_threshold,",
                                                                      "            seed=seed,",
                                                                          "        )",
                                                                              "",
                                                                                  "        row = {",
                                                                                      "            \"dataset_name\": dataset,",
                                                                                          "            \"model_name\": \"coupled_nmf\",",
                                                                                              "            \"seed\": seed,",
                                                                                                  "        }",
                                                                                                      "        row.update(metrics)",
                                                                                                          "        run_rows.append(row)",
                                                                                                              "",
                                                                                                                  "        if first_model_objects is None:",
                                                                                                                      "            first_model_objects = model_objects",
                                                                                                                          "            first_train = train",
                                                                                                                              "            first_test = test",
                                                                                                                              ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "    new_run_df = pd.DataFrame(run_rows)",
          "    run_level_path = f\"results/csv/{dataset}_run_level_metrics.csv\"",
              "",
                  "    if os.path.exists(run_level_path):",
                      "        old_run_df = pd.read_csv(run_level_path)",
                          "        old_run_df = old_run_df[old_run_df[\"model_name\"] != \"coupled_nmf\"]",
                              "        run_df = pd.concat([old_run_df, new_run_df], ignore_index=True)",
                                  "    else:",
                                      "        run_df = new_run_df",
                                          "",
                                              "    ensure_parent(run_level_path)",
                                                  "    run_df.to_csv(run_level_path, index=False)",
                                                      "",
                                                          "    summary_df = aggregate_results(run_df, dataset)",
                                                              "    ensure_parent(output_csv)",
                                                                  "    summary_df.to_csv(output_csv, index=False)",
                                                                      "",
                                                                          "    if first_model_objects is not None:",
                                                                              "        p1 = export_factor_descriptor_weights(dataset, first_model_objects)",
                                                                                  "        p2 = export_factor_keywords(dataset, first_model_objects)",
                                                                                      "        p3 = export_recommendation_traces(dataset, first_train, first_test, first_model_objects)",
                                                                                          "        write_log(log_file, f\"Saved: {p1}\")",
                                                                                              "        write_log(log_file, f\"Saved: {p2}\")",
                                                                                                  "        write_log(log_file, f\"Saved: {p3}\")",
                                                                                                      "",
                                                                                                          "    write_log(log_file, f\"Saved run-level metrics: {run_level_path}\")",
                                                                                                              "    write_log(log_file, f\"Saved main comparison: {output_csv}\")",
                                                                                                              ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def main():",
    "    parser = argparse.ArgumentParser()",
    "",
    "    parser.add_argument('--dataset', required=True)",
    "    parser.add_argument('--input_ratings', required=True)",
    "    parser.add_argument('--input_descriptors', required=True)",
    "    parser.add_argument('--seed_list', required=True)",
    "    parser.add_argument('--descriptor_source', default='tags_genres')",
    "    parser.add_argument('--top_k', default='5,10')",
    "    parser.add_argument('--relevance_threshold', type=float, default=4.0)",
    "    parser.add_argument('--output_csv', required=True)",
    "    parser.add_argument('--log_file', default=None)",
    "    parser.add_argument('--overwrite', default='false')",
    "",
    "    args = parser.parse_args()",
    "    export_coupled_nmf(**vars(args))",
    "",
    "if __name__ == '__main__':",
    "    main()"
])

In [ ]:
!python -m py_compile src/run_coupled_nmf.py

In [ ]:
!python src/run_coupled_nmf.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --seed_list 42,123,2026,7,99 \
          --descriptor_source tags_genres \
            --top_k 5,10 \
              --relevance_threshold 4.0 \
                --output_csv results/csv/ml_latest_small_main_comparison.csv \
                  --log_file results/logs/ml_latest_small_run_coupled_nmf.log \
                    --overwrite true

In [ ]:
from pathlib import Path

target = Path("src/run_coupled_nmf.py")
if target.exists():
    target.unlink()

def append_lines(path, lines):
    with open(path, "a", encoding="utf-8") as f:
        f.write("\n".join(lines))
        f.write("\n\n")

print("Global helper append_lines defined and script target reset.")

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "import argparse",
    "import os",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "from benchmark_utils import (",
    "    build_id_maps,",
    "    ensure_parent,",
    "    load_split_data,",
    "    parse_seed_list,",
    "    parse_top_k,",
    "    prediction_metrics,",
    "    ranking_metrics_for_scores,",
    "    write_log,",
    "    )",
    "",
    "",
    "def build_descriptor_matrix(descriptors, item_to_idx):",
    "    feature_cols = [",
    "        c for c in descriptors.columns",
    "        if c != 'item_id' and pd.api.types.is_numeric_dtype(descriptors[c])",
    "    ]",
    "",
    "    X = np.zeros((len(item_to_idx), len(feature_cols)), dtype=np.float32)",
    "    desc_indexed = descriptors.set_index('item_id')",
    "",
    "    for item_id, idx in item_to_idx.items():",
    "        if item_id in desc_indexed.index:",
    "            values = desc_indexed.loc[item_id, feature_cols].to_numpy(dtype=np.float32)",
    "            X[idx, :] = values",
    "",
    "    col_max = np.maximum(X.max(axis=0), 1e-8)",
    "    X = X / col_max",
    "",
    "    return X, feature_cols"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def train_coupled_nmf(",
          "    train,",
              "    descriptors,",
                  "    user_to_idx,",
                      "    item_to_idx,",
                          "    k=30,",
                              "    epochs=20,",
                                  "    lr_rating=0.005,",
                                      "    lr_semantic=0.01,",
                                          "    alpha=0.1,",
                                              "    lambda_reg=0.03,",
                                                  "    beta=0.001,",
                                                      "    seed=42,",
                                                          "):",
                                                              "    rng = np.random.default_rng(seed)",
                                                                  "",
                                                                      "    n_users = len(user_to_idx)",
                                                                          "    n_items = len(item_to_idx)",
                                                                              "",
                                                                                  "    X, feature_cols = build_descriptor_matrix(descriptors, item_to_idx)",
                                                                                      "    n_features = X.shape[1]",
                                                                                          "",
                                                                                              "    U = rng.random((n_users, k), dtype=np.float32) * 0.1",
                                                                                                  "    V = rng.random((n_items, k), dtype=np.float32) * 0.1",
                                                                                                      "    B = rng.random((n_features, k), dtype=np.float32) * 0.1",
                                                                                                          "",
                                                                                                              "    triples = [",
                                                                                                                  "        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))",
                                                                                                                      "        for r in train.itertuples(index=False)",
                                                                                                                          "        if r.user_id in user_to_idx and r.item_id in item_to_idx",
                                                                                                                              "    ]"
                                                                                                                              ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "    for _ in range(epochs):",
          "        rng.shuffle(triples)",
              "",
                  "        for u, i, rating in triples:",
                      "            pred = float(U[u] @ V[i])",
                          "            err = rating - pred",
                              "",
                                  "            old_u = U[u].copy()",
                                      "",
                                          "            U[u] += lr_rating * (err * V[i] - lambda_reg * U[u])",
                                              "            V[i] += lr_rating * (err * old_u - lambda_reg * V[i])",
                                                  "",
                                                      "            U[u] = np.maximum(U[u], 1e-8)",
                                                          "            V[i] = np.maximum(V[i], 1e-8)",
                                                              "",
                                                                  "        E = X - (V @ B.T)",
                                                                      "",
                                                                          "        grad_V_sem = (-2.0 * alpha / max(n_features, 1)) * (E @ B)",
                                                                              "        grad_B_sem = (-2.0 * alpha / max(n_items, 1)) * (E.T @ V)",
                                                                                  "",
                                                                                      "        grad_V = grad_V_sem + 2.0 * lambda_reg * V",
                                                                                          "        grad_B = grad_B_sem + 2.0 * lambda_reg * B + beta",
                                                                                              "",
                                                                                                  "        V -= lr_semantic * grad_V",
                                                                                                      "        B -= lr_semantic * grad_B",
                                                                                                          "",
                                                                                                              "        V = np.maximum(V, 1e-8)",
                                                                                                                  "        B = np.maximum(B, 1e-8)",
                                                                                                                      "",
                                                                                                                          "    return U, V, B, feature_cols",
                                                                                                                          ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def run_coupled_nmf_one_seed(",
          "    train,",
              "    test,",
                  "    descriptors,",
                      "    all_items,",
                          "    top_k_values,",
                              "    relevance_threshold,",
                                  "    seed,",
                                      "): ",
                                          "    all_data = pd.concat(",
                                              "        [",
                                                  "            train[['user_id', 'item_id', 'rating']],",
                                                      "            test[['user_id', 'item_id', 'rating']],",
                                                          "        ],",
                                                              "        ignore_index=True,",
                                                                  "    )",
                                                                      "",
                                                                          "    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)",
                                                                              "",
                                                                                  "    U, V, B, feature_cols = train_coupled_nmf(",
                                                                                      "        train=train,",
                                                                                          "        descriptors=descriptors,",
                                                                                              "        user_to_idx=user_to_idx,",
                                                                                                  "        item_to_idx=item_to_idx,",
                                                                                                      "        seed=seed,",
                                                                                                          "    )",
                                                                                                              "",
                                                                                                                  "    global_mean = train['rating'].mean()",
                                                                                                                      "    preds = []",
                                                                                                                          "",
                                                                                                                              "    for row in test.itertuples(index=False):",
                                                                                                                                  "        u = user_to_idx.get(row.user_id)",
                                                                                                                                      "        i = item_to_idx.get(row.item_id)",
                                                                                                                                          "        if u is None or i is None:",
                                                                                                                                              "            preds.append(global_mean)",
                                                                                                                                                  "        else:",
                                                                                                                                                      "            preds.append(float(U[u] @ V[i]))"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "    scores_by_user = {}",
    "    for user_id, u in user_to_idx.items():",
    "        scores = V @ U[u]",
    "        scores_by_user[user_id] = {",
    "            idx_to_item[i]: float(scores[i])",
    "            for i in range(len(idx_to_item))",
    "        }",
    "",
    "    out = {}",
    "    out['rmse'], out['mae'] = prediction_metrics(test, np.array(preds))",
    "",
    "    out.update(",
    "        ranking_metrics_for_scores(",
    "            scores_by_user=scores_by_user,",
    "            train=train,",
    "            test=test,",
    "            all_items=all_items,",
    "            top_k_values=top_k_values,",
    "            relevance_threshold=relevance_threshold,",
    "        )",
    "    )",
    "",
    "    model_objects = {",
    "        'U': U,",
    "        'V': V,",
    "        'B': B,",
    "        'feature_cols': feature_cols,",
    "        'user_to_idx': user_to_idx,",
    "        'item_to_idx': item_to_idx,",
    "        'idx_to_user': idx_to_user,",
    "        'idx_to_item': idx_to_item,",
    "    }",
    "",
    "    return out, model_objects",
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def aggregate_results(run_df, dataset_name):",
          "    metric_cols = [",
              "        'rmse', 'mae',",
                  "        'precision_at_5', 'precision_at_10',",
                      "        'recall_at_5', 'recall_at_10',",
                          "        'ndcg_at_5', 'ndcg_at_10',",
                              "    ]",
                                  "    rows = []",
                                      "    for model_name, group in run_df.groupby('model_name'):",
                                          "        row = {'dataset_name': dataset_name, 'model_name': model_name}",
                                              "        for col in metric_cols:",
                                                  "            row[col] = group[col].mean()",
                                                      "            row[f'{col}_std'] = group[col].std(ddof=0)",
                                                          "        rows.append(row)",
                                                              "    return pd.DataFrame(rows)",
                                                                  "",
                                                                      "",
                                                                          "def export_factor_descriptor_weights(dataset, model_objects):",
                                                                              "    B = model_objects['B']",
                                                                                  "    feature_cols = model_objects['feature_cols']",
                                                                                      "    rows = []",
                                                                                          "    for factor_idx in range(B.shape[1]):",
                                                                                              "        for desc_idx, desc_name in enumerate(feature_cols):",
                                                                                                  "            rows.append({",
                                                                                                      "                'dataset_name': dataset,",
                                                                                                          "                'factor_id': factor_idx + 1,",
                                                                                                              "                'descriptor_name': desc_name,",
                                                                                                                  "                'weight': float(B[desc_idx, factor_idx]),",
                                                                                                                      "            })",
                                                                                                                          "    out_path = f'results/csv/{dataset}_factor_descriptor_weights.csv'",
                                                                                                                              "    ensure_parent(out_path)",
                                                                                                                                  "    pd.DataFrame(rows).to_csv(out_path, index=False)",
                                                                                                                                      "    return out_path",
                                                                                                                                      ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def export_factor_keywords(dataset, model_objects, top_m=5):",
          "    B = model_objects['B']",
              "    feature_cols = model_objects['feature_cols']",
                  "    rows = []",
                      "    for factor_idx in range(B.shape[1]):",
                          "        weights = B[:, factor_idx]",
                              "        top_indices = np.argsort(weights)[::-1][:top_m]",
                                  "        descriptors = [feature_cols[i] for i in top_indices]",
                                      "        row = {",
                                          "            'dataset_name': dataset,",
                                              "            'factor_id': factor_idx + 1,",
                                                  "        }",
                                                      "        for j in range(top_m):",
                                                          "            row[f'descriptor_{j + 1}'] = descriptors[j] if j < len(descriptors) else ''",
                                                              "        row['interpretation'] = ''",
                                                                  "        rows.append(row)",
                                                                      "    out_path = f'results/csv/{dataset}_factor_keywords.csv'",
                                                                          "    ensure_parent(out_path)",
                                                                              "    pd.DataFrame(rows).to_csv(out_path, index=False)",
                                                                                  "    return out_path",
                                                                                  ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
      "def export_recommendation_traces(dataset, train, model_objects, top_k=10):",
          "    U = model_objects['U']",
              "    V = model_objects['V']",
                  "    idx_to_item = model_objects['idx_to_item']",
                      "    user_to_idx = model_objects['user_to_idx']",
                          "",
                              "    train_items_by_user = train.groupby('user_id')['item_id'].apply(set).to_dict()",
                                  "    rows = []",
                                      "",
                                          "    for user_id, u in list(user_to_idx.items())[:50]:",
                                              "        seen = train_items_by_user.get(user_id, set())",
                                                  "        scores = V @ U[u]",
                                                      "        candidate_pairs = []",
                                                          "        for i_idx, item_id in idx_to_item.items():",
                                                              "            if item_id not in seen:",
                                                                  "                candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))",
                                                                      "        candidate_pairs.sort(key=lambda x: x[1], reverse=True)",
                                                                          "",
                                                                              "        for item_id, score, i_idx in candidate_pairs[:top_k]:",
                                                                                  "            contrib = U[u] * V[i_idx]",
                                                                                      "            top_factors = np.argsort(contrib)[::-1][:3] + 1",
                                                                                          "            rows.append({",
                                                                                              "                'dataset_name': dataset,",
                                                                                                  "                'user_id': user_id,",
                                                                                                      "                'item_id': item_id,",
                                                                                                          "                'score': score,",
                                                                                                              "                'dominant_factors': ','.join([f'f{x}' for x in top_factors]),",
                                                                                                                  "            })",
                                                                                                                      "",
                                                                                                                          "    out_path = f'results/csv/{dataset}_recommendation_traces.csv'",
                                                                                                                              "    ensure_parent(out_path)",
                                                                                                                                  "    pd.DataFrame(rows).to_csv(out_path, index=False)",
                                                                                                                                      "    return out_path",
                                                                                                                                      ])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def export_coupled_nmf(",
    "    dataset,",
    "    input_ratings,",
    "    input_descriptors,",
    "    seed_list,",
    "    descriptor_source,",
    "    top_k,",
    "    relevance_threshold,",
    "    output_csv,",
    "    log_file=None,",
    "    overwrite='false',",
    "): ",
    "    del descriptor_source",
    "    del overwrite",
    "",
    "    ratings = pd.read_csv(input_ratings)",
    "    descriptors = pd.read_csv(input_descriptors)",
    "    split_path = f'results/csv/{dataset}_split_registry.csv'",
    "    if not os.path.exists(split_path):",
    "        raise FileNotFoundError(f'Split registry not found: {split_path}')",
    "    split_registry = pd.read_csv(split_path)",
    "    seeds = parse_seed_list(seed_list)",
    "    top_k_values = parse_top_k(top_k)",
    "    all_items = sorted(ratings['item_id'].unique())",
    "",
    "    run_rows = []",
    "    first_model_objects = None",
    "    first_train = None",
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "    for seed in seeds:",
    "        write_log(log_file, f'Running coupled_nmf seed={seed}')",
    "        train, val, test = load_split_data(",
    "            ratings=ratings,",
    "            split_registry=split_registry,",
    "            seed=seed,",
    "        )",
    "        metrics, model_objects = run_coupled_nmf_one_seed(",
    "            train=train,",
    "            test=test,",
    "            descriptors=descriptors,",
    "            all_items=all_items,",
    "            top_k_values=top_k_values,",
    "            relevance_threshold=relevance_threshold,",
    "            seed=seed,",
    "        )",
    "        row = {",
    "            'dataset_name': dataset,",
    "            'model_name': 'coupled_nmf',",
    "            'seed': seed,",
    "        }",
    "        row.update(metrics)",
    "        run_rows.append(row)",
    "        if first_model_objects is None:",
    "            first_model_objects = model_objects",
    "            first_train = train"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "    new_run_df = pd.DataFrame(run_rows)",
    "    run_level_path = f'results/csv/{dataset}_run_level_metrics.csv'",
    "",
    "    if os.path.exists(run_level_path):",
    "        old_run_df = pd.read_csv(run_level_path)",
    "        old_run_df = old_run_df[old_run_df['model_name'] != 'coupled_nmf']",
    "        run_df = pd.concat([old_run_df, new_run_df], ignore_index=True)",
    "    else:",
    "        run_df = new_run_df",
    "",
    "    ensure_parent(run_level_path)",
    "    run_df.to_csv(run_level_path, index=False)",
    "",
    "    summary_df = aggregate_results(run_df, dataset)",
    "    ensure_parent(output_csv)",
    "    summary_df.to_csv(output_csv, index=False)",
    "",
    "    if first_model_objects is not None:",
    "        p1 = export_factor_descriptor_weights(dataset, first_model_objects)",
    "        p2 = export_factor_keywords(dataset, first_model_objects)",
    "        p3 = export_recommendation_traces(dataset, first_train, first_model_objects)",
    "        write_log(log_file, f'Saved: {p1}')",
    "        write_log(log_file, f'Saved: {p2}')",
    "        write_log(log_file, f'Saved: {p3}')",
    "",
    "    write_log(log_file, f'Saved run-level metrics: {run_level_path}')",
    "    write_log(log_file, f'Saved main comparison: {output_csv}')"
])

In [ ]:
append_lines("src/run_coupled_nmf.py", [
    "def main():",
    "    parser = argparse.ArgumentParser()",
    "    parser.add_argument('--dataset', required=True)",
    "    parser.add_argument('--input_ratings', required=True)",
    "    parser.add_argument('--input_descriptors', required=True)",
    "    parser.add_argument('--seed_list', required=True)",
    "    parser.add_argument('--descriptor_source', default='tags_genres')",
    "    parser.add_argument('--top_k', default='5,10')",
    "    parser.add_argument('--relevance_threshold', type=float, default=4.0)",
    "    parser.add_argument('--output_csv', required=True)",
    "    parser.add_argument('--log_file', default=None)",
    "    parser.add_argument('--overwrite', default='false')",
    "    args = parser.parse_args()",
    "    export_coupled_nmf(**vars(args))",
    "",
    "if __name__ == '__main__':",
    "    main()"
])

In [ ]:
!wc -l src/run_coupled_nmf.py
!python -m py_compile src/run_coupled_nmf.py

In [ ]:
!pwd
!ls -lh src/benchmark_utils.py
!ls -lh src/benchmark_models.py
!ls -lh src/run_coupled_nmf.py

In [ ]:
!python -m py_compile src/benchmark_utils.py
!python -m py_compile src/benchmark_models.py
!python -m py_compile src/run_coupled_nmf.py

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target = PROJECT_ROOT / "src" / "run_coupled_nmf.py"

if target.exists():
    target.unlink()

    target.parent.mkdir(parents=True, exist_ok=True)

    print("Deleted and reset:", target)
    print("Exists now?", target.exists())

In [ ]:
import textwrap
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target = PROJECT_ROOT / "src" / "run_coupled_nmf.py"

def write_part(text, mode="a"):
    content = textwrap.dedent(text).strip() + "\n\n"
    if mode == "w":
        target.write_text(content, encoding="utf-8")
    else:
        with open(target, "a", encoding="utf-8") as f:
            f.write(content)
    print("Writing to:", target)

In [ ]:
write_part(
    """
    import argparse
    import os

    import numpy as np
    import pandas as pd

    from benchmark_utils import (
        build_id_maps,
        ensure_parent,
        load_split_data,
        parse_seed_list,
        parse_top_k,
        prediction_metrics,
        ranking_metrics_for_scores,
        write_log,
    )

    def build_descriptor_matrix(descriptors, item_to_idx):
        feature_cols = [
            c for c in descriptors.columns
            if c != 'item_id' and pd.api.types.is_numeric_dtype(descriptors[c])
        ]
        X = np.zeros((len(item_to_idx), len(feature_cols)), dtype=np.float32)
        desc_indexed = descriptors.set_index('item_id')

        for item_id, idx in item_to_idx.items():
            if item_id in desc_indexed.index:
                values = desc_indexed.loc[item_id, feature_cols].to_numpy(dtype=np.float32)
                X[idx, :] = values

        col_max = np.maximum(X.max(axis=0), 1e-8)
        X = X / col_max
        return X, feature_cols

    def train_coupled_nmf(
        train, descriptors, user_to_idx, item_to_idx, k=30, epochs=20,
        lr_rating=0.005, lr_semantic=0.01, alpha=0.1, lambda_reg=0.03, beta=0.001, seed=42
    ):
        rng = np.random.default_rng(seed)
        n_users = len(user_to_idx)
        n_items = len(item_to_idx)
        X, feature_cols = build_descriptor_matrix(descriptors, item_to_idx)
        n_features = X.shape[1]

        U = rng.random((n_users, k), dtype=np.float32) * 0.1
        V = rng.random((n_items, k), dtype=np.float32) * 0.1
        B = rng.random((n_features, k), dtype=np.float32) * 0.1

        triples = [
            (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
            for r in train.itertuples(index=False)
            if r.user_id in user_to_idx and r.item_id in item_to_idx
        ]

        for _ in range(epochs):
            rng.shuffle(triples)
            for u, i, rating in triples:
                pred = float(U[u] @ V[i])
                err = rating - pred
                old_u = U[u].copy()
                U[u] = np.maximum(U[u] + lr_rating * (err * V[i] - lambda_reg * U[u]), 1e-8)
                V[i] = np.maximum(V[i] + lr_rating * (err * old_u - lambda_reg * V[i]), 1e-8)

            E = X - (V @ B.T)
            grad_V_sem = (-2.0 * alpha / max(n_features, 1)) * (E @ B)
            grad_B_sem = (-2.0 * alpha / max(n_items, 1)) * (E.T @ V)
            V = np.maximum(V - lr_semantic * (grad_V_sem + 2.0 * lambda_reg * V), 1e-8)
            B = np.maximum(B - lr_semantic * (grad_B_sem + 2.0 * lambda_reg * B + beta), 1e-8)

        return U, V, B, feature_cols
    """,
    mode="w"
)

In [ ]:
write_part(
"""
def run_coupled_nmf_one_seed(
    train, test, descriptors, all_items, top_k_values, relevance_threshold, seed
):
    all_data = pd.concat(
        [
            train[["user_id", "item_id", "rating"]],
            test[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V, B, feature_cols = train_coupled_nmf(
        train=train,
        descriptors=descriptors,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        seed=seed,
    )

    global_mean = train["rating"].mean()
    preds = []

    for row in test.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)
        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    scores_by_user = {}
    for user_id, u_idx in user_to_idx.items():
        scores = V @ U[u_idx]
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i]) for i in range(len(idx_to_item))
        }

    out = {}
    out["rmse"], out["mae"] = prediction_metrics(test, np.array(preds))
    out.update(
        ranking_metrics_for_scores(
            scores_by_user=scores_by_user,
            train=train,
            test=test,
            all_items=all_items,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
        )
    )

    model_objects = {
        "U": U,
        "V": V,
        "B": B,
        "feature_cols": feature_cols,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user,
        "idx_to_item": idx_to_item,
    }

    return out, model_objects
"""
)

In [ ]:
write_part(
    """
    def aggregate_results(run_df, dataset_name):
        metric_cols = [
            "rmse", "mae",
            "precision_at_5", "precision_at_10",
            "recall_at_5", "recall_at_10",
            "ndcg_at_5", "ndcg_at_10",
        ]
        rows = []
        for model_name, group in run_df.groupby("model_name"):
            row = {"dataset_name": dataset_name, "model_name": model_name}
            for col in metric_cols:
                row[col] = group[col].mean()
                row[f"{col}_std"] = group[col].std(ddof=0)
            rows.append(row)
        return pd.DataFrame(rows)

    def export_factor_descriptor_weights(dataset, model_objects):
        B = model_objects["B"]
        feature_cols = model_objects["feature_cols"]
        rows = []
        for factor_idx in range(B.shape[1]):
            for desc_idx, desc_name in enumerate(feature_cols):
                rows.append({
                    "dataset_name": dataset,
                    "factor_id": factor_idx + 1,
                    "descriptor_name": desc_name,
                    "weight": float(B[desc_idx, factor_idx]),
                })
        out_path = f"results/csv/{dataset}_factor_descriptor_weights.csv"
        ensure_parent(out_path)
        pd.DataFrame(rows).to_csv(out_path, index=False)
        return out_path

    def export_factor_keywords(dataset, model_objects, top_m=5):
        B = model_objects["B"]
        feature_cols = model_objects["feature_cols"]
        rows = []
        for factor_idx in range(B.shape[1]):
            weights = B[:, factor_idx]
            top_indices = np.argsort(weights)[::-1][:top_m]
            descriptors = [feature_cols[i] for i in top_indices]
            row = {"dataset_name": dataset, "factor_id": factor_idx + 1}
            for j in range(top_m):
                row[f"descriptor_{j + 1}"] = descriptors[j] if j < len(descriptors) else ""
            row["interpretation"] = ""
            rows.append(row)
        out_path = f"results/csv/{dataset}_factor_keywords.csv"
        ensure_parent(out_path)
        pd.DataFrame(rows).to_csv(out_path, index=False)
        return out_path

    def export_recommendation_traces(dataset, train, model_objects, top_k=10):
        U = model_objects["U"]
        V = model_objects["V"]
        idx_to_item = model_objects["idx_to_item"]
        user_to_idx = model_objects["user_to_idx"]
        train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
        rows = []
        for user_id, u in list(user_to_idx.items())[:50]:
            seen = train_items_by_user.get(user_id, set())
            scores = V @ U[u]
            candidate_pairs = []
            for i_idx, item_id in idx_to_item.items():
                if item_id not in seen:
                    candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))
            candidate_pairs.sort(key=lambda x: x[1], reverse=True)
            for item_id, score, i_idx in candidate_pairs[:top_k]:
                contrib = U[u] * V[i_idx]
                top_factors = np.argsort(contrib)[::-1][:3] + 1
                rows.append({
                    "dataset_name": dataset,
                    "user_id": user_id,
                    "item_id": item_id,
                    "score": score,
                    "dominant_factors": ",".join([f"f{x}" for x in top_factors]),
                })
        out_path = f"results/csv/{dataset}_recommendation_traces.csv"
        ensure_parent(out_path)
        pd.DataFrame(rows).to_csv(out_path, index=False)
        return out_path
    """
)

In [ ]:
write_part(
    """
    def export_coupled_nmf(
        dataset,
        input_ratings,
        input_descriptors,
        seed_list,
        descriptor_source,
        top_k,
        relevance_threshold,
        output_csv,
        log_file=None,
        overwrite=\"false\",
    ):
        del descriptor_source
        del overwrite

        ratings = pd.read_csv(input_ratings)
        descriptors = pd.read_csv(input_descriptors)
        split_path = f\"results/csv/{dataset}_split_registry.csv\"

        if not os.path.exists(split_path):
            raise FileNotFoundError(f\"Split registry not found: {split_path}\")

        split_registry = pd.read_csv(split_path)
        seeds = parse_seed_list(seed_list)
        top_k_values = parse_top_k(top_k)
        all_items = sorted(ratings[\"item_id\"].unique())

        run_rows = []
        first_model_objects = None
        first_train = None

        for seed in seeds:
            write_log(log_file, f\"Running coupled_nmf seed={seed}\")
            train, val, test = load_split_data(ratings, split_registry, seed)
            metrics, model_objects = run_coupled_nmf_one_seed(
                train, test, descriptors, all_items, top_k_values, relevance_threshold, seed
            )
            row = {\"dataset_name\": dataset, \"model_name\": \"coupled_nmf\", \"seed\": seed}
            row.update(metrics)
            run_rows.append(row)

            if first_model_objects is None:
                first_model_objects = model_objects
                first_train = train

        new_run_df = pd.DataFrame(run_rows)
        run_level_path = f\"results/csv/{dataset}_run_level_metrics.csv\"

        if os.path.exists(run_level_path):
            old_run_df = pd.read_csv(run_level_path)
            old_run_df = old_run_df[old_run_df[\"model_name\"] != \"coupled_nmf\"]
            run_df = pd.concat([old_run_df, new_run_df], ignore_index=True)
        else:
            run_df = new_run_df

        ensure_parent(run_level_path)
        run_df.to_csv(run_level_path, index=False)

        summary_df = aggregate_results(run_df, dataset)
        ensure_parent(output_csv)
        summary_df.to_csv(output_csv, index=False)

        if first_model_objects is not None:
            p1 = export_factor_descriptor_weights(dataset, first_model_objects)
            p2 = export_factor_keywords(dataset, first_model_objects)
            p3 = export_recommendation_traces(dataset, first_train, first_model_objects)
            write_log(log_file, f\"Saved: {p1}\")
            write_log(log_file, f\"Saved: {p2}\")
            write_log(log_file, f\"Saved: {p3}\")

        write_log(log_file, f\"Saved run-level metrics: {run_level_path}\")
        write_log(log_file, f\"Saved main comparison: {output_csv}\")

    def main():
        parser = argparse.ArgumentParser()
        parser.add_argument(\"--dataset\", required=True)
        parser.add_argument(\"--input_ratings\", required=True)
        parser.add_argument(\"--input_descriptors\", required=True)
        parser.add_argument(\"--seed_list\", required=True)
        parser.add_argument(\"--descriptor_source\", default=\"tags_genres\")
        parser.add_argument(\"--top_k\", default=\"5,10\")
        parser.add_argument(\"--relevance_threshold\", type=float, default=4.0)
        parser.add_argument(\"--output_csv\", required=True)
        parser.add_argument(\"--log_file\", default=None)
        parser.add_argument(\"--overwrite\", default=\"false\")

        args = parser.parse_args()
        export_coupled_nmf(**vars(args))

    if __name__ == \"__main__\":
        main()
    """
)

In [ ]:
!cd /content/drive/MyDrive/xai_coupled_nmf_project && wc -l src/run_coupled_nmf.py
!cd /content/drive/MyDrive/xai_coupled_nmf_project && grep -n "def run_coupled_nmf_one_seed" src/run_coupled_nmf.py
!cd /content/drive/MyDrive/xai_coupled_nmf_project && python -m py_compile src/run_coupled_nmf.py

In [ ]:
%cd /content/drive/MyDrive/xai_coupled_nmf_project

!python src/run_coupled_nmf.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --seed_list 42,123,2026,7,99 \
          --descriptor_source tags_genres \
            --top_k 5,10 \
              --relevance_threshold 4.0 \
                --output_csv results/csv/ml_latest_small_main_comparison.csv \
                  --log_file results/logs/ml_latest_small_run_coupled_nmf.log \
                    --overwrite true

In [ ]:
import pandas as pd

main_results = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")
run_level = pd.read_csv("results/csv/ml_latest_small_run_level_metrics.csv")

display(main_results)
print("Run-level shape:", run_level.shape)
display(run_level.tail(10))

In [ ]:
!ls -lh results/csv/ml_latest_small_factor_descriptor_weights.csv
!ls -lh results/csv/ml_latest_small_factor_keywords.csv
!ls -lh results/csv/ml_latest_small_recommendation_traces.csv

In [ ]:
import pandas as pd

factor_keywords = pd.read_csv("results/csv/ml_latest_small_factor_keywords.csv")
factor_weights = pd.read_csv("results/csv/ml_latest_small_factor_descriptor_weights.csv")
traces = pd.read_csv("results/csv/ml_latest_small_recommendation_traces.csv")

display(factor_keywords.head(15))
display(traces.head(10))

print("Factor keyword shape:", factor_keywords.shape)
print("Factor weight shape:", factor_weights.shape)
print("Recommendation trace shape:", traces.shape)

In [ ]:
import shutil
from pathlib import Path

Path("results/csv/archive").mkdir(parents=True, exist_ok=True)

shutil.copy(
    "results/csv/ml_latest_small_main_comparison.csv",
    "results/csv/archive/ml_latest_small_main_comparison_v1_initial_coupled.csv"
)

shutil.copy(
    "results/csv/ml_latest_small_run_level_metrics.csv",
    "results/csv/archive/ml_latest_small_run_level_metrics_v1_initial_coupled.csv"
)

print("Initial coupled NMF results archived.")

In [ ]:
import shutil
from pathlib import Path

# Create archive directory if it doesn't exist
Path("results/csv/archive").mkdir(parents=True, exist_ok=True)

# Define files to archive
files_to_archive = [
    (
        "results/csv/ml_latest_small_main_comparison.csv",
        "results/csv/archive/ml_latest_small_main_comparison_before_tuning.csv"
    ),
    (
        "results/csv/ml_latest_small_run_level_metrics.csv",
        "results/csv/archive/ml_latest_small_run_level_metrics_before_tuning.csv"
    )
]

for src, dst in files_to_archive:
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f"Archived: {dst}")
    else:
        print(f"Warning: Source file not found: {src}")

## 5. Coupled NMF tuning and selected configuration

Original cell index starts around `156`.

In [ ]:
%%writefile src/run_coupled_nmf_tuning.py
import argparse
import os
from itertools import product
import numpy as np
import pandas as pd

from benchmark_utils import (
    build_id_maps,
    ensure_parent,
    load_split_data,
    parse_seed_list,
    parse_top_k,
    prediction_metrics,
    ranking_metrics_for_scores,
    write_log,
)

from run_coupled_nmf import (
    train_coupled_nmf,
    aggregate_results,
    export_factor_descriptor_weights,
    export_factor_keywords,
    export_recommendation_traces,
)

def parse_int_list(text):
    return [int(x.strip()) for x in text.split(",") if x.strip()]

def parse_float_list(text):
    return [float(x.strip()) for x in text.split(",") if x.strip()]

def metric_is_lower_better(metric_name):
    return metric_name in {"rmse", "mae"}

def fit_and_evaluate_coupled(
    train, eval_df, descriptors, all_items, top_k_values, relevance_threshold,
    seed, k, alpha, beta, lambda_reg, epochs
):
    all_data = pd.concat([
        train[["user_id", "item_id", "rating"]],
        eval_df[["user_id", "item_id", "rating"]]
    ], ignore_index=True)

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V, B, feature_cols = train_coupled_nmf(
        train=train, descriptors=descriptors, user_to_idx=user_to_idx,
        item_to_idx=item_to_idx, k=k, epochs=epochs, alpha=alpha,
        beta=beta, lambda_reg=lambda_reg, seed=seed
    )

    global_mean = train["rating"].mean()
    preds = []
    for row in eval_df.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)
        preds.append(float(U[u] @ V[i]) if u is not None and i is not None else global_mean)

    scores_by_user = {}
    for user_id, u in user_to_idx.items():
        scores = V @ U[u]
        scores_by_user[user_id] = {idx_to_item[idx]: float(scores[idx]) for idx in range(len(idx_to_item))}

    metrics = {}
    metrics["rmse"], metrics["mae"] = prediction_metrics(eval_df, np.array(preds))
    metrics.update(ranking_metrics_for_scores(
        scores_by_user=scores_by_user, train=train, test=eval_df,
        all_items=all_items, top_k_values=top_k_values, relevance_threshold=relevance_threshold
    ))

    model_objects = {
        "U": U, "V": V, "B": B, "feature_cols": feature_cols,
        "user_to_idx": user_to_idx, "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user, "idx_to_item": idx_to_item
    }
    return metrics, model_objects

def run_validation_grid(
    dataset, ratings, descriptors, split_registry, tune_seed, top_k_values,
    relevance_threshold, k_values, alpha_values, beta_values, lambda_values,
    tune_epochs, selection_metric, log_file
):
    train, val, test = load_split_data(ratings=ratings, split_registry=split_registry, seed=tune_seed)
    all_items = sorted(ratings["item_id"].unique())
    rows = []
    grid = list(product(k_values, alpha_values, beta_values, lambda_values))

    for idx, (k, alpha, beta, lambda_reg) in enumerate(grid, start=1):
        write_log(log_file, f"Tuning {idx}/{len(grid)}: k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}")
        metrics, _ = fit_and_evaluate_coupled(
            train, val, descriptors, all_items, top_k_values, relevance_threshold,
            tune_seed, k, alpha, beta, lambda_reg, tune_epochs
        )
        row = {"dataset_name": dataset, "seed": tune_seed, "k": k, "alpha": alpha, "beta": beta, "lambda_reg": lambda_reg}
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)

def select_best_config(grid_df, selection_metric):
    candidate_df = grid_df[grid_df["alpha"] > 0].copy()
    ascending = metric_is_lower_better(selection_metric)
    candidate_df = candidate_df.sort_values(by=[selection_metric, "rmse"], ascending=[ascending, True])
    return candidate_df.iloc[0].to_dict()

def export_tuned_coupled_nmf(
    dataset, input_ratings, input_descriptors, seed_list, descriptor_source,
    top_k, relevance_threshold, output_csv, k_values, alpha_values,
    beta_values, lambda_values, tune_seed, tune_epochs, final_epochs,
    selection_metric, log_file=None, overwrite="false"
):
    ratings = pd.read_csv(input_ratings)
    descriptors = pd.read_csv(input_descriptors)
    split_registry = pd.read_csv(f"results/csv/{dataset}_split_registry.csv")

    seeds = parse_seed_list(seed_list)
    top_k_vals = parse_top_k(top_k)
    ks, alphas, betas, lambdas = parse_int_list(k_values), parse_float_list(alpha_values), parse_float_list(beta_values), parse_float_list(lambda_values)

    grid_df = run_validation_grid(dataset, ratings, descriptors, split_registry, tune_seed, top_k_vals, relevance_threshold, ks, alphas, betas, lambdas, tune_epochs, selection_metric, log_file)
    grid_df.to_csv(f"results/csv/{dataset}_coupled_nmf_tuning_grid.csv", index=False)

    best = select_best_config(grid_df, selection_metric)
    write_log(log_file, f"Selected best config: {best}")

    run_rows = []
    first_objs, first_train = None, None
    for seed in seeds:
        write_log(log_file, f"Final evaluation seed={seed}")
        train, _, test = load_split_data(ratings, split_registry, seed)
        metrics, objs = fit_and_evaluate_coupled(train, test, descriptors, sorted(ratings.item_id.unique()), top_k_vals, relevance_threshold, seed, int(best['k']), best['alpha'], best['beta'], best['lambda_reg'], final_epochs)
        row = {"dataset_name": dataset, "model_name": "coupled_nmf", "seed": seed}
        row.update(metrics)
        run_rows.append(row)
        if first_objs is None: first_objs, first_train = objs, train

    run_df = pd.DataFrame(run_rows)
    run_level_path = f"results/csv/{dataset}_run_level_metrics.csv"
    if os.path.exists(run_level_path):
        old = pd.read_csv(run_level_path)
        run_df = pd.concat([old[old.model_name != 'coupled_nmf'], run_df], ignore_index=True)
    run_df.to_csv(run_level_path, index=False)
    aggregate_results(run_df, dataset).to_csv(output_csv, index=False)

    export_factor_descriptor_weights(dataset, first_objs)
    export_factor_keywords(dataset, first_objs)
    export_recommendation_traces(dataset, first_train, first_objs)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--input_ratings", required=True)
    parser.add_argument("--input_descriptors", required=True)
    parser.add_argument("--seed_list", required=True)
    parser.add_argument("--output_csv", required=True)
    parser.add_argument("--k_values", default="20,30")
    parser.add_argument("--alpha_values", default="0.01,0.1")
    parser.add_argument("--beta_values", default="0,0.001")
    parser.add_argument("--lambda_values", default="0.03")
    parser.add_argument("--tune_seed", type=int, default=42)
    parser.add_argument("--tune_epochs", type=int, default=10)
    parser.add_argument("--final_epochs", type=int, default=20)
    parser.add_argument("--selection_metric", default="ndcg_at_10")
    parser.add_argument("--top_k", default="5,10")
    parser.add_argument("--relevance_threshold", type=float, default=4.0)
    parser.add_argument("--log_file", default=None)
    parser.add_argument("--descriptor_source", default="tags_genres")
    parser.add_argument("--overwrite", default="false")
    args = parser.parse_args()
    export_tuned_coupled_nmf(**vars(args))

if __name__ == "__main__":
    main()

In [ ]:
write_part(
  """
  import argparse
  import os
  from itertools import product

  import numpy as np
  import pandas as pd

  from benchmark_utils import (
      build_id_maps,
      ensure_parent,
      load_split_data,
      parse_seed_list,
      parse_top_k,
      prediction_metrics,
      ranking_metrics_for_scores,
      write_log,
  )

  from run_coupled_nmf import (
      train_coupled_nmf,
      aggregate_results,
      export_factor_descriptor_weights,
      export_factor_keywords,
      export_recommendation_traces,
  )

  def parse_int_list(text):
      return [int(x.strip()) for x in text.split(",") if x.strip()]

  def parse_float_list(text):
      return [float(x.strip()) for x in text.split(",") if x.strip()]

  def metric_is_lower_better(metric_name):
      return metric_name in {"rmse", "mae"}
  """,
  mode="w"
)

In [ ]:
write_part(
    """
    def fit_and_evaluate_coupled(
        train,
        eval_df,
        descriptors,
        all_items,
        top_k_values,
        relevance_threshold,
        seed,
        k,
        alpha,
        beta,
        lambda_reg,
        epochs,
    ):
        all_data = pd.concat(
            [
                train[["user_id", "item_id", "rating"]],
                eval_df[["user_id", "item_id", "rating"]],
            ],
            ignore_index=True,
        )

        user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

        U, V, B, feature_cols = train_coupled_nmf(
            train=train,
            descriptors=descriptors,
            user_to_idx=user_to_idx,
            item_to_idx=item_to_idx,
            k=k,
            epochs=epochs,
            alpha=alpha,
            beta=beta,
            lambda_reg=lambda_reg,
            seed=seed,
        )

        global_mean = train["rating"].mean()
        preds = []

        for row in eval_df.itertuples(index=False):
            u = user_to_idx.get(row.user_id)
            i = item_to_idx.get(row.item_id)

            if u is None or i is None:
                preds.append(global_mean)
            else:
                preds.append(float(U[u] @ V[i]))

        return preds
    """
)

In [ ]:
write_part(
    """
        scores_by_user = {}
        for user_id, u in user_to_idx.items():
            scores = V @ U[u]
            scores_by_user[user_id] = {
                idx_to_item[i]: float(scores[i])
                for i in range(len(idx_to_item))
            }

        metrics = {}
        metrics[\"rmse\"], metrics[\"mae\"] = prediction_metrics(eval_df, np.array(preds))
        metrics.update(
            ranking_metrics_for_scores(
                scores_by_user=scores_by_user,
                train=train,
                test=eval_df,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
            )
        )

        model_objects = {
            \"U\": U,
            \"V\": V,
            \"B\": B,
            \"feature_cols\": feature_cols,
            \"user_to_idx\": user_to_idx,
            \"item_to_idx\": item_to_idx,
            \"idx_to_user\": idx_to_user,
            \"idx_to_item\": idx_to_item,
        }

        return metrics, model_objects
    """
)

In [ ]:
write_part(
    """
    def run_validation_grid(
        dataset,
        ratings,
        descriptors,
        split_registry,
        tune_seed,
        top_k_values,
        relevance_threshold,
        k_values,
        alpha_values,
        beta_values,
        lambda_values,
        tune_epochs,
        selection_metric,
        log_file,
    ):
        train, val, test = load_split_data(
            ratings=ratings,
            split_registry=split_registry,
            seed=tune_seed,
        )

        all_items = sorted(ratings[\"item_id\"].unique())
        rows = []

        grid = list(product(k_values, alpha_values, beta_values, lambda_values))
        write_log(log_file, f\"Total tuning configurations: {len(grid)}\")

        for idx, (k, alpha, beta, lambda_reg) in enumerate(grid, start=1):
            write_log(
                log_file,
                f\"Tuning {idx}/{len(grid)}: k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}\",
            )

            metrics, _ = fit_and_evaluate_coupled(
                train=train,
                eval_df=val,
                descriptors=descriptors,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=tune_seed,
                k=k,
                alpha=alpha,
                beta=beta,
                lambda_reg=lambda_reg,
                epochs=tune_epochs,
            )

            row = {
                \"dataset_name\": dataset,
                \"seed\": tune_seed,
                \"k\": k,
                \"alpha\": alpha,
                \"beta\": beta,
                \"lambda_reg\": lambda_reg,
                \"epochs\": tune_epochs,
            }
            row.update(metrics)
            rows.append(row)

        grid_df = pd.DataFrame(rows)
        return grid_df
    """
)

In [ ]:
write_part(
    """
    def select_best_config(grid_df, selection_metric):
        if selection_metric not in grid_df.columns:
            raise ValueError(f\"Selection metric not found in tuning grid: {selection_metric}\")

        # Select only genuinely coupled configurations for the proposed model.
        candidate_df = grid_df[grid_df[\"alpha\"] > 0].copy()

        if candidate_df.empty:
            raise ValueError(\"No positive-alpha configurations available for coupled NMF selection.\")

        lower_better = metric_is_lower_better(selection_metric)

        if lower_better:
            candidate_df = candidate_df.sort_values(
                by=[selection_metric, \"rmse\"],
                ascending=[True, True],
            )
        else:
            candidate_df = candidate_df.sort_values(
                by=[selection_metric, \"rmse\"],
                ascending=[False, True],
            )

        best = candidate_df.iloc[0].to_dict()
        return best
    """
)

In [ ]:
write_part(
    """
    def run_final_selected_config(
        dataset,
        ratings,
        descriptors,
        split_registry,
        seeds,
        top_k_values,
        relevance_threshold,
        best_config,
        final_epochs,
        log_file,
    ):
        all_items = sorted(ratings["item_id"].unique())
        run_rows = []
        first_model_objects = None
        first_train = None

        k = int(best_config["k"])
        alpha = float(best_config["alpha"])
        beta = float(best_config["beta"])
        lambda_reg = float(best_config["lambda_reg"])

        for seed in seeds:
            write_log(
                log_file,
                f"Final coupled_nmf seed={seed}, k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}",
            )

            train, val, test = load_split_data(
                ratings=ratings, split_registry=split_registry, seed=seed
            )

            metrics, model_objects = fit_and_evaluate_coupled(
                train=train,
                eval_df=test,
                descriptors=descriptors,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=seed,
                k=k,
                alpha=alpha,
                beta=beta,
                lambda_reg=lambda_reg,
                epochs=final_epochs,
            )

            row = {
                "dataset_name": dataset,
                "model_name": "coupled_nmf",
                "seed": seed,
                "k": k,
                "alpha": alpha,
                "beta": beta,
                "lambda_reg": lambda_reg,
                "epochs": final_epochs,
            }
            row.update(metrics)
            run_rows.append(row)

            if first_model_objects is None:
                first_model_objects = model_objects
                first_train = train

        return pd.DataFrame(run_rows), first_model_objects, first_train
    """
)

In [ ]:
write_part(
    """
    def export_tuned_coupled_nmf(
        dataset,
        input_ratings,
        input_descriptors,
        seed_list,
        descriptor_source,
        top_k,
        relevance_threshold,
        output_csv,
        k_values,
        alpha_values,
        beta_values,
        lambda_values,
        tune_seed,
        tune_epochs,
        final_epochs,
        selection_metric,
        log_file=None,
        overwrite=\"false\",
    ):
        del descriptor_source
        del overwrite

        ratings = pd.read_csv(input_ratings)
        descriptors = pd.read_csv(input_descriptors)

        split_path = f\"results/csv/{dataset}_split_registry.csv\"
        if not os.path.exists(split_path):
            raise FileNotFoundError(f\"Split registry not found: {split_path}\")

        split_registry = pd.read_csv(split_path)

        seeds = parse_seed_list(seed_list)
        top_k_vals = parse_top_k(top_k)

        ks = parse_int_list(k_values)
        alphas = parse_float_list(alpha_values)
        betas = parse_float_list(beta_values)
        lambdas = parse_float_list(lambda_values)
    """
)

In [ ]:
write_part(
    """
    grid_df = run_validation_grid(
        dataset=dataset,
        ratings=ratings,
        descriptors=descriptors,
        split_registry=split_registry,
        tune_seed=tune_seed,
        top_k_values=top_k_vals,
        relevance_threshold=relevance_threshold,
        k_values=ks,
        alpha_values=alphas,
        beta_values=betas,
        lambda_values=lambdas,
        tune_epochs=tune_epochs,
        selection_metric=selection_metric,
        log_file=log_file,
    )

    tuning_grid_path = f\"results/csv/{dataset}_coupled_nmf_tuning_grid.csv\"
    ensure_parent(tuning_grid_path)
    grid_df.to_csv(tuning_grid_path, index=False)

    best_config = select_best_config(grid_df, selection_metric)

    selected_config_path = f\"results/csv/{dataset}_coupled_nmf_selected_config.csv\"
    pd.DataFrame([best_config]).to_csv(selected_config_path, index=False)

    write_log(log_file, f\"Saved tuning grid: {tuning_grid_path}\")
    write_log(log_file, f\"Saved selected config: {selected_config_path}\")
    write_log(log_file, f\"Selected config: {best_config}\")
    """
)

In [ ]:
write_part(
    """
    new_coupled_df, first_model_objects, first_train = run_final_selected_config(
        dataset=dataset,
        ratings=ratings,
        descriptors=descriptors,
        split_registry=split_registry,
        seeds=seeds,
        top_k_values=top_k_vals,
        relevance_threshold=relevance_threshold,
        best_config=best_config,
        final_epochs=final_epochs,
        log_file=log_file,
    )

    run_level_path = f"results/csv/{dataset}_run_level_metrics.csv"

    if os.path.exists(run_level_path):
        old_run_df = pd.read_csv(run_level_path)
        old_run_df = old_run_df[old_run_df["model_name"] != "coupled_nmf"]
        run_df = pd.concat([old_run_df, new_coupled_df], ignore_index=True)
    else:
        run_df = new_coupled_df

    ensure_parent(run_level_path)
    run_df.to_csv(run_level_path, index=False)

    summary_df = aggregate_results(run_df, dataset)
    ensure_parent(output_csv)
    summary_df.to_csv(output_csv, index=False)

    if first_model_objects is not None:
        p1 = export_factor_descriptor_weights(dataset, first_model_objects)
        p2 = export_factor_keywords(dataset, first_model_objects)
        p3 = export_recommendation_traces(dataset, first_train, first_model_objects)
        write_log(log_file, f"Saved: {p1}")
        write_log(log_file, f"Saved: {p2}")
        write_log(log_file, f"Saved: {p3}")

    write_log(log_file, f"Saved run-level metrics: {run_level_path}")
    write_log(log_file, f"Saved main comparison: {output_csv}")
    """
)

In [ ]:
write_part(
    """
    def main():
        parser = argparse.ArgumentParser()
        parser.add_argument("--dataset", required=True)
        parser.add_argument("--input_ratings", required=True)
        parser.add_argument("--input_descriptors", required=True)
        parser.add_argument("--seed_list", required=True)
        parser.add_argument("--descriptor_source", default="tags_genres")
        parser.add_argument("--top_k", default="5,10")
        parser.add_argument("--relevance_threshold", type=float, default=4.0)
        parser.add_argument("--output_csv", required=True)

        parser.add_argument("--k_values", default="20,30,40")
        parser.add_argument("--alpha_values", default="0.01,0.05,0.1")
        parser.add_argument("--beta_values", default="0,0.0001,0.001")
        parser.add_argument("--lambda_values", default="0.01,0.03")

        parser.add_argument("--tune_seed", type=int, default=42)
        parser.add_argument("--tune_epochs", type=int, default=20)
        parser.add_argument("--final_epochs", type=int, default=30)
        parser.add_argument("--selection_metric", default="ndcg_at_10")

        parser.add_argument("--log_file", default=None)
        parser.add_argument("--overwrite", default="false")

        args = parser.parse_args()
        export_tuned_coupled_nmf(**vars(args))

    if __name__ == "__main__":
        main()
    """
)

In [ ]:
!cd /content/drive/MyDrive/xai_coupled_nmf_project && wc -l src/run_coupled_nmf_tuning.py
!cd /content/drive/MyDrive/xai_coupled_nmf_project && python -m py_compile src/run_coupled_nmf_tuning.py

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target = PROJECT_ROOT / "src" / "run_coupled_nmf_tuning.py"

if target.exists():
    target.unlink()

    target.parent.mkdir(parents=True, exist_ok=True)

    print("Deleted:", target)

In [ ]:
import textwrap
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target = PROJECT_ROOT / "src" / "run_coupled_nmf_tuning.py"

def write_part(text, mode="a"):
    content = textwrap.dedent(text).strip() + "\n\n"
    if mode == "w":
        target.write_text(content, encoding="utf-8")
    else:
        with open(target, "a", encoding="utf-8") as f:
            f.write(content)

    print(f"Writing to: {target}")

In [ ]:
write_part(
    """
    def fit_and_evaluate_coupled(
        train, eval_df, descriptors, all_items, top_k_values, relevance_threshold,
        seed, k, alpha, beta, lambda_reg, epochs
    ):
        all_data = pd.concat([
            train[["user_id", "item_id", "rating"]],
            eval_df[["user_id", "item_id", "rating"]]
        ], ignore_index=True)

        user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

        U, V, B, feature_cols = train_coupled_nmf(
            train=train, descriptors=descriptors, user_to_idx=user_to_idx,
            item_to_idx=item_to_idx, k=k, epochs=epochs, alpha=alpha,
            beta=beta, lambda_reg=lambda_reg, seed=seed
        )

        global_mean = train["rating"].mean()
        preds = []
        for row in eval_df.itertuples(index=False):
            u = user_to_idx.get(row.user_id)
            i = item_to_idx.get(row.item_id)
            if u is not None and i is not None:
                preds.append(float(U[u] @ V[i]))
            else:
                preds.append(global_mean)

        scores_by_user = {}
        for user_id, u_idx in user_to_idx.items():
            scores = V @ U[u_idx]
            scores_by_user[user_id] = {idx_to_item[i]: float(scores[i]) for i in range(len(idx_to_item))}

        metrics = {}
        metrics["rmse"], metrics["mae"] = prediction_metrics(eval_df, np.array(preds))
        metrics.update(ranking_metrics_for_scores(
            scores_by_user=scores_by_user, train=train, test=eval_df,
            all_items=all_items, top_k_values=top_k_values, relevance_threshold=relevance_threshold
        ))

        model_objects = {
            "U": U, "V": V, "B": B, "feature_cols": feature_cols,
            "user_to_idx": user_to_idx, "item_to_idx": item_to_idx,
            "idx_to_user": idx_to_user, "idx_to_item": idx_to_item
        }

        return metrics, model_objects
    """
)

In [ ]:
write_part(
    """
    def run_validation_grid(
        dataset,
        ratings,
        descriptors,
        split_registry,
        tune_seed,
        top_k_values,
        relevance_threshold,
        k_values,
        alpha_values,
        beta_values,
        lambda_values,
        tune_epochs,
        selection_metric,
        log_file,
    ):
        train, val, test = load_split_data(
            ratings=ratings,
            split_registry=split_registry,
            seed=tune_seed,
        )

        all_items = sorted(ratings["item_id"].unique())
        rows = []

        grid = list(product(k_values, alpha_values, beta_values, lambda_values))
        write_log(log_file, f"Total tuning configurations: {len(grid)}")

        for idx, (k, alpha, beta, lambda_reg) in enumerate(grid, start=1):
            write_log(
                log_file,
                f"Tuning {idx}/{len(grid)}: k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}",
            )

            metrics, _ = fit_and_evaluate_coupled(
                train=train,
                eval_df=val,
                descriptors=descriptors,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=tune_seed,
                k=k,
                alpha=alpha,
                beta=beta,
                lambda_reg=lambda_reg,
                epochs=tune_epochs,
            )

            row = {
                "dataset_name": dataset,
                "seed": tune_seed,
                "k": k,
                "alpha": alpha,
                "beta": beta,
                "lambda_reg": lambda_reg,
                "epochs": tune_epochs,
            }
            row.update(metrics)
            rows.append(row)

        return pd.DataFrame(rows)
    """
)

In [ ]:
write_part(
    """
    def select_best_config(grid_df, selection_metric):
        if selection_metric not in grid_df.columns:
            raise ValueError(f\"Selection metric not found in tuning grid: {selection_metric}\")

        candidate_df = grid_df[grid_df[\"alpha\"] > 0].copy()

        if candidate_df.empty:
            raise ValueError(\"No positive-alpha configurations available for coupled NMF selection.\")

        lower_better = metric_is_lower_better(selection_metric)

        if lower_better:
            candidate_df = candidate_df.sort_values(
                by=[selection_metric, \"rmse\"],
                ascending=[True, True],
            )
        else:
            candidate_df = candidate_df.sort_values(
                by=[selection_metric, \"rmse\"],
                ascending=[False, True],
            )

        return candidate_df.iloc[0].to_dict()
    """
)

In [ ]:
write_part(
    """
    def run_final_selected_config(
        dataset,
        ratings,
        descriptors,
        split_registry,
        seeds,
        top_k_values,
        relevance_threshold,
        best_config,
        final_epochs,
        log_file,
    ):
        all_items = sorted(ratings[\"item_id\"].unique())
        run_rows = []
        first_model_objects = None
        first_train = None

        k = int(best_config[\"k\"])
        alpha = float(best_config[\"alpha\"])
        beta = float(best_config[\"beta\"])
        lambda_reg = float(best_config[\"lambda_reg\"])

        for seed in seeds:
            write_log(
                log_file,
                f\"Final coupled_nmf seed={seed}, k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}\",
            )

            train, val, test = load_split_data(
                ratings=ratings, split_registry=split_registry, seed=seed
            )

            metrics, model_objects = fit_and_evaluate_coupled(
                train=train,
                eval_df=test,
                descriptors=descriptors,
                all_items=all_items,
                top_k_values=top_k_values,
                relevance_threshold=relevance_threshold,
                seed=seed,
                k=k,
                alpha=alpha,
                beta=beta,
                lambda_reg=lambda_reg,
                epochs=final_epochs,
            )

            row = {
                \"dataset_name\": dataset,
                \"model_name\": \"coupled_nmf\",
                \"seed\": seed,
                \"k\": k,
                \"alpha\": alpha,
                \"beta\": beta,
                \"lambda_reg\": lambda_reg,
                \"epochs\": final_epochs,
            }

            row.update(metrics)
            run_rows.append(row)

            if first_model_objects is None:
                first_model_objects = model_objects
                first_train = train

        return pd.DataFrame(run_rows), first_model_objects, first_train
    """
)

In [ ]:
write_part(
    """
    def export_tuned_coupled_nmf(
        dataset,
        input_ratings,
        input_descriptors,
        seed_list,
        descriptor_source,
        top_k,
        relevance_threshold,
        output_csv,
        k_values,
        alpha_values,
        beta_values,
        lambda_values,
        tune_seed,
        tune_epochs,
        final_epochs,
        selection_metric,
        log_file=None,
        overwrite=\"false\",
    ):
        del descriptor_source
        del overwrite

        ratings = pd.read_csv(input_ratings)
        descriptors = pd.read_csv(input_descriptors)

        split_path = f\"results/csv/{dataset}_split_registry.csv\"
        if not os.path.exists(split_path):
            raise FileNotFoundError(f\"Split registry not found: {split_path}\")

        split_registry = pd.read_csv(split_path)

        seeds = parse_seed_list(seed_list)
        top_k_values = parse_top_k(top_k)

        k_values = parse_int_list(k_values)
        alpha_values = parse_float_list(alpha_values)
        beta_values = parse_float_list(beta_values)
        lambda_values = parse_float_list(lambda_values)

        grid_df = run_validation_grid(
            dataset=dataset,
            ratings=ratings,
            descriptors=descriptors,
            split_registry=split_registry,
            tune_seed=tune_seed,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
            k_values=k_values,
            alpha_values=alpha_values,
            beta_values=beta_values,
            lambda_values=lambda_values,
            tune_epochs=tune_epochs,
            selection_metric=selection_metric,
            log_file=log_file,
        )

        tuning_grid_path = f\"results/csv/{dataset}_coupled_nmf_tuning_grid.csv\"
        ensure_parent(tuning_grid_path)
        grid_df.to_csv(tuning_grid_path, index=False)

        best_config = select_best_config(grid_df, selection_metric)

        selected_config_path = f\"results/csv/{dataset}_coupled_nmf_selected_config.csv\"
        pd.DataFrame([best_config]).to_csv(selected_config_path, index=False)

        write_log(log_file, f\"Saved tuning grid: {tuning_grid_path}\")
        write_log(log_file, f\"Saved selected config: {selected_config_path}\")
        write_log(log_file, f\"Selected config: {best_config}\")

        new_coupled_df, first_model_objects, first_train = run_final_selected_config(
            dataset=dataset,
            ratings=ratings,
            descriptors=descriptors,
            split_registry=split_registry,
            seeds=seeds,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
            best_config=best_config,
            final_epochs=final_epochs,
            log_file=log_file,
        )

        run_level_path = f\"results/csv/{dataset}_run_level_metrics.csv\"

        if os.path.exists(run_level_path):
            old_run_df = pd.read_csv(run_level_path)
            old_run_df = old_run_df[old_run_df[\"model_name\"] != \"coupled_nmf\"]
            run_df = pd.concat([old_run_df, new_coupled_df], ignore_index=True)
        else:
            run_df = new_coupled_df

        ensure_parent(run_level_path)
        run_df.to_csv(run_level_path, index=False)

        summary_df = aggregate_results(run_df, dataset)
        ensure_parent(output_csv)
        summary_df.to_csv(output_csv, index=False)

        if first_model_objects is not None:
            p1 = export_factor_descriptor_weights(dataset, first_model_objects)
            p2 = export_factor_keywords(dataset, first_model_objects)
            p3 = export_recommendation_traces(dataset, first_train, first_model_objects)

            write_log(log_file, f\"Saved: {p1}\")
            write_log(log_file, f\"Saved: {p2}\")
            write_log(log_file, f\"Saved: {p3}\")

        write_log(log_file, f\"Saved run-level metrics: {run_level_path}\")
        write_log(log_file, f\"Saved main comparison: {output_csv}\")
    """
)

In [ ]:
write_part(
    """
    def main():
        parser = argparse.ArgumentParser()
        parser.add_argument("--dataset", required=True)
        parser.add_argument("--input_ratings", required=True)
        parser.add_argument("--input_descriptors", required=True)
        parser.add_argument("--seed_list", required=True)
        parser.add_argument("--descriptor_source", default="tags_genres")
        parser.add_argument("--top_k", default="5,10")
        parser.add_argument("--relevance_threshold", type=float, default=4.0)
        parser.add_argument("--output_csv", required=True)

        parser.add_argument("--k_values", default="20,30,40")
        parser.add_argument("--alpha_values", default="0,0.01,0.05,0.1")
        parser.add_argument("--beta_values", default="0,0.0001")
        parser.add_argument("--lambda_values", default="0.01")

        parser.add_argument("--tune_seed", type=int, default=42)
        parser.add_argument("--tune_epochs", type=int, default=20)
        parser.add_argument("--final_epochs", type=int, default=30)
        parser.add_argument("--selection_metric", default="ndcg_at_10")

        parser.add_argument("--log_file", default=None)
        parser.add_argument("--overwrite", default="false")

        args = parser.parse_args()
        export_tuned_coupled_nmf(**vars(args))

    if __name__ == "__main__":
        main()
    """
)

In [ ]:
%cd /content/drive/MyDrive/xai_coupled_nmf_project

!python src/run_coupled_nmf_tuning.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --seed_list 42,123,2026,7,99 \
          --descriptor_source tags_genres \
            --top_k 5,10 \
              --relevance_threshold 4.0 \
                --output_csv results/csv/ml_latest_small_main_comparison.csv \
                  --k_values 20,30,40 \
                    --alpha_values 0,0.01,0.05,0.1 \
                      --beta_values 0,0.0001 \
                        --lambda_values 0.01 \
                          --tune_seed 42 \
                            --tune_epochs 20 \
                              --final_epochs 30 \
                                --selection_metric ndcg_at_10 \
                                  --log_file results/logs/ml_latest_small_coupled_nmf_tuning.log \
                                    --overwrite true

In [ ]:
import pandas as pd

grid = pd.read_csv("results/csv/ml_latest_small_coupled_nmf_tuning_grid.csv")
selected = pd.read_csv("results/csv/ml_latest_small_coupled_nmf_selected_config.csv")
main_results = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")
run_level = pd.read_csv("results/csv/ml_latest_small_run_level_metrics.csv")

print("Top 10 tuning configurations by validation NDCG@10:")
display(grid.sort_values("ndcg_at_10", ascending=False).head(10))

print("Selected configuration:")
display(selected)

print("Updated main comparison:")
display(main_results)

print("Run-level shape:", run_level.shape)
display(run_level.tail(10))

In [ ]:
!ls -lh results/csv/ml_latest_small_factor_descriptor_weights.csv
!ls -lh results/csv/ml_latest_small_factor_keywords.csv
!ls -lh results/csv/ml_latest_small_recommendation_traces.csv

factor_keywords = pd.read_csv("results/csv/ml_latest_small_factor_keywords.csv")
display(factor_keywords.head(15))

In [ ]:
import shutil
from pathlib import Path

# Ensure the archive directory exists
Path("results/csv/archive").mkdir(parents=True, exist_ok=True)

# Define the mapping of current files to their archive destinations
files_to_archive = [
    (
        "results/csv/ml_latest_small_main_comparison.csv",
        "results/csv/archive/ml_latest_small_main_comparison_v2_tuned_coupled.csv"
    ),
    (
        "results/csv/ml_latest_small_run_level_metrics.csv",
        "results/csv/archive/ml_latest_small_run_level_metrics_v2_tuned_coupled.csv"
    ),
    (
        "results/csv/ml_latest_small_coupled_nmf_tuning_grid.csv",
        "results/csv/archive/ml_latest_small_coupled_nmf_tuning_grid_v2.csv"
    ),
    (
        "results/csv/ml_latest_small_coupled_nmf_selected_config.csv",
        "results/csv/archive/ml_latest_small_coupled_nmf_selected_config_v2.csv"
    )
]

# Iterate and copy files if they exist
for src, dst in files_to_archive:
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f"Archived: {dst}")
    else:
        print(f"Source not found, skipped: {src}")

In [ ]:
import pandas as pd
import numpy as np

main_results = pd.read_csv("results/csv/ml_latest_small_main_comparison.csv")

plain = main_results[main_results["model_name"] == "plain_nmf"].iloc[0]
coupled = main_results[main_results["model_name"] == "coupled_nmf"].iloc[0]

metrics = [
    "rmse",
    "mae",
    "precision_at_5",
    "precision_at_10",
    "recall_at_5",
    "recall_at_10",
    "ndcg_at_5",
    "ndcg_at_10",
]

rows = []

for metric in metrics:
    plain_value = plain[metric]
    coupled_value = coupled[metric]
    absolute_change = coupled_value - plain_value

    if plain_value != 0:
        percent_change = 100.0 * absolute_change / plain_value
    else:
        percent_change = None

    if metric in ["rmse", "mae"]:
        better_model = "coupled_nmf" if coupled_value < plain_value else "plain_nmf"
    else:
        better_model = "coupled_nmf" if coupled_value > plain_value else "plain_nmf"

    rows.append({
        "metric": metric,
        "plain_nmf": plain_value,
        "coupled_nmf": coupled_value,
        "absolute_change": absolute_change,
        "percent_change": percent_change,
        "better_model": better_model,
    })

delta_df = pd.DataFrame(rows)
delta_df.to_csv("results/csv/ml_latest_small_plain_vs_coupled_delta.csv", index=False)

display(delta_df)
print("Saved: results/csv/ml_latest_small_plain_vs_coupled_delta.csv")

In [ ]:
import pandas as pd

factor_keywords = pd.read_csv("results/csv/ml_latest_small_factor_keywords.csv")
factor_weights = pd.read_csv("results/csv/ml_latest_small_factor_descriptor_weights.csv")
traces = pd.read_csv("results/csv/ml_latest_small_recommendation_traces.csv")

print("Factor keywords:")
display(factor_keywords.head(20))

print("Recommendation traces:")
display(traces.head(20))

print("Shapes:")
print("factor_keywords:", factor_keywords.shape)
print("factor_weights:", factor_weights.shape)
print("traces:", traces.shape)

## 6. Explanation metrics and extended fidelity analysis

Original cell index starts around `185`.

In [ ]:
from pathlib import Path
import textwrap

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
target = PROJECT_ROOT / "src" / "run_explanation_metrics.py"

def write_part(text, mode="a"):
    content = textwrap.dedent(text).strip() + "\n\n"
    if mode == "w":
        target.write_text(content, encoding="utf-8")
    else:
        with open(target, "a", encoding="utf-8") as f:
            f.write(content)
    print("Writing to:", target)

if target.exists():
    target.unlink()

target.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
write_part(
  """
  import argparse
  import os

  import numpy as np
  import pandas as pd

  from benchmark_utils import (
      build_id_maps,
      ensure_parent,
      load_split_data,
      parse_top_k,
      write_log,
  )

  from run_coupled_nmf import (
      train_coupled_nmf,
      build_descriptor_matrix,
  )
  """,
  mode="w"
)

In [ ]:
write_part(
  """
  def compute_mean_factor_coherence(descriptors, B, feature_cols, top_m=5):
      descriptor_matrix = descriptors[feature_cols].to_numpy(dtype=np.float32)
      norms = np.linalg.norm(descriptor_matrix, axis=0) + 1e-12
      normalized = descriptor_matrix / norms

      factor_scores = []
      for factor_idx in range(B.shape[1]):
          weights = B[:, factor_idx]
          top_indices = np.argsort(weights)[::-1][:top_m]
          pair_scores = []
          for a_pos in range(len(top_indices)):
              for b_pos in range(a_pos + 1, len(top_indices)):
                  a = top_indices[a_pos]
                  b = top_indices[b_pos]
                  sim = float(np.dot(normalized[:, a], normalized[:, b]))
                  pair_scores.append(sim)

          if pair_scores:
              factor_scores.append(float(np.mean(pair_scores)))

      if not factor_scores:
          return np.nan

      return float(np.mean(factor_scores))
  """
)

In [ ]:
write_part(
  """
  def compute_explanation_instances(
      dataset,
      train,
      descriptors,
      model_objects,
      top_k_recs=10,
      max_users=100,
  ):
      U = model_objects[\"U\"]
      V = model_objects[\"V\"]
      idx_to_item = model_objects[\"idx_to_item\"]
      user_to_idx = model_objects[\"user_to_idx\"]

      train_items_by_user = train.groupby(\"user_id\")[\"item_id\"].apply(set).to_dict()

      rows = []
      selected_users = list(user_to_idx.items())[:max_users]

      for user_id, u in selected_users:
          seen = train_items_by_user.get(user_id, set())
          scores = V @ U[u]
          candidate_pairs = []

          for i_idx, item_id in idx_to_item.items():
              if item_id not in seen:
                  candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))

          candidate_pairs.sort(key=lambda x: x[1], reverse=True)

          for rank, (item_id, score, i_idx) in enumerate(candidate_pairs[:top_k_recs], start=1):
              contributions = U[u] * V[i_idx]
              total_contribution = float(np.sum(contributions) + 1e-12)

              sorted_factors = np.argsort(contributions)[::-1]
              top2 = sorted_factors[:2]
              top3 = sorted_factors[:3]

              fidelity_at_2 = float(np.sum(contributions[top2]) / total_contribution)
              fidelity_at_3 = float(np.sum(contributions[top3]) / total_contribution)

              dominant_factors = \",\".join([f\"f{x + 1}\" for x in top3])

              rows.append({
                  \"dataset_name\": dataset,
                  \"model_name\": \"coupled_nmf\",
                  \"user_id\": user_id,
                  \"item_id\": item_id,
                  \"rank\": rank,
                  \"score\": score,
                  \"dominant_factors\": dominant_factors,
                  \"fidelity_at_2\": fidelity_at_2,
                  \"fidelity_at_3\": fidelity_at_3,
              })

      return pd.DataFrame(rows)
  """
)

In [ ]:
write_part(
    """
    def export_explanation_metrics(
        dataset,
        input_ratings,
        input_descriptors,
        selected_config_csv,
        seed,
        top_k,
        relevance_threshold,
        final_epochs,
        max_users,
        top_m_descriptors,
        output_csv,
        output_instances_csv,
        log_file=None,
        overwrite=\"false\",
    ):
        del relevance_threshold
        del overwrite

        ratings = pd.read_csv(input_ratings)
        descriptors = pd.read_csv(input_descriptors)
        selected = pd.read_csv(selected_config_csv).iloc[0]

        split_path = f\"results/csv/{dataset}_split_registry.csv\"
        if not os.path.exists(split_path):
            raise FileNotFoundError(f\"Split registry not found: {split_path}\")

        split_registry = pd.read_csv(split_path)
        train, val, test = load_split_data(ratings, split_registry, seed)

        all_data = pd.concat(
            [train[[\"user_id\", \"item_id\", \"rating\"]], test[[\"user_id\", \"item_id\", \"rating\"]]],
            ignore_index=True
        )
        user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

        k = int(selected[\"k\"])
        alpha = float(selected[\"alpha\"])
        beta = float(selected[\"beta\"])
        lambda_reg = float(selected[\"lambda_reg\"])

        write_log(log_file, f\"Training coupled_nmf for explanations: k={k}, alpha={alpha}\")

        U, V, B, feature_cols = train_coupled_nmf(
            train=train, descriptors=descriptors, user_to_idx=user_to_idx,
            item_to_idx=item_to_idx, k=k, epochs=final_epochs,
            alpha=alpha, beta=beta, lambda_reg=lambda_reg, seed=seed
        )

        model_objects = {
            \"U\": U, \"V\": V, \"B\": B, \"feature_cols\": feature_cols,
            \"user_to_idx\": user_to_idx, \"item_to_idx\": item_to_idx,
            \"idx_to_user\": idx_to_user, \"idx_to_item\": idx_to_item
        }

        top_k_values = parse_top_k(top_k)
        top_k_recs = max(top_k_values)

        instances = compute_explanation_instances(
            dataset=dataset, train=train, descriptors=descriptors,
            model_objects=model_objects, top_k_recs=top_k_recs, max_users=max_users
        )

        mean_fidelity_at_2 = float(instances[\"fidelity_at_2\"].mean())
        mean_fidelity_at_3 = float(instances[\"fidelity_at_3\"].mean())
        mean_coherence = compute_mean_factor_coherence(
            descriptors, B, feature_cols, top_m=top_m_descriptors
        )

        metrics = pd.DataFrame([{
            \"dataset_name\": dataset,
            \"model_name\": \"coupled_nmf\",
            \"fidelity_at_2\": mean_fidelity_at_2,
            \"fidelity_at_3\": mean_fidelity_at_3,
            \"mean_coherence\": mean_coherence,
            \"seed\": seed, \"k\": k, \"alpha\": alpha
        }])

        ensure_parent(output_csv)
        metrics.to_csv(output_csv, index=False)
        instances.to_csv(output_instances_csv, index=False)

        write_log(log_file, f\"Saved metrics to {output_csv}\")
    """
)

In [ ]:
write_part(
  """
  def main():
      parser = argparse.ArgumentParser()

      parser.add_argument("--dataset", required=True)
      parser.add_argument("--input_ratings", required=True)
      parser.add_argument("--input_descriptors", required=True)
      parser.add_argument("--selected_config_csv", required=True)
      parser.add_argument("--seed", type=int, default=42)
      parser.add_argument("--top_k", default="5,10")
      parser.add_argument("--relevance_threshold", type=float, default=4.0)
      parser.add_argument("--final_epochs", type=int, default=30)
      parser.add_argument("--max_users", type=int, default=100)
      parser.add_argument("--top_m_descriptors", type=int, default=5)
      parser.add_argument("--output_csv", required=True)
      parser.add_argument("--output_instances_csv", required=True)
      parser.add_argument("--log_file", default=None)
      parser.add_argument("--overwrite", default="false")

      args = parser.parse_args()
      export_explanation_metrics(**vars(args))

  if __name__ == "__main__":
      main()
  """
)

In [ ]:
!cd /content/drive/MyDrive/xai_coupled_nmf_project && wc -l src/run_explanation_metrics.py
!cd /content/drive/MyDrive/xai_coupled_nmf_project && python -m py_compile src/run_explanation_metrics.py

In [ ]:
%cd /content/drive/MyDrive/xai_coupled_nmf_project

!python src/run_explanation_metrics.py \
  --dataset ml_latest_small \
    --input_ratings data/processed/ml_latest_small_ratings.csv \
      --input_descriptors data/processed/ml_latest_small_item_descriptors.csv \
        --selected_config_csv results/csv/ml_latest_small_coupled_nmf_selected_config.csv \
          --seed 42 \
            --top_k 5,10 \
              --relevance_threshold 4.0 \
                --final_epochs 30 \
                  --max_users 100 \
                    --top_m_descriptors 5 \
                      --output_csv results/csv/ml_latest_small_explanation_metrics.csv \
                        --output_instances_csv results/csv/ml_latest_small_explanation_instances.csv \
                          --log_file results/logs/ml_latest_small_explanation_metrics.log \
                            --overwrite true

In [ ]:
import pandas as pd

explanation_metrics = pd.read_csv("results/csv/ml_latest_small_explanation_metrics.csv")
explanation_instances = pd.read_csv("results/csv/ml_latest_small_explanation_instances.csv")
factor_keywords = pd.read_csv("results/csv/ml_latest_small_factor_keywords.csv")

print("Explanation metrics:")
display(explanation_metrics)

print("Explanation instances:")
display(explanation_instances.head(20))

print("Factor keywords:")
display(factor_keywords.head(15))

print("Shapes:")
print("explanation_metrics:", explanation_metrics.shape)
print("explanation_instances:", explanation_instances.shape)
print("factor_keywords:", factor_keywords.shape)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
%cd /content/drive/MyDrive/xai_coupled_nmf_project

instances = pd.read_csv("results/csv/ml_latest_small_explanation_instances.csv")

summary_rows = []

for col in ["fidelity_at_2", "fidelity_at_3"]:
    summary_rows.append({
        "metric": col,
        "mean": instances[col].mean(),
        "std": instances[col].std(),
        "min": instances[col].min(),
        "q25": instances[col].quantile(0.25),
        "median": instances[col].median(),
        "q75": instances[col].quantile(0.75),
        "max": instances[col].max(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results/csv/ml_latest_small_explanation_fidelity_summary.csv", index=False)

print("Fidelity summary:")
display(summary_df)

print("\nHighest Fidelity@3 examples:")
high_fid = instances.sort_values("fidelity_at_3", ascending=False).head(20)
display(high_fid)

print("\nLowest Fidelity@3 examples:")
low_fid = instances.sort_values("fidelity_at_3", ascending=True).head(20)
display(low_fid)

high_fid.to_csv("results/csv/ml_latest_small_high_fidelity_examples.csv", index=False)
low_fid.to_csv("results/csv/ml_latest_small_low_fidelity_examples.csv", index=False)

print("\nSaved:")
print("results/csv/ml_latest_small_explanation_fidelity_summary.csv")
print("results/csv/ml_latest_small_high_fidelity_examples.csv")
print("results/csv/ml_latest_small_low_fidelity_examples.csv")

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"
%cd {PROJECT_ROOT}

# Ensure src is in the system path for imports
src_path = os.path.join(PROJECT_ROOT, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from benchmark_utils import build_id_maps, load_split_data
from run_coupled_nmf import train_coupled_nmf

dataset = "ml_latest_small"

ratings = pd.read_csv("data/processed/ml_latest_small_ratings.csv")
descriptors = pd.read_csv("data/processed/ml_latest_small_item_descriptors.csv")
split_registry = pd.read_csv("results/csv/ml_latest_small_split_registry.csv")
selected = pd.read_csv("results/csv/ml_latest_small_coupled_nmf_selected_config.csv").iloc[0]

seed = 42
final_epochs = 30

train, val, test = load_split_data(
    ratings=ratings,
    split_registry=split_registry,
    seed=seed,
)

all_data = pd.concat(
    [
        train[["user_id", "item_id", "rating"]],
        test[["user_id", "item_id", "rating"]],
    ],
    ignore_index=True,
)

user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

k = int(selected["k"])
alpha = float(selected["alpha"])
beta = float(selected["beta"])
lambda_reg = float(selected["lambda_reg"])

print("Selected configuration:")
print("k =", k)
print("alpha =", alpha)
print("beta =", beta)
print("lambda_reg =", lambda_reg)
print("epochs =", final_epochs)

U, V, B, feature_cols = train_coupled_nmf(
    train=train,
    descriptors=descriptors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    k=k,
    epochs=final_epochs,
    alpha=alpha,
    beta=beta,
    lambda_reg=lambda_reg,
    seed=seed,
)

print("Model trained.")
print("U shape:", U.shape)
print("V shape:", V.shape)
print("B shape:", B.shape)
print("Number of descriptor features:", len(feature_cols))

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/xai_coupled_nmf_project"

%cd /content/drive/MyDrive/xai_coupled_nmf_project

!pwd
!ls -lh src/benchmark_utils.py
!ls -lh src/run_coupled_nmf.py

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

    print("SRC_DIR added:", SRC_DIR)
    print("Exists:", SRC_DIR.exists())
    print("First Python path:", sys.path[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
SRC_DIR = PROJECT_ROOT / "src"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("SRC_DIR:", SRC_DIR)
print("SRC_DIR exists:", SRC_DIR.exists())

print("\nFiles in src:")
if SRC_DIR.exists():
    for p in sorted(SRC_DIR.glob("*.py")):
            print(" -", p.name)

            benchmark_file = SRC_DIR / "benchmark_utils.py"
            coupled_file = SRC_DIR / "run_coupled_nmf.py"

            print("\nbenchmark_utils.py exists:", benchmark_file.exists())
            print("run_coupled_nmf.py exists:", coupled_file.exists())

            if not benchmark_file.exists():
                raise FileNotFoundError(f"Missing file: {benchmark_file}")

                if not coupled_file.exists():
                    raise FileNotFoundError(f"Missing file: {coupled_file}")

                    # Force Python to search the correct src directory first
                    sys.path.insert(0, str(SRC_DIR))
                    importlib.invalidate_caches()

                    # Remove stale failed imports if any
                    for module_name in ["benchmark_utils", "run_coupled_nmf"]:
                        if module_name in sys.modules:
                                del sys.modules[module_name]

                                print("\nFirst entries in sys.path:")
                                for x in sys.path[:5]:
                                    print(x)

                                    from benchmark_utils import build_id_maps, load_split_data
                                    from run_coupled_nmf import train_coupled_nmf

                                    print("\nImports successful.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
SRC_DIR = PROJECT_ROOT / "src"

benchmark_path = SRC_DIR / "benchmark_utils.py"
coupled_path = SRC_DIR / "run_coupled_nmf.py"

print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("SRC_DIR exists:", SRC_DIR.exists())
print("benchmark_utils.py exists:", benchmark_path.exists())
print("run_coupled_nmf.py exists:", coupled_path.exists())

if not benchmark_path.exists():
    raise FileNotFoundError(f"Missing: {benchmark_path}")
if not coupled_path.exists():
    raise FileNotFoundError(f"Missing: {coupled_path}")

# Add src directory for any internal imports used by run_coupled_nmf.py
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

def load_module_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# Correctly call the loader and extract functions
benchmark_utils = load_module_from_path("benchmark_utils", benchmark_path)
run_coupled_nmf_module = load_module_from_path("run_coupled_nmf", coupled_path)

build_id_maps = benchmark_utils.build_id_maps
load_split_data = benchmark_utils.load_split_data
train_coupled_nmf = run_coupled_nmf_module.train_coupled_nmf

print("Modules loaded successfully from absolute paths.")

In [ ]:
import pandas as pd
import numpy as np

dataset = "ml_latest_small"

# Load data using PROJECT_ROOT (defined in previous cells)
ratings = pd.read_csv(PROJECT_ROOT / "data/processed/ml_latest_small_ratings.csv")
descriptors = pd.read_csv(PROJECT_ROOT / "data/processed/ml_latest_small_item_descriptors.csv")
split_registry = pd.read_csv(PROJECT_ROOT / "results/csv/ml_latest_small_split_registry.csv")
selected = pd.read_csv(PROJECT_ROOT / "results/csv/ml_latest_small_coupled_nmf_selected_config.csv").iloc[0]

seed = 42
final_epochs = 30

# load_split_data, build_id_maps, and train_coupled_nmf were loaded in cell ZC3M4VLNTpuI
train, val, test = load_split_data(
    ratings=ratings,
    split_registry=split_registry,
    seed=seed,
)

all_data = pd.concat(
    [
        train[["user_id", "item_id", "rating"]],
        test[["user_id", "item_id", "rating"]],
    ],
    ignore_index=True,
)

user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

k = int(selected["k"])
alpha = float(selected["alpha"])
beta = float(selected["beta"])
lambda_reg = float(selected["lambda_reg"])

print("Selected configuration:")
print("k =", k)
print("alpha =", alpha)
print("beta =", beta)
print("lambda_reg =", lambda_reg)
print("epochs =", final_epochs)

U, V, B, feature_cols = train_coupled_nmf(
    train=train,
    descriptors=descriptors,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    k=k,
    epochs=final_epochs,
    alpha=alpha,
    beta=beta,
    lambda_reg=lambda_reg,
    seed=seed,
)

print("Model trained.")
print("train shape:", train.shape)
print("test shape:", test.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)
print("B shape:", B.shape)
print("Number of descriptor features:", len(feature_cols))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def compute_extended_explanation_instances(
    dataset,
    train,
    U,
    V,
    user_to_idx,
    idx_to_item,
    top_k_recs=10,
    max_users=100,
):
    train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
    rows = []
    selected_users = list(user_to_idx.items())[:max_users]

    for user_id, u in selected_users:
        seen = train_items_by_user.get(user_id, set())
        scores = V @ U[u]
        candidate_pairs = []

        for i_idx, item_id in idx_to_item.items():
            if item_id not in seen:
                candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))

        candidate_pairs.sort(key=lambda x: x[1], reverse=True)

        for rank, (item_id, score, i_idx) in enumerate(candidate_pairs[:top_k_recs], start=1):
            contributions = U[u] * V[i_idx]
            total = float(np.sum(contributions) + 1e-12)

            sorted_factors = np.argsort(contributions)[::-1]
            top2 = sorted_factors[:2]
            top3 = sorted_factors[:3]
            top5 = sorted_factors[:5]
            top10 = sorted_factors[:10]

            rows.append({
                "dataset_name": dataset,
                "model_name": "coupled_nmf",
                "user_id": user_id,
                "item_id": item_id,
                "rank": rank,
                "score": score,
                "dominant_factors_3": ",".join([f"f{x + 1}" for x in top3]),
                "dominant_factors_5": ",".join([f"f{x + 1}" for x in top5]),
                "dominant_factors_10": ",".join([f"f{x + 1}" for x in top10]),
                "fidelity_at_2": float(np.sum(contributions[top2]) / total),
                "fidelity_at_3": float(np.sum(contributions[top3]) / total),
                "fidelity_at_5": float(np.sum(contributions[top5]) / total),
                "fidelity_at_10": float(np.sum(contributions[top10]) / total),
            })

    return pd.DataFrame(rows)

extended_instances = compute_extended_explanation_instances(
    dataset=dataset,
    train=train,
    U=U,
    V=V,
    user_to_idx=user_to_idx,
    idx_to_item=idx_to_item,
    top_k_recs=10,
    max_users=100,
)

out_path = PROJECT_ROOT / "results/csv/ml_latest_small_explanation_instances_extended.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

extended_instances.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", extended_instances.shape)
display(extended_instances.head(20))

In [ ]:
def compute_mean_factor_coherence(descriptors, B, feature_cols, top_m=5):
    descriptor_matrix = descriptors[feature_cols].to_numpy(dtype=np.float32)
    norms = np.linalg.norm(descriptor_matrix, axis=0) + 1e-12
    normalized = descriptor_matrix / norms

    factor_scores = []
    for factor_idx in range(B.shape[1]):
        weights = B[:, factor_idx]
        top_indices = np.argsort(weights)[::-1][:top_m]

        pair_scores = []
        for a_pos in range(len(top_indices)):
            for b_pos in range(a_pos + 1, len(top_indices)):
                a = top_indices[a_pos]
                b = top_indices[b_pos]
                sim = float(np.dot(normalized[:, a], normalized[:, b]))
                pair_scores.append(sim)

        if pair_scores:
            factor_scores.append(float(np.mean(pair_scores)))

    if not factor_scores:
        return np.nan

    return float(np.mean(factor_scores))

mean_coherence = compute_mean_factor_coherence(
    descriptors=descriptors,
    B=B,
    feature_cols=feature_cols,
    top_m=5,
)

extended_metrics = pd.DataFrame([
    {
        "dataset_name": dataset,
        "model_name": "coupled_nmf",
        "fidelity_at_2": extended_instances["fidelity_at_2"].mean(),
        "fidelity_at_3": extended_instances["fidelity_at_3"].mean(),
        "fidelity_at_5": extended_instances["fidelity_at_5"].mean(),
        "fidelity_at_10": extended_instances["fidelity_at_10"].mean(),
        "mean_coherence": mean_coherence,
        "stability_score": np.nan,
        "seed": seed,
        "k": k,
        "alpha": alpha,
        "beta": beta,
        "lambda_reg": lambda_reg,
        "epochs": final_epochs,
        "num_explanation_instances": len(extended_instances),
    }
])

extended_metrics_path = PROJECT_ROOT / "results/csv/ml_latest_small_explanation_metrics_extended.csv"
extended_metrics.to_csv(extended_metrics_path, index=False)

summary_rows = []
for col in ["fidelity_at_2", "fidelity_at_3", "fidelity_at_5", "fidelity_at_10"]:
    summary_rows.append({
        "metric": col,
        "mean": extended_instances[col].mean(),
        "std": extended_instances[col].std(),
        "min": extended_instances[col].min(),
        "q25": extended_instances[col].quantile(0.25),
        "median": extended_instances[col].median(),
        "q75": extended_instances[col].quantile(0.75),
        "max": extended_instances[col].max(),
    })

fidelity_summary = pd.DataFrame(summary_rows)
summary_path = PROJECT_ROOT / "results/csv/ml_latest_small_explanation_fidelity_summary_extended.csv"
fidelity_summary.to_csv(summary_path, index=False)

print("Extended explanation metrics:")
display(extended_metrics)

print("Extended fidelity distribution summary:")
display(fidelity_summary)

print("Saved:")
print(extended_metrics_path)
print(summary_path)

In [ ]:
high_fid = extended_instances.sort_values("fidelity_at_10", ascending=False).head(30)
low_fid = extended_instances.sort_values("fidelity_at_10", ascending=True).head(30)

high_path = PROJECT_ROOT / "results/csv/ml_latest_small_high_fidelity_examples_extended.csv"
low_path = PROJECT_ROOT / "results/csv/ml_latest_small_low_fidelity_examples_extended.csv"

high_fid.to_csv(high_path, index=False)
low_fid.to_csv(low_path, index=False)

print("Highest Fidelity@10 examples:")
display(high_fid)

print("Lowest Fidelity@10 examples:")
display(low_fid)

print("Saved:")
print(high_path)
print(low_path)

In [ ]:
import pandas as pd

extended_metrics = pd.read_csv(
    "/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_metrics_extended.csv"
)

fidelity_summary = pd.read_csv(
    "/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_fidelity_summary_extended.csv"
)

print("Extended explanation metrics:")
display(extended_metrics)

print("Extended fidelity summary:")
display(fidelity_summary)

In [ ]:
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_instances_extended.csv
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_metrics_extended.csv
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_fidelity_summary_extended.csv
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_high_fidelity_examples_extended.csv
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_low_fidelity_examples_extended.csv

In [ ]:
import pandas as pd

extended_metrics = pd.read_csv(
    "/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_metrics_extended.csv"
)

fidelity_summary = pd.read_csv(
    "/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_explanation_fidelity_summary_extended.csv"
)

display(extended_metrics)
display(fidelity_summary)

In [ ]:
from pathlib import Path

CSV_DIR = Path("/content/drive/MyDrive/xai_coupled_nmf_project/results/csv")

required_files = [
    "ml_latest_small_main_comparison.csv",
    "ml_latest_small_run_level_metrics.csv",
    "ml_latest_small_coupled_nmf_tuning_grid.csv",
    "ml_latest_small_coupled_nmf_selected_config.csv",
    "ml_latest_small_factor_descriptor_weights.csv",
    "ml_latest_small_factor_keywords.csv",
    "ml_latest_small_recommendation_traces.csv",
    "ml_latest_small_explanation_metrics.csv",
    "ml_latest_small_explanation_instances.csv",
    "ml_latest_small_explanation_instances_extended.csv",
    "ml_latest_small_explanation_metrics_extended.csv",
    "ml_latest_small_explanation_fidelity_summary_extended.csv",
    "ml_latest_small_high_fidelity_examples_extended.csv",
    "ml_latest_small_low_fidelity_examples_extended.csv",
]

for filename in required_files:
    path = CSV_DIR / filename
    print(f"{filename:70s} ->", "FOUND" if path.exists() else "MISSING")

In [ ]:
!ls -lh /content/drive/MyDrive/xai_coupled_nmf_project/results/csv/ml_latest_small_*.csv

In [ ]:
from pathlib import Path

ARCHIVE_DIR = Path("/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/archive")

v2_files = [
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
]

for filename in v2_files:
    path = ARCHIVE_DIR / filename
    print(f"{filename:70s} ->", "FOUND" if path.exists() else "MISSING")

In [ ]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

files_to_archive = [
    (
        CSV_DIR / "ml_latest_small_main_comparison.csv",
        ARCHIVE_DIR / "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_run_level_metrics.csv",
        ARCHIVE_DIR / "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_coupled_nmf_tuning_grid.csv",
        ARCHIVE_DIR / "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_coupled_nmf_selected_config.csv",
        ARCHIVE_DIR / "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_explanation_metrics_extended.csv",
        ARCHIVE_DIR / "ml_latest_small_explanation_metrics_extended_v2.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_explanation_fidelity_summary_extended.csv",
        ARCHIVE_DIR / "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",
    ),
]

for src, dst in files_to_archive:
    if src.exists():
        shutil.copy(src, dst)
        print("Archived:", dst.name)
    else:
        print("Missing source, not archived:", src.name)

In [ ]:
from pathlib import Path

ARCHIVE_DIR = Path("/content/drive/MyDrive/xai_coupled_nmf_project/results/csv/archive")

v2_files = [
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    "ml_latest_small_explanation_metrics_extended_v2.csv",
    "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",
]

for filename in v2_files:
    path = ARCHIVE_DIR / filename
    print(f"{filename:75s} ->", "FOUND" if path.exists() else "MISSING")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
SRC_DIR = PROJECT_ROOT / "src"

%cd /content/drive/MyDrive/xai_coupled_nmf_project

def load_module_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

benchmark_utils = load_module_from_path(
    "benchmark_utils",
    SRC_DIR / "benchmark_utils.py"
)

run_coupled_nmf_module = load_module_from_path(
    "run_coupled_nmf",
    SRC_DIR / "run_coupled_nmf.py"
)

build_id_maps = benchmark_utils.build_id_maps
load_split_data = benchmark_utils.load_split_data
prediction_metrics = benchmark_utils.prediction_metrics
ranking_metrics_for_scores = benchmark_utils.ranking_metrics_for_scores
train_coupled_nmf = run_coupled_nmf_module.train_coupled_nmf

print("Modules loaded successfully.")

In [ ]:
import pandas as pd
import numpy as np
from itertools import product

dataset = "ml_latest_small"

# Load data using PROJECT_ROOT
ratings = pd.read_csv(PROJECT_ROOT / "data/processed/ml_latest_small_ratings.csv")
descriptors = pd.read_csv(PROJECT_ROOT / "data/processed/ml_latest_small_item_descriptors.csv")
split_registry = pd.read_csv(PROJECT_ROOT / "results/csv/ml_latest_small_split_registry.csv")

seed = 42

# Call the loaded functions with correct indentation
train, val, test = load_split_data(
    ratings=ratings,
    split_registry=split_registry,
    seed=seed,
)

all_items = sorted(ratings["item_id"].unique())

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)
print("Number of items:", len(all_items))

## 7. Interpretability-oriented diagnostic tuning

Original cell index starts around `216`.

In [ ]:
def compute_fidelity_for_top_recs(
    train,
    U,
    V,
    user_to_idx,
    idx_to_item,
    max_users=100,
    top_k_recs=10,
):
    train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()

    rows = []
    selected_users = list(user_to_idx.items())[:max_users]

    for user_id, u in selected_users:
        seen = train_items_by_user.get(user_id, set())
        scores = V @ U[u]

        candidate_pairs = []

        for i_idx, item_id in idx_to_item.items():
            if item_id not in seen:
                candidate_pairs.append((item_id, float(scores[i_idx]), i_idx))

        candidate_pairs.sort(key=lambda x: x[1], reverse=True)

        for rank, (item_id, score, i_idx) in enumerate(candidate_pairs[:top_k_recs], start=1):
            contributions = U[u] * V[i_idx]
            total = float(np.sum(contributions) + 1e-12)

            sorted_factors = np.argsort(contributions)[::-1]

            rows.append({
                "user_id": user_id,
                "item_id": item_id,
                "rank": rank,
                "score": score,
                "fidelity_at_2": float(np.sum(contributions[sorted_factors[:2]]) / total),
                "fidelity_at_3": float(np.sum(contributions[sorted_factors[:3]]) / total),
                "fidelity_at_5": float(np.sum(contributions[sorted_factors[:5]]) / total),
                "fidelity_at_10": float(np.sum(contributions[sorted_factors[:10]]) / total),
            })

    return pd.DataFrame(rows)

In [ ]:
def evaluate_interpretability_config(
    train,
    val,
    descriptors,
    all_items,
    seed,
    k,
    alpha,
    beta,
    lambda_reg,
    epochs,
    relevance_threshold=4.0,
    top_k_values=(5, 10),
    max_users=100,
):
    all_data = pd.concat(
        [
            train[["user_id", "item_id", "rating"]],
            val[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V, B, feature_cols = train_coupled_nmf(
        train=train,
        descriptors=descriptors,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        k=k,
        epochs=epochs,
        alpha=alpha,
        beta=beta,
        lambda_reg=lambda_reg,
        seed=seed,
    )

    global_mean = train["rating"].mean()
    preds = []

    for row in val.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    scores_by_user = {}
    for user_id, u_idx in user_to_idx.items():
        scores = V @ U[u_idx]
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    rmse_value, mae_value = prediction_metrics(val, np.array(preds))

    rank_metrics = ranking_metrics_for_scores(
        scores_by_user=scores_by_user,
        train=train,
        test=val,
        all_items=all_items,
        top_k_values=list(top_k_values),
        relevance_threshold=relevance_threshold,
    )

    fidelity_df = compute_fidelity_for_top_recs(
        train=train,
        U=U,
        V=V,
        user_to_idx=user_to_idx,
        idx_to_item=idx_to_item,
        max_users=max_users,
        top_k_recs=10,
    )

    row = {
        "dataset_name": dataset,
        "seed": seed,
        "k": k,
        "alpha": alpha,
        "beta": beta,
        "lambda_reg": lambda_reg,
        "epochs": epochs,
        "rmse": rmse_value,
        "mae": mae_value,
        "fidelity_at_2": fidelity_df["fidelity_at_2"].mean(),
        "fidelity_at_3": fidelity_df["fidelity_at_3"].mean(),
        "fidelity_at_5": fidelity_df["fidelity_at_5"].mean(),
        "fidelity_at_10": fidelity_df["fidelity_at_10"].mean(),
        "num_explanation_instances": len(fidelity_df),
    }

    row.update(rank_metrics)
    return row

In [ ]:
k_values = [10, 20, 30, 40]
alpha_values = [0.05, 0.10, 0.20]
beta_values = [0.0001, 0.001, 0.005]
lambda_values = [0.01]

epochs = 30
relevance_threshold = 4.0
top_k_values = (5, 10)
max_users = 100

grid = list(product(k_values, alpha_values, beta_values, lambda_values))
rows = []

print("Total configurations:", len(grid))

for idx, (k, alpha, beta, lambda_reg) in enumerate(grid, start=1):
    print(f"Running {idx}/{len(grid)}: k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}")

    row = evaluate_interpretability_config(
        train=train,
        val=val,
        descriptors=descriptors,
        all_items=all_items,
        seed=seed,
        k=k,
        alpha=alpha,
        beta=beta,
        lambda_reg=lambda_reg,
        epochs=epochs,
        relevance_threshold=relevance_threshold,
        top_k_values=top_k_values,
        max_users=max_users
    )

    rows.append(row)

    interpretability_grid = pd.DataFrame(rows)
    out_path = PROJECT_ROOT / "results/csv/ml_latest_small_interpretability_tuning_grid.csv"
    interpretability_grid.to_csv(out_path, index=False)

    print("Saved:", out_path)
    display(interpretability_grid.sort_values("fidelity_at_10", ascending=False).head(10))

In [ ]:
grid = pd.read_csv(PROJECT_ROOT / "results/csv/ml_latest_small_interpretability_tuning_grid.csv")

best_ndcg = grid["ndcg_at_10"].max()
ndcg_threshold = 0.90 * best_ndcg

eligible = grid[grid["ndcg_at_10"] >= ndcg_threshold].copy()

selected_interpretability = eligible.sort_values(
    by=["fidelity_at_10", "ndcg_at_10", "rmse"],
    ascending=[False, False, True],
).head(1)

selected_path = PROJECT_ROOT / "results/csv/ml_latest_small_interpretability_selected_config.csv"
selected_interpretability.to_csv(selected_path, index=False)

print("Best validation NDCG@10:", best_ndcg)
print("NDCG eligibility threshold:", ndcg_threshold)
print("Number of eligible configurations:", len(eligible))

print("Selected interpretability-oriented configuration:")
display(selected_interpretability)

print("Top 10 by Fidelity@10 among eligible configurations:")
display(
    eligible.sort_values(
        by=["fidelity_at_10", "ndcg_at_10"],
        ascending=[False, False],
    ).head(10)
)

print("Saved:", selected_path)

In [ ]:
accuracy_selected = pd.read_csv(
    PROJECT_ROOT / "results/csv/ml_latest_small_coupled_nmf_selected_config.csv"
)

interpretability_selected = pd.read_csv(
    PROJECT_ROOT / "results/csv/ml_latest_small_interpretability_selected_config.csv"
)

print("Accuracy-oriented selected configuration:")
display(accuracy_selected)

print("\nInterpretability-oriented selected configuration:")
display(interpretability_selected)

In [ ]:
import pandas as pd
from pathlib import Path

# Use the same PROJECT_ROOT path that worked in your Colab session
CSV_DIR = PROJECT_ROOT / "results/csv"

grid = pd.read_csv(CSV_DIR / "ml_latest_small_interpretability_tuning_grid.csv")
v2_selected = pd.read_csv(CSV_DIR / "ml_latest_small_coupled_nmf_selected_config.csv")
v3_selected = pd.read_csv(CSV_DIR / "ml_latest_small_interpretability_selected_config.csv")

print("Accuracy-oriented selected configuration, Version 2:")
display(v2_selected)

print("Interpretability-oriented selected configuration, Version 3 candidate:")
display(v3_selected)

print("Top 10 configurations by Fidelity@10:")
display(
    grid.sort_values(
        by=["fidelity_at_10", "ndcg_at_10", "rmse"],
        ascending=[False, False, True]
    ).head(10)
)

print("Top 10 eligible-style configurations by NDCG@10:")
display(
    grid.sort_values(
        by=["ndcg_at_10", "fidelity_at_10", "rmse"],
        ascending=[False, False, True]
    ).head(10)
)

best_ndcg = grid["ndcg_at_10"].max()
selected_ndcg = float(v3_selected["ndcg_at_10"].iloc[0])
selected_fid10 = float(v3_selected["fidelity_at_10"].iloc[0])

print("Best validation NDCG@10:", best_ndcg)
print("Selected V3 validation NDCG@10:", selected_ndcg)
print("Selected V3 Fidelity@10:", selected_fid10)
print("Selected V3 NDCG retention:", selected_ndcg / best_ndcg)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())

In [ ]:
v3_files_to_archive = [
    (
        CSV_DIR / "ml_latest_small_interpretability_tuning_grid.csv",
        ARCHIVE_DIR / "ml_latest_small_interpretability_tuning_grid_v3.csv",
    ),
    (
        CSV_DIR / "ml_latest_small_interpretability_selected_config.csv",
        ARCHIVE_DIR / "ml_latest_small_interpretability_selected_config_v3.csv",
    ),
]

for src, dst in v3_files_to_archive:
    if src.exists():
        shutil.copy(src, dst)
        print("Archived:", dst.name)
    else:
        print("Missing source, not archived:", src.name)

In [ ]:
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

v3_timestamp_dir = ARCHIVE_DIR / f"v3_interpretability_{timestamp}"
v3_timestamp_dir.mkdir(parents=True, exist_ok=True)

for src, _ in v3_files_to_archive:
    if src.exists():
        dst = v3_timestamp_dir / src.name
        shutil.copy(src, dst)
        print("Timestamped copy:", dst.name)

print("Timestamped Version 3 folder:", v3_timestamp_dir)

In [ ]:
v3_expected = [
    "ml_latest_small_interpretability_tuning_grid_v3.csv",
    "ml_latest_small_interpretability_selected_config_v3.csv",
]

for filename in v3_expected:
    path = ARCHIVE_DIR / filename
    print(f"{filename:70s} ->", "FOUND" if path.exists() else "MISSING")

In [ ]:
v3_selected_path = ARCHIVE_DIR / "ml_latest_small_interpretability_selected_config_v3.csv"

if v3_selected_path.exists():
    v3_selected = pd.read_csv(v3_selected_path)
    display(v3_selected)
else:
    print("Version 3 selected config file not found.")

## 8. Archiving Version 2/Version 3 latest-small outputs

Original cell index starts around `227`.

In [ ]:
from pathlib import Path
from datetime import datetime

# Use the active path that worked in your current session.
# If your current PROJECT_ROOT already exists, this will use it.
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BUNDLE_DIR = PROJECT_ROOT / "results/export_bundles" / f"ml_latest_small_csv_bundle_{timestamp}"

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("BUNDLE_DIR:", BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())

In [ ]:
import shutil
import pandas as pd

files_to_preserve = [
    # Core dataset and benchmark outputs
    "ml_latest_small_dataset_summary.csv",
    "ml_latest_small_split_registry.csv",
    "ml_latest_small_main_comparison.csv",
    "ml_latest_small_run_level_metrics.csv",

    # Version 2: accuracy-oriented tuned coupled NMF
    "ml_latest_small_coupled_nmf_tuning_grid.csv",
    "ml_latest_small_coupled_nmf_selected_config.csv",
    "ml_latest_small_plain_vs_coupled_delta.csv",

    # Explanation outputs
    "ml_latest_small_factor_descriptor_weights.csv",
    "ml_latest_small_factor_keywords.csv",
    "ml_latest_small_recommendation_traces.csv",
    "ml_latest_small_explanation_metrics.csv",
    "ml_latest_small_explanation_instances.csv",
    "ml_latest_small_explanation_instances_extended.csv",
    "ml_latest_small_explanation_metrics_extended.csv",
    "ml_latest_small_explanation_fidelity_summary_extended.csv",
    "ml_latest_small_high_fidelity_examples_extended.csv",
    "ml_latest_small_low_fidelity_examples_extended.csv",

    # Version 3: interpretability-oriented tuning candidate
    "ml_latest_small_interpretability_tuning_grid.csv",
    "ml_latest_small_interpretability_selected_config.csv",
]

copy_report = []

for filename in files_to_preserve:
    src = CSV_DIR / filename
    dst = BUNDLE_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    copy_report.append({
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

copy_report_df = pd.DataFrame(copy_report)
copy_report_df.to_csv(BUNDLE_DIR / "bundle_copy_report.csv", index=False)

display(copy_report_df)
print("Bundle folder:", BUNDLE_DIR)

In [ ]:
ARCHIVE_DIR = CSV_DIR / "archive"

archive_files = [
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    "ml_latest_small_explanation_metrics_extended_v2.csv",
    "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",
]

ARCHIVE_BUNDLE_DIR = BUNDLE_DIR / "archive_v2"
ARCHIVE_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

archive_report = []

for filename in archive_files:
    src = ARCHIVE_DIR / filename
    dst = ARCHIVE_BUNDLE_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    archive_report.append({
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

archive_report_df = pd.DataFrame(archive_report)
archive_report_df.to_csv(BUNDLE_DIR / "archive_v2_copy_report.csv", index=False)

display(archive_report_df)
print("Archive bundle folder:", ARCHIVE_BUNDLE_DIR)

In [ ]:
manifest_text = f"""
CSV Export Bundle for ml_latest_small Experiments
================================================

Project:
xai_coupled_nmf_project

Dataset:
ml_latest_small

Bundle created:
{timestamp}

Contents:
1. Main benchmark results
2. Accuracy-oriented tuned coupled NMF outputs, Version 2
3. Extended explanation metrics and fidelity summaries
4. Interpretability-oriented tuning candidate outputs, Version 3
5. Archived Version 2 files, if available

Important files:
- ml_latest_small_main_comparison.csv
- ml_latest_small_run_level_metrics.csv
- ml_latest_small_coupled_nmf_selected_config.csv
- ml_latest_small_explanation_metrics_extended.csv
- ml_latest_small_explanation_fidelity_summary_extended.csv
- ml_latest_small_interpretability_tuning_grid.csv
- ml_latest_small_interpretability_selected_config.csv

Notes:
Version 2 is the accuracy-oriented tuned coupled NMF selected by validation NDCG@10.
Version 3 is the interpretability-oriented candidate selected by NDCG retention and Fidelity@10.
"""

manifest_path = BUNDLE_DIR / "README_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Manifest saved:", manifest_path)

In [ ]:
import shutil

zip_base = str(BUNDLE_DIR)
zip_path = shutil.make_archive(zip_base, "zip", root_dir=BUNDLE_DIR)

print("ZIP created:")
print(zip_path)

In [ ]:
print("Bundle contents:")
!find "$BUNDLE_DIR" -maxdepth 2 -type f | sort

print("\nZIP file:")
!ls -lh "$zip_path"

In [ ]:
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

PROJECT_ROOT = Path(PROJECT_ROOT)
EXPORT_DIR = PROJECT_ROOT / "results/export_bundles"
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"

print("EXPORT_DIR exists:", EXPORT_DIR.exists())
print("ARCHIVE_DIR exists:", ARCHIVE_DIR.exists())

print("\nVersion 3 files in archive:")
v3_archive_files = [
    "ml_latest_small_interpretability_tuning_grid_v3.csv",
    "ml_latest_small_interpretability_selected_config_v3.csv",
]

for filename in v3_archive_files:
    path = ARCHIVE_DIR / filename
    print(f"{filename:70s} ->", "FOUND" if path.exists() else "MISSING")

print("\nExisting export bundles:")
if EXPORT_DIR.exists():
    for p in sorted(EXPORT_DIR.glob("*")):
        print(p)
else:
    print("No export_bundles folder found.")

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BUNDLE_DIR = (
    PROJECT_ROOT
    / "results/export_bundles"
    / f"ml_latest_small_v2_v3_csv_bundle_{timestamp}"
)

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

files_to_bundle = [
    # Version 2: accuracy-oriented tuned coupled NMF
    ARCHIVE_DIR / "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    ARCHIVE_DIR / "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    ARCHIVE_DIR / "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    ARCHIVE_DIR / "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    ARCHIVE_DIR / "ml_latest_small_explanation_metrics_extended_v2.csv",
    ARCHIVE_DIR / "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",

    # Version 3: interpretability-oriented diagnostic tuning
    ARCHIVE_DIR / "ml_latest_small_interpretability_tuning_grid_v3.csv",
    ARCHIVE_DIR / "ml_latest_small_interpretability_selected_config_v3.csv",
]

report = []

for src in files_to_bundle:
    if src.exists():
        dst = BUNDLE_DIR / src.name
        shutil.copy(src, dst)
        report.append((src.name, "COPIED"))
    else:
        report.append((src.name, "MISSING"))

manifest_text = f"""CSV Bundle: ml_latest_small Version 2 and Version 3

Created: {timestamp}

Version 2:
Accuracy-oriented tuned coupled NMF selected by validation NDCG@10.

Version 3:
Interpretability-oriented diagnostic tuning selected by NDCG retention and Fidelity@10.

Files included:
""" + "\n".join([f"- {name}: {status}" for name, status in report])

manifest_path = BUNDLE_DIR / "README_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

zip_path = shutil.make_archive(str(BUNDLE_DIR), "zip", root_dir=BUNDLE_DIR)

print("Bundle folder created:")
print(BUNDLE_DIR)

print("\nZIP created:")
print(zip_path)

print("\nCopy report:")
for name, status in report:
    print(f"{name:85s} -> {status}")

In [ ]:
print("Bundle contents:")
!find "$BUNDLE_DIR" -maxdepth 1 -type f | sort

print("\nZIP file:")
!ls -lh "$zip_path"

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

# Use the active PROJECT_ROOT if already defined.
try:
    PROJECT_ROOT
except NameError:
    candidate_roots = [
        Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
        Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
    ]
    PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)
print("PAPER_TABLE_DIR:", PAPER_TABLE_DIR)
print("RESULTS_TABLE_DIR:", RESULTS_TABLE_DIR)

def find_csv(filename):
    direct_candidates = [
        ARCHIVE_DIR / filename,
        CSV_DIR / filename,
    ]

    for path in direct_candidates:
        if path.exists():
            return path

    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not locate: {filename}")

## 9. Manuscript-ready LaTeX tables for latest-small

Original cell index starts around `237`.

In [ ]:
v2_main_path = find_csv("ml_latest_small_main_comparison_v2_tuned_coupled.csv")
v2_selected_path = find_csv("ml_latest_small_coupled_nmf_selected_config_v2.csv")
v2_expl_path = find_csv("ml_latest_small_explanation_metrics_extended_v2.csv")
v2_fidelity_path = find_csv("ml_latest_small_explanation_fidelity_summary_extended_v2.csv")

v3_grid_path = find_csv("ml_latest_small_interpretability_tuning_grid_v3.csv")
v3_selected_path = find_csv("ml_latest_small_interpretability_selected_config_v3.csv")

v2_main = pd.read_csv(v2_main_path)
v2_selected = pd.read_csv(v2_selected_path)
v2_expl = pd.read_csv(v2_expl_path)
v2_fidelity = pd.read_csv(v2_fidelity_path)

v3_grid = pd.read_csv(v3_grid_path)
v3_selected = pd.read_csv(v3_selected_path)

print("Loaded files:")
for p in [
    v2_main_path,
        v2_selected_path,
            v2_expl_path,
                v2_fidelity_path,
                    v3_grid_path,
                        v3_selected_path,
                        ]:
                            print(" -", p)

In [ ]:
def tex_escape(text):
    text = str(text)
    replacements = {
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"

def fmt_pm(row, metric, digits=4):
    std_col = f"{metric}_std"
    if std_col in row.index and not pd.isna(row[std_col]):
        return rf"${float(row[metric]):.{digits}f}\pm {float(row[std_col]):.{digits}f}$"
    return rf"${float(row[metric]):.{digits}f}$"

def write_table_file(filename, caption, label, colspec, header, rows, resize=False, note=None):
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")

    if resize:
        lines.append(r"\resizebox{\textwidth}{!}{%")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    lines.append(header + r" \\")
    lines.append(r"\midrule")

    for row in rows:
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if resize:
        lines.append(r"}")

    if note:
        lines.append(r"\vspace{0.35em}")
        lines.append(r"\begin{minipage}{0.96\textwidth}")
        lines.append(r"\footnotesize " + note)
        lines.append(r"\end{minipage}")

    lines.append(r"\end{table}")
    lines.append("")

    content = "\n".join(lines)

    paper_path = PAPER_TABLE_DIR / filename
    results_path = RESULTS_TABLE_DIR / filename

    paper_path.write_text(content, encoding="utf-8")
    results_path.write_text(content, encoding="utf-8")

    print("Saved:", paper_path)
    print("Copied:", results_path)

    return paper_path

In [ ]:
model_display = {
    "biased_mf": "Biased MF",
    "plain_nmf": "Plain NMF",
    "coupled_nmf": "Coupled NMF",
    "item_cf": "Item-CF",
    "item_mean": "Item Mean",
}

model_order = ["biased_mf", "plain_nmf", "coupled_nmf", "item_cf", "item_mean"]

main_rows = []

for model in model_order:
    sub = v2_main[v2_main["model_name"] == model]
    if sub.empty:
        continue

    row = sub.iloc[0]
    main_rows.append([
        tex_escape(model_display.get(model, model)),
        fmt_pm(row, "rmse"),
        fmt_pm(row, "mae"),
        fmt_pm(row, "precision_at_5"),
        fmt_pm(row, "precision_at_10"),
        fmt_pm(row, "recall_at_5"),
        fmt_pm(row, "recall_at_10"),
        fmt_pm(row, "ndcg_at_5"),
        fmt_pm(row, "ndcg_at_10"),
    ])

table1 = write_table_file(
    filename="table_ml_latest_small_main_comparison_v2.tex",
    caption="Main recommendation performance on MovieLens latest-small. Values are reported as mean $\\pm$ standard deviation over five random seeds.",
    label="tab:ml_latest_small_main_comparison",
    colspec="lcccccccc",
    header="Model & RMSE & MAE & P@5 & P@10 & R@5 & R@10 & NDCG@5 & NDCG@10",
    rows=main_rows,
    resize=True,
)

sel = v2_selected.iloc[0]

selected_rows = [[
    fmt(sel["k"], 0),
    fmt(sel["alpha"], 4),
    fmt(sel["beta"], 4),
    fmt(sel["lambda_reg"], 4),
    fmt(sel["epochs"], 0),
    fmt(sel["rmse"]),
    fmt(sel["mae"]),
    fmt(sel["precision_at_10"]),
    fmt(sel["recall_at_10"]),
    fmt(sel["ndcg_at_10"]),
]]

table2 = write_table_file(
    filename="table_ml_latest_small_selected_config_v2.tex",
    caption="Accuracy-oriented selected coupled NMF configuration obtained by validation tuning.",
    label="tab:ml_latest_small_selected_config",
    colspec="cccccccccc",
    header="$k$ & $\\alpha$ & $\\beta$ & $\\lambda$ & Epochs & RMSE & MAE & P@10 & R@10 & NDCG@10",
    rows=selected_rows,
    resize=True,
)

In [ ]:
plain = v2_main[v2_main['model_name'] == 'plain_nmf'].iloc[0]
coupled = v2_main[v2_main['model_name'] == 'coupled_nmf'].iloc[0]

delta_metrics = [
    ("RMSE", "rmse", "lower"),
    ("MAE", "mae", "lower"),
    ("P@5", "precision_at_5", "higher"),
    ("P@10", "precision_at_10", "higher"),
    ("R@5", "recall_at_5", "higher"),
    ("R@10", "recall_at_10", "higher"),
    ("NDCG@5", "ndcg_at_5", "higher"),
    ("NDCG@10", "ndcg_at_10", "higher"),
]

delta_rows = []

for display_name, metric, direction in delta_metrics:
    plain_value = float(plain[metric])
    coupled_value = float(coupled[metric])
    change = coupled_value - plain_value
    pct = 100.0 * change / plain_value if plain_value != 0 else np.nan

    if direction == "lower":
        better = "Coupled NMF" if coupled_value < plain_value else "Plain NMF"
    else:
        better = "Coupled NMF" if coupled_value > plain_value else "Plain NMF"

    delta_rows.append([
        display_name,
        fmt(plain_value),
        fmt(coupled_value),
        fmt(change),
        fmt(pct, 2) + r"%",
        tex_escape(better),
    ])

table3 = write_table_file(
    filename="table_ml_latest_small_plain_vs_coupled_delta_v2.tex",
    caption="Direct comparison between rating-only NMF and the proposed coupled NMF on MovieLens latest-small.",
    label="tab:ml_latest_small_plain_vs_coupled_delta",
    colspec="lccccc",
    header="Metric & Plain NMF & Coupled NMF & Change & Change (%) & Better",
    rows=delta_rows,
    resize=True,
)

In [ ]:
expl = v2_expl.iloc[0]

# Prepare rows for the summary metrics table
expl_rows = [[
    fmt(expl["fidelity_at_2"]),
    fmt(expl["fidelity_at_3"]),
    fmt(expl["fidelity_at_5"]),
    fmt(expl["fidelity_at_10"]),
    fmt(expl["mean_coherence"]),
    "--" if "stability_score" not in expl.index or pd.isna(expl["stability_score"]) else fmt(expl["stability_score"]),
    fmt(expl["num_explanation_instances"], 0),
]]

table4 = write_table_file(
    filename="table_ml_latest_small_explanation_metrics_v2.tex",
    caption="Explanation-oriented metrics for the tuned coupled NMF model.",
    label="tab:ml_latest_small_explanation_metrics",
    colspec="ccccccc",
    header="Fid.@2 & Fid.@3 & Fid.@5 & Fid.@10 & Mean coherence & Stability & Instances",
    rows=expl_rows,
    resize=True,
    note="Fidelity@r measures the proportion of the recommendation score explained by the top $r$ additive latent-factor contributions.",
)

metric_display = {
    "fidelity_at_2": "Fidelity@2",
    "fidelity_at_3": "Fidelity@3",
    "fidelity_at_5": "Fidelity@5",
    "fidelity_at_10": "Fidelity@10",
}

fidelity_rows = []
for _, row in v2_fidelity.iterrows():
    fidelity_rows.append([
        tex_escape(metric_display.get(row["metric"], row["metric"])),
        fmt(row["mean"]),
        fmt(row["std"]),
        fmt(row["min"]),
        fmt(row["q25"]),
        fmt(row["median"]),
        fmt(row["q75"]),
        fmt(row["max"]),
    ])

table5 = write_table_file(
    filename="table_ml_latest_small_fidelity_distribution_v2.tex",
    caption="Distribution of explanation fidelity over 1000 recommendation instances.",
    label="tab:ml_latest_small_fidelity_distribution",
    colspec="lccccccc",
    header="Metric & Mean & Std. & Min & Q1 & Median & Q3 & Max",
    rows=fidelity_rows,
    resize=True,
)

In [ ]:
# V2-equivalent row from the Version 3 interpretability grid:
# same as V2 except evaluated in the interpretability diagnostic setting with epochs=30.
v2_equiv = v3_grid[
    (v3_grid["k"] == 40) &
    (np.isclose(v3_grid["alpha"], 0.10)) &
    (np.isclose(v3_grid["beta"], 0.0001)) &
    (np.isclose(v3_grid["lambda_reg"], 0.01))
].iloc[0]

v3_sel = v3_selected.iloc[0]
diagnostic_rows = []

for label, row in [
    ("Accuracy-oriented region", v2_equiv),
    ("Interpretability-oriented candidate", v3_sel),
]:
    diagnostic_rows.append([
        tex_escape(label),
        fmt(row["k"], 0),
        fmt(row["alpha"], 4),
        fmt(row["beta"], 4),
        fmt(row["lambda_reg"], 4),
        fmt(row["rmse"]),
        fmt(row["mae"]),
        fmt(row["fidelity_at_5"]),
        fmt(row["fidelity_at_10"]),
        fmt(row["ndcg_at_10"]),
    ])

table6 = write_table_file(
    filename="table_ml_latest_small_v3_interpretability_diagnostic.tex",
    caption="Interpretability-oriented diagnostic comparison on the validation split.",
    label="tab:ml_latest_small_v3_interpretability_diagnostic",
    colspec="lccccccccc",
    header="Configuration & $k$ & $\\alpha$ & $\\beta$ & $\\lambda$ & RMSE & MAE & Fid.@5 & Fid.@10 & NDCG@10",
    rows=diagnostic_rows,
    resize=True,
    note="The diagnostic grid shows that increasing semantic coupling from $\\alpha=0.1$ to $\\alpha=0.2$ retains validation ranking quality but produces only a negligible improvement in Fidelity@10.",
)

top_v3 = v3_grid.sort_values(
    by=["fidelity_at_10", "ndcg_at_10", "rmse"],
    ascending=[False, False, True]
).head(5)

top_v3_rows = []
for _, row in top_v3.iterrows():
    top_v3_rows.append([
        fmt(row["k"], 0),
        fmt(row["alpha"], 4),
        fmt(row["beta"], 4),
        fmt(row["lambda_reg"], 4),
        fmt(row["rmse"]),
        fmt(row["mae"]),
        fmt(row["fidelity_at_3"]),
        fmt(row["fidelity_at_5"]),
        fmt(row["fidelity_at_10"]),
        fmt(row["ndcg_at_10"]),
    ])

table7 = write_table_file(
    filename="table_ml_latest_small_top_v3_fidelity_configs.tex",
    caption="Top diagnostic configurations sorted by Fidelity@10.",
    label="tab:ml_latest_small_top_v3_fidelity_configs",
    colspec="cccccccccc",
    header="$k$ & $\\alpha$ & $\\beta$ & $\\lambda$ & RMSE & MAE & Fid.@3 & Fid.@5 & Fid.@10 & NDCG@10",
    rows=top_v3_rows,
    resize=True,
    note="Configurations with $k=10$ attain Fidelity@10 equal to one because the top ten factors exhaust all latent dimensions.",
)

In [ ]:
table_files = [
    "table_ml_latest_small_main_comparison_v2.tex",
    "table_ml_latest_small_selected_config_v2.tex",
    "table_ml_latest_small_plain_vs_coupled_delta_v2.tex",
    "table_ml_latest_small_explanation_metrics_v2.tex",
    "table_ml_latest_small_fidelity_distribution_v2.tex",
    "table_ml_latest_small_v3_interpretability_diagnostic.tex",
    "table_ml_latest_small_top_v3_fidelity_configs.tex",
]

combined_path = PAPER_TABLE_DIR / "ml_latest_small_all_manuscript_tables.tex"

combined_parts = []
combined_parts.append("% Auto-generated manuscript-ready tables for ml_latest_small.")
combined_parts.append("% Required LaTeX packages: booktabs, graphicx.")
combined_parts.append("")

for filename in table_files:
    content = (PAPER_TABLE_DIR / filename).read_text(encoding="utf-8")
    combined_parts.append("% ------------------------------------------------------------")
    combined_parts.append(f"% {filename}")
    combined_parts.append("% ------------------------------------------------------------")
    combined_parts.append(content)

combined_path.write_text("\n".join(combined_parts), encoding="utf-8")

include_path = PAPER_TABLE_DIR / "ml_latest_small_table_include_commands.tex"

include_lines = [
    "% Include these in the manuscript after loading booktabs and graphicx.",
    "% Example: \\usepackage{booktabs,graphicx}",
    "",
]

for filename in table_files:
    stem = Path(filename).stem
    include_lines.append(rf"\input{{paper/tables/{stem}}}")

include_path.write_text("\n".join(include_lines), encoding="utf-8")

shutil.copy(combined_path, RESULTS_TABLE_DIR / combined_path.name)
shutil.copy(include_path, RESULTS_TABLE_DIR / include_path.name)

print("Combined table file:", combined_path)
print("Include command file:", include_path)

In [ ]:
print("Generated paper/tables files:")
!find "$PAPER_TABLE_DIR" -maxdepth 1 -type f -name "*.tex" | sort

print("\nGenerated results/tables files:")
!find "$RESULTS_TABLE_DIR" -maxdepth 1 -type f -name "*.tex" | sort

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    PROJECT_ROOT
except NameError:
    candidate_roots = [
        Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
        Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
    ]
    PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = list(PROJECT_ROOT.rglob(filename))
    return matches[0] if matches else None

def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"

def tex_escape(text):
    return str(text).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

def write_table(filename, caption, label, colspec, header, rows, note=None):
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    lines.append(header + r" \\")
    lines.append(r"\midrule")
    for row in rows:
        lines.append(" & ".join(row) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if note:
        lines.append(r"\vspace{0.35em}")
        lines.append(r"\begin{minipage}{0.92\textwidth}")
        lines.append(r"\footnotesize " + note)
        lines.append(r"\end{minipage}")
    lines.append(r"\end{table}")
    lines.append("")

    content = "\n".join(lines)
    paper_path = PAPER_TABLE_DIR / filename
    results_path = RESULTS_TABLE_DIR / filename

    paper_path.write_text(content, encoding="utf-8")
    results_path.write_text(content, encoding="utf-8")

    print("Saved:", paper_path)
    print("Copied:", results_path)

# Table A1: Baseline smoke test
smoke_path = find_csv("ml_latest_small_baseline_smoke_test.csv")
if smoke_path is not None:
    smoke = pd.read_csv(smoke_path)
    rows = []
    for _, row in smoke.iterrows():
        rows.append([
            tex_escape(row["model_name"]),
            fmt(row["rmse"]),
            fmt(row["mae"]),
            str(int(row["num_train"])),
            str(int(row["num_test"])),
        ])

    write_table(
        filename="table_appendix_baseline_smoke_test.tex",
        caption="Evaluation-pipeline sanity check on MovieLens latest-small.",
        label="tab:appendix_baseline_smoke_test",
        colspec="lcccc",
        header="Predictor & RMSE & MAE & Train ratings & Test ratings",
        rows=rows,
        note=("This table is intended as a reproducibility sanity check. "
              "The global-mean and item-mean predictors are not used as final competing models."),
    )
else:
    print("Smoke-test CSV not found. Skipping baseline smoke-test table.")

# Table A2: Model-development/version record
v2_main_path = find_csv("ml_latest_small_main_comparison_v2_tuned_coupled.csv")
v2_expl_path = find_csv("ml_latest_small_explanation_metrics_extended_v2.csv")
v3_selected_path = find_csv("ml_latest_small_interpretability_selected_config_v3.csv")
v1_main_path = find_csv("ml_latest_small_main_comparison_v1_initial_coupled.csv")

version_rows = []
if v1_main_path is not None:
    v1 = pd.read_csv(v1_main_path)
    v1_c = v1[v1["model_name"] == "coupled_nmf"]
    if not v1_c.empty:
        r = v1_c.iloc[0]
        version_rows.append(["V1 initial coupled NMF", "Initial untuned prototype", fmt(r["rmse"]), fmt(r["mae"]), "--", "--", fmt(r["ndcg_at_10"])])

if v2_main_path is not None:
    v2 = pd.read_csv(v2_main_path)
    v2_c = v2[v2["model_name"] == "coupled_nmf"]
    if not v2_c.empty:
        r = v2_c.iloc[0]
        fid10 = "--"
        if v2_expl_path is not None:
            v2_expl_df = pd.read_csv(v2_expl_path)
            if "fidelity_at_10" in v2_expl_df.columns:
                fid10 = fmt(v2_expl_df["fidelity_at_10"].iloc[0])
        version_rows.append(["V2 accuracy-tuned coupled NMF", "Main manuscript model", fmt(r["rmse"]), fmt(r["mae"]), fid10, "--", fmt(r["ndcg_at_10"])])

if v3_selected_path is not None:
    v3 = pd.read_csv(v3_selected_path).iloc[0]
    version_rows.append(["V3 interpretability diagnostic", "Diagnostic tuning candidate", fmt(v3["rmse"]), fmt(v3["mae"]), fmt(v3["fidelity_at_10"]), fmt(v3["alpha"]), fmt(v3["ndcg_at_10"])])

if version_rows:
    write_table(
        filename="table_appendix_model_development_versions.tex",
        caption="Development record of coupled NMF configurations.",
        label="tab:appendix_model_development_versions",
        colspec="llccccc",
        header="Version & Role & RMSE & MAE & Fid.@10 & $\\alpha$ & NDCG@10",
        rows=version_rows,
        note=("This table is intended for dissertation-level reproducibility documentation. "
              "Version 2 is retained as the main model because Version 3 gives only negligible improvement in Fidelity@10."),
    )
else:
    print("No version records found. Skipping model-development table.")

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    candidate_roots = [
        Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
        Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
    ]
    PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

PROJECT_ROOT = Path(PROJECT_ROOT)
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"
EXPORT_ROOT = PROJECT_ROOT / "results/export_bundles"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

FINAL_BUNDLE_DIR = EXPORT_ROOT / f"ml_latest_small_final_csv_latex_bundle_{timestamp}"
FINAL_CSV_DIR = FINAL_BUNDLE_DIR / "csv"
FINAL_ARCHIVE_DIR = FINAL_BUNDLE_DIR / "csv_archive"
FINAL_TABLE_DIR = FINAL_BUNDLE_DIR / "latex_tables"

for folder in [FINAL_CSV_DIR, FINAL_ARCHIVE_DIR, FINAL_TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_BUNDLE_DIR:", FINAL_BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())
print("ARCHIVE_DIR exists:", ARCHIVE_DIR.exists())
print("PAPER_TABLE_DIR exists:", PAPER_TABLE_DIR.exists())

In [ ]:
# Step 1: Migration of Current CSV Files
current_csv_files = [
    "ml_latest_small_dataset_summary.csv",
    "ml_latest_small_split_registry.csv",
    "ml_latest_small_baseline_smoke_test.csv",
    "ml_latest_small_main_comparison.csv",
    "ml_latest_small_run_level_metrics.csv",
    "ml_latest_small_plain_vs_coupled_delta.csv",
    "ml_latest_small_coupled_nmf_tuning_grid.csv",
    "ml_latest_small_coupled_nmf_selected_config.csv",
    "ml_latest_small_factor_descriptor_weights.csv",
    "ml_latest_small_factor_keywords.csv",
    "ml_latest_small_recommendation_traces.csv",
    "ml_latest_small_explanation_metrics.csv",
    "ml_latest_small_explanation_instances.csv",
    "ml_latest_small_explanation_instances_extended.csv",
    "ml_latest_small_explanation_metrics_extended.csv",
    "ml_latest_small_explanation_fidelity_summary_extended.csv",
    "ml_latest_small_high_fidelity_examples_extended.csv",
    "ml_latest_small_low_fidelity_examples_extended.csv",
    "ml_latest_small_interpretability_tuning_grid.csv",
    "ml_latest_small_interpretability_selected_config.csv",
]

csv_report = []
for filename in current_csv_files:
    src, dst = CSV_DIR / filename, FINAL_CSV_DIR / filename
    if src.exists():
        shutil.copy2(src, dst)
        csv_report.append({"section": "current", "file": filename, "status": "COPIED", "kb": round(dst.stat().st_size/1024, 2)})
    else:
        csv_report.append({"section": "current", "file": filename, "status": "MISSING", "kb": None})

# Step 2: Migration of Archive Files (v2/v3 History)
archive_files = [
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    "ml_latest_small_explanation_metrics_extended_v2.csv",
    "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",
    "ml_latest_small_interpretability_tuning_grid_v3.csv",
    "ml_latest_small_interpretability_selected_config_v3.csv",
]

for filename in archive_files:
    src, dst = ARCHIVE_DIR / filename, FINAL_ARCHIVE_DIR / filename
    if src.exists():
        shutil.copy2(src, dst)
        csv_report.append({"section": "archive", "file": filename, "status": "COPIED", "kb": round(dst.stat().st_size/1024, 2)})

# Step 3: Migration of LaTeX Table Files
for tex_file in PAPER_TABLE_DIR.glob("*.tex"):
    shutil.copy2(tex_file, FINAL_TABLE_DIR / tex_file.name)
    csv_report.append({"section": "latex", "file": tex_file.name, "status": "COPIED", "kb": round((FINAL_TABLE_DIR / tex_file.name).stat().st_size/1024, 2)})

csv_report_df = pd.DataFrame(csv_report)
display(csv_report_df.groupby("section")["status"].value_counts())
display(csv_report_df)

In [ ]:
archive_csv_files = [
    # Version 1, if available
    "ml_latest_small_main_comparison_v1_initial_coupled.csv",
    "ml_latest_small_run_level_metrics_v1_initial_coupled.csv",

    # Version 2
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    "ml_latest_small_explanation_metrics_extended_v2.csv",
    "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",

    # Version 3
    "ml_latest_small_interpretability_tuning_grid_v3.csv",
    "ml_latest_small_interpretability_selected_config_v3.csv",
]

archive_report = []

for filename in archive_csv_files:
    src = ARCHIVE_DIR / filename
    dst = FINAL_ARCHIVE_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    archive_report.append({
        "section": "archive_csv",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

archive_report_df = pd.DataFrame(archive_report)
display(archive_report_df)

In [ ]:
latex_table_files = [
    # Main result tables
    "table_ml_latest_small_main_comparison_v2.tex",
    "table_ml_latest_small_selected_config_v2.tex",
    "table_ml_latest_small_plain_vs_coupled_delta_v2.tex",
    "table_ml_latest_small_explanation_metrics_v2.tex",
    "table_ml_latest_small_fidelity_distribution_v2.tex",
    "table_ml_latest_small_v3_interpretability_diagnostic.tex",
    "table_ml_latest_small_top_v3_fidelity_configs.tex",

    # Combined/include files
    "ml_latest_small_all_manuscript_tables.tex",
    "ml_latest_small_table_include_commands.tex",

    # Appendix/reproducibility tables
    "table_appendix_baseline_smoke_test.tex",
    "table_appendix_model_development_versions.tex",
]

latex_report = []

for filename in latex_table_files:
    src_candidates = [
        PAPER_TABLE_DIR / filename,
        RESULTS_TABLE_DIR / filename,
    ]

    src = next((p for p in src_candidates if p.exists()), None)
    dst = FINAL_TABLE_DIR / filename

    if src is not None:
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    latex_report.append({
        "section": "latex_tables",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

latex_report_df = pd.DataFrame(latex_report)
display(latex_report_df)

In [ ]:
full_report = pd.concat(
    [csv_report_df, archive_report_df, latex_report_df],
    ignore_index=True,
)

copy_report_path = FINAL_BUNDLE_DIR / "bundle_copy_report.csv"
full_report.to_csv(copy_report_path, index=False)

missing_files = full_report[full_report["status"] == "MISSING"]

manifest_text = f"""Final CSV and LaTeX Table Bundle
================================

Project:
xai_coupled_nmf_project

Dataset:
ml_latest_small

Bundle created:
{timestamp}

Bundle purpose:
This bundle preserves the completed MovieLens latest-small experimental stage for the coupled NMF explainable recommendation manuscript and dissertation workflow.

Included sections:
1. Current CSV outputs
2. Archived/versioned CSV outputs
3. Manuscript-ready LaTeX tables
4. Appendix/reproducibility LaTeX tables

Version notes:
- Version 1: Initial coupled NMF prototype, if available.
- Version 2: Accuracy-oriented tuned coupled NMF selected by validation NDCG@10. This is the main manuscript model.
- Version 3: Interpretability-oriented diagnostic tuning. This is retained as a diagnostic sensitivity result, not as the main model.

Main manuscript model:
Version 2 accuracy-oriented tuned coupled NMF.

Important manuscript tables:
- table_ml_latest_small_main_comparison_v2.tex
- table_ml_latest_small_selected_config_v2.tex
- table_ml_latest_small_plain_vs_coupled_delta_v2.tex
- table_ml_latest_small_explanation_metrics_v2.tex
- table_ml_latest_small_fidelity_distribution_v2.tex
- table_ml_latest_small_v3_interpretability_diagnostic.tex
- table_ml_latest_small_top_v3_fidelity_configs.tex

Appendix/dissertation tables:
- table_appendix_baseline_smoke_test.tex
- table_appendix_model_development_versions.tex

Missing files:
"""

if missing_files.empty:
    manifest_text += "\nNone.\n"
else:
    for _, row in missing_files.iterrows():
        manifest_text += f"\n- {row['section']}: {row['filename']}"

manifest_path = FINAL_BUNDLE_DIR / "README_final_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Copy report saved:", copy_report_path)
print("Manifest saved:", manifest_path)

if not missing_files.empty:
    print("\nMissing files:")
    display(missing_files)

In [ ]:
zip_path = shutil.make_archive(
    str(FINAL_BUNDLE_DIR),
    "zip",
    root_dir=FINAL_BUNDLE_DIR,
)

print("Final bundle folder:")
print(FINAL_BUNDLE_DIR)

print("\nFinal ZIP file:")
print(zip_path)

In [ ]:
print("Final bundle contents:")
!find "$FINAL_BUNDLE_DIR" -maxdepth 3 -type f | sort

print("\nZIP file:")
!ls -lh "$zip_path"

## 10. Manuscript-ready figures for latest-small

Original cell index starts around `253`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import textwrap

try:
    PROJECT_ROOT
except NameError:
    candidate_roots = [
        Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
        Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
    ]
    PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

PROJECT_ROOT = Path(PROJECT_ROOT)

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)
print("PAPER_FIG_DIR:", PAPER_FIG_DIR)
print("RESULTS_FIG_DIR:", RESULTS_FIG_DIR)

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not locate: {filename}")

def save_figure(fig, stem):
    paper_png = PAPER_FIG_DIR / f"{stem}.png"
    paper_pdf = PAPER_FIG_DIR / f"{stem}.pdf"
    results_png = RESULTS_FIG_DIR / f"{stem}.png"
    results_pdf = RESULTS_FIG_DIR / f"{stem}.pdf"

    fig.savefig(paper_png, dpi=300, bbox_inches="tight")
    fig.savefig(paper_pdf, bbox_inches="tight")
    shutil.copy(paper_png, results_png)
    shutil.copy(paper_pdf, results_pdf)

    print("Saved:", paper_png)
    print("Saved:", paper_pdf)
    print("Copied:", results_png)
    print("Copied:", results_pdf)

    return {
        "stem": stem,
        "paper_png": paper_png,
        "paper_pdf": paper_pdf,
        "results_png": results_png,
        "results_pdf": results_pdf,
    }

def directional_normalize(series, higher_is_better=True):
    series = pd.Series(series, dtype=float)
    smin, smax = series.min(), series.max()
    if np.isclose(smin, smax):
        return pd.Series(np.ones(len(series)), index=series.index)
    if higher_is_better:
        return (series - smin) / (smax - smin)
    return (smax - series) / (smax - smin)

def wrap_text(s, width=60):
    return "\n".join(textwrap.wrap(str(s), width=width))

In [ ]:
v2_main = pd.read_csv(find_csv("ml_latest_small_main_comparison_v2_tuned_coupled.csv"))
v2_selected = pd.read_csv(find_csv("ml_latest_small_coupled_nmf_selected_config_v2.csv"))
v2_expl = pd.read_csv(find_csv("ml_latest_small_explanation_metrics_extended_v2.csv"))
v2_fidelity = pd.read_csv(find_csv("ml_latest_small_explanation_fidelity_summary_extended_v2.csv"))

v3_grid = pd.read_csv(find_csv("ml_latest_small_interpretability_tuning_grid_v3.csv"))
v3_selected = pd.read_csv(find_csv("ml_latest_small_interpretability_selected_config_v3.csv"))

factor_keywords = pd.read_csv(find_csv("ml_latest_small_factor_keywords.csv"))
explanation_examples = pd.read_csv(find_csv("ml_latest_small_high_fidelity_examples_extended.csv"))

# Optional files
v1_main = None
try:
    v1_main_path = find_csv("ml_latest_small_main_comparison_v1_initial_coupled.csv")
    v1_main = pd.read_csv(v1_main_path)
except Exception:
    v1_main = None

print("Loaded core CSVs successfully.")

In [ ]:
model_display = {
    "biased_mf": "Biased MF",
    "plain_nmf": "Plain NMF",
    "coupled_nmf": "Coupled NMF",
    "item_cf": "Item-CF",
}

selected_models = ["biased_mf", "plain_nmf", "coupled_nmf", "item_cf"]
metrics_info = [
    ("rmse", False, "RMSE"),
    ("mae", False, "MAE"),
    ("ndcg_at_10", True, "NDCG@10"),
    ("recall_at_10", True, "Recall@10"),
]

plot_df = v2_main[v2_main["model_name"].isin(selected_models)].copy()
plot_df = plot_df.set_index("model_name").loc[selected_models].reset_index()

norm_data = {}
for metric, higher_better, label in metrics_info:
    norm_data[label] = directional_normalize(plot_df[metric], higher_is_better=higher_better).values

x = np.arange(len(metrics_info))
width = 0.18

fig, ax = plt.subplots(figsize=(10, 6))

for i, model in enumerate(selected_models):
    yvals = [norm_data[label][i] for _, _, label in metrics_info]
    ax.bar(x + (i - 1.5) * width, yvals, width=width, label=model_display[model])

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metrics_info])
ax.set_ylabel("Normalized directional score")
ax.set_title("Overall benchmark comparison on MovieLens latest-small")
ax.legend()
ax.set_ylim(0, 1.08)
ax.grid(axis="y", alpha=0.3)

fig1_info = save_figure(fig, "fig01_ml_latest_small_overall_benchmark_comparison")
plt.show()

In [ ]:
plain = v2_main[v2_main["model_name"] == "plain_nmf"].iloc[0]
coupled = v2_main[v2_main["model_name"] == "coupled_nmf"].iloc[0]

comparison_metrics = [
    ("rmse", "RMSE", False),
    ("mae", "MAE", False),
    ("precision_at_10", "P@10", True),
    ("recall_at_10", "R@10", True),
    ("ndcg_at_10", "NDCG@10", True),
]

labels = []
improvements = []

for metric, label, higher_better in comparison_metrics:
    p = float(plain[metric])
    c = float(coupled[metric])

    if higher_better:
        imp = 100.0 * (c - p) / p
    else:
        imp = 100.0 * (p - c) / p

    labels.append(label)
    improvements.append(imp)

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(labels, improvements, color="skyblue")

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Relative improvement of Coupled NMF over Plain NMF (%)")
ax.set_title("Focused comparison: Coupled NMF vs Plain NMF")

for rect, val in zip(bars, improvements):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height(),
            f"{val:.2f}%", ha="center", va="bottom" if val > 0 else "top")

ax.grid(axis="y", alpha=0.3)

fig2_info = save_figure(fig, "fig02_ml_latest_small_plain_vs_coupled_relative_improvement")
plt.show()

In [ ]:
instances_ext = pd.read_csv(find_csv("ml_latest_small_explanation_instances_extended.csv"))

fidelity_cols = ["fidelity_at_2", "fidelity_at_3", "fidelity_at_5", "fidelity_at_10"]
data = [instances_ext[col].dropna().values for col in fidelity_cols]

fig, ax = plt.subplots(figsize=(8, 6))
ax.boxplot(data, tick_labels=["Fid.@2", "Fid.@3", "Fid.@5", "Fid.@10"])
ax.set_ylabel("Fidelity value")
ax.set_title("Distribution of explanation fidelity over recommendation instances")
ax.grid(axis="y", alpha=0.3)

fig3_info = save_figure(fig, "fig03_ml_latest_small_explanation_fidelity_distribution")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(v3_grid["fidelity_at_10"], v3_grid["ndcg_at_10"], alpha=0.8)

# Accuracy-oriented V2-equivalent reference inside V3 grid
v2_equiv = v3_grid[
    (v3_grid["k"] == 40) &
    (np.isclose(v3_grid["alpha"], 0.10)) &
    (np.isclose(v3_grid["beta"], 0.0001)) &
    (np.isclose(v3_grid["lambda_reg"], 0.01))
].iloc[0]

v3_sel = v3_selected.iloc[0]

ax.scatter([v2_equiv["fidelity_at_10"]], [v2_equiv["ndcg_at_10"]], s=120, marker="s", label="V2 region")
ax.scatter([v3_sel["fidelity_at_10"]], [v3_sel["ndcg_at_10"]], s=120, marker="^", label="V3 selected")

ax.annotate(
    "V2 region",
    (v2_equiv["fidelity_at_10"], v2_equiv["ndcg_at_10"]),
    xytext=(6, 6),
    textcoords="offset points",
)

ax.annotate(
    "V3 selected",
    (v3_sel["fidelity_at_10"], v3_sel["ndcg_at_10"]),
    xytext=(6, -12),
    textcoords="offset points",
)

ax.set_xlabel("Fidelity@10")
ax.set_ylabel("NDCG@10")
ax.set_title("Interpretability-performance trade-off on validation grid")
ax.legend()
ax.grid(alpha=0.3)

fig4_info = save_figure(fig, "fig04_ml_latest_small_interpretability_tradeoff")
plt.show()

In [ ]:
def build_factor_keyword_lines(df, max_factors=8, keywords_per_factor=5):
    cols = [c.lower() for c in df.columns]
    original_cols = list(df.columns)

    # Try to infer likely columns
    factor_col = None
    keyword_col = None
    rank_col = None

    for c in original_cols:
        lc = c.lower()
        if factor_col is None and ("factor" in lc or lc in ["component", "topic"]):
            factor_col = c
        if keyword_col is None and ("keyword" in lc or "descriptor" in lc or "feature" in lc or lc == "term"):
            keyword_col = c
        if rank_col is None and "rank" in lc:
            rank_col = c

    if factor_col is None:
        factor_col = original_cols[0]
    if keyword_col is None:
        keyword_col = original_cols[1]

    work = df.copy()

    if rank_col is not None:
        work = work.sort_values([factor_col, rank_col])
    else:
        work = work.sort_values([factor_col])

    factor_values = list(work[factor_col].dropna().unique())[:max_factors]

    lines = []
    for f in factor_values:
        sub = work[work[factor_col] == f].head(keywords_per_factor)
        kws = [str(x) for x in sub[keyword_col].tolist()]
        line = f"Factor {f}: " + ", ".join(kws)
        lines.append(line)

    return lines

keyword_lines = build_factor_keyword_lines(factor_keywords, max_factors=8, keywords_per_factor=5)

fig, ax = plt.subplots(figsize=(12, 8))
ax.axis("off")

title = "Representative latent factors and top descriptor keywords"
ax.text(0.01, 0.98, title, fontsize=14, va="top", fontweight="bold")

y = 0.90
for line in keyword_lines:
    ax.text(0.02, y, wrap_text(line, width=90), fontsize=11, va="top", family="monospace")
    y -= 0.10

fig5_info = save_figure(fig, "fig05_ml_latest_small_factor_keyword_map")
plt.show()

In [ ]:
examples = explanation_examples.sort_values("fidelity_at_10", ascending=False).head(6).copy()

fig, ax = plt.subplots(figsize=(12, 9))
ax.axis("off")

ax.text(0.01, 0.98, "Illustrative recommendation explanation examples", fontsize=14, va="top", fontweight="bold")

y = 0.90
for idx, row in examples.iterrows():
    block = (
        f"User {row['user_id']} | Item {row['item_id']} | Rank {row['rank']} | "
        f"Score {row['score']:.4f}\n"
        f"Dominant factors (top 5): {row['dominant_factors_5']}\n"
        f"Fid.@2={row['fidelity_at_2']:.4f}, "
        f"Fid.@3={row['fidelity_at_3']:.4f}, "
        f"Fid.@5={row['fidelity_at_5']:.4f}, "
        f"Fid.@10={row['fidelity_at_10']:.4f}"
    )
    ax.text(0.02, y, wrap_text(block, width=95), fontsize=10.5, va="top", family="monospace")
    y -= 0.15

fig6_info = save_figure(fig, "fig06_ml_latest_small_recommendation_explanation_examples")
plt.show()

In [ ]:
version_rows = []

if v1_main is not None:
    v1_c = v1_main[v1_main["model_name"] == "coupled_nmf"]
    if not v1_c.empty:
        r = v1_c.iloc[0]
        version_rows.append({
            "version": "V1 initial",
            "rmse": float(r["rmse"]),
            "ndcg_at_10": float(r["ndcg_at_10"]),
            "fidelity_at_10": np.nan,
        })

r2 = v2_main[v2_main["model_name"] == "coupled_nmf"].iloc[0]
e2 = v2_expl.iloc[0]
version_rows.append({
    "version": "V2 tuned",
    "rmse": float(r2["rmse"]),
    "ndcg_at_10": float(r2["ndcg_at_10"]),
    "fidelity_at_10": float(e2["fidelity_at_10"]),
})

r3 = v3_selected.iloc[0]
version_rows.append({
    "version": "V3 diagnostic",
    "rmse": float(r3["rmse"]),
    "ndcg_at_10": float(r3["ndcg_at_10"]),
    "fidelity_at_10": float(r3["fidelity_at_10"]),
})

ver_df = pd.DataFrame(version_rows)

norm_rmse = directional_normalize(ver_df["rmse"], higher_is_better=False)
norm_ndcg = directional_normalize(ver_df["ndcg_at_10"], higher_is_better=True)
norm_fid = directional_normalize(ver_df["fidelity_at_10"].fillna(ver_df["fidelity_at_10"].min()), higher_is_better=True)

x = np.arange(len(ver_df))
width = 0.22

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(x - width, norm_rmse, width=width, label="RMSE (direction-normalized)")
ax.bar(x, norm_ndcg, width=width, label="NDCG@10 (direction-normalized)")
ax.bar(x + width, norm_fid, width=width, label="Fidelity@10 (direction-normalized)")

ax.set_xticks(x)
ax.set_xticklabels(ver_df["version"])
ax.set_ylabel("Normalized directional score")
ax.set_title("Model development across coupled NMF versions")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig7_info = save_figure(fig, "fig07_ml_latest_small_model_development_versions")
plt.show()

In [ ]:
figure_stems = [
    "fig01_ml_latest_small_overall_benchmark_comparison",
    "fig02_ml_latest_small_plain_vs_coupled_relative_improvement",
    "fig03_ml_latest_small_explanation_fidelity_distribution",
    "fig04_ml_latest_small_interpretability_tradeoff",
    "fig05_ml_latest_small_factor_keyword_map",
    "fig06_ml_latest_small_recommendation_explanation_examples",
    "fig07_ml_latest_small_model_development_versions",
]

include_lines = [
    "% Include these figure environments in the manuscript as needed.",
    "% Required package: graphicx",
    "",
]

captions = {
    "fig01_ml_latest_small_overall_benchmark_comparison": "Overall benchmark comparison on MovieLens latest-small using directionally normalized scores.",
    "fig02_ml_latest_small_plain_vs_coupled_relative_improvement": "Relative improvement of Coupled NMF over Plain NMF.",
    "fig03_ml_latest_small_explanation_fidelity_distribution": "Distribution of explanation fidelity over recommendation instances.",
    "fig04_ml_latest_small_interpretability_tradeoff": "Interpretability-performance trade-off on the validation tuning grid.",
    "fig05_ml_latest_small_factor_keyword_map": "Representative latent factors and top descriptor keywords.",
    "fig06_ml_latest_small_recommendation_explanation_examples": "Illustrative recommendation explanation examples.",
    "fig07_ml_latest_small_model_development_versions": "Development of coupled NMF variants across experimental versions.",
}

labels = {
    stem: stem.replace("fig", "fig:")
    for stem in figure_stems
}

for stem in figure_stems:
    include_lines.extend([
        r"\begin{figure}[htbp]",
        r"\centering",
        rf"\includegraphics[width=0.9\textwidth]{{paper/figures/{stem}.pdf}}",
        rf"\caption{{{captions[stem]}}}",
        rf"\label{{{labels[stem]}}}",
        r"\end{figure}",
        "",
    ])

include_path = PAPER_FIG_DIR / "ml_latest_small_figure_include_commands.tex"
include_path.write_text("\n".join(include_lines), encoding="utf-8")
shutil.copy(include_path, RESULTS_FIG_DIR / include_path.name)

print("Saved:", include_path)
print("Copied:", RESULTS_FIG_DIR / include_path.name)

In [ ]:
figure_files = []
for stem in figure_stems:
    for ext in ["png", "pdf"]:
        figure_files.append(PAPER_FIG_DIR / f"{stem}.{ext}")

report_rows = []
for p in figure_files:
    report_rows.append({
        "filename": p.name,
        "exists": p.exists(),
        "size_kb": round(p.stat().st_size / 1024, 2) if p.exists() else None,
    })

report_df = pd.DataFrame(report_rows)
report_path = PAPER_FIG_DIR / "ml_latest_small_figure_copy_report.csv"
report_df.to_csv(report_path, index=False)
shutil.copy(report_path, RESULTS_FIG_DIR / report_path.name)

manifest_text = """Manuscript-ready figures for the completed ml_latest_small stage

Figures included:
1. Overall benchmark comparison
2. Plain NMF vs Coupled NMF relative improvement
3. Explanation fidelity distribution
4. Interpretability-performance trade-off
5. Factor keyword map
6. Recommendation explanation examples
7. Model development/version comparison

Output folders:
- paper/figures/
- results/figures/
"""

manifest_path = PAPER_FIG_DIR / "README_figure_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")
shutil.copy(manifest_path, RESULTS_FIG_DIR / manifest_path.name)

display(report_df)
print("Saved:", report_path)
print("Saved:", manifest_path)

In [ ]:
print("Generated paper/figures files:")
!find "$PAPER_FIG_DIR" -maxdepth 1 -type f | sort

print("\nGenerated results/figures files:")
!find "$RESULTS_FIG_DIR" -maxdepth 1 -type f | sort

## 11. Final combined latest-small manuscript bundle

Original cell index starts around `265`.

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    candidate_roots = [
        Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
        Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
    ]
    PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

PROJECT_ROOT = Path(PROJECT_ROOT)

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"
PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"
EXPORT_ROOT = PROJECT_ROOT / "results/export_bundles"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

FINAL_BUNDLE_DIR = EXPORT_ROOT / f"ml_latest_small_complete_manuscript_bundle_{timestamp}"

FINAL_CSV_DIR = FINAL_BUNDLE_DIR / "csv_current"
FINAL_ARCHIVE_DIR = FINAL_BUNDLE_DIR / "csv_archive_versions"
FINAL_TABLE_DIR = FINAL_BUNDLE_DIR / "latex_tables"
FINAL_FIGURE_DIR = FINAL_BUNDLE_DIR / "figures"
FINAL_INCLUDE_DIR = FINAL_BUNDLE_DIR / "latex_include_commands"
FINAL_REPORT_DIR = FINAL_BUNDLE_DIR / "reports"

folders = [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_BUNDLE_DIR:", FINAL_BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())
print("ARCHIVE_DIR exists:", ARCHIVE_DIR.exists())
print("PAPER_TABLE_DIR exists:", PAPER_TABLE_DIR.exists())
print("PAPER_FIG_DIR exists:", PAPER_FIG_DIR.exists())

In [ ]:
current_csv_files = [
    "ml_latest_small_dataset_summary.csv",
    "ml_latest_small_split_registry.csv",
    "ml_latest_small_baseline_smoke_test.csv",
    "ml_latest_small_main_comparison.csv",
    "ml_latest_small_run_level_metrics.csv",
    "ml_latest_small_plain_vs_coupled_delta.csv",
    "ml_latest_small_coupled_nmf_tuning_grid.csv",
    "ml_latest_small_coupled_nmf_selected_config.csv",
    "ml_latest_small_factor_descriptor_weights.csv",
    "ml_latest_small_factor_keywords.csv",
    "ml_latest_small_recommendation_traces.csv",
    "ml_latest_small_explanation_metrics.csv",
    "ml_latest_small_explanation_instances.csv",
    "ml_latest_small_explanation_instances_extended.csv",
    "ml_latest_small_explanation_metrics_extended.csv",
    "ml_latest_small_explanation_fidelity_summary_extended.csv",
    "ml_latest_small_high_fidelity_examples_extended.csv",
    "ml_latest_small_low_fidelity_examples_extended.csv",
    "ml_latest_small_interpretability_tuning_grid.csv",
    "ml_latest_small_interpretability_selected_config.csv",
]

csv_report = []

for filename in current_csv_files:
    src = CSV_DIR / filename
    dst = FINAL_CSV_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    csv_report.append({
        "section": "csv_current",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

csv_report_df = pd.DataFrame(csv_report)
display(csv_report_df)

In [ ]:
archive_csv_files = [
    "ml_latest_small_main_comparison_v1_initial_coupled.csv",
    "ml_latest_small_run_level_metrics_v1_initial_coupled.csv",
    "ml_latest_small_main_comparison_v2_tuned_coupled.csv",
    "ml_latest_small_run_level_metrics_v2_tuned_coupled.csv",
    "ml_latest_small_coupled_nmf_tuning_grid_v2.csv",
    "ml_latest_small_coupled_nmf_selected_config_v2.csv",
    "ml_latest_small_explanation_metrics_extended_v2.csv",
    "ml_latest_small_explanation_fidelity_summary_extended_v2.csv",
    "ml_latest_small_interpretability_tuning_grid_v3.csv",
    "ml_latest_small_interpretability_selected_config_v3.csv",
]

archive_report = []

for filename in archive_csv_files:
    src = ARCHIVE_DIR / filename
    dst = FINAL_ARCHIVE_DIR / filename

    if src.exists():
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    archive_report.append({
        "section": "csv_archive_versions",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

archive_report_df = pd.DataFrame(archive_report)
display(archive_report_df)

In [ ]:
latex_table_files = [
    "table_ml_latest_small_main_comparison_v2.tex",
    "table_ml_latest_small_selected_config_v2.tex",
    "table_ml_latest_small_plain_vs_coupled_delta_v2.tex",
    "table_ml_latest_small_explanation_metrics_v2.tex",
    "table_ml_latest_small_fidelity_distribution_v2.tex",
    "table_ml_latest_small_v3_interpretability_diagnostic.tex",
    "table_ml_latest_small_top_v3_fidelity_configs.tex",
    "table_appendix_baseline_smoke_test.tex",
    "table_appendix_model_development_versions.tex",
    "ml_latest_small_all_manuscript_tables.tex",
    "ml_latest_small_table_include_commands.tex",
]

table_report = []

for filename in latex_table_files:
    candidates = [
        PAPER_TABLE_DIR / filename,
        RESULTS_TABLE_DIR / filename,
    ]

    src = next((p for p in candidates if p.exists()), None)
    dst = FINAL_TABLE_DIR / filename

    if src is not None:
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    table_report.append({
        "section": "latex_tables",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

table_report_df = pd.DataFrame(table_report)
display(table_report_df)

In [ ]:
figure_stems = [
    "fig01_ml_latest_small_overall_benchmark_comparison",
    "fig02_ml_latest_small_plain_vs_coupled_relative_improvement",
    "fig03_ml_latest_small_explanation_fidelity_distribution",
    "fig04_ml_latest_small_interpretability_tradeoff",
    "fig05_ml_latest_small_factor_keyword_map",
    "fig06_ml_latest_small_recommendation_explanation_examples",
    "fig07_ml_latest_small_model_development_versions",
]

figure_files = []
for stem in figure_stems:
    figure_files.append(f"{stem}.png")
    figure_files.append(f"{stem}.pdf")

figure_files.extend([
    "ml_latest_small_figure_include_commands.tex",
    "ml_latest_small_figure_copy_report.csv",
    "README_figure_manifest.txt",
])

figure_report = []
for filename in figure_files:
    candidates = [
        PAPER_FIG_DIR / filename,
        RESULTS_FIG_DIR / filename,
    ]
    src = next((p for p in candidates if p.exists()), None)

    if filename.endswith(".tex"):
        dst = FINAL_INCLUDE_DIR / filename
    elif filename.endswith(".csv") or filename.endswith(".txt"):
        dst = FINAL_REPORT_DIR / filename
    else:
        dst = FINAL_FIGURE_DIR / filename

    if src is not None:
        shutil.copy2(src, dst)
        status = "COPIED"
        size_kb = round(dst.stat().st_size / 1024, 2)
    else:
        status = "MISSING"
        size_kb = None

    figure_report.append({
        "section": "figures_and_figure_includes",
        "filename": filename,
        "status": status,
        "size_kb": size_kb,
    })

figure_report_df = pd.DataFrame(figure_report)
display(figure_report_df)

In [ ]:
include_files = [
    PAPER_TABLE_DIR / "ml_latest_small_table_include_commands.tex",
    RESULTS_TABLE_DIR / "ml_latest_small_table_include_commands.tex",
]

include_report = []

table_include_src = next((p for p in include_files if p.exists()), None)

if table_include_src is not None:
    dst = FINAL_INCLUDE_DIR / "ml_latest_small_table_include_commands.tex"
    shutil.copy2(table_include_src, dst)
    include_report.append({
        "section": "latex_include_commands",
        "filename": "ml_latest_small_table_include_commands.tex",
        "status": "COPIED",
        "size_kb": round(dst.stat().st_size / 1024, 2),
    })
else:
    include_report.append({
        "section": "latex_include_commands",
        "filename": "ml_latest_small_table_include_commands.tex",
        "status": "MISSING",
        "size_kb": None,
    })

include_report_df = pd.DataFrame(include_report)
display(include_report_df)

In [ ]:
full_report = pd.concat(
    [
        csv_report_df,
        archive_report_df,
        table_report_df,
        figure_report_df,
        include_report_df,
    ],
    ignore_index=True,
)

copy_report_path = FINAL_REPORT_DIR / "complete_bundle_copy_report.csv"
full_report.to_csv(copy_report_path, index=False)

missing_files = full_report[full_report["status"] == "MISSING"]

manifest_text = f"""Complete Manuscript Bundle for ml_latest_small
================================================

Project:
xai_coupled_nmf_project

Dataset:
ml_latest_small

Bundle created:
{timestamp}

Bundle purpose:
This bundle preserves the completed MovieLens latest-small stage for the explainable coupled NMF recommender-system manuscript and dissertation workflow.

Included folders:
1. csv_current
   Current experimental CSV outputs.

2. csv_archive_versions
   Versioned CSV outputs:
      - Version 1: initial coupled NMF prototype, if available.
      - Version 2: accuracy-oriented tuned coupled NMF selected by validation NDCG@10.
      - Version 3: interpretability-oriented diagnostic tuning.

3. latex_tables
   Manuscript-ready LaTeX tables and appendix/reproducibility tables.

4. figures
   Manuscript-ready figure files in PNG and PDF formats.

5. latex_include_commands
   Ready-to-use LaTeX input/includegraphics blocks for tables and figures.

6. reports
   Copy reports and figure/table manifests.

Main manuscript model:
Version 2 accuracy-oriented tuned coupled NMF.

Version 3 status:
Diagnostic only. It does not materially improve Fidelity@10 over Version 2 and is not selected as the main model.

Recommended use in manuscript:
- Use Version 2 main comparison, selected-configuration, delta, and explanation tables.
- Use Version 3 only as an interpretability diagnostic/sensitivity result.
- Use appendix tables for dissertation-level reproducibility documentation.

Essential LaTeX include files:
- latex_include_commands/ml_latest_small_table_include_commands.tex
- latex_include_commands/ml_latest_small_figure_include_commands.tex

Missing files:
"""

if missing_files.empty:
    manifest_text += "\nNone.\n"
else:
    for _, row in missing_files.iterrows():
        manifest_text += f"\n- {row['section']}: {row['filename']}"

manifest_path = FINAL_BUNDLE_DIR / "README_complete_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Copy report saved:", copy_report_path)
print("Manifest saved:", manifest_path)

if not missing_files.empty:
    print("\nMissing files:")
    display(missing_files)

In [ ]:
zip_path = shutil.make_archive(
    str(FINAL_BUNDLE_DIR),
    "zip",
    root_dir=FINAL_BUNDLE_DIR,
)

print("Final complete bundle folder:")
print(FINAL_BUNDLE_DIR)

print("\nFinal complete ZIP file:")
print(zip_path)

In [ ]:
for folder in [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]:
    files = list(folder.glob("*"))
    print(f"{folder.name:30s}: {len(files)} files")

## 12. MovieLens 1M preprocessing and smoke testing

Original cell index starts around `275`.

In [ ]:
from pathlib import Path

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

%cd "$PROJECT_ROOT"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

In [ ]:
RAW_ML1M_DIR = PROJECT_ROOT / "data/raw/ml-1m"

print("RAW_ML1M_DIR:", RAW_ML1M_DIR)
print("Exists:", RAW_ML1M_DIR.exists())

!ls -lh "$RAW_ML1M_DIR"

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ratings_path = PROJECT_ROOT / "data/raw/ml-1m/ratings.dat"
movies_path = PROJECT_ROOT / "data/raw/ml-1m/movies.dat"

ratings = pd.read_csv(
    ratings_path,
    sep="::",
    engine="python",
    names=["user_id", "item_id", "rating", "timestamp"],
    encoding="latin-1",
)

movies = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    names=["item_id", "title", "genres"],
    encoding="latin-1",
)

ratings["user_id"] = ratings["user_id"].astype(int)
ratings["item_id"] = ratings["item_id"].astype(int)
ratings["rating"] = ratings["rating"].astype(float)
ratings["timestamp"] = ratings["timestamp"].astype(int)

movies["item_id"] = movies["item_id"].astype(int)

print("Raw ratings shape:", ratings.shape)
print("Raw movies shape:", movies.shape)

display(ratings.head())
display(movies.head())

In [ ]:
def iterative_filter_ratings(df, min_user_ratings=20, min_item_ratings=20):
    current = df.copy()
    while True:
        before_shape = current.shape

        user_counts = current.groupby("user_id")["item_id"].count()
        valid_users = user_counts[user_counts >= min_user_ratings].index
        current = current[current["user_id"].isin(valid_users)]

        item_counts = current.groupby("item_id")["user_id"].count()
        valid_items = item_counts[item_counts >= min_item_ratings].index
        current = current[current["item_id"].isin(valid_items)]

        after_shape = current.shape
        if before_shape == after_shape:
            break

    return current.reset_index(drop=True)

# Filter ratings to ensure density
filtered_ratings = iterative_filter_ratings(
    ratings,
    min_user_ratings=20,
    min_item_ratings=20,
)

valid_items = sorted(filtered_ratings["item_id"].unique())
movies_filtered = movies[movies["item_id"].isin(valid_items)].copy()

all_genres = sorted({
    g.strip()
    for genre_string in movies_filtered["genres"].fillna("")
    for g in genre_string.split("|")
    if g.strip()
})

def clean_genre_name(g):
    return "genre_" + g.lower().replace("-", "_").replace(" ", "_").replace("(", "").replace(")", "").replace("'", "")

descriptor_df = movies_filtered[["item_id"]].copy()
for genre in all_genres:
    col = clean_genre_name(genre)
    descriptor_df[col] = movies_filtered["genres"].fillna("").apply(
        lambda x: 1.0 if genre in x.split("|") else 0.0
    )

metadata_df = movies_filtered[["item_id", "title", "genres"]].copy()

# Save processed outputs
filtered_ratings.to_csv(PROCESSED_DIR / "ml1m_ratings.csv", index=False)
descriptor_df.to_csv(PROCESSED_DIR / "ml1m_item_descriptors.csv", index=False)
metadata_df.to_csv(PROCESSED_DIR / "ml1m_item_metadata.csv", index=False)

print("Filtered ratings shape:", filtered_ratings.shape)
print("Descriptor shape:", descriptor_df.shape)
print("Metadata shape:", metadata_df.shape)

print("\nNumber of users:", filtered_ratings["user_id"].nunique())
print("Number of items:", filtered_ratings["item_id"].nunique())
print("Number of descriptor columns:", descriptor_df.shape[1] - 1)

display(filtered_ratings.head())
display(descriptor_df.head())
display(metadata_df.head())

In [ ]:
!ls -lh data/processed/ml1m_*.csv

In [ ]:
!python src/build_dataset_summary.py \
  --dataset ml1m \
    --input_ratings data/processed/ml1m_ratings.csv \
      --input_descriptors data/processed/ml1m_item_descriptors.csv \
        --output_csv results/csv/ml1m_dataset_summary.csv \
          --log_file results/logs/ml1m_dataset_summary.log \
            --overwrite true

In [ ]:
ml1m_summary = pd.read_csv("results/csv/ml1m_dataset_summary.csv")
display(ml1m_summary)

In [ ]:
!python src/generate_splits.py \
  --dataset ml1m \
    --input_ratings data/processed/ml1m_ratings.csv \
      --split_type random_per_user \
        --seed_list 42,123,2026,7,99 \
          --output_csv results/csv/ml1m_split_registry.csv \
            --log_file results/logs/ml1m_generate_splits.log

In [ ]:
splits = pd.read_csv("results/csv/ml1m_split_registry.csv")

print("Split registry shape:", splits.shape)
display(splits.head())

print("\nSplit counts:")
display(splits["split"].value_counts())

print("\nSeed counts:")
display(splits["seed"].value_counts())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

ratings_ml1m = pd.read_csv("data/processed/ml1m_ratings.csv")
splits_ml1m = pd.read_csv("results/csv/ml1m_split_registry.csv")

seed = 42
split_seed = splits_ml1m[splits_ml1m["seed"] == seed].copy()

train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

train = ratings_ml1m.merge(train_keys, on=["user_id", "item_id"], how="inner")
test = ratings_ml1m.merge(test_keys, on=["user_id", "item_id"], how="inner")

global_mean = train["rating"].mean()
item_means = train.groupby("item_id")["rating"].mean()

test = test.copy()
test["pred_global_mean"] = global_mean
test["pred_item_mean"] = test["item_id"].map(item_means).fillna(global_mean)

def compute_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

rows = []

for model_name, pred_col in [
    ("global_mean", "pred_global_mean"),
    ("item_mean", "pred_item_mean"),
]:
    rmse = compute_rmse(test["rating"], test[pred_col])
    mae = mean_absolute_error(test["rating"], test[pred_col])

    rows.append({
        "dataset_name": "ml1m",
        "model_name": model_name,
        "seed": seed,
        "rmse": rmse,
        "mae": mae,
        "num_train": len(train),
        "num_test": len(test),
    })

smoke_ml1m = pd.DataFrame(rows)
smoke_ml1m.to_csv("results/csv/ml1m_baseline_smoke_test.csv", index=False)

display(smoke_ml1m)
print("Saved: results/csv/ml1m_baseline_smoke_test.csv")

In [ ]:
required_ml1m_files = [
    "data/processed/ml1m_ratings.csv",
    "data/processed/ml1m_item_descriptors.csv",
    "data/processed/ml1m_item_metadata.csv",
    "results/csv/ml1m_dataset_summary.csv",
    "results/csv/ml1m_split_registry.csv",
    "results/csv/ml1m_baseline_smoke_test.csv",
]

for file in required_ml1m_files:
    path = PROJECT_ROOT / file
    print(f"{file:60s} ->", "FOUND" if path.exists() else "MISSING")

## 13. MovieLens 1M seed-42 benchmark and tuning recovery

Original cell index starts around `286`.

In [ ]:
# ============================================================
# Single-seed ml1m benchmark using existing scripts
# Dataset: ml1m
# Seed: 42
# Descriptor source: genres
# ============================================================

from pathlib import Path
import pandas as pd
import subprocess
import sys
import os

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

# ------------------------------------------------------------
# 2. Check required files
# ------------------------------------------------------------

required_files = [
    "data/processed/ml1m_ratings.csv",
    "data/processed/ml1m_item_descriptors.csv",
    "results/csv/ml1m_split_registry.csv",
    "src/run_main_benchmarks.py",
    "src/run_coupled_nmf.py",
    "src/benchmark_utils.py",
    "src/benchmark_models.py",
]

missing = []

for file in required_files:
    path = PROJECT_ROOT / file
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{file:55s} -> {status}")
    if not path.exists():
        missing.append(file)

if missing:
    raise FileNotFoundError("Missing required files: " + ", ".join(missing))

# ------------------------------------------------------------
# 3. Syntax check existing benchmark scripts
# ------------------------------------------------------------

syntax_commands = [
    ["python", "-m", "py_compile", "src/benchmark_utils.py"],
    ["python", "-m", "py_compile", "src/benchmark_models.py"],
    ["python", "-m", "py_compile", "src/run_main_benchmarks.py"],
    ["python", "-m", "py_compile", "src/run_coupled_nmf.py"],
]

print("\nRunning syntax checks...")

for cmd in syntax_commands:
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        print("FAILED:", " ".join(cmd))
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Syntax check failed.")
    else:
        print("PASSED:", " ".join(cmd))

# ------------------------------------------------------------
# 4. Run baseline benchmark: item_mean, biased_mf, plain_nmf
# ------------------------------------------------------------

print("\nRunning ml1m single-seed baseline benchmark...")

baseline_cmd = [
    "python", "src/run_main_benchmarks.py",
    "--dataset", "ml1m",
    "--input_ratings", "data/processed/ml1m_ratings.csv",
    "--input_descriptors", "data/processed/ml1m_item_descriptors.csv",
    "--seed_list", "42",
    "--descriptor_source", "genres",
    "--top_k", "5,10",
    "--relevance_threshold", "4.0",
    "--output_csv", "results/csv/ml1m_main_comparison_seed42.csv",
    "--log_file", "results/logs/ml1m_main_comparison_seed42.log",
    "--overwrite", "true",
]

result = subprocess.run(baseline_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("ml1m baseline benchmark failed.")

print("Baseline benchmark completed.")

# ------------------------------------------------------------
# 5. Run coupled NMF benchmark for seed 42
# ------------------------------------------------------------

print("\nRunning ml1m single-seed coupled NMF benchmark...")

coupled_cmd = [
    "python", "src/run_coupled_nmf.py",
    "--dataset", "ml1m",
    "--input_ratings", "data/processed/ml1m_ratings.csv",
    "--input_descriptors", "data/processed/ml1m_item_descriptors.csv",
    "--seed_list", "42",
    "--descriptor_source", "genres",
    "--top_k", "5,10",
    "--relevance_threshold", "4.0",
    "--output_csv", "results/csv/ml1m_main_comparison_seed42.csv",
    "--log_file", "results/logs/ml1m_run_coupled_nmf_seed42.log",
    "--overwrite", "true",
]

result = subprocess.run(coupled_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("ml1m coupled NMF benchmark failed.")

print("Coupled NMF benchmark completed.")

# ------------------------------------------------------------
# 6. Inspect generated outputs
# ------------------------------------------------------------

print("\nInspecting ml1m single-seed outputs...")

main_path = PROJECT_ROOT / "results/csv/ml1m_main_comparison_seed42.csv"
run_level_path = PROJECT_ROOT / "results/csv/ml1m_run_level_metrics.csv"

if not main_path.exists():
    raise FileNotFoundError(f"Missing output: {main_path}")

if not run_level_path.exists():
    raise FileNotFoundError(f"Missing output: {run_level_path}")

main_results = pd.read_csv(main_path)
run_level = pd.read_csv(run_level_path)

print("\nMain comparison, seed 42:")
display(main_results)

print("\nRun-level metrics:")
display(run_level)

print("\nRun-level shape:", run_level.shape)

print("\nModel names in run-level file:")
print(run_level["model_name"].unique())

# ------------------------------------------------------------
# 7. Check coupled NMF explanation support files
# ------------------------------------------------------------

support_files = [
    "results/csv/ml1m_factor_descriptor_weights.csv",
    "results/csv/ml1m_factor_keywords.csv",
    "results/csv/ml1m_recommendation_traces.csv",
]

print("\nCoupled NMF support files:")

for file in support_files:
    path = PROJECT_ROOT / file
    print(f"{file:55s} ->", "FOUND" if path.exists() else "MISSING")

if (PROJECT_ROOT / "results/csv/ml1m_factor_keywords.csv").exists():
    factor_keywords = pd.read_csv(PROJECT_ROOT / "results/csv/ml1m_factor_keywords.csv")
    print("\nFactor keywords preview:")
    display(factor_keywords.head(15))

# ------------------------------------------------------------
# 8. Save a copy of the single-seed benchmark as a checkpoint
# ------------------------------------------------------------

checkpoint_dir = PROJECT_ROOT / "results/csv/archive/ml1m_seed42_smoke_benchmark"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_files = [
    "results/csv/ml1m_main_comparison_seed42.csv",
    "results/csv/ml1m_run_level_metrics.csv",
    "results/csv/ml1m_factor_descriptor_weights.csv",
    "results/csv/ml1m_factor_keywords.csv",
    "results/csv/ml1m_recommendation_traces.csv",
]

for file in checkpoint_files:
    src = PROJECT_ROOT / file
    if src.exists():
        dst = checkpoint_dir / src.name
        dst.write_bytes(src.read_bytes())
        print("Checkpoint copied:", dst)

print("\nSingle-seed ml1m benchmark completed successfully.")
print("Checkpoint folder:", checkpoint_dir)

In [ ]:
from pathlib import Path

possible_mounts = [
    Path("/content/drive/MyDrive"),
    Path("/content/gdrive/MyDrive"),
]

for p in possible_mounts:
    print(p, "->", "FOUND" if p.exists() else "NOT FOUND")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
CSV_DIR = PROJECT_ROOT / "results/csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("CSV_DIR:", CSV_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())

checkpoint_path = CSV_DIR / "ml1m_coupled_nmf_tuning_grid_seed42.csv"

print("\nCheckpoint path:", checkpoint_path)
print("Checkpoint exists:", checkpoint_path.exists())

if checkpoint_path.exists():
    import pandas as pd

    grid_ckpt = pd.read_csv(checkpoint_path)
    print("Completed configurations:", len(grid_ckpt))
    print("Expected total configurations:", 30)

    display(grid_ckpt.tail(10))

    print("\nCompleted configuration keys:")
    display(
        grid_ckpt[["k", "alpha", "beta", "lambda_reg", "epochs"]]
        .sort_values(["k", "alpha", "beta"])
        .reset_index(drop=True)
    )
else:
    print("Checkpoint not found at the expected path.")

In [ ]:
from pathlib import Path

target_name = "ml1m_coupled_nmf_tuning_grid_seed42.csv"

search_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
]

found_paths = []

for root in search_roots:
    if root.exists():
        print("Searching:", root)
        try:
            matches = list(root.rglob(target_name))
            found_paths.extend(matches)
        except Exception as e:
            print("Could not search:", root, "|", e)

print("\nSearch result:")
if found_paths:
    for p in found_paths:
        print("FOUND:", p)
else:
    print("No checkpoint file found.")

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
CSV_DIR = PROJECT_ROOT / "results/csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)

expected_path = CSV_DIR / "ml1m_coupled_nmf_tuning_grid_seed42.csv"

print("Expected checkpoint path:")
print(expected_path)
print("Exists:", expected_path.exists())

# Search common locations if not found
if not expected_path.exists():
    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]

    found_paths = []
    for root in search_roots:
        if root.exists():
            found_paths.extend(list(root.rglob("ml1m_coupled_nmf_tuning_grid_seed42.csv")))

    print("\nFound paths:")
    for p in found_paths:
        print(p)

    if found_paths:
        shutil.copy(found_paths[0], expected_path)
        print("\nCopied checkpoint to expected path:")
        print(expected_path)
    else:
        print("\nCheckpoint not found in Colab file system. Upload it to Colab Files and rerun this cell.")

# Verify
if expected_path.exists():
    ckpt = pd.read_csv(expected_path)
    print("\nCheckpoint loaded successfully.")
    print("Completed configurations:", len(ckpt))
    print("Expected total configurations:", 30)
    display(ckpt.tail(10))

## 14. MovieLens 1M module-free resume and tuning completion

Original cell index starts around `292`.

In [ ]:
# ============================================================
# MODULE-FREE RESUME CELL
# ml1m coupled-NMF tuning pass, seed 42
# This cell does NOT require src/benchmark_utils.py or src/run_coupled_nmf.py.
# It resumes from:
# results/csv/ml1m_coupled_nmf_tuning_grid_seed42.csv
# ============================================================

from pathlib import Path
from itertools import product
import os
import time
import shutil

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate xai_coupled_nmf_project under /content/drive or /content/gdrive. "
        "Please mount Google Drive and check the project folder path."
    )

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
LOG_DIR = PROJECT_ROOT / "results/logs"

CSV_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("Current working directory:", Path.cwd())

# ------------------------------------------------------------
# 2. Check required data files
# ------------------------------------------------------------

required_files = [
    PROJECT_ROOT / "data/processed/ml1m_ratings.csv",
    PROJECT_ROOT / "data/processed/ml1m_item_descriptors.csv",
    PROJECT_ROOT / "results/csv/ml1m_split_registry.csv",
]

for p in required_files:
    print(f"{str(p):90s} ->", "FOUND" if p.exists() else "MISSING")

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

# ------------------------------------------------------------
# 3. Load data
# ------------------------------------------------------------

dataset = "ml1m"
seed = 42

ratings = pd.read_csv(PROJECT_ROOT / "data/processed/ml1m_ratings.csv")
descriptors = pd.read_csv(PROJECT_ROOT / "data/processed/ml1m_item_descriptors.csv")
split_registry = pd.read_csv(PROJECT_ROOT / "results/csv/ml1m_split_registry.csv")

print("Ratings shape:", ratings.shape)
print("Descriptors shape:", descriptors.shape)
print("Split registry shape:", split_registry.shape)

# ------------------------------------------------------------
# 4. Local utility functions
# ------------------------------------------------------------

def build_id_maps(ratings_df):
    users = sorted(ratings_df["user_id"].unique())
    items = sorted(ratings_df["item_id"].unique())

    user_to_idx = {u: idx for idx, u in enumerate(users)}
    item_to_idx = {i: idx for idx, i in enumerate(items)}

    idx_to_user = {idx: u for u, idx in user_to_idx.items()}
    idx_to_item = {idx: i for i, idx in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


def load_split_data(ratings_df, split_registry_df, seed):
    split_seed = split_registry_df[split_registry_df["seed"] == seed].copy()

    train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
    val_keys = split_seed[split_seed["split"] == "val"][["user_id", "item_id"]]
    test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

    train_df = ratings_df.merge(train_keys, on=["user_id", "item_id"], how="inner")
    val_df = ratings_df.merge(val_keys, on=["user_id", "item_id"], how="inner")
    test_df = ratings_df.merge(test_keys, on=["user_id", "item_id"], how="inner")

    return train_df, val_df, test_df


def build_descriptor_matrix(descriptors_df, item_to_idx):
    feature_cols = [
        c for c in descriptors_df.columns
        if c != "item_id" and pd.api.types.is_numeric_dtype(descriptors_df[c])
    ]

    X = np.zeros((len(item_to_idx), len(feature_cols)), dtype=np.float32)
    desc_indexed = descriptors_df.set_index("item_id")

    for item_id, idx in item_to_idx.items():
        if item_id in desc_indexed.index:
            X[idx, :] = desc_indexed.loc[item_id, feature_cols].to_numpy(dtype=np.float32)

    col_max = np.maximum(X.max(axis=0), 1e-8)
    X = X / col_max

    return X, feature_cols


def train_coupled_nmf(
    train_df,
    descriptors_df,
    user_to_idx,
    item_to_idx,
    k=40,
    epochs=20,
    lr_rating=0.005,
    lr_semantic=0.01,
    alpha=0.01,
    lambda_reg=0.01,
    beta=0.0001,
    seed=42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    X, feature_cols = build_descriptor_matrix(descriptors_df, item_to_idx)
    n_features = X.shape[1]

    U = rng.random((n_users, k), dtype=np.float32) * 0.1
    V = rng.random((n_items, k), dtype=np.float32) * 0.1
    B = rng.random((n_features, k), dtype=np.float32) * 0.1

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train_df.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for epoch in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = float(U[u] @ V[i])
            err = rating - pred

            old_u = U[u].copy()

            U[u] += lr_rating * (err * V[i] - lambda_reg * U[u])
            V[i] += lr_rating * (err * old_u - lambda_reg * V[i])

            U[u] = np.maximum(U[u], 1e-8)
            V[i] = np.maximum(V[i], 1e-8)

        E = X - (V @ B.T)

        grad_V_sem = (-2.0 * alpha / max(n_features, 1)) * (E @ B)
        grad_B_sem = (-2.0 * alpha / max(n_items, 1)) * (E.T @ V)

        grad_V = grad_V_sem + 2.0 * lambda_reg * V
        grad_B = grad_B_sem + 2.0 * lambda_reg * B + beta

        V -= lr_semantic * grad_V
        B -= lr_semantic * grad_B

        V = np.maximum(V, 1e-8)
        B = np.maximum(B, 1e-8)

    return U, V, B, feature_cols


def compute_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def compute_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def lightweight_ranking_metrics(
    U,
    V,
    user_to_idx,
    idx_to_item,
    train_df,
    eval_df,
    top_k_values=(5, 10),
    relevance_threshold=4.0,
):
    max_k = max(top_k_values)
    item_ids = np.array([idx_to_item[i] for i in range(len(idx_to_item))])

    train_items_by_user = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

    relevant_eval = eval_df[eval_df["rating"] >= relevance_threshold]
    relevant_by_user = relevant_eval.groupby("user_id")["item_id"].apply(set).to_dict()

    store = {}
    for k in top_k_values:
        store[f"precision_at_{k}"] = []
        store[f"recall_at_{k}"] = []
        store[f"ndcg_at_{k}"] = []

    for user_id, relevant_items in relevant_by_user.items():
        if user_id not in user_to_idx:
            continue

        u = user_to_idx[user_id]
        scores = V @ U[u]

        seen = train_items_by_user.get(user_id, set())
        candidate_mask = np.array([item_id not in seen for item_id in item_ids])

        if candidate_mask.sum() == 0:
            continue

        candidate_indices = np.where(candidate_mask)[0]
        candidate_scores = scores[candidate_indices]

        if len(candidate_scores) <= max_k:
            order = np.argsort(candidate_scores)[::-1]
        else:
            partial = np.argpartition(candidate_scores, -max_k)[-max_k:]
            order = partial[np.argsort(candidate_scores[partial])[::-1]]

        top_indices = candidate_indices[order]
        top_items_all = item_ids[top_indices].tolist()

        for k in top_k_values:
            top_items = top_items_all[:k]
            hits = [1 if item in relevant_items else 0 for item in top_items]

            precision = sum(hits) / k
            recall = sum(hits) / len(relevant_items)

            dcg = 0.0
            for rank, hit in enumerate(hits, start=1):
                if hit:
                    dcg += 1.0 / np.log2(rank + 1)

            ideal_count = min(len(relevant_items), k)
            idcg = sum(
                1.0 / np.log2(rank + 1)
                for rank in range(1, ideal_count + 1)
            )

            ndcg = dcg / idcg if idcg > 0 else 0.0

            store[f"precision_at_{k}"].append(precision)
            store[f"recall_at_{k}"].append(recall)
            store[f"ndcg_at_{k}"].append(ndcg)

    return {
        key: float(np.mean(values)) if values else 0.0
        for key, values in store.items()
    }


def compute_fidelity_for_top_recs(
    train_df,
    U,
    V,
    user_to_idx,
    idx_to_item,
    max_users=100,
    top_k_recs=10,
):
    item_ids = np.array([idx_to_item[i] for i in range(len(idx_to_item))])
    train_items_by_user = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

    rows = []
    selected_users = list(user_to_idx.items())[:max_users]

    for user_id, u in selected_users:
        seen = train_items_by_user.get(user_id, set())
        scores = V @ U[u]

        candidate_mask = np.array([item_id not in seen for item_id in item_ids])
        candidate_indices = np.where(candidate_mask)[0]

        if len(candidate_indices) == 0:
            continue

        candidate_scores = scores[candidate_indices]

        if len(candidate_scores) <= top_k_recs:
            order = np.argsort(candidate_scores)[::-1]
        else:
            partial = np.argpartition(candidate_scores, -top_k_recs)[-top_k_recs:]
            order = partial[np.argsort(candidate_scores[partial])[::-1]]

        top_indices = candidate_indices[order]

        for rank, i_idx in enumerate(top_indices[:top_k_recs], start=1):
            contrib = U[u] * V[i_idx]
            total = float(np.sum(contrib) + 1e-12)
            sorted_factors = np.argsort(contrib)[::-1]

            rows.append({
                "user_id": user_id,
                "item_id": int(item_ids[i_idx]),
                "rank": rank,
                "fidelity_at_2": float(np.sum(contrib[sorted_factors[:2]]) / total),
                "fidelity_at_3": float(np.sum(contrib[sorted_factors[:3]]) / total),
                "fidelity_at_5": float(np.sum(contrib[sorted_factors[:5]]) / total),
                "fidelity_at_10": float(np.sum(contrib[sorted_factors[:10]]) / total),
            })

    return pd.DataFrame(rows)


def fit_and_evaluate_coupled_config(
    train_df,
    eval_df,
    descriptors_df,
    seed,
    k,
    alpha,
    beta,
    lambda_reg,
    epochs,
    relevance_threshold=4.0,
    top_k_values=(5, 10),
    max_users_for_fidelity=100,
):
    all_data = pd.concat(
        [
            train_df[["user_id", "item_id", "rating"]],
            eval_df[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V, B, feature_cols = train_coupled_nmf(
        train_df=train_df,
        descriptors_df=descriptors_df,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        k=k,
        epochs=epochs,
        alpha=alpha,
        beta=beta,
        lambda_reg=lambda_reg,
        seed=seed,
    )

    global_mean = train_df["rating"].mean()
    preds = []

    for row in eval_df.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    y_true = eval_df["rating"].to_numpy(dtype=float)
    y_pred = np.clip(np.asarray(preds, dtype=float), 0.5, 5.0)

    metrics = {
        "rmse": compute_rmse(y_true, y_pred),
        "mae": compute_mae(y_true, y_pred),
    }

    metrics.update(
        lightweight_ranking_metrics(
            U=U,
            V=V,
            user_to_idx=user_to_idx,
            idx_to_item=idx_to_item,
            train_df=train_df,
            eval_df=eval_df,
            top_k_values=top_k_values,
            relevance_threshold=relevance_threshold,
        )
    )

    fidelity_df = compute_fidelity_for_top_recs(
        train_df=train_df,
        U=U,
        V=V,
        user_to_idx=user_to_idx,
        idx_to_item=idx_to_item,
        max_users=max_users_for_fidelity,
        top_k_recs=10,
    )

    if len(fidelity_df) > 0:
        metrics["fidelity_at_2"] = float(fidelity_df["fidelity_at_2"].mean())
        metrics["fidelity_at_3"] = float(fidelity_df["fidelity_at_3"].mean())
        metrics["fidelity_at_5"] = float(fidelity_df["fidelity_at_5"].mean())
        metrics["fidelity_at_10"] = float(fidelity_df["fidelity_at_10"].mean())
        metrics["num_explanation_instances"] = int(len(fidelity_df))
    else:
        metrics["fidelity_at_2"] = np.nan
        metrics["fidelity_at_3"] = np.nan
        metrics["fidelity_at_5"] = np.nan
        metrics["fidelity_at_10"] = np.nan
        metrics["num_explanation_instances"] = 0

    model_objects = {
        "U": U,
        "V": V,
        "B": B,
        "feature_cols": feature_cols,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user,
        "idx_to_item": idx_to_item,
    }

    return metrics, model_objects


# ------------------------------------------------------------
# 5. Load split
# ------------------------------------------------------------

train, val, test = load_split_data(
    ratings_df=ratings,
    split_registry_df=split_registry,
    seed=seed,
)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

# ------------------------------------------------------------
# 6. Grid and checkpoint setup
# ------------------------------------------------------------

k_values = [30, 40, 60]
alpha_values = [0.0, 0.001, 0.005, 0.01, 0.05]
beta_values = [0.0, 0.0001]
lambda_values = [0.01]
epochs = 20

grid = list(product(k_values, alpha_values, beta_values, lambda_values))

tuning_grid_path = CSV_DIR / "ml1m_coupled_nmf_tuning_grid_seed42.csv"
selected_config_path = CSV_DIR / "ml1m_coupled_nmf_selected_config_seed42.csv"
test_metrics_path = CSV_DIR / "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv"
comparison_after_tuning_path = CSV_DIR / "ml1m_main_comparison_seed42_after_coupled_tuning.csv"

print("Total configurations:", len(grid))
print("Tuning checkpoint path:", tuning_grid_path)


def make_key(k, alpha, beta, lambda_reg, epochs):
    return (
        int(k),
        f"{float(alpha):.8g}",
        f"{float(beta):.8g}",
        f"{float(lambda_reg):.8g}",
        int(epochs),
    )

# ------------------------------------------------------------
# 7. Resume-aware grid loading
# ------------------------------------------------------------

rows = []
completed_keys = set()

if tuning_grid_path.exists():
    try:
        existing_grid = pd.read_csv(tuning_grid_path)
        rows = existing_grid.to_dict("records")

        for _, r in existing_grid.iterrows():
            completed_keys.add(
                make_key(
                    r["k"],
                    r["alpha"],
                    r["beta"],
                    r["lambda_reg"],
                    r["epochs"],
                )
            )

        print("Existing checkpoint found.")
        print("Completed configurations:", len(completed_keys))

    except Exception as e:
        backup_path = tuning_grid_path.with_suffix(".corrupt_backup.csv")
        shutil.copy(tuning_grid_path, backup_path)
        print("Existing checkpoint could not be read.")
        print("Backed up corrupted checkpoint to:", backup_path)
        print("Error:", e)
        rows = []
        completed_keys = set()
else:
    print("No existing checkpoint found. Starting fresh.")

# ------------------------------------------------------------
# 8. Resume tuning grid
# ------------------------------------------------------------

start_time = time.time()

for idx, (k, alpha, beta, lambda_reg) in enumerate(grid, start=1):
    key = make_key(k, alpha, beta, lambda_reg, epochs)

    if key in completed_keys:
        print(
            f"Skipping completed {idx}/{len(grid)}: "
            f"k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}"
        )
        continue

    print("=" * 80)
    print(
        f"Running {idx}/{len(grid)}: "
        f"k={k}, alpha={alpha}, beta={beta}, lambda={lambda_reg}, epochs={epochs}"
    )

    cfg_start = time.time()

    metrics, _ = fit_and_evaluate_coupled_config(
        train_df=train,
        eval_df=val,
        descriptors_df=descriptors,
        seed=seed,
        k=k,
        alpha=alpha,
        beta=beta,
        lambda_reg=lambda_reg,
        epochs=epochs,
        relevance_threshold=4.0,
        top_k_values=(5, 10),
        max_users_for_fidelity=100,
    )

    row = {
        "dataset_name": dataset,
        "seed": seed,
        "k": k,
        "alpha": alpha,
        "beta": beta,
        "lambda_reg": lambda_reg,
        "epochs": epochs,
    }
    row.update(metrics)
    rows.append(row)

    grid_df = pd.DataFrame(rows)
    grid_df.to_csv(tuning_grid_path, index=False)

    completed_keys.add(key)

    elapsed = time.time() - cfg_start
    print(f"Completed in {elapsed/60:.2f} minutes.")
    print("Validation RMSE:", metrics.get("rmse"))
    print("Validation NDCG@10:", metrics.get("ndcg_at_10"))
    print("Fidelity@10:", metrics.get("fidelity_at_10"))
    print("Checkpoint saved:", tuning_grid_path)

total_elapsed = time.time() - start_time

print("=" * 80)
print(f"Grid resume finished in {total_elapsed/60:.2f} minutes.")

grid_df = pd.read_csv(tuning_grid_path)

# ------------------------------------------------------------
# 9. Select best positive-alpha config by validation NDCG@10
# ------------------------------------------------------------

positive_grid = grid_df[grid_df["alpha"] > 0].copy()

if positive_grid.empty:
    raise ValueError("No positive-alpha configurations available.")

selected = positive_grid.sort_values(
    by=["ndcg_at_10", "rmse"],
    ascending=[False, True],
).head(1).copy()

selected.to_csv(selected_config_path, index=False)

print("\nTop 10 validation configs by NDCG@10:")
display(
    grid_df.sort_values(
        by=["ndcg_at_10", "rmse"],
        ascending=[False, True],
    ).head(10)
)

print("\nTop 10 positive-alpha configs by NDCG@10:")
display(
    positive_grid.sort_values(
        by=["ndcg_at_10", "rmse"],
        ascending=[False, True],
    ).head(10)
)

print("\nSelected ml1m coupled-NMF configuration:")
display(selected)

print("Saved selected config:", selected_config_path)

# ------------------------------------------------------------
# 10. Evaluate selected configuration on test split
# ------------------------------------------------------------

best = selected.iloc[0]

best_k = int(best["k"])
best_alpha = float(best["alpha"])
best_beta = float(best["beta"])
best_lambda = float(best["lambda_reg"])
best_epochs = int(best["epochs"])

print("\nEvaluating selected configuration on test split:")
print("k:", best_k)
print("alpha:", best_alpha)
print("beta:", best_beta)
print("lambda_reg:", best_lambda)
print("epochs:", best_epochs)

test_metrics, _ = fit_and_evaluate_coupled_config(
    train_df=train,
    eval_df=test,
    descriptors_df=descriptors,
    seed=seed,
    k=best_k,
    alpha=best_alpha,
    beta=best_beta,
    lambda_reg=best_lambda,
    epochs=best_epochs,
    relevance_threshold=4.0,
    top_k_values=(5, 10),
    max_users_for_fidelity=100,
)

test_row = {
    "dataset_name": dataset,
    "model_name": "coupled_nmf_tuned",
    "seed": seed,
    "k": best_k,
    "alpha": best_alpha,
    "beta": best_beta,
    "lambda_reg": best_lambda,
    "epochs": best_epochs,
}
test_row.update(test_metrics)

test_df = pd.DataFrame([test_row])
test_df.to_csv(test_metrics_path, index=False)

print("\nSelected configuration test metrics:")
display(test_df)

print("Saved test metrics:", test_metrics_path)

# ------------------------------------------------------------
# 11. Create comparison after tuning
# ------------------------------------------------------------

original_comparison_path = CSV_DIR / "ml1m_main_comparison_seed42.csv"

metric_cols = [
    "dataset_name",
    "model_name",
    "rmse",
    "mae",
    "precision_at_5",
    "recall_at_5",
    "ndcg_at_5",
    "precision_at_10",
    "recall_at_10",
    "ndcg_at_10",
]

if original_comparison_path.exists():
    original_comp = pd.read_csv(original_comparison_path)

    original_metric_comp = original_comp[
        [c for c in metric_cols if c in original_comp.columns]
    ].copy()

    tuned_metric_comp = test_df[
        [c for c in metric_cols if c in test_df.columns]
    ].copy()

    comparison_after_tuning = pd.concat(
        [original_metric_comp, tuned_metric_comp],
        ignore_index=True,
    )

    comparison_after_tuning.to_csv(comparison_after_tuning_path, index=False)

    print("\nml1m comparison after coupled tuning:")
    display(comparison_after_tuning)

    print("Saved:", comparison_after_tuning_path)

else:
    print("Original comparison not found:", original_comparison_path)

# ------------------------------------------------------------
# 12. Archive checkpoint files
# ------------------------------------------------------------

checkpoint_dir = ARCHIVE_DIR / "ml1m_seed42_coupled_tuning"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

for src in [
    tuning_grid_path,
    selected_config_path,
    test_metrics_path,
    comparison_after_tuning_path,
]:
    if src.exists():
        dst = checkpoint_dir / src.name
        shutil.copy(src, dst)
        print("Archived:", dst)

print("\nml1m coupled-NMF tuning resume completed.")
print("Tuning grid:", tuning_grid_path)
print("Selected config:", selected_config_path)
print("Test metrics:", test_metrics_path)
print("Comparison after tuning:", comparison_after_tuning_path)
print("Archive folder:", checkpoint_dir)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
CSV_DIR = PROJECT_ROOT / "results/csv"

selected_path = CSV_DIR / "ml1m_coupled_nmf_selected_config_seed42.csv"
test_metrics_path = CSV_DIR / "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv"
comparison_path = CSV_DIR / "ml1m_main_comparison_seed42_after_coupled_tuning.csv"

selected = pd.read_csv(selected_path)
test_metrics = pd.read_csv(test_metrics_path)
comparison = pd.read_csv(comparison_path)

print("Selected ml1m coupled-NMF configuration:")
display(selected)

print("Selected configuration test metrics:")
display(test_metrics)

print("ml1m comparison after coupled tuning:")
display(comparison)

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/ml1m_seed42_secondary_validation"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

files_to_archive = [
    "ml1m_main_comparison_seed42.csv",
    "ml1m_coupled_nmf_tuning_grid_seed42.csv",
    "ml1m_coupled_nmf_selected_config_seed42.csv",
    "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv",
    "ml1m_main_comparison_seed42_after_coupled_tuning.csv",
]

for filename in files_to_archive:
    src = CSV_DIR / filename
    dst = ARCHIVE_DIR / filename

    if src.exists():
        shutil.copy(src, dst)
        print("Archived:", dst)
    else:
        print("Missing:", src)

## 15. MovieLens 1M seed-42 LaTeX tables

Original cell index starts around `295`.

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

input_csv = CSV_DIR / "ml1m_main_comparison_seed42_after_coupled_tuning.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Input CSV:", input_csv)
print("Input exists:", input_csv.exists())

if not input_csv.exists():
    raise FileNotFoundError(f"Missing input CSV: {input_csv}")

# ------------------------------------------------------------
# 2. Load result table
# ------------------------------------------------------------

df = pd.read_csv(input_csv)

display(df)

# ------------------------------------------------------------
# 3. Formatting helpers
# ------------------------------------------------------------

def tex_escape(text):
    text = str(text)
    replacements = {
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"


model_display = {
    "biased_mf": "Biased MF",
    "plain_nmf": "Plain NMF",
    "coupled_nmf": "Coupled NMF",
    "coupled_nmf_tuned": "Tuned Coupled NMF",
    "item_cf": "Item-CF",
    "item_mean": "Item Mean",
}

model_order = [
    "biased_mf",
    "plain_nmf",
    "coupled_nmf",
    "coupled_nmf_tuned",
    "item_cf",
    "item_mean",
]

# ------------------------------------------------------------
# 4. Create LaTeX rows
# ------------------------------------------------------------

rows = []

for model in model_order:
    sub = df[df["model_name"] == model]
    if sub.empty:
        continue

    r = sub.iloc[0]

    rows.append([
        tex_escape(model_display.get(model, model)),
        fmt(r["rmse"]),
        fmt(r["mae"]),
        fmt(r["precision_at_5"]),
        fmt(r["recall_at_5"]),
        fmt(r["ndcg_at_5"]),
        fmt(r["precision_at_10"]),
        fmt(r["recall_at_10"]),
        fmt(r["ndcg_at_10"]),
    ])

# ------------------------------------------------------------
# 5. Write manuscript-ready LaTeX table
# ------------------------------------------------------------

caption = (
    "Secondary validation on MovieLens 1M using seed 42. "
    "The tuned coupled NMF model is selected from the reduced validation grid, "
    "whereas the remaining baselines are evaluated using the same split."
)

label = "tab:ml1m_secondary_validation_seed42"

lines = []

lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{" + caption + r"}")
lines.append(r"\label{" + label + r"}")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{lcccccccc}")
lines.append(r"\toprule")
lines.append(r"Model & RMSE & MAE & P@5 & R@5 & NDCG@5 & P@10 & R@10 & NDCG@10 \\")
lines.append(r"\midrule")

for row in rows:
    lines.append(" & ".join(row) + r" \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"}")
lines.append(r"\vspace{0.35em}")
lines.append(r"\begin{minipage}{0.96\textwidth}")
lines.append(
    r"\footnotesize "
    r"MovieLens 1M is used as a larger secondary validation dataset with genre-only item descriptors. "
    r"The tuned coupled NMF improves substantially over the untuned coupled NMF and remains competitive with plain NMF on short-list ranking metrics, "
    r"although plain NMF remains stronger on RMSE, MAE, and NDCG@10."
)
lines.append(r"\end{minipage}")
lines.append(r"\end{table}")
lines.append("")

latex_content = "\n".join(lines)

paper_path = PAPER_TABLE_DIR / "table_ml1m_secondary_validation_seed42.tex"
results_path = RESULTS_TABLE_DIR / "table_ml1m_secondary_validation_seed42.tex"

paper_path.write_text(latex_content, encoding="utf-8")
results_path.write_text(latex_content, encoding="utf-8")

print("Saved:", paper_path)
print("Copied:", results_path)

# ------------------------------------------------------------
# 6. Also create an include-command snippet
# ------------------------------------------------------------

include_path = PAPER_TABLE_DIR / "ml1m_table_include_commands.tex"

include_content = r"""
% Include this in the manuscript after loading booktabs and graphicx.
% Required packages: \usepackage{booktabs,graphicx}

\input{paper/tables/table_ml1m_secondary_validation_seed42}
""".strip() + "\n"

include_path.write_text(include_content, encoding="utf-8")
shutil.copy(include_path, RESULTS_TABLE_DIR / include_path.name)

print("Saved include file:", include_path)
print("Copied include file:", RESULTS_TABLE_DIR / include_path.name)

# ------------------------------------------------------------
# 7. Preview LaTeX content
# ------------------------------------------------------------

print("\nGenerated LaTeX table:")
print(latex_content)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/ml1m_seed42_secondary_validation"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)
print("PAPER_TABLE_DIR:", PAPER_TABLE_DIR)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]
    for path in candidates:
        if path.exists():
            return path

    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]

    return None


def tex_escape(text):
    text = str(text)
    replacements = {
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"


def write_table(filename, caption, label, colspec, header, rows, resize=False, note=None):
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")

    if resize:
        lines.append(r"\resizebox{\textwidth}{!}{%")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    lines.append(header + r" \\")
    lines.append(r"\midrule")

    for row in rows:
        lines.append(" & ".join(row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")

    if resize:
        lines.append(r"}")

    if note:
        lines.append(r"\vspace{0.35em}")
        lines.append(r"\begin{minipage}{0.96\textwidth}")
        lines.append(r"\footnotesize " + note)
        lines.append(r"\end{minipage}")

    lines.append(r"\end{table}")
    lines.append("")

    content = "\n".join(lines)

    paper_path = PAPER_TABLE_DIR / filename
    results_path = RESULTS_TABLE_DIR / filename

    paper_path.write_text(content, encoding="utf-8")
    results_path.write_text(content, encoding="utf-8")

    print("Saved:", paper_path)
    print("Copied:", results_path)

    return paper_path, content


generated_tables = []

# ------------------------------------------------------------
# 3. Table: ml1m smoke-test sanity check
# ------------------------------------------------------------

smoke_path = find_csv("ml1m_baseline_smoke_test.csv")

if smoke_path is not None:
    smoke = pd.read_csv(smoke_path)

    rows = []
    for _, r in smoke.iterrows():
        rows.append([
            tex_escape(r["model_name"]),
            fmt(r["rmse"]),
            fmt(r["mae"]),
            str(int(r["num_train"])),
            str(int(r["num_test"])),
        ])

    path, content = write_table(
        filename="table_ml1m_baseline_smoke_test_seed42.tex",
        caption="Evaluation-pipeline sanity check on MovieLens 1M using seed 42.",
        label="tab:ml1m_baseline_smoke_test_seed42",
        colspec="lcccc",
        header="Predictor & RMSE & MAE & Train ratings & Test ratings",
        rows=rows,
        resize=False,
        note=(
            "This sanity check verifies the MovieLens 1M split and prediction-evaluation pipeline. "
            "The global-mean and item-mean predictors are not treated as final competing models."
        ),
    )
    generated_tables.append(path.name)
else:
    print("Skipping smoke-test table: ml1m_baseline_smoke_test.csv not found.")

# ------------------------------------------------------------
# 4. Table: selected ml1m coupled-NMF configuration
# ------------------------------------------------------------

selected_path = find_csv("ml1m_coupled_nmf_selected_config_seed42.csv")

if selected_path is not None:
    selected = pd.read_csv(selected_path).iloc[0]

    rows = [[
        fmt(selected["k"], 0),
        fmt(selected["alpha"], 4),
        fmt(selected["beta"], 4),
        fmt(selected["lambda_reg"], 4),
        fmt(selected["epochs"], 0),
        fmt(selected["rmse"]),
        fmt(selected["mae"]),
        fmt(selected["precision_at_10"]),
        fmt(selected["recall_at_10"]),
        fmt(selected["ndcg_at_10"]),
        fmt(selected["fidelity_at_10"]),
    ]]

    path, content = write_table(
        filename="table_ml1m_coupled_selected_config_seed42.tex",
        caption="Selected coupled NMF configuration for MovieLens 1M using seed 42.",
        label="tab:ml1m_coupled_selected_config_seed42",
        colspec="ccccccccccc",
        header=(
            r"$k$ & $\alpha$ & $\beta$ & $\lambda$ & Epochs & RMSE & MAE & "
            r"P@10 & R@10 & NDCG@10 & Fid.@10"
        ),
        rows=rows,
        resize=True,
        note=(
            "The selected configuration is obtained from the reduced validation grid using positive semantic coupling "
            "and validation NDCG@10 as the primary selection criterion."
        ),
    )
    generated_tables.append(path.name)
else:
    print("Skipping selected-config table: ml1m_coupled_nmf_selected_config_seed42.csv not found.")

# ------------------------------------------------------------
# 5. Table: top tuning configurations by validation NDCG@10
# ------------------------------------------------------------

grid_path = find_csv("ml1m_coupled_nmf_tuning_grid_seed42.csv")

if grid_path is not None:
    grid = pd.read_csv(grid_path)

    top_grid = grid.sort_values(
        by=["ndcg_at_10", "rmse"],
        ascending=[False, True],
    ).head(8)

    rows = []
    for _, r in top_grid.iterrows():
        rows.append([
            fmt(r["k"], 0),
            fmt(r["alpha"], 4),
            fmt(r["beta"], 4),
            fmt(r["lambda_reg"], 4),
            fmt(r["rmse"]),
            fmt(r["mae"]),
            fmt(r["precision_at_10"]),
            fmt(r["recall_at_10"]),
            fmt(r["ndcg_at_10"]),
            fmt(r["fidelity_at_10"]),
        ])

    path, content = write_table(
        filename="table_ml1m_top_tuning_configs_seed42.tex",
        caption="Top MovieLens 1M coupled NMF tuning configurations ranked by validation NDCG@10.",
        label="tab:ml1m_top_tuning_configs_seed42",
        colspec="cccccccccc",
        header=(
            r"$k$ & $\alpha$ & $\beta$ & $\lambda$ & RMSE & MAE & "
            r"P@10 & R@10 & NDCG@10 & Fid.@10"
        ),
        rows=rows,
        resize=True,
        note=(
            "This table reports the strongest configurations from the reduced MovieLens 1M tuning grid. "
            "MovieLens 1M uses genre-only descriptors, which are coarser than the tag-and-genre representation used for MovieLens latest-small."
        ),
    )
    generated_tables.append(path.name)
else:
    print("Skipping tuning-grid table: ml1m_coupled_nmf_tuning_grid_seed42.csv not found.")

# ------------------------------------------------------------
# 6. Table: tuned versus untuned coupled NMF delta
# ------------------------------------------------------------

comparison_path = find_csv("ml1m_main_comparison_seed42_after_coupled_tuning.csv")

if comparison_path is not None:
    comparison = pd.read_csv(comparison_path)

    untuned = comparison[comparison["model_name"] == "coupled_nmf"].iloc[0]
    tuned = comparison[comparison["model_name"] == "coupled_nmf_tuned"].iloc[0]

    delta_metrics = [
        ("RMSE", "rmse", "lower"),
        ("MAE", "mae", "lower"),
        ("P@5", "precision_at_5", "higher"),
        ("R@5", "recall_at_5", "higher"),
        ("NDCG@5", "ndcg_at_5", "higher"),
        ("P@10", "precision_at_10", "higher"),
        ("R@10", "recall_at_10", "higher"),
        ("NDCG@10", "ndcg_at_10", "higher"),
    ]

    rows = []
    for display_name, metric, direction in delta_metrics:
        u = float(untuned[metric])
        t = float(tuned[metric])

        if direction == "lower":
            improvement = 100.0 * (u - t) / u
        else:
            improvement = 100.0 * (t - u) / u

        rows.append([
            display_name,
            fmt(u),
            fmt(t),
            fmt(t - u),
            fmt(improvement, 2) + r"\%",
        ])

    path, content = write_table(
        filename="table_ml1m_tuned_vs_untuned_coupled_delta_seed42.tex",
        caption="Effect of coupled NMF tuning on MovieLens 1M using seed 42.",
        label="tab:ml1m_tuned_vs_untuned_coupled_delta_seed42",
        colspec="lcccc",
        header="Metric & Untuned Coupled NMF & Tuned Coupled NMF & Change & Relative improvement",
        rows=rows,
        resize=True,
        note=(
            "Positive relative improvement means that the tuned model improves over the untuned coupled NMF. "
            "For RMSE and MAE, lower values are better; for ranking metrics, higher values are better."
        ),
    )
    generated_tables.append(path.name)
else:
    print("Skipping tuned-vs-untuned delta table: comparison CSV not found.")

# ------------------------------------------------------------
# 7. Table: coupled NMF test explanation metrics
# ------------------------------------------------------------

test_metrics_path = find_csv("ml1m_coupled_nmf_tuned_test_metrics_seed42.csv")

if test_metrics_path is not None:
    test_metrics = pd.read_csv(test_metrics_path).iloc[0]

    rows = [[
        fmt(test_metrics["fidelity_at_2"]),
        fmt(test_metrics["fidelity_at_3"]),
        fmt(test_metrics["fidelity_at_5"]),
        fmt(test_metrics["fidelity_at_10"]),
        fmt(test_metrics["num_explanation_instances"], 0),
        fmt(test_metrics["k"], 0),
        fmt(test_metrics["alpha"], 4),
        fmt(test_metrics["beta"], 4),
    ]]

    path, content = write_table(
        filename="table_ml1m_coupled_test_explanation_metrics_seed42.tex",
        caption="Explanation metrics for the tuned coupled NMF on MovieLens 1M test recommendations.",
        label="tab:ml1m_coupled_test_explanation_metrics_seed42",
        colspec="cccccccc",
        header=r"Fid.@2 & Fid.@3 & Fid.@5 & Fid.@10 & Instances & $k$ & $\alpha$ & $\beta$",
        rows=rows,
        resize=True,
        note=(
            "Fidelity@r measures the proportion of the recommendation score explained by the top r additive latent-factor contributions. "
            "The MovieLens 1M explanations are based on genre-only descriptors."
        ),
    )
    generated_tables.append(path.name)
else:
    print("Skipping explanation metric table: ml1m_coupled_nmf_tuned_test_metrics_seed42.csv not found.")

# ------------------------------------------------------------
# 8. Create combined ml1m seed-42 table file and include commands
# ------------------------------------------------------------

combined_path = PAPER_TABLE_DIR / "ml1m_seed42_all_manuscript_tables.tex"
include_path = PAPER_TABLE_DIR / "ml1m_seed42_table_include_commands.tex"

combined_parts = [
    "% Auto-generated manuscript-ready tables for MovieLens 1M seed-42 secondary validation.",
    "% Required LaTeX packages: booktabs, graphicx.",
    "",
]

include_lines = [
    "% Include these in the manuscript after loading booktabs and graphicx.",
    "% Required packages: \\usepackage{booktabs,graphicx}",
    "",
]

# Include existing secondary validation table if present
existing_secondary = PAPER_TABLE_DIR / "table_ml1m_secondary_validation_seed42.tex"
if existing_secondary.exists():
    all_table_files = ["table_ml1m_secondary_validation_seed42.tex"] + generated_tables
else:
    all_table_files = generated_tables

for filename in all_table_files:
    path = PAPER_TABLE_DIR / filename
    if path.exists():
        combined_parts.append("% ------------------------------------------------------------")
        combined_parts.append(f"% {filename}")
        combined_parts.append("% ------------------------------------------------------------")
        combined_parts.append(path.read_text(encoding="utf-8"))
        include_lines.append(rf"\input{{paper/tables/{Path(filename).stem}}}")

combined_path.write_text("\n".join(combined_parts), encoding="utf-8")
include_path.write_text("\n".join(include_lines), encoding="utf-8")

shutil.copy(combined_path, RESULTS_TABLE_DIR / combined_path.name)
shutil.copy(include_path, RESULTS_TABLE_DIR / include_path.name)

print("\nGenerated ml1m table files:")
for filename in all_table_files:
    print(" -", filename)

print("\nCombined table file:", combined_path)
print("Include command file:", include_path)

print("\nCopied to results/tables as well.")

## 16. MovieLens 1M seed-42 figures

Original cell index starts around `297`.

In [ ]:
# ============================================================
# Manuscript-ready figures for ml1m seed-42 secondary validation
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import textwrap

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/ml1m_seed42_secondary_validation"

PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)
print("PAPER_FIG_DIR:", PAPER_FIG_DIR)
print("RESULTS_FIG_DIR:", RESULTS_FIG_DIR)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not locate {filename}")


def save_figure(fig, stem):
    paper_png = PAPER_FIG_DIR / f"{stem}.png"
    paper_pdf = PAPER_FIG_DIR / f"{stem}.pdf"
    results_png = RESULTS_FIG_DIR / f"{stem}.png"
    results_pdf = RESULTS_FIG_DIR / f"{stem}.pdf"

    fig.savefig(paper_png, dpi=300, bbox_inches="tight")
    fig.savefig(paper_pdf, bbox_inches="tight")

    shutil.copy(paper_png, results_png)
    shutil.copy(paper_pdf, results_pdf)

    print("Saved:", paper_png)
    print("Saved:", paper_pdf)
    print("Copied:", results_png)
    print("Copied:", results_pdf)

    return {
        "stem": stem,
        "paper_png": paper_png,
        "paper_pdf": paper_pdf,
        "results_png": results_png,
        "results_pdf": results_pdf,
    }


def directional_normalize(series, higher_is_better=True):
    series = pd.Series(series, dtype=float)
    smin = series.min()
    smax = series.max()

    if np.isclose(smin, smax):
        return pd.Series(np.ones(len(series)), index=series.index)

    if higher_is_better:
        return (series - smin) / (smax - smin)

    return (smax - series) / (smax - smin)


def fmt_config_label(row):
    return (
        f"k={int(row['k'])}, "
        f"a={float(row['alpha']):.3g}, "
        f"b={float(row['beta']):.3g}"
    )


def wrap_text(s, width=70):
    return "\n".join(textwrap.wrap(str(s), width=width))


# ------------------------------------------------------------
# 3. Load CSV files
# ------------------------------------------------------------

comparison = pd.read_csv(
    find_csv("ml1m_main_comparison_seed42_after_coupled_tuning.csv")
)

tuning_grid = pd.read_csv(
    find_csv("ml1m_coupled_nmf_tuning_grid_seed42.csv")
)

selected_config = pd.read_csv(
    find_csv("ml1m_coupled_nmf_selected_config_seed42.csv")
)

test_metrics = pd.read_csv(
    find_csv("ml1m_coupled_nmf_tuned_test_metrics_seed42.csv")
)

try:
    smoke = pd.read_csv(find_csv("ml1m_baseline_smoke_test.csv"))
except Exception:
    smoke = None

print("Loaded ml1m seed-42 CSVs.")
display(comparison)
display(selected_config)

# ------------------------------------------------------------
# 4. Figure 1: Secondary validation model comparison
# ------------------------------------------------------------

model_display = {
    "biased_mf": "Biased MF",
    "plain_nmf": "Plain NMF",
    "coupled_nmf": "Untuned Coupled NMF",
    "coupled_nmf_tuned": "Tuned Coupled NMF",
    "item_cf": "Item-CF",
}

model_order = [
    "biased_mf",
    "plain_nmf",
    "coupled_nmf",
    "coupled_nmf_tuned",
    "item_cf",
]

metrics_info = [
    ("rmse", False, "RMSE"),
    ("mae", False, "MAE"),
    ("ndcg_at_10", True, "NDCG@10"),
    ("recall_at_10", True, "Recall@10"),
]

plot_df = comparison[comparison["model_name"].isin(model_order)].copy()
plot_df = plot_df.set_index("model_name").loc[
    [m for m in model_order if m in plot_df["model_name"].values]
].reset_index()

norm_data = {}
for metric, higher_better, label in metrics_info:
    norm_data[label] = directional_normalize(
        plot_df[metric],
        higher_is_better=higher_better,
    ).values

x = np.arange(len(metrics_info))
width = 0.16

fig, ax = plt.subplots(figsize=(11, 6))

for i, model in enumerate(plot_df["model_name"].tolist()):
    yvals = [norm_data[label][i] for _, _, label in metrics_info]
    ax.bar(
        x + (i - (len(plot_df) - 1) / 2) * width,
        yvals,
        width=width,
        label=model_display.get(model, model),
    )

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metrics_info])
ax.set_ylabel("Normalized directional score")
ax.set_title("MovieLens 1M seed-42 secondary validation")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig01 = save_figure(fig, "fig_ml1m_seed42_secondary_validation_comparison")
plt.show()

# ------------------------------------------------------------
# 5. Figure 2: Tuned vs untuned coupled NMF improvement
# ------------------------------------------------------------

untuned = comparison[comparison["model_name"] == "coupled_nmf"].iloc[0]
tuned = comparison[comparison["model_name"] == "coupled_nmf_tuned"].iloc[0]

delta_metrics = [
    ("rmse", "RMSE", False),
    ("mae", "MAE", False),
    ("precision_at_5", "P@5", True),
    ("recall_at_5", "R@5", True),
    ("ndcg_at_5", "NDCG@5", True),
    ("precision_at_10", "P@10", True),
    ("recall_at_10", "R@10", True),
    ("ndcg_at_10", "NDCG@10", True),
]

labels = []
improvements = []

for metric, label, higher_better in delta_metrics:
    u = float(untuned[metric])
    t = float(tuned[metric])

    if higher_better:
        improvement = 100.0 * (t - u) / u
    else:
        improvement = 100.0 * (u - t) / u

    labels.append(label)
    improvements.append(improvement)

fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.bar(labels, improvements)

ax.axhline(0, linewidth=1)
ax.set_ylabel("Relative improvement after tuning (%)")
ax.set_title("Effect of tuning on Coupled NMF for MovieLens 1M")
ax.grid(axis="y", alpha=0.3)

for rect, val in zip(bars, improvements):
    va = "bottom" if val >= 0 else "top"
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        val,
        f"{val:.1f}%",
        ha="center",
        va=va,
        fontsize=9,
    )

fig02 = save_figure(fig, "fig_ml1m_seed42_tuned_vs_untuned_coupled_improvement")
plt.show()

# ------------------------------------------------------------
# 6. Figure 3: Tuning grid, NDCG@10 by k and alpha
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

for k_value in sorted(tuning_grid["k"].unique()):
    sub = tuning_grid[tuning_grid["k"] == k_value].copy()

    # Average over beta values for compact visualization.
    avg = (
        sub.groupby("alpha", as_index=False)["ndcg_at_10"]
        .mean()
        .sort_values("alpha")
    )

    ax.plot(
        avg["alpha"],
        avg["ndcg_at_10"],
        marker="o",
        label=f"k={int(k_value)}",
    )

ax.set_xlabel(r"Semantic coupling parameter $\alpha$")
ax.set_ylabel("Validation NDCG@10")
ax.set_title("MovieLens 1M coupled-NMF tuning grid")
ax.legend()
ax.grid(alpha=0.3)

fig03 = save_figure(fig, "fig_ml1m_seed42_tuning_grid_ndcg_by_alpha")
plt.show()

# ------------------------------------------------------------
# 7. Figure 4: Interpretability-performance trade-off
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    tuning_grid["fidelity_at_10"],
    tuning_grid["ndcg_at_10"],
    alpha=0.8,
)

selected = selected_config.iloc[0]

ax.scatter(
    [selected["fidelity_at_10"]],
    [selected["ndcg_at_10"]],
    s=130,
    marker="^",
    label="Selected configuration",
)

ax.annotate(
    "selected",
    (selected["fidelity_at_10"], selected["ndcg_at_10"]),
    xytext=(8, 8),
    textcoords="offset points",
)

ax.set_xlabel("Fidelity@10")
ax.set_ylabel("Validation NDCG@10")
ax.set_title("MovieLens 1M interpretability-performance trade-off")
ax.legend()
ax.grid(alpha=0.3)

fig04 = save_figure(fig, "fig_ml1m_seed42_fidelity_ndcg_tradeoff")
plt.show()

# ------------------------------------------------------------
# 8. Figure 5: Selected tuned model explanation fidelity profile
# ------------------------------------------------------------

tm = test_metrics.iloc[0]

fid_labels = ["Fid.@2", "Fid.@3", "Fid.@5", "Fid.@10"]
fid_values = [
    float(tm["fidelity_at_2"]),
    float(tm["fidelity_at_3"]),
    float(tm["fidelity_at_5"]),
    float(tm["fidelity_at_10"]),
]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(fid_labels, fid_values, marker="o")
ax.set_ylabel("Mean explanation fidelity")
ax.set_title("Explanation fidelity profile of tuned Coupled NMF on MovieLens 1M")
ax.set_ylim(0, max(fid_values) * 1.15)
ax.grid(alpha=0.3)

for x_pos, val in enumerate(fid_values):
    ax.text(x_pos, val, f"{val:.3f}", ha="center", va="bottom")

fig05 = save_figure(fig, "fig_ml1m_seed42_explanation_fidelity_profile")
plt.show()

# ------------------------------------------------------------
# 9. Figure 6: Baseline smoke-test sanity check
# ------------------------------------------------------------

if smoke is not None:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    labels = smoke["model_name"].astype(str).tolist()
    x = np.arange(len(labels))
    width = 0.35

    ax.bar(x - width / 2, smoke["rmse"], width=width, label="RMSE")
    ax.bar(x + width / 2, smoke["mae"], width=width, label="MAE")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Error")
    ax.set_title("MovieLens 1M smoke-test sanity check")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    fig06 = save_figure(fig, "fig_ml1m_seed42_baseline_smoke_test")
    plt.show()
else:
    print("Smoke-test CSV not found. Skipping Figure 6.")

# ------------------------------------------------------------
# 10. Figure 7: Top tuning configurations by validation NDCG@10
# ------------------------------------------------------------

top_configs = tuning_grid.sort_values(
    by=["ndcg_at_10", "rmse"],
    ascending=[False, True],
).head(8).copy()

top_configs["label"] = top_configs.apply(fmt_config_label, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

y = np.arange(len(top_configs))
ax.barh(y, top_configs["ndcg_at_10"])
ax.set_yticks(y)
ax.set_yticklabels(top_configs["label"])
ax.invert_yaxis()
ax.set_xlabel("Validation NDCG@10")
ax.set_title("Top MovieLens 1M coupled-NMF tuning configurations")
ax.grid(axis="x", alpha=0.3)

for y_pos, val in zip(y, top_configs["ndcg_at_10"]):
    ax.text(val, y_pos, f"{val:.4f}", va="center", ha="left", fontsize=9)

fig07 = save_figure(fig, "fig_ml1m_seed42_top_tuning_configurations")
plt.show()

# ------------------------------------------------------------
# 11. Create figure include-command file
# ------------------------------------------------------------

figure_stems = [
    "fig_ml1m_seed42_secondary_validation_comparison",
    "fig_ml1m_seed42_tuned_vs_untuned_coupled_improvement",
    "fig_ml1m_seed42_tuning_grid_ndcg_by_alpha",
    "fig_ml1m_seed42_fidelity_ndcg_tradeoff",
    "fig_ml1m_seed42_explanation_fidelity_profile",
    "fig_ml1m_seed42_baseline_smoke_test",
    "fig_ml1m_seed42_top_tuning_configurations",
]

captions = {
    "fig_ml1m_seed42_secondary_validation_comparison":
        "Secondary validation comparison on MovieLens 1M using seed 42.",
    "fig_ml1m_seed42_tuned_vs_untuned_coupled_improvement":
        "Relative improvement of tuned Coupled NMF over untuned Coupled NMF on MovieLens 1M.",
    "fig_ml1m_seed42_tuning_grid_ndcg_by_alpha":
        "Validation NDCG@10 across the reduced MovieLens 1M coupled-NMF tuning grid.",
    "fig_ml1m_seed42_fidelity_ndcg_tradeoff":
        "Interpretability-performance trade-off for MovieLens 1M coupled-NMF configurations.",
    "fig_ml1m_seed42_explanation_fidelity_profile":
        "Explanation fidelity profile of the selected tuned Coupled NMF on MovieLens 1M.",
    "fig_ml1m_seed42_baseline_smoke_test":
        "Smoke-test sanity check for the MovieLens 1M evaluation pipeline.",
    "fig_ml1m_seed42_top_tuning_configurations":
        "Top MovieLens 1M coupled-NMF configurations ranked by validation NDCG@10.",
}

include_lines = [
    "% Include these figure environments in the manuscript as needed.",
    "% Required package: graphicx",
    "",
]

for stem in figure_stems:
    pdf_path = PAPER_FIG_DIR / f"{stem}.pdf"

    if not pdf_path.exists():
        continue

    label = "fig:" + stem.replace("fig_", "")

    include_lines.extend([
        r"\begin{figure}[htbp]",
        r"\centering",
        rf"\includegraphics[width=0.9\textwidth]{{paper/figures/{stem}.pdf}}",
        rf"\caption{{{captions[stem]}}}",
        rf"\label{{{label}}}",
        r"\end{figure}",
        "",
    ])

include_path = PAPER_FIG_DIR / "ml1m_seed42_figure_include_commands.tex"
include_path.write_text("\n".join(include_lines), encoding="utf-8")
shutil.copy(include_path, RESULTS_FIG_DIR / include_path.name)

print("Saved include-command file:", include_path)
print("Copied include-command file:", RESULTS_FIG_DIR / include_path.name)

# ------------------------------------------------------------
# 12. Create figure copy report and manifest
# ------------------------------------------------------------

report_rows = []

for stem in figure_stems:
    for ext in ["png", "pdf"]:
        p = PAPER_FIG_DIR / f"{stem}.{ext}"
        report_rows.append({
            "filename": p.name,
            "exists": p.exists(),
            "size_kb": round(p.stat().st_size / 1024, 2) if p.exists() else None,
        })

report_df = pd.DataFrame(report_rows)
report_path = PAPER_FIG_DIR / "ml1m_seed42_figure_copy_report.csv"
report_df.to_csv(report_path, index=False)
shutil.copy(report_path, RESULTS_FIG_DIR / report_path.name)

manifest_text = """Manuscript-ready figures for MovieLens 1M seed-42 secondary validation

Figures included:
1. Secondary validation model comparison
2. Tuned versus untuned Coupled NMF improvement
3. Validation NDCG@10 across alpha and k
4. Fidelity@10 versus NDCG@10 trade-off
5. Explanation fidelity profile
6. Baseline smoke-test sanity check
7. Top tuning configurations by validation NDCG@10

Recommended use:
- Journal paper: include Figures 1, 2, and optionally 4 or 5.
- Dissertation: include all seven figures.
"""

manifest_path = PAPER_FIG_DIR / "README_ml1m_seed42_figure_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")
shutil.copy(manifest_path, RESULTS_FIG_DIR / manifest_path.name)

display(report_df)

print("Saved report:", report_path)
print("Saved manifest:", manifest_path)

# ------------------------------------------------------------
# 13. Final verification
# ------------------------------------------------------------

print("\nGenerated ml1m seed-42 figure files in paper/figures:")
for p in sorted(PAPER_FIG_DIR.glob("fig_ml1m_seed42_*")):
    print(p.name)

print("\nFigure include commands:")
print(include_path)

## 17. MovieLens 1M seed-42 secondary-validation bundle

Original cell index starts around `298`.

In [ ]:
# ============================================================
# Final ml1m seed-42 secondary-validation bundle
# Includes:
# - CSVs
# - archived CSVs
# - LaTeX tables
# - manuscript figures
# - include-command files
# - copy reports
# - manifest
# - ZIP file
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/ml1m_seed42_secondary_validation"

PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

EXPORT_ROOT = PROJECT_ROOT / "results/export_bundles"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

FINAL_BUNDLE_DIR = EXPORT_ROOT / f"ml1m_seed42_secondary_validation_bundle_{timestamp}"

FINAL_CSV_DIR = FINAL_BUNDLE_DIR / "csv_current"
FINAL_ARCHIVE_DIR = FINAL_BUNDLE_DIR / "csv_archive"
FINAL_TABLE_DIR = FINAL_BUNDLE_DIR / "latex_tables"
FINAL_FIGURE_DIR = FINAL_BUNDLE_DIR / "figures"
FINAL_INCLUDE_DIR = FINAL_BUNDLE_DIR / "latex_include_commands"
FINAL_REPORT_DIR = FINAL_BUNDLE_DIR / "reports"

for folder in [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_BUNDLE_DIR:", FINAL_BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())
print("ARCHIVE_DIR exists:", ARCHIVE_DIR.exists())
print("PAPER_TABLE_DIR exists:", PAPER_TABLE_DIR.exists())
print("PAPER_FIG_DIR exists:", PAPER_FIG_DIR.exists())


# ------------------------------------------------------------
# 2. Helper copy function
# ------------------------------------------------------------

def copy_if_exists(src_candidates, dst_dir, filename, section):
    src = None

    for candidate in src_candidates:
        if candidate.exists():
            src = candidate
            break

    dst = dst_dir / filename

    if src is not None:
        shutil.copy(src, dst)
        return {
            "section": section,
            "filename": filename,
            "status": "COPIED",
            "size_kb": round(dst.stat().st_size / 1024, 2),
            "source": str(src),
        }

    return {
        "section": section,
        "filename": filename,
        "status": "MISSING",
        "size_kb": None,
        "source": "",
    }


copy_reports = []


# ------------------------------------------------------------
# 3. Copy current ml1m seed-42 CSV outputs
# ------------------------------------------------------------

current_csv_files = [
    # Processed dataset and split outputs
    "ml1m_dataset_summary.csv",
    "ml1m_split_registry.csv",
    "ml1m_baseline_smoke_test.csv",

    # Initial single-seed benchmark
    "ml1m_main_comparison_seed42.csv",
    "ml1m_run_level_metrics.csv",

    # Coupled NMF tuning outputs
    "ml1m_coupled_nmf_tuning_grid_seed42.csv",
    "ml1m_coupled_nmf_selected_config_seed42.csv",
    "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv",
    "ml1m_main_comparison_seed42_after_coupled_tuning.csv",

    # Coupled NMF support files, if generated
    "ml1m_factor_descriptor_weights.csv",
    "ml1m_factor_keywords.csv",
    "ml1m_recommendation_traces.csv",
]

for filename in current_csv_files:
    report = copy_if_exists(
        src_candidates=[CSV_DIR / filename],
        dst_dir=FINAL_CSV_DIR,
        filename=filename,
        section="csv_current",
    )
    copy_reports.append(report)

csv_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "csv_current"])
display(csv_report_df)


# ------------------------------------------------------------
# 4. Copy archived ml1m seed-42 CSV outputs
# ------------------------------------------------------------

archive_csv_files = [
    "ml1m_main_comparison_seed42.csv",
    "ml1m_coupled_nmf_tuning_grid_seed42.csv",
    "ml1m_coupled_nmf_selected_config_seed42.csv",
    "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv",
    "ml1m_main_comparison_seed42_after_coupled_tuning.csv",
]

for filename in archive_csv_files:
    report = copy_if_exists(
        src_candidates=[
            ARCHIVE_DIR / filename,
            CSV_DIR / filename,
        ],
        dst_dir=FINAL_ARCHIVE_DIR,
        filename=filename,
        section="csv_archive",
    )
    copy_reports.append(report)

archive_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "csv_archive"])
display(archive_report_df)


# ------------------------------------------------------------
# 5. Copy ml1m LaTeX tables
# ------------------------------------------------------------

latex_table_files = [
    # Main secondary validation table
    "table_ml1m_secondary_validation_seed42.tex",

    # Supporting seed-42 tables
    "table_ml1m_baseline_smoke_test_seed42.tex",
    "table_ml1m_coupled_selected_config_seed42.tex",
    "table_ml1m_top_tuning_configs_seed42.tex",
    "table_ml1m_tuned_vs_untuned_coupled_delta_seed42.tex",
    "table_ml1m_coupled_test_explanation_metrics_seed42.tex",

    # Combined and include files
    "ml1m_seed42_all_manuscript_tables.tex",
    "ml1m_seed42_table_include_commands.tex",
    "ml1m_table_include_commands.tex",
]

for filename in latex_table_files:
    if "include_commands" in filename:
        target_dir = FINAL_INCLUDE_DIR
        section = "latex_include_commands"
    else:
        target_dir = FINAL_TABLE_DIR
        section = "latex_tables"

    report = copy_if_exists(
        src_candidates=[
            PAPER_TABLE_DIR / filename,
            RESULTS_TABLE_DIR / filename,
        ],
        dst_dir=target_dir,
        filename=filename,
        section=section,
    )
    copy_reports.append(report)

table_report_df = pd.DataFrame([
    r for r in copy_reports
    if r["section"] in ["latex_tables", "latex_include_commands"]
])
display(table_report_df)


# ------------------------------------------------------------
# 6. Copy ml1m figures and figure include-command files
# ------------------------------------------------------------

figure_stems = [
    "fig_ml1m_seed42_secondary_validation_comparison",
    "fig_ml1m_seed42_tuned_vs_untuned_coupled_improvement",
    "fig_ml1m_seed42_tuning_grid_ndcg_by_alpha",
    "fig_ml1m_seed42_fidelity_ndcg_tradeoff",
    "fig_ml1m_seed42_explanation_fidelity_profile",
    "fig_ml1m_seed42_baseline_smoke_test",
    "fig_ml1m_seed42_top_tuning_configurations",
]

figure_files = []

for stem in figure_stems:
    figure_files.append(f"{stem}.png")
    figure_files.append(f"{stem}.pdf")

figure_files.extend([
    "ml1m_seed42_figure_include_commands.tex",
    "ml1m_seed42_figure_copy_report.csv",
    "README_ml1m_seed42_figure_manifest.txt",
])

for filename in figure_files:
    if filename.endswith(".tex"):
        target_dir = FINAL_INCLUDE_DIR
        section = "latex_include_commands"
    elif filename.endswith(".csv") or filename.endswith(".txt"):
        target_dir = FINAL_REPORT_DIR
        section = "reports"
    else:
        target_dir = FINAL_FIGURE_DIR
        section = "figures"

    report = copy_if_exists(
        src_candidates=[
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
        ],
        dst_dir=target_dir,
        filename=filename,
        section=section,
    )
    copy_reports.append(report)

figure_report_df = pd.DataFrame([
    r for r in copy_reports
    if r["section"] in ["figures", "reports", "latex_include_commands"]
])
display(figure_report_df)


# ------------------------------------------------------------
# 7. Create unified copy report
# ------------------------------------------------------------

full_report = pd.DataFrame(copy_reports)

copy_report_path = FINAL_REPORT_DIR / "ml1m_seed42_bundle_copy_report.csv"
full_report.to_csv(copy_report_path, index=False)

missing_files = full_report[full_report["status"] == "MISSING"].copy()

print("Copy report saved:", copy_report_path)

print("\nMissing files:")
display(missing_files)


# ------------------------------------------------------------
# 8. Create manifest
# ------------------------------------------------------------

manifest_text = f"""Final Bundle: MovieLens 1M Seed-42 Secondary Validation
======================================================

Project:
xai_coupled_nmf_project

Dataset:
MovieLens 1M

Experiment stage:
Secondary validation using seed 42.

Bundle created:
{timestamp}

Bundle purpose:
This bundle preserves the completed MovieLens 1M seed-42 secondary-validation stage for the coupled NMF explainable recommendation manuscript and dissertation workflow.

Included folders:
1. csv_current
   Current CSV outputs from preprocessing, smoke testing, single-seed benchmark, and coupled-NMF tuning.

2. csv_archive
   Archived copies of the seed-42 secondary-validation CSV outputs.

3. latex_tables
   Manuscript-ready LaTeX tables for MovieLens 1M seed-42.

4. figures
   Manuscript-ready figures in PNG and PDF formats.

5. latex_include_commands
   Ready-to-use LaTeX include/input blocks for tables and figures.

6. reports
   Copy reports and figure/table manifests.

Main result interpretation:
MovieLens 1M is used as a larger secondary validation dataset with genre-only item descriptors.
The tuned coupled NMF improves substantially over the untuned coupled NMF.
It remains competitive with plain NMF on short-list ranking metrics, especially P@5 and NDCG@5, but plain NMF remains stronger on RMSE, MAE, and NDCG@10.

Selected tuned coupled NMF configuration:
See csv_current/ml1m_coupled_nmf_selected_config_seed42.csv.

Main comparison file:
csv_current/ml1m_main_comparison_seed42_after_coupled_tuning.csv.

Recommended journal use:
- table_ml1m_secondary_validation_seed42.tex
- table_ml1m_tuned_vs_untuned_coupled_delta_seed42.tex
- fig_ml1m_seed42_secondary_validation_comparison
- fig_ml1m_seed42_tuned_vs_untuned_coupled_improvement
- optionally fig_ml1m_seed42_fidelity_ndcg_tradeoff

Recommended dissertation use:
Use all tables and all seven figures.

Missing files:
"""

if missing_files.empty:
    manifest_text += "\nNone.\n"
else:
    for _, row in missing_files.iterrows():
        manifest_text += f"\n- {row['section']}: {row['filename']}"

manifest_path = FINAL_BUNDLE_DIR / "README_ml1m_seed42_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Manifest saved:", manifest_path)


# ------------------------------------------------------------
# 9. Create ZIP file
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(FINAL_BUNDLE_DIR),
    "zip",
    root_dir=FINAL_BUNDLE_DIR,
)

print("\nFinal ml1m seed-42 bundle folder:")
print(FINAL_BUNDLE_DIR)

print("\nFinal ml1m seed-42 ZIP file:")
print(zip_path)


# ------------------------------------------------------------
# 10. Verify bundle contents
# ------------------------------------------------------------

print("\nBundle folder counts:")

for folder in [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]:
    files = list(folder.glob("*"))
    print(f"{folder.name:28s}: {len(files)} files")

print("\nZIP file size:")
zip_file = Path(zip_path)
print(zip_file, "->", round(zip_file.stat().st_size / 1024 / 1024, 2), "MB")

print("\nFinal verification list:")
for p in sorted(FINAL_BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(FINAL_BUNDLE_DIR))

## 18. MovieLens 1M three-seed final evaluation

Original cell index starts around `299`.

In [ ]:
# ============================================================
# ml1m final 3-seed evaluation for journal manuscript
# Models:
#   biased_mf
#   plain_nmf
#   coupled_nmf_tuned
#
# Seeds:
#   42, 123, 2026
#
# Output:
#   results/csv/ml1m_3seed_run_level_metrics.csv
#   results/csv/ml1m_3seed_main_comparison.csv
# ============================================================

from pathlib import Path
import os
import time
import shutil
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])
CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/ml1m_3seed_final_evaluation"
LOG_DIR = PROJECT_ROOT / "results/logs"

CSV_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("ARCHIVE_DIR:", ARCHIVE_DIR)

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

ratings = pd.read_csv(PROJECT_ROOT / "data/processed/ml1m_ratings.csv")
descriptors = pd.read_csv(PROJECT_ROOT / "data/processed/ml1m_item_descriptors.csv")
split_registry = pd.read_csv(PROJECT_ROOT / "results/csv/ml1m_split_registry.csv")

print("Ratings shape:", ratings.shape)
print("Descriptors shape:", descriptors.shape)
print("Split registry shape:", split_registry.shape)

# ------------------------------------------------------------
# 3. Utility functions
# ------------------------------------------------------------

def build_id_maps(ratings_df):
    users = sorted(ratings_df["user_id"].unique())
    items = sorted(ratings_df["item_id"].unique())

    user_to_idx = {u: idx for idx, u in enumerate(users)}
    item_to_idx = {i: idx for idx, i in enumerate(items)}

    idx_to_user = {idx: u for u, idx in user_to_idx.items()}
    idx_to_item = {idx: i for i, idx in item_to_idx.items()}

    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


def load_split_data(ratings_df, split_registry_df, seed):
    split_seed = split_registry_df[split_registry_df["seed"] == seed].copy()

    train_keys = split_seed[split_seed["split"] == "train"][["user_id", "item_id"]]
    val_keys = split_seed[split_seed["split"] == "val"][["user_id", "item_id"]]
    test_keys = split_seed[split_seed["split"] == "test"][["user_id", "item_id"]]

    train_df = ratings_df.merge(train_keys, on=["user_id", "item_id"], how="inner")
    val_df = ratings_df.merge(val_keys, on=["user_id", "item_id"], how="inner")
    test_df = ratings_df.merge(test_keys, on=["user_id", "item_id"], how="inner")

    return train_df, val_df, test_df


def compute_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def compute_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def prediction_metrics(test_df, preds):
    y_true = test_df["rating"].to_numpy(dtype=float)
    y_pred = np.clip(np.asarray(preds, dtype=float), 0.5, 5.0)
    return compute_rmse(y_true, y_pred), compute_mae(y_true, y_pred)


def lightweight_ranking_metrics(
    scores_by_user,
    train_df,
    test_df,
    all_items,
    top_k_values=(5, 10),
    relevance_threshold=4.0,
):
    max_k = max(top_k_values)

    train_items_by_user = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

    relevant_test = test_df[test_df["rating"] >= relevance_threshold]
    relevant_by_user = relevant_test.groupby("user_id")["item_id"].apply(set).to_dict()

    store = {}

    for k in top_k_values:
        store[f"precision_at_{k}"] = []
        store[f"recall_at_{k}"] = []
        store[f"ndcg_at_{k}"] = []

    all_items_array = np.array(all_items)

    for user_id, relevant_items in relevant_by_user.items():
        if user_id not in scores_by_user:
            continue

        user_scores_dict = scores_by_user[user_id]
        seen = train_items_by_user.get(user_id, set())

        candidate_items = [item for item in all_items if item not in seen]

        if not candidate_items:
            continue

        candidate_scores = np.array(
            [user_scores_dict.get(item, -np.inf) for item in candidate_items],
            dtype=float,
        )

        if len(candidate_scores) <= max_k:
            order = np.argsort(candidate_scores)[::-1]
        else:
            partial = np.argpartition(candidate_scores, -max_k)[-max_k:]
            order = partial[np.argsort(candidate_scores[partial])[::-1]]

        top_items_all = [candidate_items[i] for i in order[:max_k]]

        for k in top_k_values:
            top_items = top_items_all[:k]
            hits = [1 if item in relevant_items else 0 for item in top_items]

            precision = sum(hits) / k
            recall = sum(hits) / len(relevant_items)

            dcg = 0.0
            for rank, hit in enumerate(hits, start=1):
                if hit:
                    dcg += 1.0 / np.log2(rank + 1)

            ideal_count = min(len(relevant_items), k)
            idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_count + 1))

            ndcg = dcg / idcg if idcg > 0 else 0.0

            store[f"precision_at_{k}"].append(precision)
            store[f"recall_at_{k}"].append(recall)
            store[f"ndcg_at_{k}"].append(ndcg)

    return {
        key: float(np.mean(values)) if values else 0.0
        for key, values in store.items()
    }


# ------------------------------------------------------------
# 4. Biased MF
# ------------------------------------------------------------

def train_biased_mf(
    train_df,
    user_to_idx,
    item_to_idx,
    k=30,
    epochs=25,
    lr=0.01,
    reg=0.05,
    seed=42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    mu = float(train_df["rating"].mean())
    bu = np.zeros(n_users, dtype=np.float32)
    bi = np.zeros(n_items, dtype=np.float32)

    P = 0.05 * rng.standard_normal((n_users, k)).astype(np.float32)
    Q = 0.05 * rng.standard_normal((n_items, k)).astype(np.float32)

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train_df.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for epoch in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = mu + bu[u] + bi[i] + float(P[u] @ Q[i])
            err = rating - pred

            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])

            old_p = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * old_p - reg * Q[i])

    return mu, bu, bi, P, Q


def evaluate_biased_mf(train_df, test_df, seed):
    all_data = pd.concat(
        [
            train_df[["user_id", "item_id", "rating"]],
            test_df[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    mu, bu, bi, P, Q = train_biased_mf(
        train_df=train_df,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        seed=seed,
    )

    preds = []

    for row in test_df.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(mu)
        else:
            preds.append(mu + bu[u] + bi[i] + float(P[u] @ Q[i]))

    scores_by_user = {}

    for user_id, u in user_to_idx.items():
        scores = mu + bu[u] + bi + (Q @ P[u])
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    rmse, mae = prediction_metrics(test_df, preds)

    rank_metrics = lightweight_ranking_metrics(
        scores_by_user=scores_by_user,
        train_df=train_df,
        test_df=test_df,
        all_items=sorted(all_data["item_id"].unique()),
        top_k_values=(5, 10),
        relevance_threshold=4.0,
    )

    out = {"rmse": rmse, "mae": mae}
    out.update(rank_metrics)
    return out


# ------------------------------------------------------------
# 5. Plain NMF
# ------------------------------------------------------------

def train_plain_nmf(
    train_df,
    user_to_idx,
    item_to_idx,
    k=30,
    epochs=35,
    lr=0.005,
    reg=0.03,
    seed=42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    U = rng.random((n_users, k), dtype=np.float32) * 0.1
    V = rng.random((n_items, k), dtype=np.float32) * 0.1

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train_df.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for epoch in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = float(U[u] @ V[i])
            err = rating - pred

            old_u = U[u].copy()

            U[u] += lr * (err * V[i] - reg * U[u])
            V[i] += lr * (err * old_u - reg * V[i])

            U[u] = np.maximum(U[u], 1e-8)
            V[i] = np.maximum(V[i], 1e-8)

    return U, V


def evaluate_plain_nmf(train_df, test_df, seed):
    all_data = pd.concat(
        [
            train_df[["user_id", "item_id", "rating"]],
            test_df[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V = train_plain_nmf(
        train_df=train_df,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        seed=seed,
    )

    global_mean = train_df["rating"].mean()
    preds = []

    for row in test_df.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    scores_by_user = {}

    for user_id, u in user_to_idx.items():
        scores = V @ U[u]
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    rmse, mae = prediction_metrics(test_df, preds)

    rank_metrics = lightweight_ranking_metrics(
        scores_by_user=scores_by_user,
        train_df=train_df,
        test_df=test_df,
        all_items=sorted(all_data["item_id"].unique()),
        top_k_values=(5, 10),
        relevance_threshold=4.0,
    )

    out = {"rmse": rmse, "mae": mae}
    out.update(rank_metrics)
    return out


# ------------------------------------------------------------
# 6. Tuned Coupled NMF
# ------------------------------------------------------------

def build_descriptor_matrix(descriptors_df, item_to_idx):
    feature_cols = [
        c for c in descriptors_df.columns
        if c != "item_id" and pd.api.types.is_numeric_dtype(descriptors_df[c])
    ]

    X = np.zeros((len(item_to_idx), len(feature_cols)), dtype=np.float32)
    desc_indexed = descriptors_df.set_index("item_id")

    for item_id, idx in item_to_idx.items():
        if item_id in desc_indexed.index:
            X[idx, :] = desc_indexed.loc[item_id, feature_cols].to_numpy(dtype=np.float32)

    col_max = np.maximum(X.max(axis=0), 1e-8)
    X = X / col_max

    return X, feature_cols


def train_coupled_nmf(
    train_df,
    descriptors_df,
    user_to_idx,
    item_to_idx,
    k=60,
    epochs=20,
    lr_rating=0.005,
    lr_semantic=0.01,
    alpha=0.05,
    lambda_reg=0.01,
    beta=0.0001,
    seed=42,
):
    rng = np.random.default_rng(seed)

    n_users = len(user_to_idx)
    n_items = len(item_to_idx)

    X, feature_cols = build_descriptor_matrix(descriptors_df, item_to_idx)
    n_features = X.shape[1]

    U = rng.random((n_users, k), dtype=np.float32) * 0.1
    V = rng.random((n_items, k), dtype=np.float32) * 0.1
    B = rng.random((n_features, k), dtype=np.float32) * 0.1

    triples = [
        (user_to_idx[r.user_id], item_to_idx[r.item_id], float(r.rating))
        for r in train_df.itertuples(index=False)
        if r.user_id in user_to_idx and r.item_id in item_to_idx
    ]

    for epoch in range(epochs):
        rng.shuffle(triples)

        for u, i, rating in triples:
            pred = float(U[u] @ V[i])
            err = rating - pred

            old_u = U[u].copy()

            U[u] += lr_rating * (err * V[i] - lambda_reg * U[u])
            V[i] += lr_rating * (err * old_u - lambda_reg * V[i])

            U[u] = np.maximum(U[u], 1e-8)
            V[i] = np.maximum(V[i], 1e-8)

        E = X - (V @ B.T)

        grad_V_sem = (-2.0 * alpha / max(n_features, 1)) * (E @ B)
        grad_B_sem = (-2.0 * alpha / max(n_items, 1)) * (E.T @ V)

        grad_V = grad_V_sem + 2.0 * lambda_reg * V
        grad_B = grad_B_sem + 2.0 * lambda_reg * B + beta

        V -= lr_semantic * grad_V
        B -= lr_semantic * grad_B

        V = np.maximum(V, 1e-8)
        B = np.maximum(B, 1e-8)

    return U, V, B, feature_cols


def compute_fidelity_for_top_recs(
    train_df,
    U,
    V,
    user_to_idx,
    idx_to_item,
    max_users=100,
    top_k_recs=10,
):
    item_ids = np.array([idx_to_item[i] for i in range(len(idx_to_item))])
    train_items_by_user = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

    rows = []
    selected_users = list(user_to_idx.items())[:max_users]

    for user_id, u in selected_users:
        seen = train_items_by_user.get(user_id, set())
        scores = V @ U[u]

        candidate_mask = np.array([item_id not in seen for item_id in item_ids])
        candidate_indices = np.where(candidate_mask)[0]

        if len(candidate_indices) == 0:
            continue

        candidate_scores = scores[candidate_indices]

        if len(candidate_scores) <= top_k_recs:
            order = np.argsort(candidate_scores)[::-1]
        else:
            partial = np.argpartition(candidate_scores, -top_k_recs)[-top_k_recs:]
            order = partial[np.argsort(candidate_scores[partial])[::-1]]

        top_indices = candidate_indices[order]

        for rank, i_idx in enumerate(top_indices[:top_k_recs], start=1):
            contrib = U[u] * V[i_idx]
            total = float(np.sum(contrib) + 1e-12)
            sorted_factors = np.argsort(contrib)[::-1]

            rows.append({
                "fidelity_at_2": float(np.sum(contrib[sorted_factors[:2]]) / total),
                "fidelity_at_3": float(np.sum(contrib[sorted_factors[:3]]) / total),
                "fidelity_at_5": float(np.sum(contrib[sorted_factors[:5]]) / total),
                "fidelity_at_10": float(np.sum(contrib[sorted_factors[:10]]) / total),
            })

    return pd.DataFrame(rows)


def evaluate_coupled_nmf_tuned(train_df, test_df, seed):
    all_data = pd.concat(
        [
            train_df[["user_id", "item_id", "rating"]],
            test_df[["user_id", "item_id", "rating"]],
        ],
        ignore_index=True,
    )

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_id_maps(all_data)

    U, V, B, feature_cols = train_coupled_nmf(
        train_df=train_df,
        descriptors_df=descriptors,
        user_to_idx=user_to_idx,
        item_to_idx=item_to_idx,
        k=60,
        epochs=20,
        alpha=0.05,
        beta=0.0001,
        lambda_reg=0.01,
        seed=seed,
    )

    global_mean = train_df["rating"].mean()
    preds = []

    for row in test_df.itertuples(index=False):
        u = user_to_idx.get(row.user_id)
        i = item_to_idx.get(row.item_id)

        if u is None or i is None:
            preds.append(global_mean)
        else:
            preds.append(float(U[u] @ V[i]))

    scores_by_user = {}

    for user_id, u in user_to_idx.items():
        scores = V @ U[u]
        scores_by_user[user_id] = {
            idx_to_item[i]: float(scores[i])
            for i in range(len(idx_to_item))
        }

    rmse, mae = prediction_metrics(test_df, preds)

    rank_metrics = lightweight_ranking_metrics(
        scores_by_user=scores_by_user,
        train_df=train_df,
        test_df=test_df,
        all_items=sorted(all_data["item_id"].unique()),
        top_k_values=(5, 10),
        relevance_threshold=4.0,
    )

    fidelity_df = compute_fidelity_for_top_recs(
        train_df=train_df,
        U=U,
        V=V,
        user_to_idx=user_to_idx,
        idx_to_item=idx_to_item,
        max_users=100,
        top_k_recs=10,
    )

    out = {"rmse": rmse, "mae": mae}
    out.update(rank_metrics)

    if len(fidelity_df) > 0:
        out["fidelity_at_2"] = float(fidelity_df["fidelity_at_2"].mean())
        out["fidelity_at_3"] = float(fidelity_df["fidelity_at_3"].mean())
        out["fidelity_at_5"] = float(fidelity_df["fidelity_at_5"].mean())
        out["fidelity_at_10"] = float(fidelity_df["fidelity_at_10"].mean())
        out["num_explanation_instances"] = int(len(fidelity_df))
    else:
        out["fidelity_at_2"] = np.nan
        out["fidelity_at_3"] = np.nan
        out["fidelity_at_5"] = np.nan
        out["fidelity_at_10"] = np.nan
        out["num_explanation_instances"] = 0

    return out


# ------------------------------------------------------------
# 7. Resume-aware final 3-seed evaluation
# ------------------------------------------------------------

seeds = [42, 123, 2026]
models = ["biased_mf", "plain_nmf", "coupled_nmf_tuned"]

run_level_path = CSV_DIR / "ml1m_3seed_run_level_metrics.csv"
summary_path = CSV_DIR / "ml1m_3seed_main_comparison.csv"

if run_level_path.exists():
    run_df = pd.read_csv(run_level_path)
    rows = run_df.to_dict("records")
    completed = set(zip(run_df["seed"], run_df["model_name"]))
    print("Existing run-level checkpoint found.")
    print("Completed model-seed runs:", len(completed))
else:
    rows = []
    completed = set()
    print("No existing run-level checkpoint found. Starting fresh.")

for seed in seeds:
    print("=" * 80)
    print("Seed:", seed)

    train_df, val_df, test_df = load_split_data(
        ratings_df=ratings,
        split_registry_df=split_registry,
        seed=seed,
    )

    print("Train:", train_df.shape, "Test:", test_df.shape)

    for model_name in models:
        key = (seed, model_name)

        if key in completed:
            print(f"Skipping completed: seed={seed}, model={model_name}")
            continue

        print("-" * 80)
        print(f"Running model={model_name}, seed={seed}")

        start = time.time()

        if model_name == "biased_mf":
            metrics = evaluate_biased_mf(train_df, test_df, seed)

        elif model_name == "plain_nmf":
            metrics = evaluate_plain_nmf(train_df, test_df, seed)

        elif model_name == "coupled_nmf_tuned":
            metrics = evaluate_coupled_nmf_tuned(train_df, test_df, seed)

        else:
            raise ValueError(f"Unknown model: {model_name}")

        row = {
            "dataset_name": "ml1m",
            "model_name": model_name,
            "seed": seed,
        }

        if model_name == "coupled_nmf_tuned":
            row.update({
                "k": 60,
                "alpha": 0.05,
                "beta": 0.0001,
                "lambda_reg": 0.01,
                "epochs": 20,
            })
        else:
            row.update({
                "k": np.nan,
                "alpha": np.nan,
                "beta": np.nan,
                "lambda_reg": np.nan,
                "epochs": np.nan,
            })

        row.update(metrics)
        rows.append(row)

        run_df = pd.DataFrame(rows)
        run_df.to_csv(run_level_path, index=False)

        completed.add(key)

        elapsed = time.time() - start
        print(f"Completed model={model_name}, seed={seed} in {elapsed/60:.2f} minutes.")
        print("Checkpoint saved:", run_level_path)
        print(metrics)

# ------------------------------------------------------------
# 8. Aggregate mean ± std
# ------------------------------------------------------------

run_df = pd.read_csv(run_level_path)

metric_cols = [
    "rmse",
    "mae",
    "precision_at_5",
    "recall_at_5",
    "ndcg_at_5",
    "precision_at_10",
    "recall_at_10",
    "ndcg_at_10",
    "fidelity_at_2",
    "fidelity_at_3",
    "fidelity_at_5",
    "fidelity_at_10",
]

summary_rows = []

for model_name, group in run_df.groupby("model_name"):
    row = {
        "dataset_name": "ml1m",
        "model_name": model_name,
        "num_seeds": group["seed"].nunique(),
    }

    for col in metric_cols:
        if col in group.columns:
            row[col] = group[col].mean(skipna=True)
            row[f"{col}_std"] = group[col].std(ddof=0, skipna=True)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

model_order = {
    "biased_mf": 1,
    "plain_nmf": 2,
    "coupled_nmf_tuned": 3,
}

summary_df["sort_order"] = summary_df["model_name"].map(model_order)
summary_df = summary_df.sort_values("sort_order").drop(columns=["sort_order"])

summary_df.to_csv(summary_path, index=False)

print("\nFinal ml1m 3-seed run-level metrics:")
display(run_df)

print("\nFinal ml1m 3-seed main comparison:")
display(summary_df)

print("Saved run-level:", run_level_path)
print("Saved summary:", summary_path)

# ------------------------------------------------------------
# 9. Archive final 3-seed outputs
# ------------------------------------------------------------

for src in [run_level_path, summary_path]:
    if src.exists():
        dst = ARCHIVE_DIR / src.name
        shutil.copy(src, dst)
        print("Archived:", dst)

print("\nml1m 3-seed final evaluation completed.")

## 19. MovieLens 1M three-seed table and figure

Original cell index starts around `300`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"
PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

for d in [PAPER_TABLE_DIR, RESULTS_TABLE_DIR, PAPER_FIG_DIR, RESULTS_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

summary_path = CSV_DIR / "ml1m_3seed_main_comparison.csv"
run_path = CSV_DIR / "ml1m_3seed_run_level_metrics.csv"

summary = pd.read_csv(summary_path)
run_level = pd.read_csv(run_path)

display(summary)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def tex_escape(text):
    return str(text).replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")

def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"

def fmt_pm(row, metric, digits=4):
    std_col = f"{metric}_std"
    if std_col in row.index and not pd.isna(row[std_col]):
        return rf"${float(row[metric]):.{digits}f}\pm {float(row[std_col]):.{digits}f}$"
    return rf"${float(row[metric]):.{digits}f}$"

model_display = {
    "biased_mf": "Biased MF",
    "plain_nmf": "Plain NMF",
    "coupled_nmf_tuned": "Tuned Coupled NMF",
}

model_order = ["biased_mf", "plain_nmf", "coupled_nmf_tuned"]

# ------------------------------------------------------------
# 3. Generate LaTeX table
# ------------------------------------------------------------

rows = []

for model in model_order:
    sub = summary[summary["model_name"] == model]
    if sub.empty:
        continue

    r = sub.iloc[0]

    rows.append([
        tex_escape(model_display.get(model, model)),
        str(int(r["num_seeds"])),
        fmt_pm(r, "rmse"),
        fmt_pm(r, "mae"),
        fmt_pm(r, "precision_at_5"),
        fmt_pm(r, "recall_at_5"),
        fmt_pm(r, "ndcg_at_5"),
        fmt_pm(r, "precision_at_10"),
        fmt_pm(r, "recall_at_10"),
        fmt_pm(r, "ndcg_at_10"),
    ])

caption = (
    "Three-seed secondary validation on MovieLens 1M. "
    "Values are reported as mean $\\pm$ standard deviation over seeds 42, 123, and 2026."
)

label = "tab:ml1m_3seed_secondary_validation"

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{" + caption + r"}")
lines.append(r"\label{" + label + r"}")
lines.append(r"\resizebox{\textwidth}{!}{%")
lines.append(r"\begin{tabular}{lccccccccc}")
lines.append(r"\toprule")
lines.append(r"Model & Seeds & RMSE & MAE & P@5 & R@5 & NDCG@5 & P@10 & R@10 & NDCG@10 \\")
lines.append(r"\midrule")

for row in rows:
    lines.append(" & ".join(row) + r" \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"}")
lines.append(r"\vspace{0.35em}")
lines.append(r"\begin{minipage}{0.96\textwidth}")
lines.append(
    r"\footnotesize "
    r"MovieLens 1M is used as a larger secondary validation dataset with genre-only item descriptors. "
    r"The tuned coupled NMF is evaluated with $k=60$, $\alpha=0.05$, $\beta=0.0001$, $\lambda=0.01$, and 20 training epochs. "
    r"The coupled model is competitive with plain NMF in ranking quality and slightly improves short-list ranking metrics, "
    r"although plain NMF remains stronger on prediction error and NDCG@10."
)
lines.append(r"\end{minipage}")
lines.append(r"\end{table}")
lines.append("")

latex_content = "\n".join(lines)

paper_table_path = PAPER_TABLE_DIR / "table_ml1m_3seed_secondary_validation.tex"
results_table_path = RESULTS_TABLE_DIR / "table_ml1m_3seed_secondary_validation.tex"

paper_table_path.write_text(latex_content, encoding="utf-8")
results_table_path.write_text(latex_content, encoding="utf-8")

print("Saved:", paper_table_path)
print("Copied:", results_table_path)

# ------------------------------------------------------------
# 4. Generate figure: 3-seed secondary validation comparison
# ------------------------------------------------------------

def directional_normalize(series, higher_is_better=True):
    series = pd.Series(series, dtype=float)
    smin = series.min()
    smax = series.max()

    if np.isclose(smin, smax):
        return pd.Series(np.ones(len(series)), index=series.index)

    if higher_is_better:
        return (series - smin) / (smax - smin)

    return (smax - series) / (smax - smin)

plot_df = summary.set_index("model_name").loc[model_order].reset_index()

metrics_info = [
    ("rmse", False, "RMSE"),
    ("mae", False, "MAE"),
    ("precision_at_5", True, "P@5"),
    ("ndcg_at_5", True, "NDCG@5"),
    ("precision_at_10", True, "P@10"),
    ("ndcg_at_10", True, "NDCG@10"),
]

norm_data = {}

for metric, higher_better, label in metrics_info:
    norm_data[label] = directional_normalize(
        plot_df[metric],
        higher_is_better=higher_better,
    ).values

x = np.arange(len(metrics_info))
width = 0.23

fig, ax = plt.subplots(figsize=(11, 6))

for i, model in enumerate(model_order):
    yvals = [norm_data[label][i] for _, _, label in metrics_info]
    ax.bar(
        x + (i - 1) * width,
        yvals,
        width=width,
        label=model_display[model],
    )

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metrics_info])
ax.set_ylabel("Normalized directional score")
ax.set_title("MovieLens 1M three-seed secondary validation")
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig_stem = "fig_ml1m_3seed_secondary_validation_comparison"

paper_png = PAPER_FIG_DIR / f"{fig_stem}.png"
paper_pdf = PAPER_FIG_DIR / f"{fig_stem}.pdf"
results_png = RESULTS_FIG_DIR / f"{fig_stem}.png"
results_pdf = RESULTS_FIG_DIR / f"{fig_stem}.pdf"

fig.savefig(paper_png, dpi=300, bbox_inches="tight")
fig.savefig(paper_pdf, bbox_inches="tight")
shutil.copy(paper_png, results_png)
shutil.copy(paper_pdf, results_pdf)

print("Saved:", paper_png)
print("Saved:", paper_pdf)
print("Copied:", results_png)
print("Copied:", results_pdf)

plt.show()

# ------------------------------------------------------------
# 5. Create include commands
# ------------------------------------------------------------

include_table_path = PAPER_TABLE_DIR / "ml1m_3seed_table_include_commands.tex"
include_figure_path = PAPER_FIG_DIR / "ml1m_3seed_figure_include_commands.tex"

include_table_content = r"""
% Required packages: \usepackage{booktabs,graphicx}
\input{paper/tables/table_ml1m_3seed_secondary_validation}
""".strip() + "\n"

include_figure_content = r"""
% Required package: \usepackage{graphicx}

\begin{figure}[htbp]
\centering
\includegraphics[width=0.9\textwidth]{paper/figures/fig_ml1m_3seed_secondary_validation_comparison.pdf}
\caption{Three-seed secondary validation comparison on MovieLens 1M.}
\label{fig:ml1m_3seed_secondary_validation_comparison}
\end{figure}
""".strip() + "\n"

include_table_path.write_text(include_table_content, encoding="utf-8")
include_figure_path.write_text(include_figure_content, encoding="utf-8")

shutil.copy(include_table_path, RESULTS_TABLE_DIR / include_table_path.name)
shutil.copy(include_figure_path, RESULTS_FIG_DIR / include_figure_path.name)

print("Saved table include:", include_table_path)
print("Saved figure include:", include_figure_path)

## 20. MovieLens 1M three-seed journal-ready bundle

Original cell index starts around `301`.

In [ ]:
# ============================================================
# Final journal-ready ml1m 3-seed bundle
# Includes:
# - ml1m_3seed CSVs
# - 3-seed LaTeX table
# - 3-seed manuscript figure
# - include-command files
# - copy report
# - manifest
# - ZIP file
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_ROOT = CSV_DIR / "archive"
ARCHIVE_DIR = ARCHIVE_ROOT / "ml1m_3seed_final_evaluation"

PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

EXPORT_ROOT = PROJECT_ROOT / "results/export_bundles"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

FINAL_BUNDLE_DIR = EXPORT_ROOT / f"ml1m_3seed_journal_ready_bundle_{timestamp}"

FINAL_CSV_DIR = FINAL_BUNDLE_DIR / "csv"
FINAL_ARCHIVE_DIR = FINAL_BUNDLE_DIR / "csv_archive"
FINAL_TABLE_DIR = FINAL_BUNDLE_DIR / "latex_tables"
FINAL_FIGURE_DIR = FINAL_BUNDLE_DIR / "figures"
FINAL_INCLUDE_DIR = FINAL_BUNDLE_DIR / "latex_include_commands"
FINAL_REPORT_DIR = FINAL_BUNDLE_DIR / "reports"

for folder in [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_BUNDLE_DIR:", FINAL_BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())
print("PAPER_TABLE_DIR exists:", PAPER_TABLE_DIR.exists())
print("PAPER_FIG_DIR exists:", PAPER_FIG_DIR.exists())

# ------------------------------------------------------------
# 2. Helper copy function
# ------------------------------------------------------------

copy_reports = []

def copy_if_exists(src_candidates, dst_dir, filename, section):
    src = None

    for candidate in src_candidates:
        if candidate.exists():
            src = candidate
            break

    dst = dst_dir / filename

    if src is not None:
        shutil.copy(src, dst)
        report = {
            "section": section,
            "filename": filename,
            "status": "COPIED",
            "size_kb": round(dst.stat().st_size / 1024, 2),
            "source": str(src),
        }
    else:
        report = {
            "section": section,
            "filename": filename,
            "status": "MISSING",
            "size_kb": None,
            "source": "",
        }

    copy_reports.append(report)
    return report

# ------------------------------------------------------------
# 3. Copy 3-seed CSV files
# ------------------------------------------------------------

csv_files = [
    "ml1m_3seed_main_comparison.csv",
    "ml1m_3seed_run_level_metrics.csv",

    # Useful context files
    "ml1m_dataset_summary.csv",
    "ml1m_split_registry.csv",
    "ml1m_coupled_nmf_selected_config_seed42.csv",
    "ml1m_coupled_nmf_tuning_grid_seed42.csv",
    "ml1m_coupled_nmf_tuned_test_metrics_seed42.csv",
    "ml1m_main_comparison_seed42_after_coupled_tuning.csv",
]

for filename in csv_files:
    copy_if_exists(
        src_candidates=[
            CSV_DIR / filename,
            ARCHIVE_DIR / filename,
            ARCHIVE_ROOT / "ml1m_seed42_secondary_validation" / filename,
        ],
        dst_dir=FINAL_CSV_DIR,
        filename=filename,
        section="csv",
    )

csv_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "csv"])
display(csv_report_df)

# ------------------------------------------------------------
# 4. Copy archived 3-seed files separately, if present
# ------------------------------------------------------------

archive_files = [
    "ml1m_3seed_main_comparison.csv",
    "ml1m_3seed_run_level_metrics.csv",
]

for filename in archive_files:
    copy_if_exists(
        src_candidates=[
            ARCHIVE_DIR / filename,
            CSV_DIR / filename,
        ],
        dst_dir=FINAL_ARCHIVE_DIR,
        filename=filename,
        section="csv_archive",
    )

archive_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "csv_archive"])
display(archive_report_df)

# ------------------------------------------------------------
# 5. Copy LaTeX table and include-command file
# ------------------------------------------------------------

table_files = [
    "table_ml1m_3seed_secondary_validation.tex",
]

table_include_files = [
    "ml1m_3seed_table_include_commands.tex",
]

for filename in table_files:
    copy_if_exists(
        src_candidates=[
            PAPER_TABLE_DIR / filename,
            RESULTS_TABLE_DIR / filename,
        ],
        dst_dir=FINAL_TABLE_DIR,
        filename=filename,
        section="latex_tables",
    )

for filename in table_include_files:
    copy_if_exists(
        src_candidates=[
            PAPER_TABLE_DIR / filename,
            RESULTS_TABLE_DIR / filename,
        ],
        dst_dir=FINAL_INCLUDE_DIR,
        filename=filename,
        section="latex_include_commands",
    )

table_report_df = pd.DataFrame([
    r for r in copy_reports
    if r["section"] in ["latex_tables", "latex_include_commands"]
])
display(table_report_df)

# ------------------------------------------------------------
# 6. Copy 3-seed figure files and include-command file
# ------------------------------------------------------------

figure_files = [
    "fig_ml1m_3seed_secondary_validation_comparison.png",
    "fig_ml1m_3seed_secondary_validation_comparison.pdf",
]

figure_include_files = [
    "ml1m_3seed_figure_include_commands.tex",
]

for filename in figure_files:
    copy_if_exists(
        src_candidates=[
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
        ],
        dst_dir=FINAL_FIGURE_DIR,
        filename=filename,
        section="figures",
    )

for filename in figure_include_files:
    copy_if_exists(
        src_candidates=[
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
        ],
        dst_dir=FINAL_INCLUDE_DIR,
        filename=filename,
        section="latex_include_commands",
    )

figure_report_df = pd.DataFrame([
    r for r in copy_reports
    if r["section"] in ["figures", "latex_include_commands"]
])
display(figure_report_df)

# ------------------------------------------------------------
# 7. Save complete copy report
# ------------------------------------------------------------

full_report = pd.DataFrame(copy_reports)

copy_report_path = FINAL_REPORT_DIR / "ml1m_3seed_bundle_copy_report.csv"
full_report.to_csv(copy_report_path, index=False)

missing_files = full_report[full_report["status"] == "MISSING"].copy()

print("Copy report saved:", copy_report_path)
print("\nMissing files:")
display(missing_files)

# ------------------------------------------------------------
# 8. Create manifest
# ------------------------------------------------------------

manifest_text = f"""Final Journal-Ready Bundle: MovieLens 1M Three-Seed Secondary Validation
=======================================================================

Project:
xai_coupled_nmf_project

Dataset:
MovieLens 1M

Experiment:
Three-seed secondary validation

Seeds:
42, 123, 2026

Bundle created:
{timestamp}

Purpose:
This bundle preserves the journal-ready MovieLens 1M secondary-validation outputs for the coupled NMF explainable recommendation manuscript.

Included folders:
1. csv
   Main 3-seed CSV outputs and selected supporting seed-42 tuning files.

2. csv_archive
   Archived copies of final 3-seed CSV outputs.

3. latex_tables
   Manuscript-ready LaTeX table for the 3-seed secondary validation.

4. figures
   Manuscript-ready PNG and PDF figure files.

5. latex_include_commands
   Ready-to-use LaTeX table and figure include-command files.

6. reports
   Copy report and manifest.

Main CSV files:
- csv/ml1m_3seed_main_comparison.csv
- csv/ml1m_3seed_run_level_metrics.csv

Main LaTeX table:
- latex_tables/table_ml1m_3seed_secondary_validation.tex

Main figure:
- figures/fig_ml1m_3seed_secondary_validation_comparison.pdf
- figures/fig_ml1m_3seed_secondary_validation_comparison.png

Include files:
- latex_include_commands/ml1m_3seed_table_include_commands.tex
- latex_include_commands/ml1m_3seed_figure_include_commands.tex

Selected tuned coupled NMF configuration:
k = 60
alpha = 0.05
beta = 0.0001
lambda_reg = 0.01
epochs = 20

Manuscript interpretation:
MovieLens 1M is used as a larger secondary validation dataset with genre-only item descriptors.
The tuned coupled NMF remains competitive with plain NMF in ranking quality and slightly improves short-list ranking metrics such as P@5 and NDCG@5.
Plain NMF remains stronger on prediction error and NDCG@10, which is consistent with the limited semantic expressiveness of genre-only descriptors.

Recommended journal use:
Use this bundle as the final MovieLens 1M secondary-validation evidence.

Missing files:
"""

if missing_files.empty:
    manifest_text += "\nNone.\n"
else:
    for _, row in missing_files.iterrows():
        manifest_text += f"\n- {row['section']}: {row['filename']}"

manifest_path = FINAL_BUNDLE_DIR / "README_ml1m_3seed_journal_ready_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Manifest saved:", manifest_path)

# ------------------------------------------------------------
# 9. Create ZIP file
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(FINAL_BUNDLE_DIR),
    "zip",
    root_dir=FINAL_BUNDLE_DIR,
)

print("\nFinal ml1m 3-seed journal-ready bundle folder:")
print(FINAL_BUNDLE_DIR)

print("\nFinal ZIP file:")
print(zip_path)

# ------------------------------------------------------------
# 10. Verify bundle contents
# ------------------------------------------------------------

print("\nBundle folder counts:")

for folder in [
    FINAL_CSV_DIR,
    FINAL_ARCHIVE_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
]:
    files = list(folder.glob("*"))
    print(f"{folder.name:28s}: {len(files)} files")

zip_file = Path(zip_path)
print("\nZIP file size:")
print(zip_file, "->", round(zip_file.stat().st_size / 1024 / 1024, 2), "MB")

print("\nFinal verification list:")
for p in sorted(FINAL_BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(FINAL_BUNDLE_DIR))

## 21. Statistical significance testing

Original cell index starts around `302`.

In [ ]:
# ============================================================
# Statistical significance testing for:
# 1. ml_latest_small, 5 seeds
# 2. ml1m, 3 seeds
#
# Tests:
# - Paired t-test
# - Wilcoxon signed-rank test
# - Cohen's dz effect size
#
# Output:
# - CSV files
# - manuscript-ready LaTeX tables
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import shutil

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive"
PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("PAPER_TABLE_DIR:", PAPER_TABLE_DIR)

# ------------------------------------------------------------
# 2. Imports with safe fallbacks
# ------------------------------------------------------------

try:
    from scipy.stats import ttest_rel, wilcoxon
    SCIPY_AVAILABLE = True
except Exception as e:
    SCIPY_AVAILABLE = False
    print("SciPy not available. Only descriptive paired differences will be computed.")
    print("Import error:", e)

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not locate: {filename}")


def fmt(x, digits=4):
    if pd.isna(x):
        return "--"
    return f"{float(x):.{digits}f}"


def fmt_p(x):
    if pd.isna(x):
        return "--"
    x = float(x)
    if x < 0.001:
        return r"$<0.001$"
    return rf"${x:.4f}$"


def tex_escape(text):
    text = str(text)
    replacements = {
        "_": r"\_",
        "%": r"\%",
        "&": r"\&",
        "#": r"\#",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def significance_marker(p):
    if pd.isna(p):
        return "n/a"
    p = float(p)
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def metric_direction(metric):
    lower_better = {"rmse", "mae"}
    return "lower" if metric in lower_better else "higher"


def compute_cohens_dz(diff_values):
    diff_values = np.asarray(diff_values, dtype=float)

    if len(diff_values) < 2:
        return np.nan

    sd = np.std(diff_values, ddof=1)

    if np.isclose(sd, 0.0):
        return np.nan

    return float(np.mean(diff_values) / sd)


def paired_test_for_metric(df, dataset_name, model_a, model_b, metric):
    """
    model_a = baseline/reference model
    model_b = target/comparison model

    raw_diff = model_b - model_a
    improvement = positive means model_b is better
       lower-is-better metrics: model_a - model_b
       higher-is-better metrics: model_b - model_a
    """

    sub_a = df[df["model_name"] == model_a][["seed", metric]].rename(
        columns={metric: "value_a"}
    )
    sub_b = df[df["model_name"] == model_b][["seed", metric]].rename(
        columns={metric: "value_b"}
    )

    merged = sub_a.merge(sub_b, on="seed", how="inner").dropna()

    n = len(merged)

    if n == 0:
        return None

    value_a = merged["value_a"].to_numpy(dtype=float)
    value_b = merged["value_b"].to_numpy(dtype=float)

    raw_diff = value_b - value_a

    direction = metric_direction(metric)

    if direction == "lower":
        improvement = value_a - value_b
    else:
        improvement = value_b - value_a

    mean_a = float(np.mean(value_a))
    mean_b = float(np.mean(value_b))
    mean_raw_diff = float(np.mean(raw_diff))
    mean_improvement = float(np.mean(improvement))
    std_improvement = float(np.std(improvement, ddof=1)) if n > 1 else np.nan

    dz = compute_cohens_dz(improvement)

    paired_t_p = np.nan
    wilcoxon_p = np.nan

    if SCIPY_AVAILABLE and n >= 2:
        try:
            paired_t_p = float(ttest_rel(value_b, value_a, nan_policy="omit").pvalue)
        except Exception:
            paired_t_p = np.nan

        try:
            # Wilcoxon on improvement values. If all differences are zero, scipy raises.
            if not np.allclose(improvement, 0.0):
                wilcoxon_p = float(wilcoxon(improvement, zero_method="wilcox").pvalue)
            else:
                wilcoxon_p = 1.0
        except Exception:
            wilcoxon_p = np.nan

    result = {
        "dataset_name": dataset_name,
        "model_a_reference": model_a,
        "model_b_target": model_b,
        "metric": metric,
        "direction": direction,
        "num_paired_seeds": n,
        "mean_model_a": mean_a,
        "mean_model_b": mean_b,
        "mean_raw_difference_b_minus_a": mean_raw_diff,
        "mean_improvement_of_b_over_a": mean_improvement,
        "std_improvement": std_improvement,
        "cohens_dz": dz,
        "paired_t_p_value": paired_t_p,
        "wilcoxon_p_value": wilcoxon_p,
        "paired_t_significance": significance_marker(paired_t_p),
        "wilcoxon_significance": significance_marker(wilcoxon_p),
        "model_b_better_by_mean": bool(mean_improvement > 0),
    }

    return result


def run_significance_tests(df, dataset_name, comparisons, metrics):
    rows = []

    for model_a, model_b in comparisons:
        for metric in metrics:
            if metric not in df.columns:
                continue

            result = paired_test_for_metric(
                df=df,
                dataset_name=dataset_name,
                model_a=model_a,
                model_b=model_b,
                metric=metric,
            )

            if result is not None:
                rows.append(result)

    return pd.DataFrame(rows)


def write_significance_latex_table(
    sig_df,
    filename,
    caption,
    label,
    max_rows=None,
    note=None,
):
    if max_rows is not None:
        work = sig_df.head(max_rows).copy()
    else:
        work = sig_df.copy()

    metric_display = {
        "rmse": "RMSE",
        "mae": "MAE",
        "precision_at_5": "P@5",
        "precision_at_10": "P@10",
        "recall_at_5": "R@5",
        "recall_at_10": "R@10",
        "ndcg_at_5": "NDCG@5",
        "ndcg_at_10": "NDCG@10",
    }

    model_display = {
        "biased_mf": "Biased MF",
        "plain_nmf": "Plain NMF",
        "coupled_nmf": "Coupled NMF",
        "coupled_nmf_tuned": "Tuned Coupled NMF",
        "item_cf": "Item-CF",
    }

    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{lllcccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"Reference & Target & Metric & Ref. mean & Target mean & Improvement & $p_t$ & $p_w$ & $d_z$ \\"
    )
    lines.append(r"\midrule")

    for _, row in work.iterrows():
        ref = model_display.get(row["model_a_reference"], row["model_a_reference"])
        target = model_display.get(row["model_b_target"], row["model_b_target"])
        metric = metric_display.get(row["metric"], row["metric"])

        latex_row = [
            tex_escape(ref),
            tex_escape(target),
            metric,
            fmt(row["mean_model_a"]),
            fmt(row["mean_model_b"]),
            fmt(row["mean_improvement_of_b_over_a"]),
            fmt_p(row["paired_t_p_value"]),
            fmt_p(row["wilcoxon_p_value"]),
            fmt(row["cohens_dz"]),
        ]

        lines.append(" & ".join(latex_row) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")

    if note:
        lines.append(r"\vspace{0.35em}")
        lines.append(r"\begin{minipage}{0.96\textwidth}")
        lines.append(r"\footnotesize " + note)
        lines.append(r"\end{minipage}")

    lines.append(r"\end{table}")
    lines.append("")

    content = "\n".join(lines)

    paper_path = PAPER_TABLE_DIR / filename
    results_path = RESULTS_TABLE_DIR / filename

    paper_path.write_text(content, encoding="utf-8")
    results_path.write_text(content, encoding="utf-8")

    print("Saved:", paper_path)
    print("Copied:", results_path)

    return paper_path


# ------------------------------------------------------------
# 4. Load run-level files
# ------------------------------------------------------------

ml_latest_path = find_csv("ml_latest_small_run_level_metrics_v2_tuned_coupled.csv")
ml1m_path = find_csv("ml1m_3seed_run_level_metrics.csv")

ml_latest = pd.read_csv(ml_latest_path)
ml1m = pd.read_csv(ml1m_path)

print("Loaded:")
print("ml_latest_small:", ml_latest_path, ml_latest.shape)
print("ml1m:", ml1m_path, ml1m.shape)

print("\nml_latest_small models:", sorted(ml_latest["model_name"].dropna().unique()))
print("ml1m models:", sorted(ml1m["model_name"].dropna().unique()))

# ------------------------------------------------------------
# 5. Define comparisons and metrics
# ------------------------------------------------------------

common_metrics = [
    "rmse",
    "mae",
    "precision_at_5",
    "precision_at_10",
    "recall_at_5",
    "recall_at_10",
    "ndcg_at_5",
    "ndcg_at_10",
]

ml_latest_comparisons = [
    ("plain_nmf", "coupled_nmf"),
    ("biased_mf", "coupled_nmf"),
    ("item_cf", "coupled_nmf"),
]

ml1m_comparisons = [
    ("plain_nmf", "coupled_nmf_tuned"),
    ("biased_mf", "coupled_nmf_tuned"),
]

# ------------------------------------------------------------
# 6. Run tests
# ------------------------------------------------------------

ml_latest_sig = run_significance_tests(
    df=ml_latest,
    dataset_name="ml_latest_small",
    comparisons=ml_latest_comparisons,
    metrics=common_metrics,
)

ml1m_sig = run_significance_tests(
    df=ml1m,
    dataset_name="ml1m",
    comparisons=ml1m_comparisons,
    metrics=common_metrics,
)

all_sig = pd.concat([ml_latest_sig, ml1m_sig], ignore_index=True)

# ------------------------------------------------------------
# 7. Save CSV outputs
# ------------------------------------------------------------

ml_latest_out = CSV_DIR / "ml_latest_small_significance_tests.csv"
ml1m_out = CSV_DIR / "ml1m_significance_tests.csv"
all_out = CSV_DIR / "statistical_tests_all_datasets.csv"

ml_latest_sig.to_csv(ml_latest_out, index=False)
ml1m_sig.to_csv(ml1m_out, index=False)
all_sig.to_csv(all_out, index=False)

print("\nSaved CSV files:")
print(ml_latest_out)
print(ml1m_out)
print(all_out)

print("\nml_latest_small significance tests:")
display(ml_latest_sig)

print("\nml1m significance tests:")
display(ml1m_sig)

# ------------------------------------------------------------
# 8. Generate compact manuscript tables
# ------------------------------------------------------------

# For main manuscript, keep the most important comparisons.
ml_latest_main = ml_latest_sig[
    (
        (ml_latest_sig["model_a_reference"] == "plain_nmf")
        & (ml_latest_sig["model_b_target"] == "coupled_nmf")
    )
    & ml_latest_sig["metric"].isin(["rmse", "mae", "precision_at_10", "recall_at_10", "ndcg_at_10"])
].copy()

ml1m_main = ml1m_sig[
    (
        (ml1m_sig["model_a_reference"] == "plain_nmf")
        & (ml1m_sig["model_b_target"] == "coupled_nmf_tuned")
    )
    & ml1m_sig["metric"].isin(["rmse", "mae", "precision_at_5", "ndcg_at_5", "precision_at_10", "ndcg_at_10"])
].copy()

write_significance_latex_table(
    sig_df=ml_latest_main,
    filename="table_ml_latest_small_significance_tests.tex",
    caption="Paired seed-level significance tests for MovieLens latest-small.",
    label="tab:ml_latest_small_significance_tests",
    note=(
        r"The reference model is plain NMF and the target model is the coupled NMF. "
        r"Positive improvement means that the target model is better. "
        r"$p_t$ denotes the paired $t$-test p-value, $p_w$ denotes the Wilcoxon signed-rank p-value, "
        r"and $d_z$ is Cohen's paired effect size."
    ),
)

write_significance_latex_table(
    sig_df=ml1m_main,
    filename="table_ml1m_significance_tests.tex",
    caption="Paired seed-level significance tests for MovieLens 1M.",
    label="tab:ml1m_significance_tests",
    note=(
        r"The reference model is plain NMF and the target model is the tuned coupled NMF. "
        r"MovieLens 1M has only three seeds, so the tests should be interpreted as supporting stability evidence rather than strong inferential evidence."
    ),
)

# ------------------------------------------------------------
# 9. Include-command file
# ------------------------------------------------------------

include_content = r"""
% Required packages: \usepackage{booktabs,graphicx}
% Statistical significance tables

\input{paper/tables/table_ml_latest_small_significance_tests}

\input{paper/tables/table_ml1m_significance_tests}
""".strip() + "\n"

include_path = PAPER_TABLE_DIR / "statistical_tests_table_include_commands.tex"
include_path.write_text(include_content, encoding="utf-8")
shutil.copy(include_path, RESULTS_TABLE_DIR / include_path.name)

print("Saved include file:", include_path)
print("Copied include file:", RESULTS_TABLE_DIR / include_path.name)

# ------------------------------------------------------------
# 10. Archive significance outputs
# ------------------------------------------------------------

sig_archive_dir = ARCHIVE_DIR / "statistical_significance_tests"
sig_archive_dir.mkdir(parents=True, exist_ok=True)

for src in [
    ml_latest_out,
    ml1m_out,
    all_out,
    PAPER_TABLE_DIR / "table_ml_latest_small_significance_tests.tex",
    PAPER_TABLE_DIR / "table_ml1m_significance_tests.tex",
    include_path,
]:
    if src.exists():
        shutil.copy(src, sig_archive_dir / src.name)
        print("Archived:", sig_archive_dir / src.name)

print("\nStatistical significance testing completed.")
print("Archive folder:", sig_archive_dir)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

files_to_check = [
    "paper/tables/table_ml_latest_small_significance_tests.tex",
    "paper/tables/table_ml1m_significance_tests.tex",
    "paper/tables/statistical_tests_table_include_commands.tex",
    "results/tables/table_ml_latest_small_significance_tests.tex",
    "results/tables/table_ml1m_significance_tests.tex",
    "results/tables/statistical_tests_table_include_commands.tex",
    "results/csv/archive/statistical_significance_tests/table_ml_latest_small_significance_tests.tex",
    "results/csv/archive/statistical_significance_tests/table_ml1m_significance_tests.tex",
    "results/csv/archive/statistical_significance_tests/statistical_tests_table_include_commands.tex",
]

for file in files_to_check:
    path = PROJECT_ROOT / file
    print(f"{file:95s} ->", "FOUND" if path.exists() else "MISSING")

## 22. Statistical significance figures

Original cell index starts around `304`.

In [ ]:
# ============================================================
# Manuscript-ready figures for statistical significance testing
# Datasets:
#   1. ml_latest_small
#   2. ml1m
#
# Figures:
#   fig_ml_latest_small_significance_effects
#   fig_ml1m_significance_effects
#
# Output folders:
#   paper/figures/
#   results/figures/
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/statistical_significance_tests"

PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_DIR:", CSV_DIR)
print("PAPER_FIG_DIR:", PAPER_FIG_DIR)
print("RESULTS_FIG_DIR:", RESULTS_FIG_DIR)

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def find_csv(filename):
    candidates = [
        CSV_DIR / filename,
        ARCHIVE_DIR / filename,
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = list(PROJECT_ROOT.rglob(filename))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not locate: {filename}")


def p_label(p):
    if pd.isna(p):
        return "p=n/a"
    p = float(p)
    if p < 0.001:
        return "p<0.001"
    return f"p={p:.3f}"


def metric_label(metric):
    labels = {
        "rmse": "RMSE",
        "mae": "MAE",
        "precision_at_5": "P@5",
        "precision_at_10": "P@10",
        "recall_at_5": "R@5",
        "recall_at_10": "R@10",
        "ndcg_at_5": "NDCG@5",
        "ndcg_at_10": "NDCG@10",
    }
    return labels.get(metric, metric)


def save_figure(fig, stem):
    paper_png = PAPER_FIG_DIR / f"{stem}.png"
    paper_pdf = PAPER_FIG_DIR / f"{stem}.pdf"
    results_png = RESULTS_FIG_DIR / f"{stem}.png"
    results_pdf = RESULTS_FIG_DIR / f"{stem}.pdf"

    fig.savefig(paper_png, dpi=300, bbox_inches="tight")
    fig.savefig(paper_pdf, bbox_inches="tight")

    shutil.copy(paper_png, results_png)
    shutil.copy(paper_pdf, results_pdf)

    print("Saved:", paper_png)
    print("Saved:", paper_pdf)
    print("Copied:", results_png)
    print("Copied:", results_pdf)

    return paper_png, paper_pdf


def make_significance_effect_figure(
    sig_df,
    reference_model,
    target_model,
    metrics,
    title,
    stem,
):
    work = sig_df[
        (sig_df["model_a_reference"] == reference_model)
        & (sig_df["model_b_target"] == target_model)
        & (sig_df["metric"].isin(metrics))
    ].copy()

    if work.empty:
        raise ValueError(f"No matching rows found for {reference_model} vs {target_model}")

    metric_order = {m: i for i, m in enumerate(metrics)}
    work["metric_order"] = work["metric"].map(metric_order)
    work = work.sort_values("metric_order")

    labels = [metric_label(m) for m in work["metric"]]
    improvements = work["mean_improvement_of_b_over_a"].astype(float).to_numpy()
    p_values = work["paired_t_p_value"].to_numpy()

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(11, 5.8))
    bars = ax.bar(x, improvements)

    ax.axhline(0, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Mean improvement of target over reference")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

    for rect, improvement, p in zip(bars, improvements, p_values):
        if improvement >= 0:
            va = "bottom"
            y_pos = improvement
        else:
            va = "top"
            y_pos = improvement

        ax.text(
            rect.get_x() + rect.get_width() / 2,
            y_pos,
            p_label(p),
            ha="center",
            va=va,
            fontsize=9,
            rotation=0,
        )

    fig.text(
        0.5,
        -0.02,
        "Positive values indicate that the target model is better. "
        "For RMSE and MAE, improvement is computed after reversing the direction because lower values are better.",
        ha="center",
        fontsize=9,
    )

    save_figure(fig, stem)
    plt.show()

    return stem


# ------------------------------------------------------------
# 3. Load significance CSV files
# ------------------------------------------------------------

ml_latest_sig = pd.read_csv(find_csv("ml_latest_small_significance_tests.csv"))
ml1m_sig = pd.read_csv(find_csv("ml1m_significance_tests.csv"))

print("Loaded significance CSVs:")
print("ml_latest_small:", ml_latest_sig.shape)
print("ml1m:", ml1m_sig.shape)

# ------------------------------------------------------------
# 4. Generate figures
# ------------------------------------------------------------

main_metrics_latest = [
    "rmse",
    "mae",
    "precision_at_10",
    "recall_at_10",
    "ndcg_at_10",
]

main_metrics_ml1m = [
    "rmse",
    "mae",
    "precision_at_5",
    "ndcg_at_5",
    "precision_at_10",
    "ndcg_at_10",
]

fig1_stem = make_significance_effect_figure(
    sig_df=ml_latest_sig,
    reference_model="plain_nmf",
    target_model="coupled_nmf",
    metrics=main_metrics_latest,
    title="Seed-level significance summary: MovieLens latest-small",
    stem="fig_ml_latest_small_significance_effects",
)

fig2_stem = make_significance_effect_figure(
    sig_df=ml1m_sig,
    reference_model="plain_nmf",
    target_model="coupled_nmf_tuned",
    metrics=main_metrics_ml1m,
    title="Seed-level significance summary: MovieLens 1M",
    stem="fig_ml1m_significance_effects",
)

# ------------------------------------------------------------
# 5. Create figure include-command file
# ------------------------------------------------------------

include_lines = [
    "% Include these figure environments in the manuscript or appendix as needed.",
    "% Required package: graphicx",
    "",
    r"\begin{figure}[htbp]",
    r"\centering",
    r"\includegraphics[width=0.9\textwidth]{paper/figures/fig_ml_latest_small_significance_effects.pdf}",
    r"\caption{Seed-level significance summary for MovieLens latest-small. Positive values indicate improvement of coupled NMF over plain NMF.}",
    r"\label{fig:ml_latest_small_significance_effects}",
    r"\end{figure}",
    "",
    r"\begin{figure}[htbp]",
    r"\centering",
    r"\includegraphics[width=0.9\textwidth]{paper/figures/fig_ml1m_significance_effects.pdf}",
    r"\caption{Seed-level significance summary for MovieLens 1M. Positive values indicate improvement of tuned coupled NMF over plain NMF.}",
    r"\label{fig:ml1m_significance_effects}",
    r"\end{figure}",
    "",
]

include_path = PAPER_FIG_DIR / "statistical_tests_figure_include_commands.tex"
include_path.write_text("\n".join(include_lines), encoding="utf-8")
shutil.copy(include_path, RESULTS_FIG_DIR / include_path.name)

print("Saved include-command file:", include_path)
print("Copied include-command file:", RESULTS_FIG_DIR / include_path.name)

# ------------------------------------------------------------
# 6. Copy report and manifest
# ------------------------------------------------------------

figure_files = [
    "fig_ml_latest_small_significance_effects.png",
    "fig_ml_latest_small_significance_effects.pdf",
    "fig_ml1m_significance_effects.png",
    "fig_ml1m_significance_effects.pdf",
    "statistical_tests_figure_include_commands.tex",
]

report_rows = []

for filename in figure_files:
    if filename.endswith(".tex"):
        path = PAPER_FIG_DIR / filename
    else:
        path = PAPER_FIG_DIR / filename

    report_rows.append({
        "filename": filename,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
    })

report_df = pd.DataFrame(report_rows)

report_path = PAPER_FIG_DIR / "statistical_tests_figure_copy_report.csv"
report_df.to_csv(report_path, index=False)
shutil.copy(report_path, RESULTS_FIG_DIR / report_path.name)

manifest_text = """Manuscript-ready figures for statistical significance testing

Figures included:
1. fig_ml_latest_small_significance_effects
2. fig_ml1m_significance_effects

Interpretation:
Positive bars indicate that the target model improves over the reference model.
For RMSE and MAE, improvement is direction-corrected because lower values are better.
The p-value annotation reports the paired t-test p-value.

Recommended use:
- Journal paper: optional appendix or supporting figure.
- Dissertation: include both figures in the statistical analysis subsection.
"""

manifest_path = PAPER_FIG_DIR / "README_statistical_tests_figure_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")
shutil.copy(manifest_path, RESULTS_FIG_DIR / manifest_path.name)

display(report_df)

print("Saved report:", report_path)
print("Saved manifest:", manifest_path)

# ------------------------------------------------------------
# 7. Archive significance figures and reports
# ------------------------------------------------------------

sig_archive_dir = ARCHIVE_DIR
sig_archive_dir.mkdir(parents=True, exist_ok=True)

for src in [
    PAPER_FIG_DIR / "fig_ml_latest_small_significance_effects.png",
    PAPER_FIG_DIR / "fig_ml_latest_small_significance_effects.pdf",
    PAPER_FIG_DIR / "fig_ml1m_significance_effects.png",
    PAPER_FIG_DIR / "fig_ml1m_significance_effects.pdf",
    PAPER_FIG_DIR / "statistical_tests_figure_include_commands.tex",
    PAPER_FIG_DIR / "statistical_tests_figure_copy_report.csv",
    PAPER_FIG_DIR / "README_statistical_tests_figure_manifest.txt",
]:
    if src.exists():
        shutil.copy(src, sig_archive_dir / src.name)
        print("Archived:", sig_archive_dir / src.name)

print("\nStatistical significance figures generated successfully.")

## 23. Final statistical significance bundle

Original cell index starts around `305`.

In [ ]:
# ============================================================
# Final statistical significance bundle
# Includes:
# - significance CSVs
# - LaTeX significance tables
# - significance figures
# - include-command files
# - copy reports
# - manifest
# - ZIP file
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd

# ------------------------------------------------------------
# 1. Set paths
# ------------------------------------------------------------

candidate_roots = [
    Path("/content/drive/MyDrive/xai_coupled_nmf_project"),
    Path("/content/gdrive/MyDrive/xai_coupled_nmf_project"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])

CSV_DIR = PROJECT_ROOT / "results/csv"
ARCHIVE_DIR = CSV_DIR / "archive/statistical_significance_tests"

PAPER_TABLE_DIR = PROJECT_ROOT / "paper/tables"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results/tables"

PAPER_FIG_DIR = PROJECT_ROOT / "paper/figures"
RESULTS_FIG_DIR = PROJECT_ROOT / "results/figures"

EXPORT_ROOT = PROJECT_ROOT / "results/export_bundles"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

FINAL_BUNDLE_DIR = EXPORT_ROOT / f"statistical_significance_bundle_{timestamp}"

FINAL_CSV_DIR = FINAL_BUNDLE_DIR / "csv"
FINAL_TABLE_DIR = FINAL_BUNDLE_DIR / "latex_tables"
FINAL_FIGURE_DIR = FINAL_BUNDLE_DIR / "figures"
FINAL_INCLUDE_DIR = FINAL_BUNDLE_DIR / "latex_include_commands"
FINAL_REPORT_DIR = FINAL_BUNDLE_DIR / "reports"
FINAL_ARCHIVE_DIR = FINAL_BUNDLE_DIR / "archive_copy"

for folder in [
    FINAL_CSV_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
    FINAL_ARCHIVE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_BUNDLE_DIR:", FINAL_BUNDLE_DIR)
print("CSV_DIR exists:", CSV_DIR.exists())
print("ARCHIVE_DIR exists:", ARCHIVE_DIR.exists())
print("PAPER_TABLE_DIR exists:", PAPER_TABLE_DIR.exists())
print("PAPER_FIG_DIR exists:", PAPER_FIG_DIR.exists())

# ------------------------------------------------------------
# 2. Helper copy function
# ------------------------------------------------------------

copy_reports = []

def copy_if_exists(src_candidates, dst_dir, filename, section):
    src = None

    for candidate in src_candidates:
        if candidate.exists():
            src = candidate
            break

    dst = dst_dir / filename

    if src is not None:
        shutil.copy(src, dst)
        report = {
            "section": section,
            "filename": filename,
            "status": "COPIED",
            "size_kb": round(dst.stat().st_size / 1024, 2),
            "source": str(src),
        }
    else:
        report = {
            "section": section,
            "filename": filename,
            "status": "MISSING",
            "size_kb": None,
            "source": "",
        }

    copy_reports.append(report)
    return report

# ------------------------------------------------------------
# 3. Copy statistical CSVs
# ------------------------------------------------------------

csv_files = [
    "ml_latest_small_significance_tests.csv",
    "ml1m_significance_tests.csv",
    "statistical_tests_all_datasets.csv",
]

for filename in csv_files:
    copy_if_exists(
        src_candidates=[
            CSV_DIR / filename,
            ARCHIVE_DIR / filename,
        ],
        dst_dir=FINAL_CSV_DIR,
        filename=filename,
        section="csv",
    )

csv_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "csv"])
display(csv_report_df)

# ------------------------------------------------------------
# 4. Copy LaTeX significance tables
# ------------------------------------------------------------

table_files = [
    "table_ml_latest_small_significance_tests.tex",
    "table_ml1m_significance_tests.tex",
]

for filename in table_files:
    copy_if_exists(
        src_candidates=[
            PAPER_TABLE_DIR / filename,
            RESULTS_TABLE_DIR / filename,
            ARCHIVE_DIR / filename,
        ],
        dst_dir=FINAL_TABLE_DIR,
        filename=filename,
        section="latex_tables",
    )

table_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "latex_tables"])
display(table_report_df)

# ------------------------------------------------------------
# 5. Copy significance figures
# ------------------------------------------------------------

figure_files = [
    "fig_ml_latest_small_significance_effects.png",
    "fig_ml_latest_small_significance_effects.pdf",
    "fig_ml1m_significance_effects.png",
    "fig_ml1m_significance_effects.pdf",
]

for filename in figure_files:
    copy_if_exists(
        src_candidates=[
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
            ARCHIVE_DIR / filename,
        ],
        dst_dir=FINAL_FIGURE_DIR,
        filename=filename,
        section="figures",
    )

figure_report_df = pd.DataFrame([r for r in copy_reports if r["section"] == "figures"])
display(figure_report_df)

# ------------------------------------------------------------
# 6. Copy include-command files
# ------------------------------------------------------------

include_files = [
    "statistical_tests_table_include_commands.tex",
    "statistical_tests_figure_include_commands.tex",
]

for filename in include_files:
    if filename == "statistical_tests_table_include_commands.tex":
        src_candidates = [
            PAPER_TABLE_DIR / filename,
            RESULTS_TABLE_DIR / filename,
            ARCHIVE_DIR / filename,
        ]
    else:
        src_candidates = [
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
            ARCHIVE_DIR / filename,
        ]

    copy_if_exists(
        src_candidates=src_candidates,
        dst_dir=FINAL_INCLUDE_DIR,
        filename=filename,
        section="latex_include_commands",
    )

include_report_df = pd.DataFrame([
    r for r in copy_reports
    if r["section"] == "latex_include_commands"
])
display(include_report_df)

# ------------------------------------------------------------
# 7. Copy reports and manifests, if available
# ------------------------------------------------------------

report_files = [
    "statistical_tests_figure_copy_report.csv",
    "README_statistical_tests_figure_manifest.txt",
]

for filename in report_files:
    copy_if_exists(
        src_candidates=[
            PAPER_FIG_DIR / filename,
            RESULTS_FIG_DIR / filename,
            ARCHIVE_DIR / filename,
        ],
        dst_dir=FINAL_REPORT_DIR,
        filename=filename,
        section="reports",
    )

report_df_partial = pd.DataFrame([r for r in copy_reports if r["section"] == "reports"])
display(report_df_partial)

# ------------------------------------------------------------
# 8. Copy full archive folder contents, if available
# ------------------------------------------------------------

archive_copy_rows = []

if ARCHIVE_DIR.exists():
    for src in sorted(ARCHIVE_DIR.glob("*")):
        if src.is_file():
            dst = FINAL_ARCHIVE_DIR / src.name
            shutil.copy(src, dst)
            archive_copy_rows.append({
                "section": "archive_copy",
                "filename": src.name,
                "status": "COPIED",
                "size_kb": round(dst.stat().st_size / 1024, 2),
                "source": str(src),
            })

copy_reports.extend(archive_copy_rows)

archive_copy_df = pd.DataFrame(archive_copy_rows)
display(archive_copy_df)

# ------------------------------------------------------------
# 9. Save unified copy report
# ------------------------------------------------------------

full_report = pd.DataFrame(copy_reports)

copy_report_path = FINAL_REPORT_DIR / "statistical_significance_bundle_copy_report.csv"
full_report.to_csv(copy_report_path, index=False)

missing_files = full_report[full_report["status"] == "MISSING"].copy()

print("Copy report saved:", copy_report_path)

print("\nMissing files:")
display(missing_files)

# ------------------------------------------------------------
# 10. Create manifest
# ------------------------------------------------------------

manifest_text = f"""Final Statistical Significance Bundle
=====================================

Project:
xai_coupled_nmf_project

Datasets:
1. MovieLens latest-small
2. MovieLens 1M

Bundle created:
{timestamp}

Bundle purpose:
This bundle preserves the statistical significance testing outputs for the coupled NMF explainable recommendation manuscript and dissertation workflow.

Included folders:
1. csv
   Statistical test CSV outputs for both datasets.

2. latex_tables
   Manuscript-ready LaTeX tables for significance testing.

3. figures
   Manuscript-ready significance figures in PNG and PDF formats.

4. latex_include_commands
   Ready-to-use LaTeX include commands for tables and figures.

5. reports
   Copy reports and figure manifests.

6. archive_copy
   Copy of archived significance-test outputs.

Main CSV files:
- csv/ml_latest_small_significance_tests.csv
- csv/ml1m_significance_tests.csv
- csv/statistical_tests_all_datasets.csv

Main LaTeX tables:
- latex_tables/table_ml_latest_small_significance_tests.tex
- latex_tables/table_ml1m_significance_tests.tex

Main figures:
- figures/fig_ml_latest_small_significance_effects.pdf
- figures/fig_ml1m_significance_effects.pdf

Include files:
- latex_include_commands/statistical_tests_table_include_commands.tex
- latex_include_commands/statistical_tests_figure_include_commands.tex

Interpretation summary:
For MovieLens latest-small, coupled NMF is statistically comparable to plain NMF while providing descriptor-grounded explanations. The observed gains in MAE, Precision@10, Recall@10, and NDCG@10 are positive but not statistically significant.

For MovieLens 1M, tuned coupled NMF is weaker than plain NMF on RMSE and MAE, but remains competitive in ranking quality. The three-seed results should be interpreted as stability evidence rather than strong inferential evidence.

Recommended journal use:
Use the LaTeX significance tables in the Results or Appendix section. Use the figures as optional supplementary or appendix figures.

Recommended dissertation use:
Include both tables and figures in the statistical analysis subsection.

Missing files:
"""

if missing_files.empty:
    manifest_text += "\nNone.\n"
else:
    for _, row in missing_files.iterrows():
        manifest_text += f"\n- {row['section']}: {row['filename']}"

manifest_path = FINAL_BUNDLE_DIR / "README_statistical_significance_bundle_manifest.txt"
manifest_path.write_text(manifest_text, encoding="utf-8")

print("Manifest saved:", manifest_path)

# ------------------------------------------------------------
# 11. Create ZIP file
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(FINAL_BUNDLE_DIR),
    "zip",
    root_dir=FINAL_BUNDLE_DIR,
)

print("\nFinal statistical significance bundle folder:")
print(FINAL_BUNDLE_DIR)

print("\nFinal ZIP file:")
print(zip_path)

# ------------------------------------------------------------
# 12. Verify final bundle
# ------------------------------------------------------------

print("\nBundle folder counts:")

for folder in [
    FINAL_CSV_DIR,
    FINAL_TABLE_DIR,
    FINAL_FIGURE_DIR,
    FINAL_INCLUDE_DIR,
    FINAL_REPORT_DIR,
    FINAL_ARCHIVE_DIR,
]:
    files = list(folder.glob("*"))
    print(f"{folder.name:28s}: {len(files)} files")

zip_file = Path(zip_path)

print("\nZIP file size:")
print(zip_file, "->", round(zip_file.stat().st_size / 1024 / 1024, 2), "MB")

print("\nFinal verification list:")
for p in sorted(FINAL_BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(FINAL_BUNDLE_DIR))

## 24. Repository-ready package creation and audit

Original cell index starts around `306`.

In [ ]:
# ============================================================
# Create repository-ready package:
# xai-coupled-nmf-recommendation/
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import textwrap
import pandas as pd

# ------------------------------------------------------------
# 1. Source project and repository destination
# ------------------------------------------------------------

SOURCE_PROJECT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")

REPO_PARENT = Path("/content/drive/MyDrive/repository_exports")
REPO_ROOT = REPO_PARENT / "xai-coupled-nmf-recommendation"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if not SOURCE_PROJECT.exists():
    raise FileNotFoundError(f"Source project not found: {SOURCE_PROJECT}")

REPO_PARENT.mkdir(parents=True, exist_ok=True)

# Clean previous generated copy safely by renaming it
if REPO_ROOT.exists():
    backup = REPO_PARENT / f"xai-coupled-nmf-recommendation_backup_{timestamp}"
    shutil.move(str(REPO_ROOT), str(backup))
    print("Existing repository folder moved to backup:")
    print(backup)

# ------------------------------------------------------------
# 2. Create repository folder structure
# ------------------------------------------------------------

folders = [
    REPO_ROOT / "configs",
    REPO_ROOT / "src/preprocessing",
    REPO_ROOT / "src/models",
    REPO_ROOT / "src/evaluation",
    REPO_ROOT / "src/explanation",
    REPO_ROOT / "src/statistics",
    REPO_ROOT / "src/manuscript_outputs",
    REPO_ROOT / "scripts",
    REPO_ROOT / "results/csv",
    REPO_ROOT / "results/tables",
    REPO_ROOT / "results/figures",
    REPO_ROOT / "paper_assets/tables",
    REPO_ROOT / "paper_assets/figures",
    REPO_ROOT / "docs",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Created repository root:")
print(REPO_ROOT)

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

copy_report = []

def copy_file(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if src.exists() and src.is_file():
        shutil.copy(src, dst)
        copy_report.append({
            "source": str(src),
            "destination": str(dst.relative_to(REPO_ROOT)),
            "status": "COPIED",
            "size_kb": round(dst.stat().st_size / 1024, 2),
        })
        return True

    copy_report.append({
        "source": str(src),
        "destination": str(dst.relative_to(REPO_ROOT)),
        "status": "MISSING",
        "size_kb": None,
    })
    return False


def copy_matching_files(src_dir, dst_dir, patterns):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)

    if not src_dir.exists():
        return []

    copied = []
    for pattern in patterns:
        for src in sorted(src_dir.glob(pattern)):
            if src.is_file():
                dst = dst_dir / src.name
                copy_file(src, dst)
                copied.append(dst)
    return copied


def write_text(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    copy_report.append({
        "source": "GENERATED",
        "destination": str(path.relative_to(REPO_ROOT)),
        "status": "GENERATED",
        "size_kb": round(path.stat().st_size / 1024, 2),
    })


# ------------------------------------------------------------
# 4. Generate README.md
# ------------------------------------------------------------

write_text(
    REPO_ROOT / "README.md",
    r"""
    # Descriptor-Grounded Coupled NMF for Explainable Recommendation

    This repository contains the source code, configuration files, processed experimental outputs, manuscript-ready tables, and figures for the study on descriptor-grounded coupled non-negative matrix factorization for explainable recommendation.

    ## Summary

    The project evaluates a coupled NMF recommender that jointly models user-item ratings and item-side semantic descriptors. The model preserves the additive structure of NMF and supports descriptor-grounded explanations through factor-wise recommendation score decomposition.

    ## Datasets

    The experiments use publicly available MovieLens datasets:

    - MovieLens latest-small: primary dataset, using tag-and-genre descriptors.
    - MovieLens 1M: secondary validation dataset, using genre-only descriptors.

    Raw MovieLens files are not redistributed in this repository. Please download the datasets from the official MovieLens website and use the preprocessing scripts or documentation to reconstruct the processed inputs.

    ## Final experimental design

    Primary dataset:

    - Dataset: MovieLens latest-small
    - Descriptor setting: tags + genres
    - Seeds: 42, 123, 2026, 7, 99
    - Models: Item-CF, biased MF, plain NMF, coupled NMF

    Secondary dataset:

    - Dataset: MovieLens 1M
    - Descriptor setting: genres only
    - Seeds: 42, 123, 2026
    - Models: biased MF, plain NMF, tuned coupled NMF

    Final tuned MovieLens 1M coupled NMF configuration:

    - k = 60
    - alpha = 0.05
    - beta = 0.0001
    - lambda_reg = 0.01
    - epochs = 20

    ## Repository contents

    ```text
    configs/                Experiment configuration files.
    src/                    Source code modules.
    scripts/                Reproduction and execution scripts.
    results/csv/            Final result CSV files.
    results/tables/         Manuscript-ready LaTeX tables.
    results/figures/        Manuscript-ready figures.
    paper_assets/           Tables and figures for direct manuscript upload.
    docs/                   Reproduction guide and table/figure mapping.
    ```

    ## Reproducibility

    The processed result tables, generated figures, and statistical significance outputs are included. Raw datasets are excluded. The repository is intended to support reproducibility of the manuscript tables and figures and to document the experimental workflow.

    ## Citation

    Please cite the corresponding manuscript when using this code or results.
    """
)

# ------------------------------------------------------------
# 5. LICENSE, requirements.txt, environment.yml, .gitignore
# ------------------------------------------------------------

write_text(
    REPO_ROOT / "LICENSE",
    """
    MIT License

    Copyright (c) 2026 Dr. Saikat Kanjilal

    Permission is hereby granted, free of charge, to any person obtaining a copy
    of this software and associated documentation files (the "Software"), to deal
    in the Software without restriction, including without limitation the rights
    to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
    copies of the Software, and to permit persons to whom the Software is
    furnished to do so, subject to the following conditions:

    The above copyright notice and this permission notice shall be included in all
    copies or substantial portions of the Software.

    THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
    IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
    FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
    AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
    LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
    OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
    SOFTWARE.
    """
)

write_text(
    REPO_ROOT / "requirements.txt",
    """
    numpy>=1.24
    pandas>=2.0
    scipy>=1.10
    scikit-learn>=1.2
    matplotlib>=3.7
    pyyaml>=6.0
    """
)

write_text(
    REPO_ROOT / "environment.yml",
    """
    name: xai-coupled-nmf
    channels:
      - conda-forge
      - defaults
    dependencies:
      - python>=3.10
      - numpy
      - pandas
      - scipy
      - scikit-learn
      - matplotlib
      - pyyaml
      - pip
    """
)

write_text(
    REPO_ROOT / ".gitignore",
    """
    # Python
    __pycache__/
    *.pyc
    .ipynb_checkpoints/

    # Data
    data/raw/
    data/external/
    *.zip

    # Large or temporary files
    *.log
    *.tmp
    .DS_Store

    # Virtual environments
    .venv/
    venv/
    env/

    # Overleaf/LaTeX build files
    *.aux
    *.bbl
    *.blg
    *.log
    *.out
    *.toc
    *.synctex.gz
    """
)

# ------------------------------------------------------------
# 6. Config YAML files
# ------------------------------------------------------------

write_text(
    REPO_ROOT / "configs/ml_latest_small_config.yaml",
    """
    dataset_name: ml_latest_small
    descriptor_source: tags_genres
    seeds: [42, 123, 2026, 7, 99]
    relevance_threshold: 4.0
    top_k: [5, 10]

    models:
      - item_cf
      - biased_mf
      - plain_nmf
      - coupled_nmf

    outputs:
      main_comparison: results/csv/ml_latest_small_main_comparison.csv
      run_level_metrics: results/csv/ml_latest_small_run_level_metrics.csv
      explanation_metrics: results/csv/ml_latest_small_explanation_metrics_extended.csv
      significance_tests: results/csv/ml_latest_small_significance_tests.csv
    """
)

write_text(
    REPO_ROOT / "configs/ml1m_config.yaml",
    """
    dataset_name: ml1m
    descriptor_source: genres
    seeds: [42, 123, 2026]
    relevance_threshold: 4.0
    top_k: [5, 10]

    models:
      - biased_mf
      - plain_nmf
      - coupled_nmf_tuned

    outputs:
      main_comparison: results/csv/ml1m_3seed_main_comparison.csv
      run_level_metrics: results/csv/ml1m_3seed_run_level_metrics.csv
      significance_tests: results/csv/ml1m_significance_tests.csv
    """
)

write_text(
    REPO_ROOT / "configs/selected_hyperparameters.yaml",
    """
    ml_latest_small:
      selected_model: coupled_nmf
      descriptor_source: tags_genres
      selection_basis: validation ranking performance
      notes: >
        Final configuration was selected from the coupled NMF validation experiments.
        See the selected configuration CSV in results/csv.

    ml1m:
      selected_model: coupled_nmf_tuned
      descriptor_source: genres
      k: 60
      alpha: 0.05
      beta: 0.0001
      lambda_reg: 0.01
      epochs: 20
      selection_basis: reduced seed-42 validation grid using positive semantic coupling
    """
)

# ------------------------------------------------------------
# 7. Copy source code files from existing project
# ------------------------------------------------------------

SOURCE_SRC = SOURCE_PROJECT / "src"

if SOURCE_SRC.exists():
    # Copy known files to logical folders
    known_mapping = {
        "benchmark_utils.py": "src/evaluation/benchmark_utils.py",
        "benchmark_models.py": "src/models/benchmark_models.py",
        "run_main_benchmarks.py": "scripts/run_main_benchmarks_original.py",
        "run_coupled_nmf.py": "src/models/run_coupled_nmf.py",
        "run_coupled_nmf_tuning.py": "src/models/run_coupled_nmf_tuning.py",
    }

    for filename, rel_dst in known_mapping.items():
        copy_file(SOURCE_SRC / filename, REPO_ROOT / rel_dst)

    # Copy any remaining Python files into src/models/original_*
    for src in sorted(SOURCE_SRC.glob("*.py")):
        if src.name not in known_mapping:
            copy_file(src, REPO_ROOT / "src/models" / src.name)

# Add README files inside source subfolders
for subfolder in [
    "src/preprocessing",
    "src/models",
    "src/evaluation",
    "src/explanation",
    "src/statistics",
    "src/manuscript_outputs",
]:
    write_text(
        REPO_ROOT / subfolder / "README.md",
        f"""
        # {subfolder}

        This folder contains source files related to `{subfolder}`.
        Some scripts were generated from the Colab-based experimental workflow and may require path adjustment before standalone execution.
        """
    )

# ------------------------------------------------------------
# 8. Generate top-level scripts
# ------------------------------------------------------------

write_text(
    REPO_ROOT / "scripts/run_ml_latest_small_pipeline.py",
    """
    \"\"\"Run or document the MovieLens latest-small experimental pipeline.

    The original experiments were executed in Google Colab. This script is a
    repository entry point documenting the expected pipeline order.

    Recommended stages:
    1. Preprocess MovieLens latest-small ratings and descriptors.
    2. Generate fixed train-validation-test split registries.
    3. Run baseline models and coupled NMF.
    4. Generate explanation metrics.
    5. Generate LaTeX tables and manuscript figures.
    6. Run statistical significance tests.

    See docs/reproduction_guide.md for details.
    \"\"\"

    def main():
        print("MovieLens latest-small pipeline entry point.")
        print("See docs/reproduction_guide.md for the Colab-based workflow.")

    if __name__ == "__main__":
        main()
    """
)

write_text(
    REPO_ROOT / "scripts/run_ml1m_3seed_evaluation.py",
    """
    \"\"\"Run or document the MovieLens 1M three-seed evaluation.

    Final tuned coupled NMF configuration:
    k=60, alpha=0.05, beta=0.0001, lambda_reg=0.01, epochs=20.

    See configs/ml1m_config.yaml and docs/reproduction_guide.md.
    \"\"\"

    def main():
        print("MovieLens 1M three-seed evaluation entry point.")
        print("Use configs/ml1m_config.yaml and the saved scripts/source modules.")

    if __name__ == "__main__":
        main()
    """
)

write_text(
    REPO_ROOT / "scripts/generate_tables.py",
    """
    \"\"\"Generate or collect manuscript-ready LaTeX tables.

    The final generated LaTeX tables are already preserved in:
    - results/tables/
    - paper_assets/tables/

    This script documents the intended table-generation stage.
    \"\"\"

    def main():
        print("Manuscript tables are available in results/tables and paper_assets/tables.")

    if __name__ == "__main__":
        main()
    """
)

write_text(
    REPO_ROOT / "scripts/generate_figures.py",
    """
    \"\"\"Generate or collect manuscript-ready figures.

    The final generated figures are already preserved in:
    - results/figures/
    - paper_assets/figures/

    This script documents the intended figure-generation stage.
    \"\"\"

    def main():
        print("Manuscript figures are available in results/figures and paper_assets/figures.")

    if __name__ == "__main__":
        main()
    """
)

write_text(
    REPO_ROOT / "scripts/run_significance_tests.py",
    """
    \"\"\"Run or document statistical significance testing.

    Final significance outputs are preserved in:
    - results/csv/ml_latest_small_significance_tests.csv
    - results/csv/ml1m_significance_tests.csv
    - results/csv/statistical_tests_all_datasets.csv
    \"\"\"

    def main():
        print("Statistical significance outputs are available in results/csv.")

    if __name__ == "__main__":
        main()
    """
)

# ------------------------------------------------------------
# 9. Copy result CSV files
# ------------------------------------------------------------

SOURCE_CSV_DIR = SOURCE_PROJECT / "results/csv"

csv_patterns = [
    "ml_latest_small*.csv",
    "ml1m*.csv",
    "statistical_tests_all_datasets.csv",
]

copy_matching_files(SOURCE_CSV_DIR, REPO_ROOT / "results/csv", csv_patterns)

# ------------------------------------------------------------
# 10. Copy LaTeX tables
# ------------------------------------------------------------

SOURCE_PAPER_TABLES = SOURCE_PROJECT / "paper/tables"
SOURCE_RESULTS_TABLES = SOURCE_PROJECT / "results/tables"

table_patterns = ["*.tex"]

copy_matching_files(SOURCE_RESULTS_TABLES, REPO_ROOT / "results/tables", table_patterns)
copy_matching_files(SOURCE_PAPER_TABLES, REPO_ROOT / "paper_assets/tables", table_patterns)

# ------------------------------------------------------------
# 11. Copy figures
# ------------------------------------------------------------

SOURCE_PAPER_FIGS = SOURCE_PROJECT / "paper/figures"
SOURCE_RESULTS_FIGS = SOURCE_PROJECT / "results/figures"

figure_patterns = ["*.pdf", "*.png"]

copy_matching_files(SOURCE_RESULTS_FIGS, REPO_ROOT / "results/figures", figure_patterns)
copy_matching_files(SOURCE_PAPER_FIGS, REPO_ROOT / "paper_assets/figures", figure_patterns)

# ------------------------------------------------------------
# 12. Documentation files
# ------------------------------------------------------------

write_text(
    REPO_ROOT / "docs/reproduction_guide.md",
    """
    # Reproduction Guide

    ## 1. Data

    Raw MovieLens datasets are not included in this repository. Download them from the official MovieLens website.

    Required datasets:

    - MovieLens latest-small
    - MovieLens 1M

    ## 2. Preprocessing

    Convert ratings into a tabular format with columns:

    - user_id
    - item_id
    - rating
    - timestamp

    Construct item descriptor matrices:

    - MovieLens latest-small: tag-and-genre descriptors
    - MovieLens 1M: genre-only descriptors

    ## 3. Splits

    Use fixed seed-specific train-validation-test split registries.

    Primary dataset seeds:

    - 42
    - 123
    - 2026
    - 7
    - 99

    Secondary dataset seeds:

    - 42
    - 123
    - 2026

    ## 4. Final evaluation

    MovieLens latest-small:

    - Item-CF
    - Biased MF
    - Plain NMF
    - Coupled NMF

    MovieLens 1M:

    - Biased MF
    - Plain NMF
    - Tuned Coupled NMF

    Final MovieLens 1M tuned configuration:

    - k = 60
    - alpha = 0.05
    - beta = 0.0001
    - lambda_reg = 0.01
    - epochs = 20

    ## 5. Outputs

    Final result CSVs are stored in `results/csv/`.

    Manuscript-ready LaTeX tables are stored in:

    - `results/tables/`
    - `paper_assets/tables/`

    Manuscript-ready figures are stored in:

    - `results/figures/`
    - `paper_assets/figures/`

    ## 6. Statistical tests

    Paired seed-level tests were performed using:

    - paired t-test
    - Wilcoxon signed-rank test
    - Cohen's paired effect size dz

    See:

    - `results/csv/ml_latest_small_significance_tests.csv`
    - `results/csv/ml1m_significance_tests.csv`
    - `results/csv/statistical_tests_all_datasets.csv`
    """
)

# Auto-generate table/figure mapping from copied files

table_files = sorted((REPO_ROOT / "paper_assets/tables").glob("*.tex"))
figure_files = sorted((REPO_ROOT / "paper_assets/figures").glob("*.pdf"))

mapping_lines = ["# Table and Figure Mapping", ""]

mapping_lines.append("## LaTeX Tables")
mapping_lines.append("")
for p in table_files:
    mapping_lines.append(f"- `{p.name}`")

mapping_lines.append("")
mapping_lines.append("## Figures")
mapping_lines.append("")
for p in figure_files:
    mapping_lines.append(f"- `{p.name}`")

write_text(REPO_ROOT / "docs/table_figure_mapping.md", "\n".join(mapping_lines))

write_text(
    REPO_ROOT / "docs/experiment_summary.md",
    """
    # Experiment Summary

    ## Primary dataset

    Dataset: MovieLens latest-small

    Descriptor setting: tags + genres

    Seeds: 42, 123, 2026, 7, 99

    Main finding:
    Coupled NMF remains statistically comparable to plain NMF while providing descriptor-grounded additive explanations.

    ## Secondary dataset

    Dataset: MovieLens 1M

    Descriptor setting: genres only

    Seeds: 42, 123, 2026

    Main finding:
    Tuned coupled NMF is weaker than plain NMF on RMSE and MAE but remains competitive on ranking metrics.

    ## Interpretation

    The proposed model is not positioned as a universal accuracy winner. Its contribution is competitive recommendation performance with intrinsic descriptor-grounded explanations. Descriptor richness is important.
    """
)

# ------------------------------------------------------------
# 13. Save copy report
# ------------------------------------------------------------

copy_report_df = pd.DataFrame(copy_report)
copy_report_path = REPO_ROOT / "docs/repository_copy_report.csv"
copy_report_df.to_csv(copy_report_path, index=False)

print("\nCopy report:")
display(copy_report_df)

missing = copy_report_df[copy_report_df["status"] == "MISSING"]
print("\nMissing files:")
display(missing)

# ------------------------------------------------------------
# 14. Create ZIP file
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(REPO_ROOT),
    "zip",
    root_dir=REPO_PARENT,
    base_dir="xai-coupled-nmf-recommendation",
)

print("\nRepository-ready folder:")
print(REPO_ROOT)

print("\nRepository ZIP:")
print(zip_path)

print("\nZIP size:")
zip_file = Path(zip_path)
print(round(zip_file.stat().st_size / 1024 / 1024, 2), "MB")

# ------------------------------------------------------------
# 15. Final tree preview
# ------------------------------------------------------------

print("\nFinal repository contents:")
for p in sorted(REPO_ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(REPO_ROOT))

In [ ]:
from pathlib import Path
import pandas as pd

SOURCE_PROJECT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
REPO_ROOT = Path("/content/drive/MyDrive/repository_exports/xai-coupled-nmf-recommendation")

checks = {
    "CSV files": (
        SOURCE_PROJECT / "results/csv",
        REPO_ROOT / "results/csv",
        ["ml_latest_small*.csv", "ml1m*.csv", "statistical_tests_all_datasets.csv"],
    ),
    "LaTeX tables": (
        SOURCE_PROJECT / "paper/tables",
        REPO_ROOT / "paper_assets/tables",
        ["*.tex"],
    ),
    "Figures": (
        SOURCE_PROJECT / "paper/figures",
        REPO_ROOT / "paper_assets/figures",
        ["*.pdf", "*.png"],
    ),
    "Source scripts": (
        SOURCE_PROJECT / "src",
        REPO_ROOT / "src",
        ["*.py"],
    ),
}

rows = []

for label, (src_dir, repo_dir, patterns) in checks.items():
    src_files = []
    repo_files = []

    if src_dir.exists():
        for pattern in patterns:
            src_files.extend([p.name for p in src_dir.glob(pattern)])

    if repo_dir.exists():
        for pattern in patterns:
            repo_files.extend([p.name for p in repo_dir.rglob(pattern)])

    src_set = set(src_files)
    repo_set = set(repo_files)

    for name in sorted(src_set):
        rows.append({
            "category": label,
            "filename": name,
            "included_in_repo": name in repo_set,
        })

audit = pd.DataFrame(rows)

display(audit)

print("Missing important files:")
display(audit[audit["included_in_repo"] == False])

In [ ]:
from pathlib import Path
import shutil

SOURCE_PROJECT = Path("/content/drive/MyDrive/xai_coupled_nmf_project")
REPO_ROOT = Path("/content/drive/MyDrive/repository_exports/xai-coupled-nmf-recommendation")

src_file = SOURCE_PROJECT / "src/run_main_benchmarks.py"

dst_1 = REPO_ROOT / "src/models/run_main_benchmarks.py"
dst_2 = REPO_ROOT / "scripts/run_main_benchmarks_original.py"

if not src_file.exists():
    raise FileNotFoundError(f"Source file not found: {src_file}")

dst_1.parent.mkdir(parents=True, exist_ok=True)
dst_2.parent.mkdir(parents=True, exist_ok=True)

shutil.copy(src_file, dst_1)
shutil.copy(src_file, dst_2)

print("Copied:")
print("FROM:", src_file)
print("TO:  ", dst_1)
print("TO:  ", dst_2)

print("\nVerification:")
print(dst_1, "->", dst_1.exists())
print(dst_2, "->", dst_2.exists())

In [ ]:
from pathlib import Path
import shutil

REPO_PARENT = Path("/content/drive/MyDrive/repository_exports")
REPO_ROOT = REPO_PARENT / "xai-coupled-nmf-recommendation"

zip_path = shutil.make_archive(
    str(REPO_ROOT),
    "zip",
    root_dir=REPO_PARENT,
    base_dir="xai-coupled-nmf-recommendation",
)

print("Updated ZIP created:")
print(zip_path)

zip_file = Path(zip_path)
print("ZIP size:", round(zip_file.stat().st_size / 1024 / 1024, 2), "MB")

## Cleaning Report

A separate cleaning report is provided with counts of kept and removed cells. The removed cells were either failed cells, empty cells, or exact duplicates.